# Chronotome: A Reproducible Bibliometric Workflow Notebook

Chronotome is an end-to-end, reproducible workflow for bibliometric and scientometric analysis using **Scopus** and **Web of Science** exports.

**What you get at the end:**
- A merged and deduplicated bibliographic dataset (**CSV + Excel**)  
- PRISMA-style reporting counts (identification → screening → included)  
- Country, journal, and collaboration analyses (publication-ready figures; **Cividis** palette)

> Tip (non-coders): You can run this notebook top-to-bottom with **Run All**.  
> You will only edit a small number of “⚙️ User Settings” fields early in Phase 1.


##  Open‑science release (Zenodo)

This notebook is part of the **Chronotome / Open Bibliometrics** open‑science release archived on **Zenodo**:

- **Zenodo DOI:** `10.5281/zenodo.17514930`

### What you get in the archive
The Zenodo package contains the notebook plus supporting resources used by the pipeline, including:

- `institutions.json` — a curated **institution alias map** used by the entity‑resolution step (see Cell 22).


### Suggested citation text (for users of the notebook)
If you use this notebook for a study or project, cite the Zenodo record:

> Popescu-Apreutesei, L.-E., & Iosupescu, M.-S. (2025). Chronotome. Zenodo. https://doi.org/10.5281/zenodo.17514930



# Phase 1: Data Acquisition and Harmonization Protocol

This workflow implements a rigorous and reproducible **ETL pipeline for bibliometric analysis**, aligned with **PRISMA 2020** reporting logic (**Identification → Screening → Included**).

---

### What Phase 1 does

### 1) Ingestion
Loads raw bibliographic exports from heterogeneous sources (**Scopus** and/or **Web of Science**).

### 2) Harmonization
Maps different metadata schemas into a single unified structure (consistent columns, formats).

### 3) Preprocessing
Standardizes key fields (e.g., authors, DOI/title normalization, references, affiliations) so downstream indicators and networks are valid.

---
## 🚨 REQUIRED USER CHANGE (MOST IMPORTANT):

### What you need before you run this notebook

### Required inputs (choose 1 or both)

**Scopus export**
- Recommended: **CSV** (`.csv`)
- Typical filenames:
  - `scopus.csv`
  - `scopus_export.csv`
  - `scopus_export_<date>.csv`

**Web of Science export**
- Common formats: **TXT**, **CSV**, **XLS/XLSX**
  - WoS sometimes exports files with `.xls` extension that are actually **tab-separated text** (this notebook handles that).

### Optional: Split export “appendage” inputs (if your export is split)
If your export was split into multiple files by the platform (very common), you can use **appendage mode** to stitch files like:
- `WoS_export_A_1.xls`, `WoS_export_A_2.xls`, `WoS_export_A_3.xls`
- `scopus_export_A_1.csv`, `scopus_export_A_2.csv`, ...

**Important:** appendage mode only *stacks parts of the same export series* (same base name + same extension).  
It does **not** merge Scopus with WoS. It also won’t mix Query A with Query B unless the base name matches.

---

### Where your files should be (by platform)

### A) Kaggle Notebooks
Attach your files as a **Kaggle Dataset** (right panel → “Upload”).  
Your files will appear under:

```python
/kaggle/input/<your-dataset-name>/...
```

**Examples**
```python
/kaggle/input/my-bibliometric-dataset/scopus_export.csv
/kaggle/input/my-bibliometric-dataset/WoS_export.txt
/kaggle/input/my-bibliometric-dataset/WoS_export_A_1.xls
```

### B) Google Colab
You have two common options:

**Option 1 — Upload files directly to the Colab runtime**
Files usually land under:
```python
/content/<filename>
```

Examples:
```python
/content/scopus_export.csv
/content/WoS_export.txt
```

**Option 2 — Mount Google Drive (recommended for larger exports)**
After mounting Drive, paths look like:
```python
/content/drive/MyDrive/<folder>/<filename>
```

Examples:
```python
/content/drive/MyDrive/bibliometrics/scopus_export.csv
/content/drive/MyDrive/bibliometrics/WoS_export_A_1.xls
```

### C) Local Jupyter Notebook
Use an absolute path or a path relative to your notebook.

**Examples (macOS / Linux)**
```python
/Users/<you>/Downloads/scopus_export.csv
/home/<you>/data/WoS_export.txt
```

**Examples (Windows)**
```python
C:\Users\<you>\Downloads\scopus_export.csv
C:\Users\<you>\Documents\data\WoS_export.txt
```

---

## 1.1 Environment Setup and File Verification

The next cell initializes the computational environment:

- Data manipulation: `pandas`, `numpy`
- Text processing: `re`
- System operations: `os`
- Notebook display: `IPython.display`

### What the verification step does
It prints all files found under **`/kaggle/input/`** so you can confirm your dataset attachment and copy-paste the correct paths later.

### Example output (Kaggle)
You should see lines like:

```python
/kaggle/input/my-bibliometric-dataset/scopus_export.csv
/kaggle/input/my-bibliometric-dataset/WoS_export_A_1.xls
```

### If you’re NOT running on Kaggle
- In Colab or local Jupyter, `/kaggle/input/` won’t exist, so this listing may be empty.
- That’s OK — you will set file paths later in the **USER CONFIG** cell (the notebook prints clear prompts there).

If you want an equivalent listing in Colab/local, here’s a safe example you can run in a scratch cell:

```python
import os

ROOT = "."  # current folder
print("Listing files under:", os.path.abspath(ROOT))
for dirname, _, filenames in os.walk(ROOT):
    for fn in filenames:
        print(os.path.join(dirname, fn))
```

---

### ✅ What most users will change later
Most users only need to edit one configuration cell (the “USER CONFIG” cell) where you specify either:
- a single Scopus/WoS file path, **or**
- a folder + seed filename for appendage stitching.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import numpy as np # Linear algebra
import pandas as pd # Data processing, CSV file I/O
import re # Regular expressions for text cleaning
import os # Operating system interaction
from datetime import datetime # Date/Time handling
from IPython.display import display # Rich notebook output

# --- Verify Input Files ---
# The loop below walks through the /kaggle/input/ directory and prints every file it finds.
# This helps you confirm your datasets are uploaded and get their exact paths.
print("Listing all files in /kaggle/input/...")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        # Print the full path of each file
        print(os.path.join(dirname, filename))
print("File listing complete.\n")

## 1.2 Visualization Dependencies

Some plots in Phase 2 (especially dense collaboration and network maps) require **adjustText**.

**Why it’s used:**  
Network figures often have overlapping country/institution labels. `adjustText` applies a force-based adjustment to labels in Matplotlib so figures remain readable and publication-ready.

### What you should do
- If you are running on **Kaggle**: we will install it with `pip` in the next cell.
- If you are running locally: you can also install it in your environment.

### Expected output
- If it is already installed: you’ll see something like `Requirement already satisfied`
- If it installs now: you may be asked to **restart the kernel** (rare on Kaggle, more common locally)


In [ ]:
!pip -q install adjustText
print("✅ adjustText installed (used to prevent label overlap in network figures).")


## 2. Data Ingestion (Scopus & Web of Science)

This phase ingests bibliographic exports from **Scopus** and/or **Web of Science (WoS)** and prepares them for harmonization and deduplication.

Because exports are often **inconsistently formatted** (even with the same file extension), Chronotome uses a **fault-tolerant loader** to avoid silent data loss.

---

## 🚨 REQUIRED USER CHANGE (MOST IMPORTANT):

### What you need to change (non-coders)

In the next configuration cell, you will set:

1) **Which sources you have** (Scopus / WoS / both)  
2) Whether your export is **one file** or **split into parts** (*appendage mode*)  
3) The **file path(s)** or **folder + seed filename**

> If you’re on Kaggle, your files must be under `/kaggle/input/...` (Cell 3 showed the paths).

---

### Robustness strategy (why the loader is reliable)

The loader detects and handles common export anomalies:

- **Format detection** by extension (`.csv`, `.txt`, `.xls`, `.xlsx`)
- **Tab-delimited fallback parsing** if CSV/Excel parsing fails
- **WoS `.xls` anomaly handling:** WoS sometimes exports files named `.xls` that are actually **tab-separated text**  
  → Chronotome tries Excel first, then safely falls back to tab parsing

This prevents ingestion failures and avoids downstream distortions in collaboration networks, journal indicators, and normalized citation analyses.

---

## 2.1 Two ingestion modes

### Mode A — Single file (most common)
Use this if you exported *one file per source* (or manually combined parts yourself).

#### **Example: Scopus only**
```python
APPEND_SCOPUS = False
APPEND_WOS    = False

SCOPUS_FILE = "/kaggle/input/dataset/scopus.csv"
WOS_FILE    = ""
```

#### **Example (WoS Only):**
```python
APPEND_SCOPUS = False
APPEND_WOS    = False

SCOPUS_FILE = ""  # Leave empty to skip
WOS_FILE    = "/kaggle/input/dataset/savedrecs.xls"
```

#### **Example (Both Sources, Single File Each):**
```python
APPEND_SCOPUS = False
APPEND_WOS    = False

SCOPUS_FILE = "/kaggle/input/dataset/scopus.csv"
WOS_FILE    = "/kaggle/input/dataset/savedrecs.xls"
```

---

### 2) Appendage Mode (Split Export Stitching)

Use this when your database export was split into multiple parts (common due to export limits):

- **Scopus:** often limited per export batch (e.g., 20,000 records)
- **WoS:** often limited per export batch (e.g., 1,000 records)

#### What “appendage” means here

We **only append split exports** that follow a `..._NUMBER.ext` pattern (e.g., `_1`, `_2`, `_3`) and **only within the same extension**:

- `.xls` appends only with `.xls`
- `.txt` appends only with `.txt`
- `.csv` appends only with `.csv`

✅ This is **not** merging Scopus with WoS.  
✅ This is **not** merging Query A with Query B unless the base name is identical.  
✅ It is purely a **vertical append** (row stacking) for split exports.

---

### How to Configure Cell 1

You control appendage using two flags:

- `APPEND_SCOPUS = True/False`
- `APPEND_WOS    = True/False`

#### If appendage is OFF (`False`)

Provide exact file path(s):

```python
APPEND_SCOPUS = False
APPEND_WOS    = False

SCOPUS_FILE = "/path/to/scopus_export.csv"
WOS_FILE    = "/path/to/WoS_export.xls"
```

### If appendage is ON (`True`)

Provide:

1) A folder containing the split exports  
2) A seed name/path that contains a `_NUMBER` pattern

The seed can be any of the following:

- Full path: `/kaggle/input/.../WoS_export_Dec 23-2025_A_1.xls`
- Filename only: `WoS_export_Dec 23-2025_A_1.xls`
- Stem only: `WoS_export_Dec 23-2025_A_1`

#### **Example (WoS appendage only):**
```python
APPEND_SCOPUS = False
APPEND_WOS    = True

WOS_APPEND_FOLDER = "/kaggle/input/new-spatial-test-dataset/"
WOS_APPEND_SEED   = "WoS_export_Dec 23-2025_A_1.xls"

SCOPUS_FILE = ""  # not used
WOS_FILE    = ""  # not used when appendage is enabled
```

#### **Example (Scopus appendage only):**
```python
APPEND_SCOPUS = True
APPEND_WOS    = False

SCOPUS_APPEND_FOLDER = "/kaggle/input/new-spatial-test-dataset/"
SCOPUS_APPEND_SEED   = "scopus_export_Dec 23-2025_A_1.csv"

SCOPUS_FILE = ""  # not used when appendage is enabled
WOS_FILE    = ""  # not used
```

#### **Example (Append both Scopus and WoS):**
```python
APPEND_SCOPUS = True
APPEND_WOS    = True

SCOPUS_APPEND_FOLDER = "/kaggle/input/new-spatial-test-dataset/"
SCOPUS_APPEND_SEED   = "scopus_export_Dec 23-2025_A_1.csv"

WOS_APPEND_FOLDER = "/kaggle/input/new-spatial-test-dataset/"
WOS_APPEND_SEED   = "WoS_export_Dec 23-2025_A_1.xls"

SCOPUS_FILE = ""  # not used
WOS_FILE    = ""  # not used
```

---

### What Each Cell Does (Pipeline Map)

### Cell 1 — Configuration
- You choose **Single File** vs **Appendage** using boolean flags
- You provide either:
  - direct file path(s), or
  - folder + seed reference(s)

### Cell 2 — Appendage + Save (only if enabled)
- Finds the matching split-series using the seed pattern (`_1`, `_2`, ...)
- Appends parts in numeric order
- Saves the appended file(s) into `OUTPUT_FOLDER`
- Prints the final saved path(s)

### Cell 3 — Load for Analysis
Loads final file(s) into:
- `df_scopus`
- `df_wos`

Includes tab-delimited fallbacks to handle export anomalies.

---

### Download / Re-upload Tip

The printed output paths are sufficient for local notebooks.  
For Kaggle and Colab, follow the platform-specific hint printed at the end of Cell 2 to download the appended export(s) and reuse them as clean single-file inputs.


In [ ]:
import os

# ========================================================================
# USER CONFIGURATION (no prompts; just flip True/False)
# ========================================================================

# --- 1) Do you need APPENDAGE (stitching split exports) for each source? ---
APPEND_SCOPUS = False
APPEND_WOS    = True

# --- 2) If NOT appending: set the exact file you want to load ---
SCOPUS_FILE = "/kaggle/input/new-spatial-test-dataset/scopus_export_Dec 23-2025_B.csv"  # e.g. "/path/to/scopus_export.csv"
WOS_FILE    = ""  # e.g. "/path/to/WoS_export.xls"

# --- 3) If appending: set a folder + a "seed" name/path that contains a _NUMBER pattern ---
# Seed can be: full path OR just filename OR just stem (with or without extension)
SCOPUS_APPEND_FOLDER = ""  # e.g. "/kaggle/input/new-spatial-test-dataset/"
SCOPUS_APPEND_SEED   = ""  # e.g. "scopus_export_Dec 23-2025_A_1.csv" or "..._A_1"

WOS_APPEND_FOLDER = "/kaggle/input/new-spatial-test-dataset/"
WOS_APPEND_SEED   = "WoS_export_Dec 23-2025_B_1.xls"  # or "/.../WoS_export_Dec 23-2025_A_1.xls"

# --- 4) Where to SAVE appended output files (must be writable) ---
# Kaggle: "./" points to /kaggle/working (writable)
# Colab: "./" points to /content (writable)
# Local: current notebook directory
OUTPUT_FOLDER = os.path.abspath("./stitched_exports")

# --- 5) These will be used by Cell 3 (Cell 2 may overwrite them if appending) ---
scopus_file_final = SCOPUS_FILE
wos_file_final    = WOS_FILE

print("✅ Cell 1 config loaded.")
print(f"Output folder for appended files: {OUTPUT_FOLDER}")


## 2.2. Appendage (Optional): Stitching Split Export Files

This step runs **only if**:
- `APPEND_SCOPUS=True` and/or `APPEND_WOS=True`

### What it does
1. Scans the folder you gave (e.g., `/kaggle/input/your-dataset/`)
2. Detects split exports matching the pattern:
   - `NAME_1.ext`, `NAME_2.ext`, `NAME_3.ext`, ...
3. Loads each part safely (with fallbacks for WoS export anomalies)
4. Concatenates them (“row stacking”)
5. Saves the stitched file into:
   - `./stitched_exports/`

### What you should expect to see
- A short log confirming whether stitching ran
- The final file path that will be used in Cell 9

### When to ignore this cell
If both flags are `False`, this cell will print that appendage is disabled — that’s fine.

In [ ]:
import pandas as pd
import os
import re

# ========================================================================
# APPENDAGE (stitching) + SAVING OUTPUT FILES
# ========================================================================

# -----------------------------
# INTERNAL TOOLS (UNCHANGED)
# -----------------------------
def parse_wos_plaintext(filepath):
    try:
        with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
            lines = f.readlines()
    except Exception: return None
    records = []; current_record = {}; current_tag = None
    for line in lines:
        line = line.rstrip('\n')
        if not line: continue
        if line.startswith('ER'):
            if current_record: records.append(current_record)
            current_record = {}; current_tag = None; continue
        if re.match(r'^[A-Z][A-Z0-9] ', line):
            tag = line[:2]; current_tag = tag; current_record[tag] = line[3:]
        elif line.startswith('   ') and current_tag:
            current_record[current_tag] += " " + line.strip()
    return pd.DataFrame(records)


def smart_loader(path, fname):
    # Strategy A: CSV (Scopus)
    if fname.lower().endswith('.csv'):
        try: return pd.read_csv(path, sep=',') 
        except: return pd.read_csv(path, sep='\t', on_bad_lines='skip', quoting=3)
    # Strategy B: Excel (WoS/Scopus)
    elif fname.lower().endswith(('.xls', '.xlsx')):
        try: return pd.read_excel(path)
        except: return pd.read_csv(path, sep='\t', on_bad_lines='skip', quoting=3)
    # Strategy C: Text (WoS)
    elif fname.lower().endswith('.txt'):
        try:
            df = pd.read_csv(path, sep='\t', on_bad_lines='skip', quoting=3)
            if df.shape[1] < 2: return parse_wos_plaintext(path)
            return df
        except: pass
    return None


# -----------------------------
# NEW HELPERS (around unchanged logic)
# -----------------------------
def build_file_groups(target_folder):
    print(f"🚀 Scanning folder: {target_folder}\n")

    all_files = os.listdir(target_folder)
    file_groups = {}

    # Matches "Name" + "_" + "Number" + ".ext"
    split_pattern = re.compile(r'^(.*)_(\d+)\.(xls|xlsx|txt|csv)$', re.IGNORECASE)

    # --- Grouping (same logic as your original cell) ---
    for filename in all_files:
        if not filename.endswith(('.xls','.xlsx','.txt','.csv')): 
            continue

        match = split_pattern.match(filename)
        full_path = os.path.join(target_folder, filename)

        if match:
            base_name = match.group(1)
            part_num = int(match.group(2))
            ext = match.group(3).lower()
            key = f"{base_name}.{ext}"  # keeps A distinct from B
            if key not in file_groups: 
                file_groups[key] = []
            file_groups[key].append((part_num, full_path, filename))
        else:
            key = filename
            if key not in file_groups: 
                file_groups[key] = []
            file_groups[key].append((0, full_path, filename))

    return file_groups


def normalize_seed(seed):
    """
    Accepts:
      - full path or filename or stem
    Returns:
      - seed_stem (no extension)
      - seed_ext (or None if not provided)
    """
    seed = seed.strip()
    seed_name = os.path.basename(seed)  # if full path, take just filename
    stem, ext = os.path.splitext(seed_name)
    seed_ext = ext.lower().lstrip('.') if ext else None
    return stem, seed_ext


def choose_group_key(file_groups, seed_stem, seed_ext, source_hint):
    """
    Find the group key that matches the seed pattern:
    seed must contain a trailing _NUMBER in its stem.
    We strip _NUMBER to get base_name and then match base_name.ext keys.
    """
    m = re.match(r'^(.*)_(\d+)$', seed_stem)
    if not m:
        raise ValueError(f"Seed must contain a trailing _NUMBER pattern, like *_1. Got: {seed_stem}")

    base_name = m.group(1)

    # Candidate keys by base_name
    candidates = [k for k in file_groups.keys() if k.lower().startswith((base_name + ".").lower())]

    if seed_ext:
        # If extension given, narrow to it
        candidates_ext = [k for k in candidates if k.lower().endswith("." + seed_ext.lower())]
        if candidates_ext:
            candidates = candidates_ext

    if not candidates:
        raise FileNotFoundError(f"No matching split-series found for base '{base_name}' in that folder.")

    # If multiple remain (e.g., both .xls and .txt exist), choose deterministically:
    # - WoS: prefer xls/xlsx over txt
    # - Scopus: prefer csv over txt over xls/xlsx
    pref = []
    if source_hint.lower() == "wos":
        pref = [".xls", ".xlsx", ".txt", ".csv"]
    else:
        pref = [".csv", ".txt", ".xls", ".xlsx"]

    # pick the candidate with best preferred ext; tie-break by most parts
    def rank_key(k):
        ext = "." + k.split(".")[-1].lower()
        ext_rank = pref.index(ext) if ext in pref else 999
        parts = len(file_groups.get(k, []))
        return (ext_rank, -parts, k.lower())

    candidates.sort(key=rank_key)
    return candidates[0]


def stitch_group(file_groups, key):
    files = file_groups[key]
    files.sort(key=lambda x: x[0])  # ensure 1,2,3 order

    data_parts = []
    for _, path, fname in files:
        df = smart_loader(path, fname)
        if df is not None and not df.empty:
            data_parts.append(df)

    if not data_parts:
        return None

    final_df = pd.concat(data_parts, ignore_index=True)  # append only
    return final_df


def save_stitched(final_df, key, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    base_name, ext = key.rsplit(".", 1)
    ext = ext.lower()

    # Safer writing choices:
    # - excel-like inputs -> save as .xlsx
    # - txt -> save as tab-delimited .txt
    # - csv -> save as .csv
    if ext in ["xls", "xlsx"]:
        out_path = os.path.join(output_folder, f"{base_name}__APPENDED.xlsx")
        final_df.to_excel(out_path, index=False)
    elif ext == "txt":
        out_path = os.path.join(output_folder, f"{base_name}__APPENDED.txt")
        final_df.to_csv(out_path, sep="\t", index=False)
    else:  # csv or other -> csv
        out_path = os.path.join(output_folder, f"{base_name}__APPENDED.csv")
        final_df.to_csv(out_path, index=False)

    return out_path


# ========================================================================
# RUN APPENDAGE ONLY IF FLAGS ARE TRUE
# ========================================================================

# These globals come from Cell 1:
# APPEND_SCOPUS, APPEND_WOS, SCOPUS_APPEND_FOLDER, SCOPUS_APPEND_SEED, WOS_APPEND_FOLDER, WOS_APPEND_SEED
# OUTPUT_FOLDER, scopus_file_final, wos_file_final

if APPEND_SCOPUS:
    if not SCOPUS_APPEND_FOLDER or not SCOPUS_APPEND_SEED:
        print("⚠️ APPEND_SCOPUS=True but folder/seed not set. Skipping Scopus appendage.")
    else:
        fg = build_file_groups(SCOPUS_APPEND_FOLDER)
        seed_stem, seed_ext = normalize_seed(SCOPUS_APPEND_SEED)
        key = choose_group_key(fg, seed_stem, seed_ext, source_hint="scopus")

        stitched = stitch_group(fg, key)
        if stitched is None:
            print(f"❌ Scopus stitching failed for key: {key} (no readable parts).")
        else:
            out_path = save_stitched(stitched, key, OUTPUT_FOLDER)
            scopus_file_final = out_path
            print(f"✅ Scopus appended: '{key}' ({len(stitched)} rows)")
            print(f"📁 Saved here: {out_path}")

else:
    print("ℹ️ Scopus appendage disabled (APPEND_SCOPUS=False).")


if APPEND_WOS:
    if not WOS_APPEND_FOLDER or not WOS_APPEND_SEED:
        print("⚠️ APPEND_WOS=True but folder/seed not set. Skipping WoS appendage.")
    else:
        fg = build_file_groups(WOS_APPEND_FOLDER)
        seed_stem, seed_ext = normalize_seed(WOS_APPEND_SEED)
        key = choose_group_key(fg, seed_stem, seed_ext, source_hint="wos")

        stitched = stitch_group(fg, key)
        if stitched is None:
            print(f"❌ WoS stitching failed for key: {key} (no readable parts).")
        else:
            out_path = save_stitched(stitched, key, OUTPUT_FOLDER)
            wos_file_final = out_path
            print(f"✅ WoS appended: '{key}' ({len(stitched)} rows)")
            print(f"📁 Saved here: {out_path}")

else:
    print("ℹ️ WoS appendage disabled (APPEND_WOS=False).")


print("\n--------------------------------------------------")
print("✅ Cell 2 complete.")
print(f"Scopus file to load in Cell 3: {scopus_file_final}")
print(f"WoS file to load in Cell 3:    {wos_file_final}")
print("--------------------------------------------------")

# ========================================================================
# OPTIONAL: Download hints (Kaggle / Colab / Local)
# ========================================================================
def detect_environment():
    # Kaggle sets KAGGLE_URL_BASE or KAGGLE_KERNEL_RUN_TYPE in many kernels
    if os.environ.get("KAGGLE_URL_BASE") or os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    # Colab sets COLAB_GPU and generally has /content
    if os.environ.get("COLAB_GPU") or os.path.exists("/content"):
        # Avoid false positives: local machines might also have /content rarely,
        # but this heuristic is good enough for a hint.
        return "colab"
    return "local"

env = detect_environment()

print("\n==================================================")
print("⬇️ Download / Re-upload hint")
print("==================================================")
print(f"Appended exports were saved in:\n  {OUTPUT_FOLDER}\n")

if env == "kaggle":
    print("Kaggle:")
    print("  - Open the right sidebar -> 'Output' section")
    print("  - Your files in the working/output directory should appear there")
    print("  - Click the file(s) to download, then re-upload as a single dataset if needed.")
elif env == "colab":
    print("Google Colab:")
    print("  - In the left sidebar, open the 'Files' panel")
    print("  - Navigate to the folder printed above")
    print("  - Right-click the file(s) -> Download")
    print("  - Re-upload them later via the same Files panel or mount Drive.")
else:
    print("Local Jupyter:")
    print("  - The folder printed above is on your machine.")
    print("  - You can copy/move the appended file(s) anywhere and load them directly next time.")

print("==================================================")


## 2.3. Load Final Files (Single or Stitched)

This cell loads the “final” dataset(s) into memory:

- `df_scopus` (if Scopus file is provided)
- `df_wos` (if WoS file is provided)

### What it prints
- The path it attempted to load
- ✅ success messages (or warnings if a file was missing)
- Final shape:
  - rows × columns

### Next step
After this cell runs successfully, the workflow moves into **schema harmonization** (merging Scopus/WoS fields into one unified dataframe).

In [ ]:
import pandas as pd
import os

# ========================================================================
# LOAD FINAL FILES (direct or appended)
# ========================================================================

# Uses scopus_file_final / wos_file_final from Cells 1-2
scopus_file = scopus_file_final
wos_file    = wos_file_final

df_scopus = None
df_wos = None

# --- Load Scopus ---
if scopus_file and os.path.exists(scopus_file):
    try:
        print(f"Loading Scopus file: {scopus_file}")

        if scopus_file.lower().endswith(('.xlsx', '.xls')):
            try:
                df_scopus = pd.read_excel(scopus_file)
            except Exception:
                print("...Excel read failed, retrying as tab-separated text...")
                df_scopus = pd.read_csv(scopus_file, sep='\t', on_bad_lines='skip', quoting=3)

        elif scopus_file.lower().endswith('.csv'):
            try:
                df_scopus = pd.read_csv(scopus_file, sep=',')
            except Exception:
                print("...Comma CSV read failed, retrying as tab-separated...")
                df_scopus = pd.read_csv(scopus_file, sep='\t', on_bad_lines='skip', quoting=3)

        elif scopus_file.lower().endswith('.txt'):
            # Tab-delimited fallback for Scopus text exports too
            df_scopus = pd.read_csv(scopus_file, sep='\t', on_bad_lines='skip', quoting=3)

        print("✅ Scopus file loaded successfully.")
    except Exception as e:
        print(f"❌ Error loading Scopus file {scopus_file}: {e}. Continuing without Scopus data.")
        df_scopus = None
else:
    print("ℹ️ Scopus file not found or path is empty. Continuing without Scopus data.")


# --- Load WoS ---
if wos_file and os.path.exists(wos_file):
    try:
        print(f"Loading WoS file: {wos_file}")

        if wos_file.lower().endswith('.xls'):
            try:
                df_wos = pd.read_excel(wos_file)
            except Exception:
                print("...Excel read failed, retrying as tab-separated text (WoS anomaly)...")
                df_wos = pd.read_csv(wos_file, sep='\t', on_bad_lines='skip', quoting=3)

        elif wos_file.lower().endswith('.xlsx'):
            df_wos = pd.read_excel(wos_file)

        elif wos_file.lower().endswith('.csv'):
            # WoS CSV is often tab-delimited in practice; try comma then tab
            try:
                df_wos = pd.read_csv(wos_file, sep=',')
            except Exception:
                df_wos = pd.read_csv(wos_file, sep='\t', on_bad_lines='skip', quoting=3)

        elif wos_file.lower().endswith('.txt'):
            df_wos = pd.read_csv(wos_file, sep='\t', on_bad_lines='skip', quoting=3)

        print("✅ WoS file loaded successfully.")
    except Exception as e:
        print(f"❌ Error loading WoS file {wos_file}: {e}. Continuing without WoS data.")
        df_wos = None
else:
    print("ℹ️ WoS file not found or path is empty. Continuing without WoS data.")


# --- Final Loading Check ---
if df_scopus is None and df_wos is None:
    print("\n!!! CRITICAL WARNING: No data files (Scopus or WoS) were loaded. !!!")
    print("Please check paths and ensure at least one file is present.")
else:
    print("\nData loading complete. Proceeding with analysis.")
    if df_scopus is not None:
        print(f"Scopus: {df_scopus.shape[0]:,} rows × {df_scopus.shape[1]:,} cols")
    if df_wos is not None:
        print(f"WoS:    {df_wos.shape[0]:,} rows × {df_wos.shape[1]:,} cols")


## 3. Preprocessing & Standardization (Reproducible Rules)

Bibliometric indicators (e.g., author counts, collaboration networks, MNCS) are highly sensitive to **metadata formatting differences** between Scopus and Web of Science.

This section defines *deterministic* cleaning rules (“heuristics”) so that downstream analyses are reproducible and not driven by database-specific quirks.

---

### 3.1 Author Name Normalization (for accurate collaboration networks)

Different exports encode authors differently:

- **Scopus example:** `Doe J.; Smith A.; Wang Y.`
- **WoS example:** `DOE J; SMITH A; WANG Y`

**Rule:** Convert all author strings to a consistent `"; "` delimited format.

> Why it matters: if delimiters or spacing differ, the same author can be counted as multiple people, inflating collaboration measures.

---

### 3.2 Remove Scopus Internal Author IDs (duplicate-node prevention)

Scopus sometimes includes internal numeric identifiers in parentheses:

- Example: `Smith, John(57211158827)`

**Rule:** Remove patterns like `(123456...)` using regex so that:
- `"Smith, John(572...)"` becomes `"Smith, John"`

---

### 3.3 Reference Parsing → *Cited Reference Count*

We derive a **Cited Reference Count** variable by counting references in bibliography strings:

- Scopus: typically `References`
- WoS: typically `CR`

**Interpretation:** A rough proxy for the breadth of the cited intellectual base (not a quality metric).

---

### ✅ Who should edit this phase?
Most users **do not need** to edit these functions.

You should only modify them if:
- your exports use unusual delimiters, or
- you are working with non-standard author/reference encodings.




In [ ]:
# ========================================================================
# Define Helper Functions for Preprocessing
# ========================================================================

def clean_scopus_authors(authors_str):
    """
    Convert a semicolon-delimited Scopus Authors string 
    into a consistent semicolon-delimited format.
    E.g., "Doe J.; Smith A." -> "Doe J.; Smith A."
    """
    if pd.isna(authors_str):
        return ""
    # Split authors by semicolon, strip whitespace, filter empty strings
    authors_list = [a.strip() for a in authors_str.split(';') if a.strip()]
    return "; ".join(authors_list)

def clean_wos_authors(authors_str):
    """
    Unify WoS authors into a semicolon-delimited string.
    WoS authors may appear as "DOE J; SMITH A" or "DOE J, SMITH A".
    This function ensures each author is separated by a semicolon.
    """
    if pd.isna(authors_str):
        return ""
    # Normalize by replacing multiple spaces with a single space.
    authors_str = re.sub(r"\s+", " ", authors_str)
    
    # WoS most commonly uses semicolons. If not, commas are second.
    if ";" in authors_str:
        delimiter = ";"
    elif "," in authors_str:
        delimiter = ","
    else:
        # If no common delimiter, assume it's a single author
        return authors_str.strip()
        
    # Split by the detected delimiter and clean up
    authors_list = [a.strip() for a in authors_str.split(delimiter) if a.strip()]
    return "; ".join(authors_list)

def remove_id_codes_from_full_names(full_names_str):
    """
    Remove numeric codes in parentheses from Scopus 'Author full names'.
    Example: "Smith, John(57211158827)" -> "Smith, John"
    """
    if pd.isna(full_names_str):
        return ""
    # Use a regular expression to find and remove parentheses containing only digits
    cleaned = re.sub(r"\(\d+\)", "", full_names_str)
    # Normalize whitespace (e.g., "Smith, John " -> "Smith, John")
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned

def scopus_cited_reference_count(ref_str):
    """
    Count references in the Scopus 'References' column.
    """
    if pd.isna(ref_str):
        return 0
    ref_list = [r.strip() for r in ref_str.split(';') if r.strip()]
    return len(ref_list)

def wos_cited_reference_count(ref_str):
    """
    Count references in the WoS 'CR' column.
    """
    if pd.isna(ref_str):
        return 0
    ref_list = [r.strip() for r in ref_str.split(';') if r.strip()]
    return len(ref_list)

## 4. Metadata Schema Harmonization (Unified Bibliometric Schema)

Scopus and WoS provide overlapping information but use **different column names** and sometimes different conventions.

To enable cross-database analysis, we map both sources into a single **Unified Bibliometric Schema**.

---

### 4.1 Schema Mapping Strategy

We use two explicit dictionaries:

- `scopus_to_final`: Scopus → unified column names  
- `wos_to_final`: WoS → unified column names

This is intentionally explicit (not automatic inference), to ensure **reproducibility** and easy auditing.

---

### 4.2 Attribute Selection (memory + comparability)

We filter to a `final_columns` list to:
- reduce memory usage,
- keep analyses comparable across datasets,
- avoid accidental reliance on source-specific columns.

---

### 4.3 PRISMA-style accounting (traceability)

We initialize `prisma_report` to track record counts across stages.

This makes it possible to report:
- how many records were imported per source,
- how many were removed during de-duplication,
- and how many records remain for analysis.

> Note: PRISMA is originally designed for systematic reviews.  
> Here we adapt the *count-tracking principle* to bibliometric ETL for transparency.

### 4.4 Schema Dictionaries (Edit Only If Your Export Has Different Column Names)

Most users should not change this.

You should modify this mapping **only if your export column names differ** from the defaults.


In [ ]:
# ========================================================================
# Map Columns to the Final Common Schema
# ========================================================================

# Define the mapping from Scopus original column names to our final standard names.
# Key = Original Scopus Name, Value = Final Standard Name
scopus_to_final = {
    "Authors": "Authors",
    "Author full names": "Author Full Names",
    "Title": "Title",
    "Source title": "Source Title",
    "Year": "Publication Year",
    "Volume": "Volume",
    "Issue": "Issue",
    "Art. No.": "Article Number",
    "Page start": "Start Page",
    "Page end": "End Page",
    "Page count": "Number of Pages",
    "Cited by": "Cited by", 
    "DOI": "DOI",
    "Abstract": "Abstract",
    "Author Keywords": "Author Keywords",
    "Index Keywords": "Keywords Plus",
    "Language of Original Document": "Language",
    "Document Type": "Document Type",
    "Conference name": "Conference Title",
    "Conference date": "Conference Date",
    "Conference location": "Conference Location",
    "Funding Details": "Funding Orgs",
    "Funding Texts": "Funding Text",
    "Publisher": "Publisher",
    "ISSN": "ISSN",
    "ISBN": "ISBN",
    "Abbreviated Source Title": "Journal Abbreviation",
    "Affiliations": "Affiliations",
    "Cited Reference Count": "Cited Reference Count" 
}

# Define the mapping from Web of Science (WoS) original column names to our final standard names.
wos_to_final = {
    "Authors": "Authors",
    "Author Full Names": "Author Full Names",
    "Article Title": "Title",
    "Source Title": "Source Title",
    "Publication Year": "Publication Year",
    "Volume": "Volume",
    "Issue": "Issue",
    "Article Number": "Article Number",
    "Start Page": "Start Page",
    "End Page": "End Page",
    "Number of Pages": "Number of Pages",
    "Times Cited, WoS Core": "Cited by",
    "Times Cited, All Databases": "Cited by_All",
    "DOI": "DOI",
    "Abstract": "Abstract",
    "Author Keywords": "Author Keywords",
    "Keywords Plus": "Keywords Plus",
    "Language": "Language",
    "Document Type": "Document Type",
    "Conference Title": "Conference Title",
    "Conference Date": "Conference Date",
    "Conference Location": "Conference Location",
    "Funding Orgs": "Funding Orgs",
    "Funding Text": "Funding Text",
    "Publisher": "Publisher",
    "ISSN": "ISSN",
    "ISBN": "ISBN",
    "Journal Abbreviation": "Journal Abbreviation",
    "WoS Categories": "WoS Categories",
    "Web of Science Index": "Web of Science Index",
    "Research Areas": "Research Areas",
    "Addresses": "Affiliations",
    "Cited Reference Count": "Cited Reference Count",
    "CR": "References_Raw_WoS" 
}

# Define the final, ordered list of columns for the merged DataFrame.
final_columns = [
    "Authors", "Author Full Names", "Title", "Source Title", "Publication Year", 
    "Volume", "Issue", "Article Number", "Start Page", "End Page", "Number of Pages", 
    "Cited by", "DOI", "Abstract", "Author Keywords", "Keywords Plus", "Language", 
    "Document Type", "Conference Title", "Conference Date", "Conference Location", 
    "Funding Orgs", "Funding Text", "Publisher", "ISSN", "ISBN", 
    "Journal Abbreviation", "WoS Categories", "Web of Science Index", 
    "Research Areas", "Affiliations", "Cited Reference Count", "Database"
]

# This dictionary will store our numbers for the final report (PRISMA-style)
prisma_report = {
    'scopus_initial': 0,
    'wos_initial': 0,
    'merged_total': 0,
    'removed_by_doi': 0,
    'records_with_doi': 0,
    'records_for_title_check': 0,
    'removed_by_title_year': 0,
    'final_total': 0
}

## 5. Data Standardization and Enrichment

Following ingestion, raw datasets undergo a **standardization protocol** to ensure full interoperability.  
This phase transforms source-specific metadata into the unified schema defined earlier.

---

### Processing Protocol

#### **1. Record Counting**  
Initial record counts are logged for the **PRISMA flow diagram**, ensuring traceability throughout the pipeline.

#### **2. Author Name Normalization**  
The `clean_..._authors` heuristics are applied to harmonize name formats:  
- Internal IDs are stripped out (e.g., `Name(12345)` → `Name`)  
- Delimiters are standardized to support accurate author disambiguation and network analysis.

#### **3. Reference Quantification**  
A **Cited Reference Count** variable is generated by parsing the bibliography strings.  
This acts as a proxy for the *intellectual base* or depth of each article.

#### **4. Schema Enforcement**  
Attributes are systematically renamed and filtered to match the **final unified schema**.  
A *Database provenance tag* is appended to each record to retain source traceability.

This cell processes Scopus and/or WoS depending on what you loaded earlier.

✅ If you loaded only Scopus, WoS will be skipped automatically.  
✅ If you loaded only WoS, Scopus will be skipped automatically.  
✅ If you loaded both, both will be standardized and prepared for merging.

**You should not edit this cell** unless you are:
- adding extra fields to the unified schema, or
- adapting to a non-standard export format.

In [ ]:
# ========================================================================
# Process Scopus Data 
# ========================================================================
all_dfs = [] # This list will hold our processed dataframes

if df_scopus is not None:
    print("Processing Scopus data...")
    # --- REPORTING ---
    # Store the initial count of records
    prisma_report['scopus_initial'] = len(df_scopus)
    
    # Create a copy to avoid 'SettingWithCopyWarning'
    df_scopus_processed = df_scopus.copy()

    # --- Preprocessing ---
    # Calculate cited reference count if 'References' column exists
    if 'References' in df_scopus_processed.columns:
        df_scopus_processed["Cited Reference Count"] = df_scopus_processed["References"].apply(scopus_cited_reference_count)
    else:
        # If no 'References' column, fill with NaN (Not a Number)
        df_scopus_processed["Cited Reference Count"] = np.nan

    # Clean the 'Authors' column
    if 'Authors' in df_scopus_processed.columns:
        df_scopus_processed['Authors'] = df_scopus_processed['Authors'].apply(clean_scopus_authors)

    # Clean the 'Author full names' column
    if 'Author full names' in df_scopus_processed.columns:
        df_scopus_processed['Author full names'] = df_scopus_processed['Author full names'].apply(remove_id_codes_from_full_names)

    # --- Select & Rename ---
    # Get a list of columns that exist in BOTH our dataframe AND our mapping
    scopus_existing_cols = [col for col in scopus_to_final.keys() if col in df_scopus_processed.columns]
    # Filter the dataframe to only these columns and rename them
    df_scopus_processed = df_scopus_processed[scopus_existing_cols].rename(columns=scopus_to_final)
    
    # Add database identifier
    df_scopus_processed['Database'] = 'Scopus'
    
    # Add the processed dataframe to our list for merging
    all_dfs.append(df_scopus_processed)
    print(f"Scopus processing complete. Found {prisma_report['scopus_initial']} initial records.")

# ========================================================================
# 7. Process WoS Data 
# ========================================================================

if df_wos is not None:
    print("Processing WoS data...")
    # --- REPORTING ---
    prisma_report['wos_initial'] = len(df_wos)

    df_wos_processed = df_wos.copy()

    # --- Preprocessing ---
    # Clean the 'Authors' column
    if 'Authors' in df_wos_processed.columns:
        df_wos_processed['Authors'] = df_wos_processed['Authors'].apply(clean_wos_authors)
    
    # WoS doesn't always have 'Author Full Names', so we add it as an empty column if missing
    if 'Author Full Names' not in df_wos_processed.columns:
        df_wos_processed['Author Full Names'] = np.nan
        
    # Create 'Cited Reference Count' for WoS from the 'CR' column
    if 'CR' in df_wos_processed.columns:
        try:
            df_wos_processed["Cited Reference Count"] = df_wos_processed["CR"].apply(wos_cited_reference_count)
        except NameError:
            print("Warning: 'wos_cited_reference_count' function not found. WoS CR count will be NaN.")
            df_wos_processed["Cited Reference Count"] = np.nan
    else:
        df_wos_processed["Cited Reference Count"] = np.nan
        
    # --- Handle WoS Citation Count ---
    # If 'WoS Core' citations aren't present, use 'All Databases' as the main 'Cited by' field.
    if 'Times Cited, WoS Core' not in df_wos_processed.columns and 'Times Cited, All Databases' in df_wos_processed.columns:
         wos_to_final['Times Cited, All Databases'] = 'Cited by'
    elif 'Times Cited, WoS Core' in df_wos_processed.columns:
        # Ensure citation counts are integers, filling missing values with 0
         df_wos_processed['Times Cited, WoS Core'] = df_wos_processed['Times Cited, WoS Core'].fillna(0).astype(int)

    # --- Select & Rename ---
    wos_existing_cols = [col for col in wos_to_final.keys() if col in df_wos_processed.columns]
    df_wos_processed = df_wos_processed[wos_existing_cols].rename(columns=wos_to_final)

    # Add database identifier
    df_wos_processed['Database'] = 'WoS'
    
    # Add to our list for merging
    all_dfs.append(df_wos_processed)
    print(f"WoS processing complete. Found {prisma_report['wos_initial']} initial records.")

## 6. Record Linkage & Data Fusion (Deduplication Without Data Loss)

When using multiple databases (Scopus + WoS), the same paper can appear multiple times.
A naïve approach (“drop duplicates”) often discards secondary records and causes **metadata loss**
(e.g., missing abstracts, keywords, affiliations).

This notebook uses a **Prioritized Data Fusion** strategy:
we keep a single **unique record per publication**, but we **enrich** it using duplicates.

---

### 6.1 Prioritization Rule (Citation Max-Pooling)

We sort records by **Cited by** (descending) so the record with the most citation impact is processed first.

**Interpretation:**  
The top record is treated as the **Primary Record** and tends to have richer metadata (often Scopus, but not always).

---

### 6.2 Hierarchical Deduplication (Precision → Recall)

**Stage 1 — DOI matching (highest precision)**  
If DOI exists, we match on normalized DOI.

**Stage 2 — Title + Year matching (fallback for missing DOI)**  
For records lacking DOI, we match on:
- normalized title (`Title_norm`)
- `Publication Year`

> Note: Title+Year matching is less precise than DOI matching, so we use it only when DOI is missing.

---

### 6.3 Data Fusion (Imputation / Enrichment)

Instead of discarding duplicates, we fuse them:

- For each duplicate group, we keep one final record
- Missing fields (e.g., Abstract / Keywords / Affiliations) are filled from other matched records

**Fusion rule used here:** the group-wise **first non-null value per column**  
(after sorting by citations so the most cited record is first).

---

### ✅ What you should edit in this stage (usually nothing)

Most users should not edit this section.

You might edit it only if:
- your DOI field uses unusual formatting, or
- your dataset has frequent missing years/titles.

Example (rare) change:
```python
# If you want stronger title normalization (more aggressive cleanup):
Title_norm = Title.lower().replace('-', '').replace(':', '')
```



In [ ]:
# ========================================================================
# Merge the DataFrames and Clean Up
# ========================================================================
import pandas as pd
import numpy as np

df_final = None # Initialize final dataframe

if len(all_dfs) > 0:
    print(f"\nCombining {len(all_dfs)} dataset(s)...")
    # Concat all processed dataframes (Scopus, WoS) into a single one
    df_merged = pd.concat(all_dfs, ignore_index=True, sort=False)
    
    # Fill NaN 'Cited by' with 0 before sorting
    if 'Cited by' in df_merged.columns:
        df_merged['Cited by'] = df_merged['Cited by'].fillna(0).astype(int)

    # Reindex all combined data to the final schema (final_columns list)
    # This ensures consistency and drops extra columns
    df_merged = df_merged.reindex(columns=final_columns)
    
    # --- REPORTING ---
    prisma_report['merged_total'] = len(df_merged)
    print(f"Total records before deduplication: {prisma_report['merged_total']}")

    # ========================================================================
    # Deduplicate Merged DataFrame 
    # ========================================================================
    
    # --- Create normalized columns for matching ---
    if 'DOI' in df_merged.columns:
        # Normalize DOI: lowercase and remove whitespace
        df_merged['DOI_norm'] = df_merged['DOI'].str.lower().str.strip()
    else:
        df_merged['DOI_norm'] = pd.NA
        
    if 'Title' in df_merged.columns:
        # Simple normalization: lower, strip, and remove non-alphanumeric chars
        df_merged['Title_norm'] = df_merged['Title'].astype(str).str.lower().str.strip().str.replace('[^a-z0-9]', '', regex=True)
    else:
        df_merged['Title_norm'] = pd.NA

    # --- Prioritize records ---
    # When duplicates exist, we want to keep the one with the most citations.
    # So, we sort the *entire* dataframe by 'Cited by' in descending order first.
    df_merged = df_merged.sort_values(by='Cited by', ascending=False)

    # --- Step 1: Deduplicate based on DOI (most reliable) ---
    # We explicitly exclude NaNs from this step
    doi_mask = df_merged['DOI_norm'].notna()
    df_with_doi = df_merged[doi_mask]
    df_no_doi = df_merged[~doi_mask] # These will be checked in Step 2

    # --- UPGRADE: Data Fusion Strategy (Corrected) ---
    print("\n--- Performing Data Fusion on Duplicates ---")
    
    # 1. Identify duplicates for reporting BEFORE fusion
    # We find groups that have more than 1 row
    duplicate_groups = df_with_doi[df_with_doi.duplicated(subset=['DOI_norm'], keep=False)]
    
    # 2. Sort: Primary by DOI (to group), Secondary by Citations (descending)
    df_with_doi = df_with_doi.sort_values(by=['DOI_norm', 'Cited by'], ascending=[True, False])

    # 3. Capture the "Before" state (the top record only, with its gaps)
    # nth(0) gets the first row of each group
    before_fusion = df_with_doi.groupby('DOI_norm').nth(0)

    # 4. Perform Fusion: 'first()' returns the first NON-NULL value for each column.
    # as_index=False ensures 'DOI_norm' remains a column.
    df_dedup_doi = df_with_doi.groupby('DOI_norm', as_index=False).first()
    
    # 5. Capture the "After" state (to compare)
    # We set index to DOI_norm to match 'before_fusion' structure for comparison
    after_fusion = df_dedup_doi.set_index('DOI_norm')

    # --- REPORTING: Calculate Data Fusion Impact ---
    if not before_fusion.empty:
        # We only care about records that actually *had* duplicates
        dupe_dois = duplicate_groups['DOI_norm'].unique()
        
        # Filter both snapshots to only the duplicate DOIs
        before_snap = before_fusion.loc[before_fusion.index.intersection(dupe_dois)]
        after_snap = after_fusion.loc[after_fusion.index.intersection(dupe_dois)]
        
        # Identify cells that were NaN before but have data now
        filled_cells = (before_snap.isna()) & (after_snap.notna())
        cols_filled = filled_cells.sum()
        cols_filled = cols_filled[cols_filled > 0].sort_values(ascending=False)
        
        total_filled_cells = cols_filled.sum()
        
        if total_filled_cells > 0:
            print(f"Data Fusion Report: Successfully filled {total_filled_cells} missing data points across {len(dupe_dois)} duplicate records.")
            print("Columns enriched (Count of missing values filled):")
            print(cols_filled.to_string())
        else:
            print("Data Fusion Report: No missing metadata was filled (duplicates were identical or top record was complete).")
    # -------------------------------------
    
    # --- REPORTING ---
    prisma_report['records_with_doi'] = len(df_dedup_doi)
    removed_doi = len(df_with_doi) - len(df_dedup_doi)
    prisma_report['removed_by_doi'] = removed_doi
    print(f"\nDeduplication Step 1 (DOI): Removed {removed_doi} duplicates.")
    
    # --- Step 2: For records without a DOI, deduplicate by Title + Year ---
    # --- REPORTING ---
    prisma_report['records_for_title_check'] = len(df_no_doi)
    print(f"Deduplication Step 2 (Title/Year): Checking {len(df_no_doi)} records without DOIs.")

    # Sort the no-DOI records again (redundant, but safe) and drop duplicates
    df_no_doi = df_no_doi.sort_values(by='Cited by', ascending=False)
    
    # For Title/Year, we stick to simple drop_duplicates as "fusion" is riskier without a precise identifier
    df_no_doi_dedup = df_no_doi.drop_duplicates(subset=['Title_norm', 'Publication Year'], keep='first')
    
    # --- REPORTING ---
    removed_title = len(df_no_doi) - len(df_no_doi_dedup)
    prisma_report['removed_by_title_year'] = removed_title
    print(f"Deduplication Step 2 (Title/Year): Removed {removed_title} duplicates.")

    # --- Step 3: Recombine the two sets ---
    # We now have our final, unique set of records.
    df_final = pd.concat([df_dedup_doi, df_no_doi_dedup], ignore_index=True)
    
    # --- REPORTING ---
    prisma_report['final_total'] = len(df_final)

    # Clean up the temporary helper columns
    df_final = df_final.drop(columns=['DOI_norm', 'Title_norm'], errors='ignore')
    
    print(f"\nFinal unique records after deduplication: {prisma_report['final_total']}")
    print("--- Final Combined DataFrame (df_final) is ready for analysis. ---")

else:
    print("\n!!! WARNING: No data was processed. !!!")
    print("Creating an empty DataFrame with the final columns.")
    df_final = pd.DataFrame(columns=final_columns)

## 6.5 Temporal Delimitation (Indexing Lag Correction)

### Why this matters
Scopus and Web of Science (WoS) have an **indexing lag**: the most recent year is often incomplete because new publications are still being released and indexed.

If you plot “articles per year” including the current or incomplete year, you may see a misleading final-year decline.  
That decline is usually a **database artifact**, not a real collapse in research output.

---

### Protocol
We apply a time cut-off to exclude incomplete years. You have two options in the next cell:

#### Option A — Use the **current system year** (automatic)
This excludes the current year and later based on your runtime environment date.

```python
ENABLE_TIME_FILTER = True
MANUAL_COLLECTION_YEAR = None  # or leave empty / falsy
```

**When to use:** you’re running the notebook “today” and you want an automatic cut-off.

---

#### Option B — Use a **manual collection year** (recommended for reproducibility)
This excludes your specified year and later (e.g., if you exported data in 2026 and you know 2026 is incomplete).

```python
ENABLE_TIME_FILTER = True
MANUAL_COLLECTION_YEAR = 2026
```

**When to use:** you want results that are stable and reproducible, independent of system date.

---

### What exactly gets excluded?
The code applies:

> keep rows where `Publication Year < CUTOFF_YEAR`

So if `MANUAL_COLLECTION_YEAR = 2026`, then **2026 and later** are removed.

---

### Option C — Disable the time filter (keep all years)
Use this only if you intentionally want to include the incomplete year (e.g., for “up-to-now” dashboards).

```python
ENABLE_TIME_FILTER = False
MANUAL_COLLECTION_YEAR = 2026   # optional; ignored when ENABLE_TIME_FILTER=False
```

**What it does:** keeps the full dataset unchanged.  
⚠️ **Warning:** your time-series plots may show an artificial final-year decline.

---

### ✅ What you should change (most users only edit these)
In the next code cell:

```python
ENABLE_TIME_FILTER = True
MANUAL_COLLECTION_YEAR = 2026
```

If you’re unsure, keep `ENABLE_TIME_FILTER = True` and set `MANUAL_COLLECTION_YEAR` to the year you exported your records (or the last incomplete year).

---

### Example: expected output (what you should see printed)
You should see messages like:

- If records were removed:
```
--- Applying Time Cut-off (Enabled) ---
    Target: Exclude year 2026 and later.
    Source: MANUAL input
 Success: Removed 123 documents from 2026 or later.
   Reason: Data for the collection year is usually incomplete.

 Final Analysis Range: 1998 to 2025
 Total Documents: 4,567
```

- If nothing was removed:
```
--- Applying Time Cut-off (Enabled) ---
    Target: Exclude year 2026 and later.
    Source: MANUAL input
  No documents found for 2026. Data was already clean.
```

---

### Notes for non-coders (safe defaults)
- If you don’t know what this means: **leave it enabled** and set the manual year to your export year.
- This step does **not delete files**. It only filters the in-memory dataset used for analysis.
- After filtering, the notebook updates:
  - `df_final` (so all downstream plots use the same filtered dataset)
  - `prisma_report['excluded_by_time_filter']`

---

In [ ]:
# ========================================================================
# Analysis Parameters: Time-Based Cut-off  (MOVED HERE)
#    NOTE: We do NOT change the filter logic. We only run it on df_final.
# ========================================================================

# Make the time-filter cell operate on df_final (without editing its internals)
df = df_final.copy()

# -------------------- PASTE YOUR TIME FILTER CELL BELOW (UNCHANGED) --------------------
from datetime import datetime

# --- USER CONFIGURATION -------------------------------------------------
ENABLE_TIME_FILTER = True
# MANUAL_COLLECTION_YEAR = None
MANUAL_COLLECTION_YEAR = 2026
#MANUAL_COLLECTION_YEAR = 2025

# ------------------------------------------------------------------------

if 'Publication Year' in df.columns:
    # Ensure Year is numeric
    df['Publication Year'] = pd.to_numeric(df['Publication Year'], errors='coerce')
    initial_count = len(df)

    if ENABLE_TIME_FILTER:
        # --- Determine the Cut-off Year ---
        if MANUAL_COLLECTION_YEAR:
            CUTOFF_YEAR = int(MANUAL_COLLECTION_YEAR)
            source_msg = "MANUAL input"
        else:
            CUTOFF_YEAR = datetime.now().year
            source_msg = "CURRENT system date"

        print(f"\n--- Applying Time Cut-off (Enabled) ---")
        print(f"    Target: Exclude year {CUTOFF_YEAR} and later.")
        print(f"    Source: {source_msg}")

        # --- Apply Filter ---
        df = df[df['Publication Year'] < CUTOFF_YEAR].copy()
        
        # --- Report ---
        final_count = len(df)
        removed_count = initial_count - final_count
        
        if removed_count > 0:
            print(f" Success: Removed {removed_count} documents from {CUTOFF_YEAR} or later.")
            print(f"   Reason: Data for the collection year is usually incomplete.")
        else:
            print(f"  No documents found for {CUTOFF_YEAR}. Data was already clean.")
            
    else:
        # --- Filter Disabled ---
        print(f"\n--- Time Cut-off DISABLED ---")
        print(f" WARNING: You are including potentially incomplete data from the final year.")
        print(f"    Your 'Articles per Year' charts may show a false decline at the end.")
    
    # Final Stats
    if not df.empty:
        min_y = int(df['Publication Year'].min())
        max_y = int(df['Publication Year'].max())
        print(f"\n Final Analysis Range: {min_y} to {max_y}")
        print(f" Total Documents: {len(df)}")

else:
    print("!!! Warning: 'Publication Year' column not found. Cannot apply time cut-off.")

# Push filtered data back into df_final (so everything downstream uses it)
df_final = df.copy()

# Add PRISMA bookkeeping
if ENABLE_TIME_FILTER and 'removed_count' in locals():
    prisma_report['excluded_by_time_filter'] = int(removed_count)
else:
    prisma_report['excluded_by_time_filter'] = 0

# Update final total to reflect the INCLUDED set after time filtering
prisma_report['final_total'] = len(df_final)

## 7. Data Verification, PRISMA-style Reporting, and Export

This final ETL step does three things:

1) **Prints PRISMA-style record counts**  
   - how many records were loaded from Scopus and WoS  
   - how many duplicates were removed (DOI + Title/Year)  
   - how many records were excluded by the time filter

2) **Sanity check (integrity validation)**  
   Confirms that:
   > merged_total − removed_duplicates − excluded_time == final_total

3) **Exports the final dataset** (`df_final`) as a CSV  
   This exported file is the clean input for all downstream analyses (countries, institutions, keywords, networks, etc.).

---

### Example: expected PRISMA output
You should see a console block similar to:

```
================================================
     BIBLIOMETRIC REPORTING (PRISMA Style)
================================================
1. IDENTIFICATION
------------------------------------------------
Records identified from Scopus:   2,345
Records identified from WoS:      1,876
Total records loaded:            4,221
Total records after merge:       4,221

2. SCREENING (DEDUPLICATION + TIME FILTER)
------------------------------------------------
Total records before deduplication: 4,221
Duplicates removed by DOI:          312
Duplicates removed by Title/Year:   88
Excluded by Time Filter:            123
------------------------------------------------
Total records removed:              523

3. INCLUDED
------------------------------------------------
Final records in dataset:           3,698
================================================
Sanity Check: Passed.
```

If the sanity check fails, it prints a warning; that indicates something unusual happened and should be inspected.

---

### ✅ What you might change (optional)
Most users won’t need to change anything.

If you want a different filename/location, edit the `output_file` path in the next code cell.

---

### Output paths by environment

### Kaggle (default)
Kaggle uses `/kaggle/working` as the writable directory.

```python
output_file = "/kaggle/working/merged_bibliometric_dataset.csv"
```

**Where to download it:** Kaggle sidebar → **Output**.

---

### Google Colab (default)
Colab uses `/content` as the writable directory.

```python
output_file = "/content/merged_bibliometric_dataset.csv"
```

---

### Google Colab + Google Drive (recommended for persistence)
If you want the file to persist across sessions, mount Drive:

```python
from google.colab import drive
drive.mount("/content/drive")
```

Then save into Drive:

```python
output_file = "/content/drive/MyDrive/ChronotomeOutputs/merged_bibliometric_dataset.csv"
```

Tip: ensure the folder exists (the code below is safe to paste above the save line):
```python
import os
os.makedirs(os.path.dirname(output_file), exist_ok=True)
```

---

### Local Jupyter / VS Code / Desktop (recommended)
Use a relative folder near your notebook:

```python
import os
output_file = os.path.join("output", "merged_bibliometric_dataset.csv")
os.makedirs(os.path.dirname(output_file), exist_ok=True)
```

This creates an `output/` folder next to your notebook (if it doesn’t exist) and saves the CSV inside it.

---

### Example: expected export output
After saving, you should see something like:

```
Final merged dataset saved to: /content/merged_bibliometric_dataset.csv
```

(or your chosen path).

---

### Non-coder guidance (safe defaults)
- If you’re unsure, keep the default path for your platform:
  - Kaggle: `/kaggle/working/...`
  - Colab: `/content/...`
  - Local: `output/...` (recommended)
- The exported CSV is the **main artifact** most users will want to download and reuse.

In [ ]:
# ========================================================================
# PRISMA / Reporting Summary
# ========================================================================

excluded_time = prisma_report.get('excluded_by_time_filter', 0)

print("\n\n================================================")
print("     BIBLIOMETRIC REPORTING (PRISMA Style)")
print("================================================")
print("1. IDENTIFICATION")
print("------------------------------------------------")
print(f"Records identified from Scopus:   {prisma_report['scopus_initial']}")
print(f"Records identified from WoS:      {prisma_report['wos_initial']}")
print(f"Total records loaded:         {prisma_report['scopus_initial'] + prisma_report['wos_initial']}")
print(f"Total records after merge:       {prisma_report['merged_total']}")

print("\n2. SCREENING (DEDUPLICATION + TIME FILTER)")
print("------------------------------------------------")
print(f"Total records before deduplication: {prisma_report['merged_total']}")
print(f"Duplicates removed by DOI:          {prisma_report['removed_by_doi']}")
print(f"Duplicates removed by Title/Year:   {prisma_report['removed_by_title_year']}")
print(f"Excluded by Time Filter:            {excluded_time}")
print("------------------------------------------------")

total_removed = prisma_report['removed_by_doi'] + prisma_report['removed_by_title_year'] + excluded_time
print(f"Total records removed:              {total_removed}")

print("\n3. INCLUDED")
print("------------------------------------------------")
print(f"Final records in dataset:           {prisma_report['final_total']}")
print("================================================")

# --- Sanity Check ---
if prisma_report['merged_total'] > 0:
    try:
        assert prisma_report['merged_total'] - total_removed == prisma_report['final_total']
        print("Sanity Check: Passed.")
    except AssertionError:
        print("!!! Sanity Check: FAILED. Final count does not match. !!!")

# ========================================================================
# Save the Final Merged Data 
# ========================================================================

if df_final is not None and not df_final.empty:
    # Define the output file path in the Kaggle working directory
    output_file = "/kaggle/working/merged_bibliometric_dataset.csv"
    
    # Save to CSV. 'utf-8-sig' ensures special characters and Excel compatibility.
    df_final.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"\nFinal merged dataset saved to: {output_file}")
else:
    print("\nNo data to save. Final dataframe is empty.")

## 8. Institutional and Geographic Entity Resolution 

This section turns **raw affiliation strings** into standardized, analysis-ready entities:

- **Institution_Extracted** (canonical institution label)
- **Country_Standardized** (ISO-consistent country name via `pycountry`)
- **Affiliation-level table** (one row per affiliation) + **article-level summary** (countries/institutions per paper)

It is designed for the reality of bibliographic data: abbreviations, mixed languages, punctuation noise, and “address-like” fragments.

---

### What inputs this stage needs

From earlier ETL steps you should already have:

- `df_final` — the merged, deduplicated dataset  
- An **Affiliations** column in `df_final` (Scopus/WoS typically provide this; naming may vary)

If your dataset uses a different affiliation column name, rename it earlier (or adapt the upstream mapping).  
This stage assumes:

```python
df_final["Affiliations"]
```

exists.

---

### Example: what affiliation strings look like (realistic)

Affiliations are often long and inconsistent, e.g.:

```text
Dept Astron, Univ. of Calif., Los Angeles, CA 90095, USA
California Inst. of Tech. (Caltech), Pasadena, United States of America
CNRS; Univ Paris Saclay, Orsay, FRANCE
European Space Agency (ESA), ESAC Campus, Madrid, Spain
Max Planck Inst for Solar System Research, Göttingen, Germany
```

Notice the problems:

- Abbreviations: `Univ.` / `Inst.` / `Tech.`
- Mixed casing: `FRANCE` / `United States of America`
- Diacritics: `Göttingen`
- Addresses mixed with org names
- Multiple institutions in one field, often separated by `;`

---

### What this stage produces

### Output 1 — Affiliation-level table (`df_affiliations_final`)
**One row per affiliation** (exploded from article-level data). Typical columns:

- `Article_Index` (links back to the article)
- `Affiliations_Cleaned`
- `Institution_Extracted`
- `Country_Standardized`

Saved as:

- **Kaggle**  
  - `/kaggle/working/exploded_affiliations_list.csv`  
  - `/kaggle/working/exploded_affiliations_list.xlsx`

- **Google Colab** (recommended folder pattern)  
  - `/content/exploded_affiliations_list.csv`  
  - `/content/exploded_affiliations_list.xlsx`

- **Local Jupyter** (recommended)  
  - `./output/exploded_affiliations_list.csv`  
  - `./output/exploded_affiliations_list.xlsx`

> Your current code saves to `/kaggle/working/…`.  
> If you run outside Kaggle, edit those save paths in **Cell 25**.

---

### Output 2 — Article-level summary (`df_article_summary`)
This keeps the original article rows and adds:

- `Countries_Extracted` (list of unique standardized countries)
- `Institutions_Extracted` (list of unique standardized institutions)
- `Country_Count`
- `Collaboration_Type` → *International / National / Undefined*

Saved as:

- **Kaggle**  
  - `/kaggle/working/final_article_summary_with_countries.csv`  
  - `/kaggle/working/final_article_summary_with_countries.xlsx`

- **Google Colab** (recommended)  
  - `/content/final_article_summary_with_countries.csv`  
  - `/content/final_article_summary_with_countries.xlsx`

- **Local Jupyter** (recommended)  
  - `./output/final_article_summary_with_countries.csv`  
  - `./output/final_article_summary_with_countries.xlsx`

---

### Where users typically customize (✅ edit-only sections)

This stage is intentionally rule-based and transparent. Most users only adjust **three things**:

### 1) Institution alias JSON path (highly recommended)

In **Cell 25**, you have:

```python
INSTITUTION_ALIASES = load_and_merge_aliases(
    "/kaggle/input/spatial-planetary-movement/institutions.json",
    INSTITUTION_ALIASES_BACKUP
)
```

Change the first argument depending on platform:

**Kaggle**
```python
"/kaggle/input/<your-dataset>/institutions.json"
```

**Google Colab** (file uploaded to session)
```python
"/content/institutions.json"
```

**Google Drive mounted**
```python
"/content/drive/MyDrive/<folder>/institutions.json"
```

**Local Jupyter**
```python
"./data/institutions.json"
```

If the JSON does not exist, the script automatically falls back to the embedded
`INSTITUTION_ALIASES_BACKUP`.

**Where does `institutions.json` come from?**  
- It ships with the **Zenodo archive** (recommended), and it is also included in the **example Kaggle dataset** bundled with the demo notebook.

If `institutions.json` is missing, the pipeline falls back to a built-in backup alias list so the notebook can still run.

---



### 2) Country aliases (optional)
The `COUNTRY_ALIASES` dictionary (Cell 24) is where you fix recurring variants:

```python
"USA": "United States",
"UNITED STATES OF AMERICA": "United States",
"TURKEY": "Republic of Türkiye",
```

If your dataset has consistent country labels already, you may not need to expand this.

---

### 3) Output folder / filenames (platform portability)
In **Cell 25**, the saves are currently hard-coded to Kaggle:

```python
df_affiliations_final.to_csv("/kaggle/working/exploded_affiliations_list.csv", ...)
df_article_summary.to_csv("/kaggle/working/final_article_summary_with_countries.csv", ...)
```

For portability, users running outside Kaggle should replace `/kaggle/working/` with:

- Colab: `/content/`
- Local: `./output/` (create this folder first)

---

### Python mini-demo (conceptual) — explode affiliations

The explode logic converts article-level rows into affiliation-level rows:

```python
df_exploded = df_final.assign(
    Affiliations_Cleaned=df_final["Affiliations"].str.split(";")
).explode("Affiliations_Cleaned")

df_exploded["Affiliations_Cleaned"] = df_exploded["Affiliations_Cleaned"].str.strip()
df_exploded = df_exploded.dropna(subset=["Affiliations_Cleaned"])
df_exploded = df_exploded[df_exploded["Affiliations_Cleaned"] != ""]
```

If an article has 3 affiliations, that one article becomes **3 rows** in `df_exploded`.

---

In [ ]:
# ========================================================================
# Install Required Package
# ========================================================================
# We need 'pycountry' to standardize country names

!pip install pycountry

In [ ]:
COUNTRY_ALIASES = {
    "PEOPLES R CHINA": "China", "PEOPLES R. CHINA": "China", "PEOPLES REP CHINA": "China",
    "PEOPLES REPUBLIC CHINA": "China", "UNITED STATES OF AMERICA": "United States",
    "USA": "United States", "U.S.A.": "United States", "UNITED KINGDOM": "United Kingdom",
    "ENGLAND": "United Kingdom", "SCOTLAND": "United Kingdom", "NORTHERN IRELAND": "United Kingdom",
    "UK": "United Kingdom", "U.K.": "United Kingdom", "WALES": "United Kingdom",
    "NORTH IRELAND": "United Kingdom", "TURKEY": "Republic of Türkiye", "TURKIYE": "Republic of Türkiye",
    "U ARAB EMIRATES": "United Arab Emirates", "RUSSIA": "Russia", "RUSSIAN FEDERATION": "Russia",
    "VATICAN": "Vatican", "BOSNIA & HERCEG": "Bosnia and Herzegovina", "BOSNIA & HERCEGOVINA": "Bosnia and Herzegovina",
    "BRUNEI": "Brunei", "DOMINICAN REP": "Dominican Republic", "SOUTH KOREA": "South Korea",
    "KOREA": "South Korea", "REP OF KOREA": "South Korea",
}

In [ ]:

import re
import os
import json
import unicodedata
import pycountry
import pandas as pd
from IPython.display import display
from collections import Counter

# ========================================================================
# Global Statistics Parsers
# ========================================================================
stats_abbrev = Counter()
stats_aliases = Counter()
stats_complex = Counter()
stats_alias_collisions = Counter()  # NEW: detect key->different canonical collisions
stats_backup_ignored = Counter()    # NEW: backup canonicals ignored when JSON exists

# ========================================================================
# Helper: Consistent Key Generation
# ========================================================================
def create_lookup_key(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'\b(of|the|for|and|an|a)\b', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ========================================================================
# Configuration: Institution Aliases (BACKUP)
# ========================================================================
INSTITUTION_ALIASES_BACKUP = {
    "California Institute of Technology": ["caltech", "california institute technology"],
    "Chinese Academy of Sciences": ["chinese academy science", "university chinese academy science", "ucas"],
    "University of California System": ["university california"],
    "University of California Berkeley": ["university calif berkeley", "uc berkeley"],
    "University of California Los Angeles": ["ucla"],
    "University of California Irvine": ["uc irvine"],
    "Johns Hopkins University": ["johns hopkins university", "johns hopkins univ"],
    "Leiden University": ["leiden university", "leiden univ"],
    "NASA": ["nasa"],
    "NASA Ames Research Center": ["nasa ames research center"],
    "INAF": ["inaf"],
    "University of Arizona": ["university arizona", "univ arizona"],
    "Cornell University": ["cornell university", "cornell univ"],
    "University Paris Saclay": ["university paris saclay"],
    "MIT": ["mit"],
    "ETH Zurich": ["eth zurich", "eth zuerich", "eidgenossische technische hochschule zurich"],
    "KU Leuven": ["ku leuven", "katholieke universiteit leuven"],
    "DLR": ["dlr", "german aerospace center", "deutsches zentrum fur luft und raumfahrt"],
    "European Space Agency (ESA)": ["esa", "european space agency"],
    "CNRS": ["cnrs"],
    "CNR": ["cnr"],
    "UFMG": ["ufmg", "federal university of minas gerais"],
    "Sternberg Astronomical Institute": ["sternberg astron inst"],
    "Toulouse School of Economics": ["toulouse sch econ"],
    "Institute of Planetology and Astrophysics of Grenoble": ["inst planetol & astrophys grenoble osug a"],
    "American Museum of Natural History": ["american museum natural history"],
    "Natural History Museum": ["natural history museum"],
    "University of Munster": ["westfalische wilhelms universitat"],
    "Academy of Sciences of the Czech Republic": ["academy of sciences cr"],
    "National Autonomous University of Mexico": ["univ nacl autonoma mexico"],
    "European Space Operations Centre": ["esoc", "european space operat ctr esa esoc"],
    "University of La Laguna": ["university la laguna"],
    "Instituto de Astrofísica de Canarias": ["institute astrofis canarias"],
    "Beihang University": ["beihang university"],
    "Center for Astrophysics | Harvard & Smithsonian": ["center astrophys harvard & smithsonian"],
    "Chalmers University of Technology": ["chalmers university technology"],
    "Delft University of Technology": ["delft university technology"],
    "Austrian Academy of Sciences": ["austrian academy science"],
    "Dublin Institute for Advanced Studies": ["dublin institute adv studies"],
    "European Space Astronomy Centre (ESAC)": ["esac campus"],
    "Complutense University of Madrid": ["university complutense madrid"],
    "Carnegie Institution for Science": ["carnegie institute science"],
    "Aristotle University of Thessaloniki": ["aristotle university thessaloniki"],
    "Eötvös Loránd University": ["eotvos lorand university"],
    "European Southern Observatory (ESO)": ["european southern observatory"]
}

# ========================================================================
# JSON Loader (Robust merge: JSON authoritative, backup enriches only)
# ========================================================================
def _canonicalize_display_name(name: str) -> str:
    """
    Only used for display polishing of the canonical label from JSON.
    IMPORTANT: we do NOT title-case blindly because it can break acronyms.
    We only fix the degenerate cases where the whole string is lower case.
    """
    if not isinstance(name, str):
        return name
    clean = name.strip()
    if clean.islower():
        # acronyms like 'nasa' -> 'NASA', but longer names -> Title Case
        return clean.upper() if len(clean) <= 4 else clean.title()
    return clean

def _safe_put(flattened: dict, key: str, canonical: str):
    """
    Detect alias collisions where the same normalized key maps to different canonicals.
    We keep the first mapping and log collisions (so you can review).
    """
    if key in flattened and flattened[key] != canonical:
        stats_alias_collisions[f"Key '{key}' : '{flattened[key]}' vs '{canonical}'"] += 1
        return
    flattened[key] = canonical

def load_and_merge_aliases(json_path, backup_dict):
    flattened = {}

    # ---- If JSON exists: authoritative
    if os.path.exists(json_path):
        try:
            with open(json_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            # 1) Load canonicals + aliases from JSON first (source of truth)
            json_canonicals = set()
            json_alias_count = 0

            for official_name, aliases in data.items():
                canonical = _canonicalize_display_name(official_name)
                json_canonicals.add(canonical)

                _safe_put(flattened, create_lookup_key(canonical), canonical)

                # aliases can be list-like; be defensive
                if isinstance(aliases, (list, tuple, set)):
                    for alias in aliases:
                        if not isinstance(alias, str) or not alias.strip():
                            continue
                        _safe_put(flattened, create_lookup_key(alias), canonical)
                        json_alias_count += 1

            print(f"✅ Loaded JSON canonicals={len(json_canonicals)} aliases={json_alias_count} from '{json_path}'.")

            # 2) Backup only enriches existing JSON canonicals (no new canonicals)
            backup_added = 0
            backup_ignored = 0

            for backup_canonical, backup_aliases in backup_dict.items():
                # Match backup canonical against JSON canonicals by lookup-key
                # (handles minor casing/article differences)
                backup_key = create_lookup_key(backup_canonical)

                # Find the "true" JSON canonical by key
                # (since flattened already contains canonical key -> canonical)
                if backup_key in flattened:
                    true_canonical = flattened[backup_key]
                    if isinstance(backup_aliases, (list, tuple, set)):
                        for alias in backup_aliases:
                            if not isinstance(alias, str) or not alias.strip():
                                continue
                            k = create_lookup_key(alias)
                            if k not in flattened:
                                flattened[k] = true_canonical
                                backup_added += 1
                            elif flattened[k] != true_canonical:
                                stats_alias_collisions[f"Key '{k}' : '{flattened[k]}' vs '{true_canonical}'"] += 1
                else:
                    stats_backup_ignored[backup_canonical] += 1
                    backup_ignored += 1

            print(f"🧩 Backup enrichment: added={backup_added} ignored_canonicals={backup_ignored} (not present in JSON).")

            # Optional: show a few collision examples
            if stats_alias_collisions:
                print(f"⚠️  Alias collisions detected: {sum(stats_alias_collisions.values())} (showing up to 10)")
                for item, cnt in stats_alias_collisions.most_common(10):
                    print(f"   - {item} : {cnt}")

            return flattened

        except Exception as e:
            print(f"⚠️  Error loading JSON: {e}. Falling back to backup-only.")

    # ---- JSON missing or failed: backup authoritative
    for official_name, aliases in backup_dict.items():
        canonical = _canonicalize_display_name(official_name)
        _safe_put(flattened, create_lookup_key(canonical), canonical)
        if isinstance(aliases, (list, tuple, set)):
            for alias in aliases:
                if not isinstance(alias, str) or not alias.strip():
                    continue
                _safe_put(flattened, create_lookup_key(alias), canonical)

    print(f"ℹ️  Using backup dictionary only. canonicals={len({v for v in flattened.values()})} entries={len(flattened)}")
    return flattened


In [ ]:
# IMPORTANT: set this path to your Kaggle input path (or local path)
INSTITUTION_ALIASES = load_and_merge_aliases(
    "/kaggle/input/spatial-planetary-movement/institutions.json",
    INSTITUTION_ALIASES_BACKUP
)


In [ ]:
# ========================================================================
# Complex Patterns & Fixes
# ========================================================================
MAX_PLANCK_PATTERN = re.compile(r'\bmax\s*(?:planck)?\s*(?:institute|inst)?\b', re.IGNORECASE)
CSIC_PATTERN = re.compile(r'\b(?:csic|inta|consejo superior de investigaciones|centro de astrobiologia|CSIC|INTA)\b', re.IGNORECASE)

ACRONYM_FIXES = {
    "Nasa": "NASA", "Cnrs": "CNRS", "Esa": "ESA", "Jaxa": "JAXA",
    "Cnr": "CNR", "Csic": "CSIC", "Inaf": "INAF", "Unesp": "UNESP", "Unifesp": "UNIFESP",
    "Mit": "MIT", "Inta": "INTA", "Isas": "ISAS", "Cnes": "CNES",
    "Ukri": "UKRI", "Stfc": "STFC", "Ucl": "UCL", "Eth": "ETH", "Ku": "KU",
    "Osug": "OSUG", "Ufmg": "UFMG", "Ucla": "UCLA", "Dlr": "DLR",
    "Kfki": "KFKI", "Cicese": "CICESE", "Esac": "ESAC", "Eso": "ESO", "Asi": "ASI"
}

# ========================================================================
# Abbreviation Replacements
# ========================================================================
ABBREV_REPLACEMENTS = {
    r"\bEngn\b": "Engineering",
    r"\bSci\b": "Science",
    r"\bTech\b": "Technology",
    r"\bTechnol\b": "Technology",
    r"\bTecnol\b": "Tecnologia",
    r"\bColl\b": "College",
    r"\bAcad\b": "Academy",
    r"\bInst\b": "Institute",
    r"\bUniv\b": "University",
    r"\bHosp\b": "Hospital",
    r"\bDist\b": "District",
    r"\bGen\b": "General",
    r"\bNatl\b": "National",
    r"\bPolytech\b": "Polytechnic",
    r"\bProv\b": "Provincial",
    r"\bDept\b": "Department",
    r"\bLab\b": "Laboratory",
    r"\bPolitecn\b": "Politecnico",
    r"\bRes\b": "Research",
    r"\bFed\b": "Federal",
    r"\bAerosp\b": "Aerospace",
    r"\bAgcy\b": "Agency",
    r"\bCtr\b": "Center",
    r"\bInt\b": "International",
    r"\bEduc\b": "Education",
    r"\bNacl\b": "Nacional",
    r"\bNazl\b": "Nazionale",
    r"\bGeofis\b": "Geofisica",
    r"\bVulcanol\b": "Vulcanologia",
    r"\bObsv\b": "Observatory",
    r"\bObserv\b": "Observatory",
    r"\bOperat\b": "Operations",
    r"\bIst\b": "Istituto",
    r"\bSez\b": "Sezione",
    r"\bDef\b": "Defense",
    r"\bGeosci\b": "Geoscience",
    r"\bGeol\b": "Geology",
    r"\bFis\b": "Fisica",
    r"\bSch\b": "School",
    r"\bEcon\b": "Economics",
    r"\bAppl\b": "Applied",
    r"\bNat\b": "Natural",
    r"\bHist\b": "History",
    r"\bAssoc\b": "Association",
    r"\bAstrobiol\b": "Astrobiology",
    r"\bMed\b": "Medicine",
    r"\bMin\b": "Ministry",
    r"\bEnvironm\b": "Environmental",
    r"\bAstrofis\b": "Astrofisica",
}

# ============================
# Helper Functions
# ============================

def normalize_diacritics(text: str) -> str:
    if not isinstance(text, str):
        return text
    nfkd_form = unicodedata.normalize('NFD', text)
    return "".join([c for c in nfkd_form if not unicodedata.combining(c)])

def remove_random_punctuation(text: str) -> str:
    if not isinstance(text, str):
        return text
    text = re.sub(r"\.\s+(of|the|a|an)\b", r" \1", text, flags=re.IGNORECASE)
    text = re.sub(r"\.+", ".", text)
    text = re.sub(r"\s+\.", ".", text)
    text = re.sub(r"\.(\s+|$)", "", text)
    text = re.sub(r"^\)", "", text)
    text = re.sub(r"\($", "", text)
    text = re.sub(r"\s{2,}", " ", text).strip()
    return text

def apply_abbrev_replacements(text: str) -> str:
    if not isinstance(text, str):
        return text
    for pattern, replacement in ABBREV_REPLACEMENTS.items():
        text, count = re.subn(pattern, replacement, text, flags=re.IGNORECASE)
        if count > 0:
            clean_pat = pattern.replace(r'\b', '')
            log_key = f"Abbrev: '{clean_pat}' -> '{replacement}'"
            stats_abbrev[log_key] += count
    return text

def standardize_institution_name(institution: str) -> str:
    if pd.isna(institution) or not str(institution).strip():
        return None

    inst_clean = str(institution)
    inst_clean = normalize_diacritics(inst_clean)
    inst_clean = remove_random_punctuation(inst_clean)
    inst_clean = apply_abbrev_replacements(inst_clean)
    inst_lower = inst_clean.lower()

    # --- Regex-based fixes ---
    if "eth" in inst_lower and "zurich" in inst_lower:
        return "ETH Zurich"
    if "ku leuven" in inst_lower or "katholieke universiteit leuven" in inst_lower:
        return "KU Leuven"
    if "academy of sciences cr" in inst_lower:
        return "Academy of Sciences of the Czech Republic"
    if "esoc" in inst_lower or "european space operat" in inst_lower:
        return "European Space Operations Centre"

    if MAX_PLANCK_PATTERN.search(inst_lower):
        stats_complex["Max Planck Pattern"] += 1
        return "Max Planck Institute"

    if CSIC_PATTERN.search(inst_lower):
        stats_complex["CSIC/INTA Pattern"] += 1
        return "CSIC"

    if "ucla" in inst_lower:
        return "University of California Los Angeles"
    if "dlr" in inst_lower:
        return "DLR"

    # --- LOOKUP LOGIC ---
    inst_lookup = create_lookup_key(inst_lower)
    if inst_lookup in INSTITUTION_ALIASES:
        canonical = INSTITUTION_ALIASES[inst_lookup]
        stats_aliases[f"Alias: '{inst_lookup}' -> '{canonical}'"] += 1
        return canonical

    # --- FALLBACK LOGIC ---
    inst_final = inst_clean.title()
    inst_final = re.sub(r"\s{2,}", " ", inst_final).strip()

    non_acronyms = {
        "The", "And", "For", "New", "Los", "San", "Via", "Rue", "Des", "Les", "Der", "Und", "Fur",
        "Univ", "Inst", "Coll", "Dept", "Lab", "Cent", "Sect", "Unit", "Area", "City", "Town", "Park",
        "Road", "Lane", "Blvd", "Pkwy", "East", "West", "Main", "High"
    }

    words = inst_final.split()
    final_words = []
    for w in words:
        clean_w = re.sub(r'[^\w]', '', w)
        if 3 <= len(clean_w) <= 4 and w not in non_acronyms:
            final_words.append(w.upper())
        else:
            final_words.append(w)
    inst_final = " ".join(final_words)

    for wrong, right in ACRONYM_FIXES.items():
        inst_final = re.sub(r'\b' + wrong + r'\b', right, inst_final)

    return inst_final if inst_final else None

# Keep your existing COUNTRY_ALIASES dict above these functions, unchanged.
def unify_country_alias(raw_string: str) -> str:
    if not raw_string:
        return None
    candidate = raw_string.upper().strip()
    candidate = re.sub(r"^(?:[A-Z]{2}\s+)+", "", candidate)
    candidate = re.sub(r"\b\d{5}(?:-\d{4})?\b", "", candidate)
    candidate = re.sub(r"[,\-]", " ", candidate)
    candidate = re.sub(r"\s{2,}", " ", candidate).strip()

    if 'COUNTRY_ALIASES' in globals() and candidate in COUNTRY_ALIASES:
        return COUNTRY_ALIASES[candidate]

    try:
        return pycountry.countries.lookup(candidate).name
    except LookupError:
        pass

    candidate = candidate.replace("&", "AND")
    candidate = re.sub(r"[^A-Z ]", "", candidate).strip()
    try:
        return pycountry.countries.lookup(candidate).name
    except LookupError:
        if 'COUNTRY_ALIASES' in globals() and candidate in COUNTRY_ALIASES:
            return COUNTRY_ALIASES[candidate]
        return None

def unify_countries_in_row(country_name: str) -> str:
    if pd.isnull(country_name) or not isinstance(country_name, str):
        return None
    return unify_country_alias(country_name)

def clean_affiliations(affiliation):
    if pd.isna(affiliation):
        return None
    affiliation = affiliation.replace("&amp;", "&")
    affiliation = re.sub(r"\[.*?\]", "", affiliation)
    affiliation = normalize_diacritics(str(affiliation))
    affiliation = remove_random_punctuation(affiliation)
    affiliation = re.sub(r"\s*,\s*", ", ", affiliation.strip())
    return affiliation

def extract_institution_custom(affiliation):
    if pd.isna(affiliation) or not affiliation.strip():
        return None

    parts = [part.strip() for part in re.split(r'[,]', affiliation) if part.strip()]
    if not parts:
        return None
    parts = [re.sub(r'\s*\(.*?\)', '', part) for part in parts]

    inst_keywords = [
        "university", "college", "institute", "polytechnic", "academy",
        "ecole", "school", "universidad", "universitas", "universiti",
        "company", "center", "hospital", "agency", "universidade", "universitat",
        "universita", "universite", "unesp",
        "centro", "observatory", "observatorio", "systems", "instituto",
        "institution", "politecnico", "minister", "ministero",
        "museum", "museo", "zentrum"
    ]

    high_confidence_acronyms_or_names = [
        "eth zurich", "dlr", "ucla", "ufmg", "cicese", "kfki", "uc irvine", "ku leuven",
        "caltech", "mit"
    ]

    def candidate_score(part):
        score = 0
        lower = part.lower()

        if re.match(r'^\s*\d+', lower):
            score -= 50

        address_indicators = [
            "boulevard", "street", "avenue", "box", "blvd", "pkwy", "hwy",
            "road", "rd", "drive", "dr", "suite", "floor", "ste",
            "rue", "way", "viale", "esp", "pr", "str", "strasse",
            "ave", "av", "avenida", "allee"
        ]
        for ind in address_indicators:
            if re.search(r'\b' + ind + r'\b', lower):
                score -= 20

        # Weights
        if re.search(r'\bnasa\b', lower): score += 20
        if re.search(r'\bcnrs\b', lower): score += 15
        if re.search(r'\bcnr\b', lower) and len(part) < 10: score += 15
        if re.search(r'\beth\b', lower) and "zurich" in lower: score += 15
        if "ku leuven" in lower: score += 15
        if re.search(r'\bunifesp\b', lower): score += 15
        if "museum" in lower or "museo" in lower: score += 15
        if "zentrum" in lower: score += 10
        if re.search(r'\bdlr\b', lower): score += 15
        if re.search(r'\bucla\b', lower): score += 15
        if re.search(r'\bufmg\b', lower): score += 15
        if re.search(r'\bcicese\b', lower): score += 15
        if re.search(r'\bkfki\b', lower): score += 15
        if "uc irvine" in lower: score += 15
        if "ministero" in lower: score += 15
        if "eth zurich" in lower: score += 35
        if re.search(r'\besa\b', lower): score += 20
        if re.search(r'\bjaxa\b', lower): score += 20

        if re.search(r'\buniversity\b', lower): score += 40
        if re.search(r'\buniversidad\b', lower): score += 40
        if re.search(r'\buniversitas\b', lower): score += 40
        if re.search(r'\buniversidade\b', lower): score += 40
        if re.search(r'\buniversiti\b', lower): score += 40
        if re.search(r'\buniversitat\b', lower): score += 40
        if re.search(r'\buniversita\b', lower): score += 40
        if re.search(r'\buniversite\b', lower): score += 40

        if re.search(r'\buniv\b', lower) and score > -15:
            score += 40

        if "college" in lower: score += 2
        if "institute" in lower: score += 2
        if "instituto" in lower: score += 2
        if "polytechnic" in lower: score += 2
        if "polytecnico" in lower: score += 2
        if "academy" in lower: score += 2
        if "ecole" in lower: score += 2
        if "school" in lower or "sch " in lower or lower.endswith(" sch"): score += 3
        if "center" in lower: score += 1.5
        if "centro" in lower: score += 1.5
        if "observatory" in lower or "observatorio" in lower: score += 1.5
        if "institution" in lower: score += 0.5
        if "hospital" in lower: score += 1
        if "company" in lower: score += 0.5
        if "agency" in lower: score += 0.5
        if "systems" in lower: score += 0.5
        if "technology" in lower: score += 0.1
        if "minister" in lower or "ministero" in lower: score += 2

        dept_keywords = [
            "department", "faculty", "division", "research", "lab", "group", "laboratory",
            "departamento", "dipartimento", "grupo", "section", "sezione", "unité", "unidad"
        ]
        if any(lower.startswith(w) for w in dept_keywords):
            score -= 3
        for dept_word in dept_keywords:
            if dept_word in lower:
                score -= 0.5

        score += len(part) / 100.0
        return score

    candidate_list = []
    for i, part in enumerate(parts):
        lower_p = part.lower()

        is_inst_keyword = any(kw in lower_p for kw in inst_keywords)
        is_acronym_or_name = any(acronym in lower_p for acronym in high_confidence_acronyms_or_names)
        is_global_acronym_or_univ = ("nasa" in lower_p or "esa" in lower_p or "jaxa" in lower_p or "univ" in lower_p)

        if is_inst_keyword or is_acronym_or_name or is_global_acronym_or_univ:
            candidate_list.append((i, part, candidate_score(part)))

    if candidate_list:
        candidate_list.sort(key=lambda x: x[2], reverse=True)
        return candidate_list[0][1]
    else:
        return parts[0]

def extract_country(affiliation):
    if pd.isna(affiliation) or not affiliation.strip():
        return None
    parts = [part.strip() for part in affiliation.split(",") if part.strip()]
    return parts[-1] if len(parts) > 1 else None




In [ ]:
# ========================================================================
# Main Execution 
# ========================================================================
if 'df_final' not in locals() or df_final is None or df_final.empty:
    print("Error: 'df_final' is not loaded.")
else:
    print(f"Starting affiliation processing on {len(df_final)} articles...")

    df = df_final.copy()
    df["Affiliations_Cleaned"] = df["Affiliations"].apply(clean_affiliations)
    df["Article_Index"] = df.index

    df_exploded = df.assign(
        Affiliations_Cleaned=df["Affiliations_Cleaned"].str.split(";")
    ).explode("Affiliations_Cleaned")

    df_exploded["Affiliations_Cleaned"] = df_exploded["Affiliations_Cleaned"].str.strip()
    df_exploded = df_exploded.dropna(subset=["Affiliations_Cleaned"])
    df_exploded = df_exploded[df_exploded["Affiliations_Cleaned"] != ""]

    print(f"Exploded into {len(df_exploded)} affiliations.")

    df_exploded["Institution_Extracted"] = df_exploded["Affiliations_Cleaned"].apply(extract_institution_custom)
    df_exploded["Country_Raw"] = df_exploded["Affiliations_Cleaned"].apply(extract_country)

    print("Standardizing Institution names...")
    df_exploded["Institution_Extracted"] = df_exploded["Institution_Extracted"].apply(standardize_institution_name)
    df_exploded["Country_Standardized"] = df_exploded["Country_Raw"].apply(unify_countries_in_row)

    cols_to_keep = [
        "Article_Index", "Title", "Publication Year", "DOI",
        "Affiliations_Cleaned", "Institution_Extracted", "Country_Standardized"
    ]
    final_exploded_cols = [col for col in cols_to_keep if col in df_exploded.columns]
    df_affiliations_final = df_exploded[final_exploded_cols].copy()
    df_affiliations_final = df_affiliations_final.dropna(subset=["Institution_Extracted"])

    df_affiliations_final.to_csv("/kaggle/working/exploded_affiliations_list.csv", index=False, encoding="utf-8-sig")
    df_affiliations_final.to_excel("/kaggle/working/exploded_affiliations_list.xlsx", index=False, engine="openpyxl")

    country_lists = df_exploded.groupby("Article_Index")["Country_Standardized"].apply(
        lambda x: sorted(list(set(filter(None, x))))
    ).reset_index().rename(columns={"Country_Standardized": "Countries_Extracted"})

    institution_lists = df_exploded.groupby("Article_Index")["Institution_Extracted"].apply(
        lambda x: sorted(list(set(filter(None, x))))
    ).reset_index().rename(columns={"Institution_Extracted": "Institutions_Extracted"})

    df_article_summary = df_final.copy()
    df_article_summary["Article_Index"] = df_article_summary.index
    df_article_summary = df_article_summary.merge(country_lists, on="Article_Index", how="left")
    df_article_summary = df_article_summary.merge(institution_lists, on="Article_Index", how="left")

    df_article_summary["Countries_Extracted"] = df_article_summary["Countries_Extracted"].apply(lambda x: x if isinstance(x, list) else [])
    df_article_summary["Institutions_Extracted"] = df_article_summary["Institutions_Extracted"].apply(lambda x: x if isinstance(x, list) else [])
    df_article_summary["Country_Count"] = df_article_summary["Countries_Extracted"].apply(len)
    df_article_summary["Collaboration_Type"] = df_article_summary["Country_Count"].apply(
        lambda x: "International" if x > 1 else ("National" if x == 1 else "Undefined")
    )
    df_article_summary = df_article_summary.drop(columns=["Article_Index"], errors="ignore")

    df_article_summary.to_csv("/kaggle/working/final_article_summary_with_countries.csv", index=False, encoding="utf-8-sig")
    df_article_summary.to_excel("/kaggle/working/final_article_summary_with_countries.xlsx", index=False, engine="openpyxl")

    print("\n--- Example: Final Article Summary (df_article_summary) ---")
    display_cols = ["Title", "Publication Year", "Countries_Extracted", "Institutions_Extracted", "Collaboration_Type"]
    display_cols = [col for col in display_cols if col in df_article_summary.columns]
    display(df_article_summary[display_cols].head())

    print("\n\nAffiliation processing complete! Two files have been saved to /kaggle/working/")

    # ============================
    # 9. PRINT DATA CLEANING REPORT
    # ============================
    print("\n" + "=" * 50)
    print("DATA CLEANING & STANDARDIZATION REPORT")
    print("=" * 50)

    print("\n--- 1. Top 10 Abbreviation Expansions ---")
    if stats_abbrev:
        for item, count in stats_abbrev.most_common(10):
            print(f"{item} : {count} occurrences")
    else:
        print("No abbreviations matched.")

    print("\n--- 2. Top 30 Institution Alias Mappings ---")
    if stats_aliases:
        for item, count in stats_aliases.most_common(30):
            print(f"{item} : {count} occurrences")
    else:
        print("No aliases matched.")

    print(f"\n--- 3. Complex Patterns (RegEx) ---")
    for pat, count in stats_complex.items():
        print(f"{pat} matches: {count}")

    if stats_alias_collisions:
        print("\n--- 4. Alias Collisions (Review these) ---")
        for item, count in stats_alias_collisions.most_common(20):
            print(f"{item} : {count}")

    if stats_backup_ignored:
        print("\n--- 5. Backup Canonicals Ignored (Not in JSON) ---")
        for item, count in stats_backup_ignored.most_common(20):
            print(f"{item} : {count}")

    print("=" * 50 + "\n")

# PART 2: DESCRIPTIVE AND EVALUATIVE BIBLIOMETRICS

## 9. Analytical Environment Configuration

After **Phase 1 (ETL + harmonization + affiliation processing)**, we move to **Phase 2**, where we compute:
- productivity and growth trends (documents per year, CAGR, linear trend reliability),
- citation impact indicators (mean citations, time-normalized metrics like MNCS),
- corpus structure summaries (sources, time span, reference base).

This section initializes the analysis environment and creates a **reproducible output folder structure**.

---

### 9.1 Libraries loaded in the next cell

The next code cell imports standard scientometric tooling:

- **Data & parsing:** `pandas`, `numpy`, `re`
- **Stats / modeling:** `scipy.stats` (for linear regression)
- **Plots:** `matplotlib`, `seaborn`
- **Networks (later sections):** `networkx`
- **Utilities:** `os`, `datetime`, `Counter`, etc.

You do **not** need to edit these imports in normal use.

---

### 9.2 Output directories (IMPORTANT)

The next cell creates two folders:

- `plots_dir` → **figures** (PNG/PDF)
- `results_dir` → **tables / CSV outputs**

✅ **In the current notebook code**, the defaults are set for **Kaggle**:

```python
plots_dir   = "/kaggle/working/output/Plots/"
results_dir = "/kaggle/working/output/Results/"
```

#### If you run outside Kaggle (Colab / local Jupyter)

You only need to change **these two variables** in the next cell.

**Google Colab examples**
```python
plots_dir   = "/content/output/Plots/"
results_dir = "/content/output/Results/"
```

**Local Jupyter examples**
```python
plots_dir   = "./output/Plots/"
results_dir = "./output/Results/"
```

> Tip: keep these two folders separate so your notebook stays reproducible and your outputs remain organized.

---

### 9.3 Publication-ready visualization defaults (Accessibility)

The next cell configures plotting for publication-quality figures:

- high DPI (`savefig.dpi = 300`)
- clean style (`seaborn-v0_8-whitegrid`)
- **Cividis** palette as the global color cycle (more robust for color-vision deficiency and grayscale printing)

✅ This is applied globally via `plt.rcParams`, so all downstream plots inherit the same standard.

---

### ✅ What you should change

Most users only change **output paths** if they are not on Kaggle:

```python
plots_dir   = ".../output/Plots/"
results_dir = ".../output/Results/"
```

Everything else can remain as-is.

---

In [ ]:
# ==============================================================================
# Phase 2: Analytical Environment Setup
# ==============================================================================

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import networkx as nx 
from itertools import combinations
from collections import Counter
from datetime import datetime
from scipy import stats # For parametric Linear Regression
from cycler import cycler # For setting the global color cycle

print("\nSetting up analytical environment...")

# ==============================================================================
# Directory Structure for Reproducibility
# ==============================================================================
# We define clear output paths to separate raw results from visualizations.
plots_dir = '/kaggle/working/output/Plots/'
os.makedirs(plots_dir, exist_ok=True)

results_dir = '/kaggle/working/output/Results/'
os.makedirs(results_dir, exist_ok=True)

# ==============================================================================
# Data Loading
# ==============================================================================
# We verify the existence of the source dataframe before proceeding.
try:
    df = df_article_summary.copy()
    print(f"✅ Data loaded successfully. Ready to analyze {len(df)} documents.")
except NameError:
    print("⚠️ CRITICAL WARNING: 'df_article_summary' not found in memory.")
    print("Please ensure the Data Harmonization (Phase 1) steps were run successfully.")

# ==============================================================================
# Publication-Ready Visualization Settings (Accessibility Focused)
# ==============================================================================
# 1. High DPI for print.
# 2. Cividis colormap for colorblind accessibility and grayscale printing.
# 3. Clean layout without chartjunk.

plt.style.use('seaborn-v0_8-whitegrid')

# Define a subset of the 'cividis' colormap for default line/bar colors
# This extracts 5 distinct colors from the cividis spectrum
cividis_colors = plt.cm.cividis(np.linspace(0, 0.9, 5))

plt.rcParams.update({
    # Output Quality
    'figure.figsize': (10, 6),
    'figure.dpi': 150,            
    'savefig.dpi': 300,           # Required 300dpi for journal submission
    
    # Fonts & Text (Sans-serif is standard for figures)
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    
    # Grid & Axes
    'grid.alpha': 0.3,
    'axes.grid': True,
    
    # ACCESSIBILITY: Set Global Color Cycle to Cividis
    # This ensures ALL plots automatically use the accessible palette.
    'axes.prop_cycle': cycler('color', cividis_colors)
})

print("✅ Visualization parameters set to 'Cividis' (Accessibility Standard).")

## 10. Computational Resource Auditing (Optional)

Some steps in bibliometrics and scientometrics become **RAM-heavy** (e.g., exploded affiliations, co-authorship graphs, institution networks).
This optional step prints the **in-memory size** of all `pandas` DataFrames currently loaded.

Use it when:
- you run on limited hardware (small laptop / free Colab / Kaggle),
- your dataset is large (tens/hundreds of thousands of records),
- you want to diagnose slowdowns or kernel crashes.

---

### What you should expect

The next code cell scans global variables and prints one line per DataFrame, like:

```text
df_final: shape=(12450, 33), memory=18,420.5 KB
df_article_summary: shape=(12450, 40), memory=22,310.1 KB
df_exploded: shape=(65000, 7), memory=61,002.7 KB
```

This is a *diagnostic* report only — it does not modify your data.

---

### If you run into memory limits (optional cleanup)

If a large intermediate DataFrame is no longer needed, you can free memory in a **new** cell:

```python
del df_exploded
import gc
gc.collect()
```

(Only do this if you are sure you won’t need that object later.)

---

### ✅ What you should change

Nothing. This step is optional and safe.

---

In [ ]:
# Get all objects in memory, filter for pandas DataFrames
dfs = {k: v for k, v in globals().items() if isinstance(v, pd.DataFrame)}

print("--- Current DataFrame Memory Usage ---")
for name, df_obj in dfs.items():
    # memory_usage(deep=True) gives a more accurate size
    mem_kb = df_obj.memory_usage(deep=True).sum() / 1024
    print(f"{name}: shape={df_obj.shape}, memory={mem_kb:.1f} KB")

## 11. Corpus Characteristics and Production Trends

We begin with a **descriptive statistical summary** to establish the foundational boundaries and structural properties of the research collection.

---

### Computed Metrics

#### **1. Temporal Coverage**  
The active publication window, capturing the earliest and most recent years represented in the corpus.

#### **2. Volume**  
The total number of **unique documents (N)** after deduplication.

#### **3. Compound Annual Growth Rate (CAGR)**  
Measures the **geometric progression rate** at which the field expands over time.

#### **4. Linear Growth Trend (R²)**  
A goodness-of-fit statistic indicating the consistency of annual productivity growth.  
- Values closer to **1.0** reflect steady, predictable development.  
- Lower values indicate volatility or fragmented growth patterns.

#### **5. Obsolescence**  
The **average age** of documents—an indicator of how frequently knowledge in the field is refreshed.

#### **6. Citation Density**  
Two complementary indicators:

- **Mean citations per document:** `TC / N`  
- **Total intellectual base size:** sum of all cited references across the corpus

Together, these metrics establish the corpus’s scale, maturity, growth trajectory, and citation structure.

### Outputs produced

The next cell:
1) prints a readable summary table, and  
2) saves it to:

```python
output_path = os.path.join(results_dir, "main_information_summary.csv")
```

#### Expected output paths (depending on platform)

**Kaggle**
- `/kaggle/working/output/Results/main_information_summary.csv`

**Google Colab (if you changed `results_dir` earlier)**
- `/content/output/Results/main_information_summary.csv`

**Local Jupyter (if you changed `results_dir` earlier)**
- `./output/Results/main_information_summary.csv`

---

### ✅ What you should change

Usually nothing.

If you want to rename the output file, change only the filename string in the next cell:

```python
"main_information_summary.csv"
```

---

In [ ]:
# ---------------------------
# Main Dataset Information
# ---------------------------
print("Calculating Descriptive Statistics...")

# Initialize metrics
metrics = {}

# Document Count
metrics['Total Documents'] = len(df)

# --- Year-based metrics ---
if 'Publication Year' in df.columns:
    years = df['Publication Year'].dropna().astype(int)
    if not years.empty:
        metrics['Timespan'] = f"{years.min()} - {years.max()}"
        
        # Growth Calculation
        documents_per_year = years.value_counts().sort_index()
        first_year, last_year = documents_per_year.index.min(), documents_per_year.index.max()
        
        # 1. CAGR
        if (last_year > first_year) and (documents_per_year.iloc[0] > 0):
            n_years = last_year - first_year
            cagr = ((documents_per_year.iloc[-1] / documents_per_year.iloc[0]) ** (1 / n_years) - 1) * 100
            metrics['Annual Growth Rate (CAGR %)'] = cagr
        else:
            metrics['Annual Growth Rate (CAGR %)'] = np.nan
            
        # 2. Linear Growth Trend (R2)
        # We fill missing years with 0 to get a true timeline
        full_timeline = pd.Series(0, index=range(first_year, last_year + 1))
        full_timeline = full_timeline.add(documents_per_year, fill_value=0)
        
        slope, intercept, r_value, p_value, std_err = stats.linregress(full_timeline.index, full_timeline.values)
        metrics['Linear Growth Reliability (R²)'] = r_value**2

        # 3. Age
        current_year = datetime.now().year
        metrics['Average Document Age'] = (current_year - years).mean()
    else:
        print("Warning: 'Publication Year' contains no valid data.")

# --- Other metrics ---
if 'Source Title' in df.columns:
    metrics['Unique Sources'] = df['Source Title'].nunique()

if 'Cited by' in df.columns:
    metrics['Avg. Citations per Doc'] = df['Cited by'].fillna(0).mean()

if 'Cited Reference Count' in df.columns:
    metrics['Total References (Intellectual Base)'] = df['Cited Reference Count'].fillna(0).sum()

# --- Formatting & Output ---
main_info = pd.DataFrame(list(metrics.items()), columns=['Metric', 'Value'])

# Format numbers
def format_val(x):
    if isinstance(x, float): return f"{x:,.2f}"
    if isinstance(x, (int, np.integer)): return f"{x:,}"
    return str(x)

main_info['Value'] = main_info['Value'].apply(format_val)

print("\n--- Descriptive Statistics Summary ---")
print(main_info.to_string(index=False))

# Save
output_path = os.path.join(results_dir, "main_information_summary.csv")
main_info.to_csv(output_path, index=False)
print(f"\nSummary report saved to: {output_path}")

## 12. Metric Definition: Time-Normalized Citations (MNCS)

Standard citation counts suffer from a **temporal bias**:  
older publications have had more time to accumulate citations than recent ones.  
Thus, directly comparing a paper from *2005* with one from *2023* is methodologically invalid.

To correct for this, we compute the **Mean Normalized Citation Score (MNCS)**, implemented here as a **time-normalized citation metric**.

---

### Formula (Time-Normalized MNCS)

MNCS_paper = Citations_paper / AverageCitations_year  

Where:

- `Citations_paper` = total citations received by the focal paper  
- `AverageCitations_year` = average citations of all papers in the corpus published in the same year

---

### Interpretation

- **MNCS > 1.0** → above-average impact compared to papers from the **same year** in this corpus  
- **MNCS = 1.0** → at the yearly average  
- **MNCS < 1.0** → below-average for that year

---

### Practical example (small intuition)

If a 2020 paper has 30 citations and the dataset’s average for 2020 is 10:

\[
MNCS=\frac{30}{10}=3.0
\]

Meaning it is ~3× more cited than the average 2020 paper **in this corpus**.

---

### Note on Interpretation

This is an **internal normalization**, meaning:

- The benchmark is **this corpus**, not the entire Scopus/WoS database.  
- MNCS identifies **relative high-performers within your specific research topic**,  
  not global citation influence across all disciplines.

### What the next cell does

The next cell:
1) computes yearly citation baselines (`groupby('Publication Year')`),  
2) creates a new column: `df['MNCS']`,  
3) prints a preview of:

```python
['Title', 'Publication Year', 'Cited by', 'MNCS']
```

If required columns are missing, the notebook prints a warning and assigns a safe fallback (`MNCS = 1.0`).

---

### ✅ What you should change

Nothing for default use.

If you plan to export MNCS later, you can save the updated dataframe in a later export cell (optional).

---

In [ ]:
# ========================================================================
# METRIC: Calculate Time-Normalized Citations (Internal MNCS)
# ========================================================================

print("Calculating Time-Normalized Citation Scores...")

if 'Publication Year' in df.columns and 'Cited by' in df.columns:
    # 1. Calculate the Internal Baseline: Average Citations for every Year
    year_averages = df.groupby('Publication Year')['Cited by'].mean().to_dict()

    # 2. Function to calculate MNCS
    def calculate_mncs(row):
        year = row['Publication Year']
        citations = row['Cited by']
        
        # Get baseline for this year (default to 1 to avoid division by zero)
        baseline = year_averages.get(year, 1)
        if baseline == 0: baseline = 1 
        
        return citations / baseline

    # 3. Apply to the main dataframe
    df['MNCS'] = df.apply(calculate_mncs, axis=1)

    print("✅ 'MNCS' column added to DataFrame.")
    print("   Methodology: Internal Field-Normalization (relative to corpus averages per year).")
    
    # Preview
    print(df[['Title', 'Publication Year', 'Cited by', 'MNCS']].head())
else:
    print("⚠️ Warning: Missing Year or Citation columns. Cannot calculate MNCS.")
    df['MNCS'] = 1.0 # Default fallback

## 13. Structural Morphology: Document Typology

This section analyzes the **distribution of publication types** to understand the communication channels, maturity,  
and structural characteristics of the research field.

---

### **Methodological Challenge**

Bibliometric databases frequently use **inconsistent and overlapping taxonomies**:

- Web of Science: `"Proceedings Paper"`
- Scopus: `"Conference Paper"`
- Mixed labels in a single record: `"Article; Proceedings Paper"`

Such inconsistencies introduce classification noise that undermines descriptive statistics and comparative analysis.

---

### **Normalization Protocol**

To ensure statistical validity and interpretability, we apply a **dictionary-based harmonization protocol** that  
maps heterogeneous document-type labels into a **standardized typology**:

#### **Article**  
Original research contributions — the primary unit of scholarly knowledge.

#### **Review**  
Secondary syntheses including reviews, surveys, and meta-analyses.

#### **Conference**  
Proceedings papers, meeting abstracts, and any conference-associated publications.

#### **Book / Chapter**  
Books, monographs, edited volumes, and book chapters.

#### **Other**  
Editorials, commentary, letters, corrections, data papers, and miscellaneous unclassified items.

---

**Important implementation details:**
- Missing values are filled as: `Missing/Incomplete data` → mapped to **Other**.
- Composite labels are split using the delimiter `"; "` (semicolon + space).  
  Example: `Article; Proceedings Paper` becomes two items and both are counted.
- Percentages are computed **relative to total unique documents** (`total_docs = len(df)`), even after exploding.

> **Note:** In the code, the horizontal bar length represents **Counts**, and the **percentage** is shown in the annotation label `count (pct%)`. The x-axis label may still mention “percentage” from an earlier draft, but interpret bar lengths as **counts**.

---

### ✅ What you may want to change (optional)
Most users do **not** need to change anything. If your database exports use different labels, edit only the mapping dictionary in Cell 35:

```python
DOC_TYPE_MAP = {
    "article": "Article",
    "review": "Review",
    "conference paper": "Conference",
    "proceeding paper": "Conference",
    "book": "Book",
    "book chapter": "Book Chapter",
    # ...
}
```

Common additions you might include:
- `"meeting abstract"` → `"Conference"` or `"Other"`
- `"early access"` → often best treated as `"Article"` (depends on your dataset)
- `"editorial material"` → `"Other"`

If your exports use a different separator than `"; "` (e.g., `"|"`), you’ll need to adapt the split rule in code.  
(For the public notebook, the recommended practice is to keep exports consistent with `"; "`.)

---



In [ ]:
# ========================================================================
# Document Type Analysis (Standardized Taxonomy)
# ========================================================================
# Purpose: Normalize heterogeneous database tags into a clean taxonomy.

import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np

print("\nStarting Document Type Analysis...")

# --- 1. Taxonomy Definition ---
# Mapping raw database terms to standardized Scientometric categories
DOC_TYPE_MAP = {
    # Primary Knowledge Units
    'article': 'Article',
    'journal article': 'Article',
    
    # Secondary Synthesis
    'review': 'Review',
    'review article': 'Review',
    'survey': 'Review',
    
    # Conference Output
    'conference paper': 'Conference',
    'proceeding paper': 'Conference',
    'conference proceeding': 'Conference',
    
    # Monographs
    'book chapter': 'Book Chapter',
    'book': 'Book',
    
    # Editorial/Paratextual (Mapped to 'Other')
    'editorial': 'Other', 'editorial material': 'Other',
    'letter': 'Other', 'note': 'Other', 'short survey': 'Other',
    'erratum': 'Other', 'correction': 'Other', 'retraction': 'Other',
    'data paper': 'Other', 'missing/incomplete data': 'Other'
}

def normalize_doc_type(doc_type_str):
    """Normalizes raw document type strings using the taxonomy map."""
    if pd.isna(doc_type_str): return 'Other'
    cleaned_str = str(doc_type_str).lower().strip()
    return DOC_TYPE_MAP.get(cleaned_str, 'Other')

# --- 2. Data Processing ---
if 'Document Type' not in df.columns:
    print("Error: 'Document Type' column not found. Skipping analysis.")
else:
    # Handle missing data
    df_clean = df.copy()
    df_clean['Document Type'] = df_clean['Document Type'].fillna('Missing/Incomplete data')
    
    # Explode composite types (e.g., "Article; Proceedings") to count both
    # This ensures we don't lose data on dual-classified items
    df_clean['Document Types List'] = df_clean['Document Type'].str.split('; ')
    df_exploded = df_clean.explode('Document Types List')
    
    # Apply Normalization
    df_exploded['Standardized_Type'] = df_exploded['Document Types List'].apply(normalize_doc_type)
    
    # Count Frequencies
    type_counts = df_exploded['Standardized_Type'].value_counts()
    
    # Calculate Percentages relative to total UNIQUE documents
    total_docs = len(df)
    summary_df = pd.DataFrame({
        'Count': type_counts,
        'Percentage': (type_counts / total_docs * 100)
    }).sort_values(by='Count', ascending=False)
    
    print("--- Document Type Distribution (Standardized) ---")
    print(summary_df.to_string(formatters={'Percentage': '{:,.2f}%'.format}))
    
    # --- 3. Visualization (Monochrome Academic Style) ---
    plt.figure(figsize=(10, 6))
    ax = plt.gca()
    
    # Horizontal Bar Chart
    bars = ax.barh(summary_df.index, summary_df['Count'], color='#333333')
    
    # Labels and Titles
    ax.set_xlabel('Percentage of Total Collection (%)')
    ax.set_title(f'Structural Morphology: Distribution of Document Types ($N={total_docs}$)', pad=20, fontweight='bold')
    
    # Invert y-axis so the largest bar is at the top
    ax.invert_yaxis()
    
    # Remove chartjunk
    ax.spines[['top', 'right', 'left']].set_visible(False)
    ax.tick_params(axis='y', length=0)
    
    # Annotate bars with Count and Percentage
    # We explicitly calculate the max width to set axis limits dynamically
    max_val = summary_df['Count'].max()
    ax.set_xlim(0, max_val * 1.2) # Add breathing room for labels

    for bar, count, pct in zip(bars, summary_df['Count'], summary_df['Percentage']):
        width = bar.get_width()
        label_text = f" {count} ({pct:.1f}%)"
        ax.text(width, bar.get_y() + bar.get_height()/2, label_text, 
                ha='left', va='center', fontsize=10, color='black')

    plt.tight_layout()
    
    # Save Outputs
    plot_path_png = os.path.join(plots_dir, 'document_types.png')
    plot_path_pdf = os.path.join(plots_dir, 'document_types.pdf')
    
    plt.savefig(plot_path_png, dpi=600, bbox_inches='tight')
    plt.savefig(plot_path_pdf, format='pdf', bbox_inches='tight')
    
    print(f"\nDocument Type plot saved to: {plot_path_png}")
    plt.show()
    
    # Save Data Table
    summary_path = os.path.join(results_dir, 'document_type_summary.csv')
    summary_df.to_csv(summary_path)
    print(f"Summary table saved to: {summary_path}")

## 14. Author Productivity and Impact Assessment

This section identifies the leading researchers in the field.  
To ensure a fair assessment aligned with **DORA (Declaration on Research Assessment)** principles, we move beyond simple counting toward a **multi-dimensional evaluation**.

---

### Methodological Approach

**1. Productivity (Full Counting)**  
- Measures the *presence* of an author.  
- If an author is one of 100 on a paper, they receive **+1**.  
- Highlights leaders of **large consortia**.

**2. Contribution (Fractional Counting)**  
- Measures the *effort* of an author.  
- If an author is one of 100, they receive **+0.01**.  
- Highlights key contributors in **smaller teams**.

**3. Impact (MNCS)**  
- Measures the *quality* of research output.  
- Computed as the **Mean Normalized Citation Score**, time-normalized per publication.

---

### Metric Definitions

**Fractionalized Credit:**  
Sum of *1 divided by the number of authors* for each publication.

**Average MNCS:**  
The mean field-normalized citation impact of the author's publications.  
- Values **> 1.0** indicate **above-average impact**.

---

### Implementation details 
- The code selects the author column automatically:
  - uses `Author Full Names` if present and non-empty,
  - otherwise falls back to `Authors`.

- The code assumes authors are **semicolon-separated** in the export:  
  `Author A; Author B; Author C`

- Team size is computed as:  
  `num_authors = len(str(authors).split(';'))`

- If `MNCS` is missing, the code creates it as `NaN` and warns that the impact plot may be empty.  
  For best results, run the MNCS cell first.

---

### ✅ What you may want to change (optional)
Most users won’t need to.

You might edit:
- The **Top N** shown in tables/plots (currently Top 10):
  - `head(10)` in the code
- The delimiter handling if your exports separate authors differently.  
  Recommended: keep exports standardized to semicolons for this public notebook.

---


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np

# ============================
# Authors Analysis
# ============================
print("\nStarting Authors Analysis...")

# --- 1. Column Selection Strategy ---
author_col_name = None 
if 'Author Full Names' in df.columns and not df['Author Full Names'].dropna().empty:
    author_col_name = 'Author Full Names'
    print(f"✅ Using '{author_col_name}' for analysis.")
elif 'Authors' in df.columns and not df['Authors'].dropna().empty:
    author_col_name = 'Authors'
    print(f"⚠️ Warning: 'Author Full Names' missing. Using '{author_col_name}'.")
else:
    print("❌ Error: No suitable author column found. Skipping analysis.")

# --- 2. Data Processing (Fractional & Impact) ---
if author_col_name:
    # Handle missing MNCS gracefully
    if 'MNCS' not in df.columns:
        df['MNCS'] = np.nan
        print("ℹ️ Note: MNCS data missing. Impact plot will be empty.")
        
    df_authors = df[[author_col_name, 'Cited by', 'MNCS']].dropna(subset=[author_col_name]).copy()
    
    # A. Calculate Team Size (N)
    df_authors['num_authors'] = df_authors[author_col_name].apply(lambda x: len(str(x).split(';')))
    
    # B. Calculate Fractional Credit (1/N)
    df_authors['fractional_credit'] = 1 / df_authors['num_authors']
    
    # C. Explode to Author Level
    df_authors['Authors_Split'] = df_authors[author_col_name].str.split(';')
    df_exploded = df_authors.explode('Authors_Split')
    
    # Clean whitespace
    df_exploded['Authors_Split'] = df_exploded['Authors_Split'].str.strip()
    df_exploded = df_exploded[df_exploded['Authors_Split'] != '']
    
    # --- 3. Aggregation ---
    print("Calculating fractional credit and normalized impact...")
    author_stats = df_exploded.groupby('Authors_Split').agg(
        Total_Papers=('Authors_Split', 'size'),            # Full Count
        Total_Citations=('Cited by', 'sum'),               # Raw Citations
        Fractional_Credit=('fractional_credit', 'sum'),    # Fractional Count
        Avg_MNCS=('MNCS', 'mean')                          # Normalized Impact
    ).reset_index().rename(columns={'Authors_Split': 'Author'})
    
    total_authors = len(author_stats)
    print(f"\nFound {total_authors} unique authors.")

    # =========================================================
    # PART 1: PRODUCTIVITY RANKING (Full Counting)
    # =========================================================
    print("\n" + "="*50)
    print("TOP AUTHORS BY VOLUME (Full Counting)")
    print("="*50)
    
    # Rank by Total Papers
    top_10_trad = author_stats.sort_values(by=['Total_Papers', 'Total_Citations'], ascending=False).head(10).copy()
    print(top_10_trad[['Author', 'Total_Papers', 'Total_Citations']].to_string(index=False))
    
    # --- Plot 1: Productivity ---
    plt.figure(figsize=(10, 6))
    ax = plt.gca()
    plot_data_trad = top_10_trad.sort_values(by='Total_Papers', ascending=True)
    
    # Monochrome Grey/Black for standard counts
    bars = ax.barh(plot_data_trad['Author'], plot_data_trad['Total_Papers'], color='#333333')
    
    ax.set_xlabel('Total Publications ($N_{papers}$)')
    # Standalone Title with N context
    ax.set_title(f'Top 10 Authors by Total Publication Volume\n(Full Counting, Dataset Total: $N={total_authors}$ Authors)', fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='x', linestyle='--', alpha=0.5)
    
    # Annotate
    for bar in bars:
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, 
                f'{int(bar.get_width())}', va='center', fontsize=10)
                    
    plt.tight_layout()
    
    # --- SAVE PDF & PNG ---
    plot_path_png = os.path.join(plots_dir, 'author_productivity_ranking.png')
    plot_path_pdf = os.path.join(plots_dir, 'author_productivity_ranking.pdf')
    plt.savefig(plot_path_png, dpi=600, bbox_inches='tight')
    plt.savefig(plot_path_pdf, format='pdf', bbox_inches='tight')
    print(f"Productivity plot saved to: {plot_path_png}")
    plt.show()

    # =========================================================
    # PART 2: CONTRIBUTION & IMPACT (Fractional & Normalized)
    # =========================================================
    print("\n" + "="*50)
    print("TOP AUTHORS BY CONTRIBUTION & IMPACT")
    print("="*50)
    
    # Rank by Fractional Credit (The "Real" Contribution)
    top_10_sci = author_stats.sort_values(by=['Fractional_Credit', 'Avg_MNCS'], ascending=False).head(10).copy()
    
    # Formatting
    top_10_sci['Fractional_Credit'] = top_10_sci['Fractional_Credit'].round(2)
    top_10_sci['Avg_MNCS'] = top_10_sci['Avg_MNCS'].round(2)
    print(top_10_sci[['Author', 'Fractional_Credit', 'Avg_MNCS']].to_string(index=False))
    
    # --- Plot 2: Impact Panel ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
    plot_data_sci = top_10_sci.sort_values(by='Fractional_Credit', ascending=True)
    
    # Define Cividis Blue for Impact
    cividis_blue = plt.cm.cividis(0.0)

    # Panel 1: Fractional (Grey for Quantity)
    ax1.barh(plot_data_sci['Author'], plot_data_sci['Fractional_Credit'], color='#333333')
    ax1.set_xlabel('Fractionalized Credit ($N_{frac}$)')
    ax1.set_title('Author Contribution\n(Adjusted for Team Size)', fontweight='bold', fontsize=11)
    ax1.grid(axis='x', linestyle='--', alpha=0.5)
    ax1.spines[['top', 'right']].set_visible(False)
    
    for i, v in enumerate(plot_data_sci['Fractional_Credit']):
        ax1.text(v + 0.05, i, f"{v:.2f}", color='black', va='center', fontweight='bold', fontsize=9)

    # Panel 2: Impact (Lollipop with Cividis Blue for Quality)
    ax2.hlines(y=plot_data_sci['Author'], xmin=0, xmax=plot_data_sci['Avg_MNCS'], color='gray', alpha=0.5, linewidth=1)
    ax2.plot(plot_data_sci['Avg_MNCS'], plot_data_sci['Author'], 'o', markersize=9, color=cividis_blue)
    
    # Reference Line
    ax2.axvline(x=1.0, color='black', linestyle='--', linewidth=1.2)
    ax2.text(1.05, 0.02, 'Global Avg (1.0)', transform=ax2.get_xaxis_transform(), color='black', fontsize=9)
    
    ax2.set_xlabel('Mean Normalized Citation Score ($MNCS$)')
    ax2.set_title('Citation Impact\n(Field-Normalized)', fontweight='bold', fontsize=11)
    ax2.grid(axis='x', linestyle='--', alpha=0.5)
    ax2.spines[['top', 'right', 'left']].set_visible(False)
    ax2.tick_params(left=False)
    
    for i, v in enumerate(plot_data_sci['Avg_MNCS']):
        ax2.text(v + 0.1, i, f"{v:.2f}", color=cividis_blue, va='center', fontweight='bold', fontsize=9)

    # Standalone Title with N context
    plt.suptitle(f'Author Performance: Fractionalized Contribution vs. Normalized Impact\n(Top 10 of $N={total_authors}$ Authors)', fontsize=14, y=1.05, fontweight='bold')
    plt.tight_layout()
    
    # --- SAVE PDF & PNG ---
    plot_path_png = os.path.join(plots_dir, 'author_impact_panel_chart.png')
    plot_path_pdf = os.path.join(plots_dir, 'author_impact_panel_chart.pdf')
    plt.savefig(plot_path_png, dpi=600, bbox_inches='tight')
    plt.savefig(plot_path_pdf, format='pdf', bbox_inches='tight')
    print(f"Impact panel plot saved to: {plot_path_png}")
    plt.show()
    
    # Save Data
    author_stats.to_csv(os.path.join(results_dir, 'author_metrics_complete.csv'), index=False)

else:
    print("Skipping author analysis.")

## 15. Annual Scientific Production and Growth Dynamics

This analysis reconstructs the historical development of the field.

---

### Methodological Notes

**Gap Filling**  
- Years with zero publications are explicitly assigned a value of **0**, ensuring a continuous time series.

**Linear Growth Trend (R²)**  
- We compute the **Coefficient of Determination (R²)** to assess the stability of growth over time.

**Interpretation of R²:**  
- **R² ≈ 1.0:** Indicates a predictable, steady, linear expansion of the field.  
- **Low R²:** Indicates volatility, irregular development, or exponential/explosive phases.

**Visualization**  
- The graph displays:
  - **Annual publication volume** (bars)  
  - **Linear trend line** (dashed)  
- This provides an immediate visual understanding of the field's trajectory.

---
### ✅ What you may want to change (optional)
Most users do not need to change anything.

If you want different filenames/locations, change the `plots_dir` / `results_dir` definitions in **Cell 27**.  
All Phase 2 plots/tables reuse those paths.

---

In [ ]:
from matplotlib.ticker import MaxNLocator
from scipy import stats # Required for Linear Regression
import numpy as np
import os # Ensure os is imported for saving files

print("\nAnalyzing Scientific Production Trends...")

# --- 1. Data Preparation ---
if df.empty or 'Publication Year' not in df.columns:
    print("Warning: Missing data or 'Publication Year' column. Skipping.")
    articles_per_year = pd.DataFrame()
else:
    # Ensure year is integer
    df_plot = df.dropna(subset=['Publication Year']).copy()
    df_plot['Publication Year'] = df_plot['Publication Year'].astype(int)
    
    # Count per year
    articles_per_year = df_plot.groupby('Publication Year').size().reset_index(name='Count')
    
    # --- Gap Filling (Continuous Timeline) ---
    if not articles_per_year.empty:
        min_year = articles_per_year['Publication Year'].min()
        max_year = articles_per_year['Publication Year'].max()
        all_years = pd.DataFrame({'Publication Year': range(min_year, max_year + 1)})
        
        # Merge and fill gaps with 0
        articles_per_year = all_years.merge(articles_per_year, on='Publication Year', how='left').fillna(0)
        articles_per_year['Count'] = articles_per_year['Count'].astype(int)
    else:
        articles_per_year = pd.DataFrame()

# --- 2. Statistical Analysis & Plotting ---
if not articles_per_year.empty:
    # Calculate Summary Metrics for Title
    total_n = articles_per_year['Count'].sum()
    
    # Calculate CAGR (Compound Annual Growth Rate)
    # Formula: (End_Value / Start_Value)^(1/n) - 1
    start_val_df = articles_per_year[articles_per_year['Count'] > 0].iloc[0] if not articles_per_year[articles_per_year['Count'] > 0].empty else None
    
    if start_val_df is not None:
        start_val = start_val_df['Count']
        start_year = start_val_df['Publication Year']
        end_val = articles_per_year.iloc[-1]['Count']
        years_diff = max_year - start_year
        
        # Only calculate CAGR if we have non-zero starting data and a time span
        if start_val > 0 and years_diff >= 1:
            cagr = (end_val / start_val) ** (1 / years_diff) - 1
            cagr_str = f"{cagr:.1%}"
        else:
            cagr_str = "N/A"
    else:
         cagr_str = "N/A"
         
    # Print Table
    print("\n--- Annual Production Data (Last 10 Years) ---")
    print(articles_per_year.tail(10).to_string(index=False)) 
    
    # Save CSV
    results_path = os.path.join(results_dir, 'annual_scientific_production.csv')
    articles_per_year.to_csv(results_path, index=False)

    # --- Plotting ---
    plt.figure(figsize=(12, 6))
    ax = plt.gca()

    # Define Cividis Colors
    cividis_blue = plt.cm.cividis(0.0)

    # 1. The Data Line (Actual Volume)
    ax.plot(
        articles_per_year['Publication Year'], 
        articles_per_year['Count'], 
        color='#555555',  # Neutral Grey
        marker='o', 
        linewidth=1.5,
        alpha=0.7,
        label='Annual Volume (Raw)'
    )
    
    # 2. The Linear Trend Line (Regression)
    slope, intercept, r_value, p_value, std_err = stats.linregress(
        articles_per_year['Publication Year'], 
        articles_per_year['Count']
    )
    line = slope * articles_per_year['Publication Year'] + intercept
    
    # Use Cividis Blue for the Trend (High Contrast, "The Signal")
    ax.plot(
        articles_per_year['Publication Year'], 
        line, 
        linestyle='--', 
        linewidth=2.5,
        color=cividis_blue, 
        label=f'Linear Trend ($R^2={r_value**2:.2f}$)'
    )

    # Titles and Labels (Including N and CAGR)
    ax.set_title(f'Annual Scientific Production: Growth Trends ({min_year}–{max_year})\nTotal $N={total_n}$ Articles | CAGR: {cagr_str}', 
                 fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Year')
    ax.set_ylabel('Number of Articles')
    
    # Format Axis
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    
    # *** MODIFIED LINE HERE: Changed +0.5 to +1.0 for better visibility of the last point ***
    ax.set_xlim(min_year - 0.5, max_year + 1.0)
    ax.set_ylim(bottom=0)
    
    # Aesthetics
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(True, linestyle='--', alpha=0.5)
    
    # Legend
    ax.legend(loc='upper left', frameon=True, facecolor='white', framealpha=0.95)

    plt.tight_layout()
    
    # --- SAVE PDF & PNG ---
    plot_path_png = os.path.join(plots_dir, 'annual_scientific_production.png')
    plot_path_pdf = os.path.join(plots_dir, 'annual_scientific_production.pdf')
    plt.savefig(plot_path_png, dpi=600, bbox_inches='tight')
    plt.savefig(plot_path_pdf, format='pdf', bbox_inches='tight')
    
    print(f"\nGrowth plot saved to: {plot_path_png}")
    plt.show()

else:
    print("Skipping production analysis (no data).")

In [ ]:
# Note: Uses 'Mean Total Citations' (Accumulated) vs 'Annualized' (Velocity)
from matplotlib.ticker import MaxNLocator

print("\nAnalyzing Citation Impact Dynamics...")

# --- 1. Determine Cutoff (for Annualization) ---
try:
    # Use global cutoff if defined in Cell 28, else current year
    calc_cutoff = CUTOFF_YEAR if 'CUTOFF_YEAR' in locals() else datetime.now().year
except:
    calc_cutoff = datetime.now().year

if 'Publication Year' in df.columns and 'Cited by' in df.columns:
    # Group by year
    citations_per_year = df.groupby('Publication Year').agg(
        MeanTotalCitations=('Cited by', 'mean'),
        N=('Publication Year', 'size')
    ).reset_index()

    # Gap Filling
    if not citations_per_year.empty:
        min_year = int(citations_per_year['Publication Year'].min())
        max_year = int(citations_per_year['Publication Year'].max())
        all_years = pd.DataFrame({'Publication Year': range(min_year, max_year + 1)})
        citations_per_year = all_years.merge(citations_per_year, on='Publication Year', how='left')
        citations_per_year['N'] = citations_per_year['N'].fillna(0)
        # We leave MeanTotalCitations as NaN for empty years to break the line

    # Calculate Annualized Citations (Citations / Age)
    # Adding 0.5 to age avoids division by zero for the most recent year
    citations_per_year['Age'] = (calc_cutoff - citations_per_year['Publication Year']) + 0.5
    citations_per_year['MeanAnnualized'] = citations_per_year['MeanTotalCitations'] / citations_per_year['Age']

    # --- Plotting (Cividis Palette) ---
    fig, ax1 = plt.subplots(figsize=(12, 7))

    # UPDATED: Calculate Total N for Title
    total_docs = int(citations_per_year['N'].sum())
    ax1.set_title(f'Citation Lifecycle: Accumulated Prestige vs. Impact Velocity ($N={total_docs}$ Articles)', fontsize=14, fontweight='bold', pad=20)

    # --- Color Strategy (Cividis Blue vs. Gold) ---
    # We use the extremes of the Cividis colormap for maximum contrast.
    # cividis(0.0) = Dark Blue (Good for Line/Text)
    # cividis(0.9) = Gold/Yellow (Good for Bars)
    
    color_bars = plt.cm.cividis(0.95) # Gold/Yellow
    color_line = plt.cm.cividis(0.0)  # Dark Blue

    # Axis 1: Accumulated (Bars) - Using Gold
    # We add an edgecolor to ensure the yellow bars are visible against white paper
    ax1.bar(citations_per_year['Publication Year'], citations_per_year['MeanTotalCitations'], 
            color=color_bars, alpha=0.7, edgecolor='grey', linewidth=0.5, 
            label='Mean Total Citations (Accumulated)')
            
    ax1.set_xlabel('Publication Year', fontsize=12)
    ax1.set_ylabel('Mean Total Citations', color='#8c7b00', fontsize=12, fontweight='bold') # Darker Gold for text legibility
    ax1.tick_params(axis='y', labelcolor='#8c7b00')
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=20))
    ax1.tick_params(axis='x', rotation=45)

    # Axis 2: Velocity (Line) - Using Dark Blue
    ax2 = ax1.twinx()
    ax2.plot(citations_per_year['Publication Year'], citations_per_year['MeanAnnualized'], 
             color=color_line, marker='o', linewidth=2.5, markersize=6, 
             label='Mean Annualized Citations (Velocity)')
             
    ax2.set_ylabel('Mean Annualized Citations (Cites/Year)', color=color_line, fontsize=12, fontweight='bold')
    ax2.tick_params(axis='y', labelcolor=color_line)
    ax2.set_ylim(bottom=0)

    # Legend
    lines, labels = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax2.legend(lines + lines2, labels + labels2, loc='upper left', facecolor='white', framealpha=0.9)

    ax1.grid(True, axis='x', linestyle='--', alpha=0.3)
    ax2.spines['top'].set_visible(False)
    
    plt.tight_layout()
    
    # --- SAVE PDF & PNG ---
    plot_path_png = os.path.join(plots_dir, 'citation_dynamics_dual_axis.png')
    plot_path_pdf = os.path.join(plots_dir, 'citation_dynamics_dual_axis.pdf')
    plt.savefig(plot_path_png, dpi=600, bbox_inches='tight')
    plt.savefig(plot_path_pdf, format='pdf', bbox_inches='tight')
    print(f"Citation dynamics plot saved to: {plot_path_png}")
    plt.show()
    
    # Save Data
    citations_per_year.to_csv(os.path.join(results_dir, 'citation_dynamics.csv'), index=False)

else:
    print("Skipping citation dynamics (missing data).")

## 16. Source Impact Analysis (Journals)

This analysis evaluates publication venues to identify core journals and high-impact outlets.  
We apply a multi-dimensional assessment strategy that combines traditional "prestige" metrics with modern "efficiency" metrics.

---
> **Important:** Metrics here are **local to your dataset** (not global JCR/SJR metrics).  
> They are still very useful for comparing sources *within the same corpus* and for describing the field’s venue structure.


### Methodological Definitions

#### 1. Accumulated Prestige Metrics (The "Old Way")
These metrics rely on raw citation counts within the downloaded corpus.  
They are sensitive to both the volume and the age of each journal.

**Local H-Index**  
- The number of papers (h) in the corpus that have received at least h citations.  
- Captures consistent impact but does not account for isolated highly-cited "blockbuster" papers.

**Local G-Index**  
- The largest number of papers (g) such that the top g articles collectively received at least g² citations.  
- Rewards highly-cited papers and highlights journals that publish breakthrough or influential research.

**Local M-Index**  
- The h-index divided by the journal's "age" in the dataset (years since first recorded publication).  
- Normalizes for time, allowing newer journals to be compared fairly against older, established venues.

---

#### 2. Efficiency Metrics (The "New Way")

**MNCS (Field-Normalized Impact)**  
- The average time-normalized citation score across the journal's publications.  
- Identifies journals that consistently publish high-efficiency research, independent of their total output or age.

---

### Methodology

**Source Normalization**  
- Journal names are harmonized (e.g., `"Nature & Science"` → `"nature and science"`) to prevent fragmentation of citation counts.

**Aggregation**  
- For every source, we compute the following metrics:  
  **Volume**, **H-Index**, **G-Index**, **M-Index**, and **MNCS**.

**Visualization**  
- A composite panel displays these indicators side-by-side, revealing complementary dimensions of journal performance.

---
### What you may want to change (safe edits)

Inside the code cell, these are the main knobs:

- `min_papers = 5`  
  Minimum number of papers required for a journal to be considered in **MNCS ranking** (reduces noise).

- `max_length = 30`  
  Truncation length for journal titles in plots (helps readability).

If you want to change the normalization rules, edit only the `Source_Key` construction block.

---

In [ ]:
from matplotlib.ticker import MaxNLocator
import math
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime

# ============================
# Helper Functions
# ============================

def print_markdown_table(title, df, columns, float_cols=None):
    """Prints a formatted markdown table to the console."""
    markdown_title = f"### {title}\n"
    table = df[columns].copy()
    if float_cols:
        for col_name in float_cols: 
            table[col_name] = table[col_name].map(lambda x: f"{x:.2f}")
    markdown_table = table.to_markdown(index=False)
    print(markdown_title + markdown_table + "\n")

def calculate_h_index(citations):
    """Calculates the Local H-index from a list of citation counts."""
    sorted_citations = sorted(citations, reverse=True)
    h = 0
    for i, c in enumerate(sorted_citations, 1):
        if c >= i: h = i
        else: break
    return h

def calculate_g_index(citations):
    """Calculates the Local g-index from a list of citation counts."""
    sorted_citations = sorted(citations, reverse=True)
    g, cumulative = 0, 0
    for i, c in enumerate(sorted_citations, 1):
        cumulative += c
        if cumulative >= i**2: g = i
        else: break
    return g

PROTECTED_ACRONYMS = {'ieee', 'acm', 'spie', 'nasa', 'mnras', 'pasp', 'aaas', 'plos'}
LOWERCASE_WORDS = {'a', 'an', 'the', 'and', 'or', 'but', 'for', 'nor', 'on', 'at', 'to', 'from', 'by', 'of', 'in'}

def format_journal_title(title_str):
    if pd.isna(title_str): return "Unknown Source"
    words = str(title_str).split()
    formatted_words = []
    for i, word in enumerate(words):
        word_clean = re.sub(r'[^\w\s]', '', word).lower()
        if word_clean in PROTECTED_ACRONYMS:
            formatted_words.append(word_clean.upper())
        elif word_clean in LOWERCASE_WORDS and i > 0:
            formatted_words.append(word.lower())
        else:
            formatted_words.append(word.title())
    return " ".join(formatted_words)

def create_top_10_panel(ax, data, column, title, xlabel, is_float=False):
    """Helper function to create a single styled h-bar plot panel (Monochrome)."""
    plot_data = data.sort_values(by=column, ascending=True)
    
    # Monochrome Grey for rankings (Clean Academic Style)
    bars = ax.barh(plot_data['Source_Display'], plot_data[column], color='#333333')
    
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel(xlabel, fontsize=10)
    
    max_val = plot_data[column].max()
    ax.set_xlim(0, max_val * 1.2) 
    
    for bar in bars:
        width = bar.get_width()
        label = f' {width:.2f}' if is_float else f' {int(width)}'
        ax.annotate(label, xy=(width, bar.get_y() + bar.get_height() / 2),
                    xytext=(3, 0), textcoords="offset points", ha='left', va='center', fontsize=9)
    
    ax.spines[['top', 'right', 'left']].set_visible(False)
    ax.tick_params(axis='y', length=0) 
    if not is_float:
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))

# ============================
# Source-level Aggregation
# ============================
print("\nCalculating Source-level Bibliometric Indices...")

if 'Source Title' not in df.columns:
    print("Warning: 'Source Title' column not found. Skipping source analysis.")
    SOURCE_COL = None
else:
    print("Normalizing 'Source Title'...")
    # Create a clean 'key' for grouping
    df['Source_Key'] = df['Source Title'].astype(str).str.lower() \
        .str.replace(' & ', ' and ', regex=False).str.replace('&', 'and', regex=False) \
        .str.replace('-', ' ', regex=False).str.replace(r'[^\w\s]', '', regex=True).str.strip()
    
    SOURCE_COL = 'Source_Key'
    SOURCE_DISPLAY_COL = 'Source_Display' 
    max_length = 30 

if SOURCE_COL:
    # Ensure MNCS exists
    if 'MNCS' not in df.columns:
        df['MNCS'] = np.nan
        
    required_cols = [SOURCE_COL, 'Publication Year', 'Cited by', 'Source Title']

    if not all(col in df.columns for col in required_cols):
        print(f"Warning: Missing columns. Skipping.")
        source_stats = pd.DataFrame() 
    else:
        df_clean = df.dropna(subset=['Source Title', 'Publication Year']).copy()
        df_clean['Publication Year'] = pd.to_numeric(df_clean['Publication Year'], errors='coerce')
        df_clean = df_clean.dropna(subset=['Publication Year'])
        
        current_year = datetime.now().year

        # --- AGGREGATION ---
        source_stats = df_clean.groupby(SOURCE_COL).agg(
            OriginalTitle=('Source Title', lambda x: x.mode()[0]), 
            h_index=('Cited by', calculate_h_index), # Local H-Index
            g_index=('Cited by', calculate_g_index),
            Total_Citations=('Cited by', 'sum'),
            Number_of_Publications=(SOURCE_COL, 'size'), 
            Avg_MNCS=('MNCS', 'mean'), 
            First_Publication_Year=('Publication Year', 'min')
        ).reset_index()

        source_stats['OriginalTitle'] = source_stats['OriginalTitle'].apply(format_journal_title)
        source_stats['Years_Active'] = source_stats['First_Publication_Year'].apply(lambda x: max(1, current_year - int(x) + 1))
        source_stats['m_index'] = source_stats['h_index'] / source_stats['Years_Active']
        
        # Save Results
        results_path = os.path.join(results_dir, 'all_sources_ranked.csv')
        source_stats.to_csv(results_path, index=False)
        print(f"Full source ranking data saved to: {results_path}")

        # Create Display Names
        source_stats[SOURCE_DISPLAY_COL] = source_stats['OriginalTitle'].apply(lambda x: x if len(x) <= max_length else x[:max_length] + '...')
        
        # --- PREPARE RANKINGS ---
        top_volume = source_stats.sort_values(by='Number_of_Publications', ascending=False).head(10).copy()
        top_h = source_stats.sort_values(by='h_index', ascending=False).head(10).copy()
        top_g = source_stats.sort_values(by='g_index', ascending=False).head(10).copy()
        top_m = source_stats.sort_values(by='m_index', ascending=False).head(10).copy()

        # Map Titles back to main df
        title_map = source_stats.set_index('Source_Key')['OriginalTitle']
        df['OriginalTitle'] = df['Source_Key'].map(title_map)

# ============================
# Descriptive Indicators (Composite Plot)
# ============================
if ('top_volume' in locals() and not top_volume.empty):
    print("\nGenerating Descriptive Indicators Plot...")
    
    fig, axes = plt.subplots(2, 2, figsize=(20, 12)) 
    
    # UPDATED: Added N to title for context
    total_n = len(source_stats)
    fig.suptitle(f'Descriptive Bibliometric Indicators: Top Sources (Total $N={total_n}$)', fontsize=18, fontweight='bold', y=0.98)

    # Note: Explicitly labeling "Local" for all corpus-dependent metrics
    create_top_10_panel(axes[0, 0], top_volume, 'Number_of_Publications', 'Most Productive Sources', 'Total Publications')
    create_top_10_panel(axes[0, 1], top_h, 'h_index', 'Highest Local Impact', 'Local H-Index') 
    create_top_10_panel(axes[1, 0], top_g, 'g_index', 'Highest Cumulative Impact', 'Local G-Index')
    create_top_10_panel(axes[1, 1], top_m, 'm_index', 'Highest Sustained Impact', 'Local M-Index', is_float=True)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) 
    
    # Save PDF & PNG
    plot_path_png = os.path.join(plots_dir, 'composite_top_sources_descriptive.png')
    plot_path_pdf = os.path.join(plots_dir, 'composite_top_sources_descriptive.pdf')
    plt.savefig(plot_path_png, dpi=600, bbox_inches='tight')
    plt.savefig(plot_path_pdf, format='pdf', bbox_inches='tight')
    plt.show()

# ============================
# Field-Normalized Impact Analysis
# ============================
if ('source_stats' in locals() and not source_stats.empty):
    print("\n" + "="*50)
    print("ANALYSIS: FIELD-NORMALIZED IMPACT")
    print("="*50)
    
    # Filter for minimum papers to avoid noise (e.g., > 5 papers)
    min_papers = 5
    sci_stats = source_stats[source_stats['Number_of_Publications'] >= min_papers].copy()
    
    # Rank by MNCS (Primary) then Local H (Secondary)
    top_10_sci = sci_stats.sort_values(by=['Avg_MNCS', 'h_index'], ascending=False).head(10).copy()
    top_10_sci['Avg_MNCS'] = top_10_sci['Avg_MNCS'].round(2)
    
    print(f"Top 10 Sources by Field-Normalized Impact (Min {min_papers} papers):")
    print(top_10_sci[['OriginalTitle', 'Avg_MNCS', 'h_index']].to_string(index=False))
    
    # --- Plot: Local H-Index vs MNCS Panel ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
    
    # Sort for plotting (highest MNCS at top)
    plot_data_sci = top_10_sci.sort_values(by='Avg_MNCS', ascending=True)
    
    # Panel 1: Local H-Index (Accumulated Prestige)
    # Style: Monochrome Grey (Neutral Context)
    ax1.barh(plot_data_sci['Source_Display'], plot_data_sci['h_index'], color='#333333')
    ax1.set_xlabel('Local H-Index (Corpus Specific)')
    ax1.set_title('Accumulated Prestige (Local H-Index)', fontweight='bold')
    ax1.grid(axis='x', linestyle='--', alpha=0.5)
    ax1.spines[['top', 'right']].set_visible(False)
    
    for i, v in enumerate(plot_data_sci['h_index']):
        ax1.text(v + 0.1, i, str(v), color='black', va='center', fontweight='bold', fontsize=9)

    # Panel 2: MNCS (Normalized Efficiency)
    # Style: Cividis Blue (High Contrast vs Grey)
    # cividis(0.0) is the darkest blue in the palette
    cividis_blue = plt.cm.cividis(0.0)
    
    # Lollipop style
    ax2.hlines(y=plot_data_sci['Source_Display'], xmin=0, xmax=plot_data_sci['Avg_MNCS'], color='gray', alpha=0.5, linewidth=1)
    ax2.plot(plot_data_sci['Avg_MNCS'], plot_data_sci['Source_Display'], 'o', markersize=9, color=cividis_blue)
    
    # Global Avg Line
    ax2.axvline(x=1.0, color='black', linestyle='--', linewidth=1.2)
    trans = ax2.get_xaxis_transform()
    ax2.text(1.05, 0.02, 'Global Avg (1.0)', transform=trans, color='black', fontsize=9, verticalalignment='bottom')
    
    ax2.set_xlabel('Mean Normalized Citation Score (MNCS)')
    ax2.set_title('Normalized Impact Efficiency (MNCS)', fontweight='bold')
    ax2.grid(axis='x', linestyle='--', alpha=0.5)
    ax2.spines[['top', 'right', 'left']].set_visible(False)
    ax2.tick_params(left=False)
    
    # Color text match the marker
    for i, v in enumerate(plot_data_sci['Avg_MNCS']):
        ax2.text(v + 0.1, i, f"{v:.2f}", color=cividis_blue, va='center', fontweight='bold', fontsize=9)

    # UPDATED: Replaced \ge with \geq to fix Matplotlib ParseFatalException
    filtered_n = len(sci_stats)
    plt.suptitle(f'Journal Impact Analysis: Accumulated Prestige vs. Normalized Efficiency\n(Subset: $N={filtered_n}$ Sources with $\\geq${min_papers} Articles)', fontsize=16, y=1.05)
    plt.tight_layout()
    
    # Save PDF & PNG
    plot_path_b_png = os.path.join(plots_dir, 'top_10_journals_impact_analysis.png')
    plot_path_b_pdf = os.path.join(plots_dir, 'top_10_journals_impact_analysis.pdf')
    plt.savefig(plot_path_b_png, dpi=600, bbox_inches='tight')
    plt.savefig(plot_path_b_pdf, format='pdf', bbox_inches='tight')
    print(f"Impact Analysis plot saved to: {plot_path_b_png}")
    plt.show()

# ============================
# Print Summary Tables
# ============================
print("\n--- Summary Tables for Top Sources ---")

if 'top_volume' in locals() and not top_volume.empty:
    print_markdown_table("Top 10 Sources by Number of Articles", top_volume, ['OriginalTitle', 'Number_of_Publications'])

if 'top_h' in locals() and not top_h.empty:
    # UPDATED COLUMN HEADER to Local H-index
    print_markdown_table("Top 10 Sources by Local H-Index", top_h, ['OriginalTitle', 'h_index']) 
    print_markdown_table("Top 10 Sources by Local G-Index", top_g, ['OriginalTitle', 'g_index'])
    print_markdown_table("Top 10 Sources by Local M-Index", top_m, ['OriginalTitle', 'm_index'])

## 17. Geographic Distribution and Collaboration Patterns

This analysis explores the *mode* of collaboration at the country level. It classifies each article as either:
* **SCP (Single-Country Publication)**: All affiliations on the paper are from a single country.
* **MCP (Multi-Country Publication)**: The paper has affiliations from two or more different countries (i.e., it is an internationally collaborative paper).

This section analyzes the spatial dimension of the research field, focusing on two key aspects:
* Collaboration Mode: Determining whether research is domestic (Single-Country Publications, SCP) or international (Multi-Country Publications, MCP).
* Impact Efficiency: Evaluating whether high-volume countries are also producing high-impact work (MNCS).

Methodology:
* Classification: Each paper is tagged as SCP or MCP based on the diversity of country affiliations.
* Ranking: Countries are ranked by Volume (Total Appearances) and Field-Normalized Impact (MNCS).
### What you may want to change

- **Top-N shown in plots:** currently Top 10 by `Articles`
- **Filtering of Unknown:** you can keep Unknown for auditing, but it is excluded by default
- **Minimum paper threshold for impact plots:** `min_papers = 5` (later in the cell)

---

In [ ]:
import ast 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from IPython.display import display

# ============================
# Country Analysis Setup
# ============================
print("\nStarting Country-level Analysis...")

# --- Helper Functions ---
def ensure_list(x):
    if isinstance(x, str):
        try: return ast.literal_eval(x)
        except (ValueError, SyntaxError): return []
    elif isinstance(x, list): return x
    return []

def classify_publication(country_list):
    if isinstance(country_list, list):
        clean_list = set(c for c in country_list if c)
        if len(clean_list) == 1: return 'SCP' 
        elif len(clean_list) > 1: return 'MCP'
    return 'Unknown'

# --- Data Prep ---
if 'df' not in locals() or 'Countries_Extracted' not in df.columns:
    print("Error: 'df' or 'Countries_Extracted' missing. Run setup/affiliation steps first.")
else:
    # Ensure MNCS exists
    if 'MNCS' not in df.columns:
        df['MNCS'] = np.nan
        print("Note: 'MNCS' column missing. Impact scores will be NaN.")

    # Prepare Data
    df['Countries_Extracted_List'] = df['Countries_Extracted'].apply(ensure_list)
    df['Country_Classification'] = df['Countries_Extracted_List'].apply(classify_publication)

    print("'Country_Classification' column added to df.")
    print(df['Country_Classification'].value_counts()) 

    # Filter out 'Unknown'
    df_collab = df[df['Country_Classification'] != 'Unknown'].copy()

    if len(df_collab) < len(df):
        print(f"Removed {len(df) - len(df_collab)} articles with 'Unknown' country data.")

    if df_collab.empty:
        print("Warning: No valid country data found.")
        agg_df = pd.DataFrame()
    else:
        # --- EXPLOSION STEP (Legacy Compatible) ---
        df_exploded_final = df_collab.explode('Countries_Extracted_List').copy()
        df_exploded_final.rename(columns={'Countries_Extracted_List': 'Country_Exploded'}, inplace=True)
        df_exploded_final.dropna(subset=['Country_Exploded'], inplace=True)
        df_exploded_final['Country_Exploded'] = df_exploded_final['Country_Exploded'].astype(str).str.strip()
        df_exploded_final = df_exploded_final[df_exploded_final['Country_Exploded'] != '']

        total_docs = len(df_collab)
        print(f"Analyzing {total_docs} total documents.")

        # --- 1c. Aggregation ---
        agg_df = (
            df_exploded_final
            .groupby('Country_Exploded')['Country_Classification']
            .agg([
                ('Articles', 'size'), 
                ('SCP', lambda x: (x == 'SCP').sum()), 
                ('MCP', lambda x: (x == 'MCP').sum())
            ])
            .reset_index()
            .rename(columns={'Country_Exploded': 'Country'})
        )

        total_countries = len(agg_df)

        # Percentages
        agg_df[['Articles','SCP','MCP']] = agg_df[['Articles','SCP','MCP']].astype(int)
        agg_df['SCP %'] = (agg_df['SCP'] / agg_df['Articles'].replace(0, np.nan) * 100).round(1).fillna(0)
        agg_df['MCP %'] = (agg_df['MCP'] / agg_df['Articles'].replace(0, np.nan) * 100).round(1).fillna(0)
        agg_df['MCP_Ratio'] = agg_df['MCP'] / agg_df['Articles'] # Helper for plot

        agg_df.sort_values(by='Articles', ascending=False, inplace=True)
        agg_df.reset_index(drop=True, inplace=True)

        # Save Basic Results
        output_table_path_csv = os.path.join(results_dir, 'country_collaboration_summary.csv')
        agg_df.to_csv(output_table_path_csv, index=False, encoding='utf-8-sig')

        # --- 1d. Display Tables ---
        print("\n--- Top 10 Countries by Volume ---")
        cols_to_show = ['Country', 'Articles', 'SCP', 'MCP', 'MCP %']
        print(agg_df.head(10)[cols_to_show].to_string(index=False))

        # --- Plot A: Collaboration Patterns (Cividis Style) ---
        print("\nGenerating Collaboration Patterns Plot...")
        plot_data = agg_df.head(10).sort_values(by='Articles', ascending=True)
        
        plt.figure(figsize=(12, 8))
        ax = plt.gca()
        
        # Color Strategy: Cividis Extremes
        cividis_gold = plt.cm.cividis(0.95)
        cividis_blue = plt.cm.cividis(0.0)
        
        # Stacked Bar Chart
        ax.barh(plot_data['Country'], plot_data['SCP'], label='Single-Country (SCP)', color=cividis_gold)
        ax.barh(plot_data['Country'], plot_data['MCP'], left=plot_data['SCP'], label='Multi-Country (MCP)', color=cividis_blue)
        
        for i, row in enumerate(plot_data.itertuples()):
            # Label total articles at the end of the bar (Monochrome Grey)
            ax.annotate(f' {row.Articles}', xy=(row.Articles, i), ha='left', va='center', fontsize=10, fontweight='bold', color='#333333')
            
            if row.MCP > row.Articles * 0.10: 
                mcp_pct_val = (row.MCP / row.Articles) * 100
                ax.annotate(f'{mcp_pct_val:.0f}%', xy=(row.SCP + row.MCP/2, i), ha='center', va='center', color='white', fontsize=9, fontweight='bold')

        ax.set_title(f'Country Production: Domestic (SCP) vs. International (MCP)\n(Top 10 of $N={total_countries}$ Active Countries)', fontweight='bold', pad=20)
        ax.set_xlabel('Number of Article Appearances')
        ax.legend(loc='lower right', frameon=True, facecolor='white', framealpha=1)
        ax.spines[['top', 'right']].set_visible(False)
        plt.tight_layout()
        
        plot_path_png = os.path.join(plots_dir, 'top_10_countries_collaboration.png')
        plot_path_pdf = os.path.join(plots_dir, 'top_10_countries_collaboration.pdf')
        plt.savefig(plot_path_png, dpi=600, bbox_inches='tight')
        plt.savefig(plot_path_pdf, format='pdf', bbox_inches='tight')
        print(f"Collaboration plot saved to: {plot_path_png}")
        plt.show()

        # ============================
        # 2. IMPACT ANALYSIS (MNCS)
        # ============================
        print("\n--- Calculating Impact (MNCS) ---")
        
        if 'MNCS' not in df_exploded_final.columns:
             df_impact_base = df_collab[['Countries_Extracted_List', 'MNCS', 'Cited by']].explode('Countries_Extracted_List')
             df_impact_base = df_impact_base.rename(columns={'Countries_Extracted_List': 'Country_Exploded'})
        else:
             df_impact_base = df_exploded_final

        impact_stats = df_impact_base.groupby('Country_Exploded').agg(
            MNCS=('MNCS', 'mean'),
            Total_Citations=('Cited by', 'sum')
        ).reset_index().rename(columns={'Country_Exploded': 'Country'})

        # Merge with Volume stats
        country_stats = agg_df.merge(impact_stats, on='Country', how='left')
        country_stats['MNCS'] = country_stats['MNCS'].fillna(0)
        
        # Save Master Stats
        master_stats_path = os.path.join(results_dir, 'country_master_stats.csv')
        country_stats.to_csv(master_stats_path, index=False)
        print(f"Master stats (Volume + Impact) saved to: {master_stats_path}")

        # --- Plot B: Impact Efficiency (Sorted by MNCS) ---
        print("\nGenerating Impact Efficiency Plot...")
        
        # 1. Select: Top 15 by MNCS (Highest Impact), Min 5 Papers
        min_papers = 5
        significant_countries = country_stats[country_stats['Articles'] >= min_papers].copy()
        top_15_impact = significant_countries.sort_values(by='MNCS', ascending=False).head(15).copy()
        
        # 2. Sort for Plotting: Lowest to Highest MNCS (so highest is at top)
        plot_data_b = top_15_impact.sort_values(by='MNCS', ascending=True)

        print(f"\nTop 15 Countries by Impact (Min {min_papers} papers):")
        print(plot_data_b[['Country', 'Articles', 'MNCS']].sort_values(by='MNCS', ascending=False).to_string(index=False))

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8), sharey=True)

        # Panel 1: Volume (Neutral Grey) - Inverted Axis
        ax1.barh(plot_data_b['Country'], plot_data_b['Articles'], color='#555555')
        ax1.set_xlabel('Total Publications')
        ax1.set_title('Quantity (Volume)', fontweight='bold')
        ax1.grid(axis='x', linestyle='--', alpha=0.5)
        ax1.invert_xaxis() 
        ax1.spines[['top', 'left']].set_visible(False)
        
        # Annotate Volume (FIXED)
        # Because axis is inverted (Right to Left), ha='right' aligns text to the left side of the coordinate
        # We add a small offset to ensure it doesn't overlap the bar
        max_vol = plot_data_b['Articles'].max()
        ax1.set_xlim(max_vol * 1.2, 0) # Extend limits to the left for labels
        
        for i, v in enumerate(plot_data_b['Articles']):
            ax1.text(v, i, f" {v}", color='black', va='center', ha='right', fontsize=9, fontweight='bold')

        # Panel 2: Impact (Cividis Diverging)
        norm = plt.Normalize(vmin=plot_data_b['MNCS'].min(), vmax=plot_data_b['MNCS'].max())
        colors = [plt.cm.cividis(norm(x)) for x in plot_data_b['MNCS']]
        
        ax2.barh(plot_data_b['Country'], plot_data_b['MNCS'], color=colors)
        ax2.axvline(x=1.0, color='black', linestyle='--', linewidth=1.5) 
        ax2.text(1.05, -1, 'Global Avg (1.0)', color='black', fontsize=10, fontweight='bold')

        ax2.set_xlabel('Mean Normalized Citation Score (MNCS)')
        ax2.set_title('Quality (Field-Normalized Impact)', fontweight='bold')
        ax2.grid(axis='x', linestyle='--', alpha=0.5)
        ax2.spines[['top', 'right', 'left']].set_visible(False)
        ax2.tick_params(left=False)

        # Annotate Impact
        for i, v in enumerate(plot_data_b['MNCS']):
            ax2.text(v + 0.05, i, f"{v:.2f}", color='black', va='center', fontsize=9, fontweight='bold')

        plt.suptitle(f'Country Performance: High-Impact Leaders\n(Top 15 by MNCS, Min. {min_papers} Articles, Analysis of $N={total_docs}$ Documents)', fontsize=16, y=1.02, fontweight='bold')
        plt.tight_layout()

        # Save PDF & PNG
        plot_path_b_png = os.path.join(plots_dir, 'country_impact_comparison.png')
        plot_path_b_pdf = os.path.join(plots_dir, 'country_impact_comparison.pdf')
        plt.savefig(plot_path_b_png, dpi=600, bbox_inches='tight')
        plt.savefig(plot_path_b_pdf, format='pdf', bbox_inches='tight')
        print(f"Impact plot saved to: {plot_path_b_png}")
        plt.show()

## 18. Advanced Country Analysis (Impact & Networks)

This section goes beyond SCP/MCP counting to analyze **impact** and the **structure** of international collaborations.  
It produces a master `country_master_stats.csv` file and three “publication-ready” visualizations:

1) **Citation distribution by country** (Top 15 by total citations)  
2) **Scientometric performance matrix** (Productivity vs MNCS, bubble size = citations)  
3) **International collaboration network topology** (Top 30 countries by collaboration strength)

---

### 0) Master data preparation (what happens first)

The code rebuilds a `country_stats` table (wide format):

1. **Global yearly averages (baseline for normalization)**  
   Compute the mean citations for each publication year in the corpus.

2. **Normalized citations per article**  
   `Normalized_Citations = citations / global_avg_citations_year`

3. **Explode by country and aggregate**  
   For each country:
   - `MNCS` = mean normalized citations  
   - `Total_Citations` = summed raw citations

4. **Merge with productivity counts**  
   Merge the impact metrics into `agg_df` from the previous country cell.

5. **Save master file**  
   `country_master_stats.csv` is written to your results directory.

> **Note:** This MNCS is still *corpus-internal* normalization (not global WoS/Scopus field normalization).  
> It is designed for robust comparisons inside your chosen topic/corpus.

---

### 1) Plot A — Global citation distribution (Top 15)

A horizontal bar chart (Cividis gradient) of:
- `Total_Citations` per country (Top 15)

This gives a quick view of “where the accumulated prestige sits”.

**Saved as**
- `Country_Citation_Impact_Distribution.(png|pdf)`

---

### 2) Plot B — Scientometric performance matrix (quadrant scatter)

A country-level “strategy map”:

- **X-axis:** Productivity (`Total_Articles`, log scale)  
- **Y-axis:** Impact (`MNCS`)  
- **Bubble size:** Total citations (sqrt-scaled to avoid dominance)  
- **Color:** MNCS (Cividis)

Reference lines:
- MNCS = 1.0 (corpus average)
- vertical median productivity line (splits “low vs high productivity”)

**Saved as**
- `Scientometric_Performance_Matrix_Productivity_vs_Impact.(png|pdf)`

---

### 3) Plot C — International collaboration network topology

A network graph where:
- **Nodes:** countries  
- **Edges:** co-occurrence in multi-country papers  
- **Edge weight:** number of co-authorship links (counted across the corpus)  
- **Node size:** weighted degree (collaboration strength)  
- **Node color:** MNCS (from `country_stats`)

To avoid an unreadable “hairball”, the plot is automatically thinned:
- keep only the **Top 30** countries by weighted degree
- keep **top-k strongest ties per node** (`TOP_K_EDGES_PER_NODE = 5`)

Optional label de-overlap:
- uses `adjustText` if available, otherwise labels may overlap slightly

**Saved as**
- `International_Collaboration_Network_Topology.(png|pdf)`

---

### What you may want to change (safe edits)

Inside the code cell:

- `min_pubs = 5`  
  Minimum output threshold for the performance matrix (reduces noise).

- `TOP_K_EDGES_PER_NODE = 5`  
  Controls how aggressively the network is thinned.

- Subgraph sizes  
  - Top 30 nodes for network: `[:30]`  
  - Top 40 labels: `[:40]`

If the network plot is too slow for very large corpora, reduce:
- `iterations=250` in `spring_layout`
- number of nodes (e.g., `[:20]`)

---

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.ticker as mtick
import seaborn as sns
import networkx as nx
import itertools
import textwrap
import os
import ast
from IPython.display import display

# --- Safe Import for adjustText ---
try:
    from adjustText import adjust_text
    ADJUST_TEXT_AVAILABLE = True
    print("Loaded 'adjustText' library.")
except ImportError:
    ADJUST_TEXT_AVAILABLE = False
    print("Warning: 'adjustText' not found.")

# =============================================================================
# Publication-ready Matplotlib defaults 
# =============================================================================
plt.style.use('seaborn-v0_8-whitegrid')

plt.rcParams.update({
    # Output quality
    "figure.dpi": 200,
    "savefig.dpi": 600,

    # Typography (safe, consistent across Kaggle)
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,

    # Subtle axes look
    "axes.linewidth": 0.8,
    "grid.alpha": 0.25,
    "grid.linestyle": ":",
    "axes.spines.top": False,
    "axes.spines.right": False,

    # Legend
    "legend.frameon": True,
    "legend.framealpha": 0.95,
})

CIVIDIS = plt.cm.cividis

# ============================
# Master Data Preparation (ORIGINAL LOGIC)
# ============================
print("\nStarting Country-Level Analysis (Wide Format)...")

def ensure_list_safe(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except Exception:
            return []
    elif isinstance(x, list):
        return x
    return []

# Check prerequisites
if 'df' not in locals() or 'agg_df' not in locals():
    print("Error: Missing 'df' or 'agg_df'. Run previous cells first.")
    country_stats = pd.DataFrame()
else:
    # -------------------------------------------------------------------------
    # 1) Calculate MNCS (Original Logic)
    # -------------------------------------------------------------------------
    print("Calculating global yearly citation averages (for MNCS)...")

    df['Cited by'] = pd.to_numeric(df['Cited by'], errors='coerce').fillna(0)

    df_year_avg = (
        df.groupby('Publication Year')['Cited by']
        .mean()
        .reset_index(name='Global_Avg_Citations')
    )

    df_norm = df.merge(df_year_avg, on='Publication Year', how='left')
    df_norm['Global_Avg_Citations'] = df_norm['Global_Avg_Citations'].fillna(1).replace(0, 1)
    df_norm['Normalized_Citations'] = df_norm['Cited by'] / df_norm['Global_Avg_Citations']

    # -------------------------------------------------------------------------
    # 2) Explode for Impact Stats
    # -------------------------------------------------------------------------
    df_norm['Countries_List'] = df_norm['Countries_Extracted'].apply(ensure_list_safe)
    df_norm_exploded = df_norm.explode('Countries_List').rename(columns={'Countries_List': 'Country'})
    df_norm_exploded = df_norm_exploded[df_norm_exploded['Country'].astype(str).str.strip() != '']

    # -------------------------------------------------------------------------
    # 3) Aggregate Impact
    # -------------------------------------------------------------------------
    country_impact_stats = (
        df_norm_exploded.groupby('Country')
        .agg(
            MNCS=('Normalized_Citations', 'mean'),
            Total_Citations=('Cited by', 'sum')
        )
        .reset_index()
    )

    # -------------------------------------------------------------------------
    # 4) Merge with Productivity (agg_df from previous cell)
    # -------------------------------------------------------------------------
    country_stats = agg_df.merge(country_impact_stats, on='Country', how='left')
    country_stats['MNCS'] = country_stats['MNCS'].fillna(0)
    country_stats['Total_Citations'] = country_stats['Total_Citations'].fillna(0).astype(int)

    # Alias 'Articles' to 'Total_Articles' (your downstream plots)
    country_stats['Total_Articles'] = country_stats['Articles']

    total_countries = len(country_stats)

    # Save master stats
    results_path = os.path.join(results_dir, 'country_master_stats.csv')
    country_stats.to_csv(results_path, index=False, encoding='utf-8-sig')
    print(f"Master stats saved to: {results_path}")

    # =============================================================================
    # Helper formatting
    # =============================================================================
    comma_fmt = mtick.FuncFormatter(lambda x, pos: f"{int(x):,}" if x >= 1 else f"{x:g}")

    # =============================================================================
    # PLOT A: CITATION DISTRIBUTION (Top 15) — Cividis gradient, clean labels
    # =============================================================================
    print("Generating Citation Distribution Plot (publication-ready)...")

    top_cited = country_stats.sort_values(by='Total_Citations', ascending=False).head(15).copy()
    top_cited = top_cited.sort_values(by='Total_Citations', ascending=True)  # barh reads better ascending

    fig, ax = plt.subplots(figsize=(10.5, 6.5))

    # Cividis gradient (subtle, print-safe)
    colors = CIVIDIS(np.linspace(0.20, 0.95, len(top_cited)))
    ax.barh(top_cited['Country'], top_cited['Total_Citations'], color=colors, edgecolor='none')

    ax.set_xlabel('Total Accumulated Citations ($TC$)')
    ax.set_ylabel('')
    ax.set_title(f'Global Citation Distribution (Top 15 of $N={total_countries}$ Countries)',
                 fontweight='bold', pad=12)

    ax.xaxis.set_major_formatter(comma_fmt)

    # tight x-limits + annotation padding
    xmax = top_cited['Total_Citations'].max() if len(top_cited) else 1
    ax.set_xlim(0, xmax * 1.12)

    # annotate with commas
    for i, v in enumerate(top_cited['Total_Citations'].values):
        ax.text(v + xmax * 0.01, i, f"{int(v):,}", va='center', ha='left', fontsize=9)

    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, 'Country_Citation_Impact_Distribution.pdf'), bbox_inches='tight')
    plt.savefig(os.path.join(plots_dir, 'Country_Citation_Impact_Distribution.png'), bbox_inches='tight')
    plt.show()

    # =========================================================
    # PLOT B: STRATEGIC QUADRANTS (SCATTER) - Journal-ready
    #   x = productivity (log)
    #   y = impact (MNCS)
    #   size = total citations (robust sqrt scaling)
    #   color = MNCS (cividis)
    # =========================================================
    print("Generating Strategic Matrix Plot...")
    
    min_pubs = 5
    plot_data = country_stats[country_stats['Total_Articles'] >= min_pubs].copy()
    filtered_n = len(plot_data)
    
    if filtered_n == 0:
        print(f"Not enough countries with >= {min_pubs} articles to plot.")
    else:
        fig, ax = plt.subplots(figsize=(12, 9))
    
        # --- Robust size scaling (prevents giant bubbles dominating) ---
        tc = plot_data['Total_Citations'].fillna(0).clip(lower=0).astype(float)
        tc_sqrt = np.sqrt(tc + 1)  # +1 avoids 0
        s_min, s_max = 40, 1400
        sizes = s_min + (tc_sqrt / tc_sqrt.max()) * (s_max - s_min)
    
        # --- Robust color scaling (avoid one extreme washing out colormap) ---
        mncs = plot_data['MNCS'].fillna(0).astype(float)
        vmin = np.nanpercentile(mncs, 2)
        vmax = np.nanpercentile(mncs, 98)
        if vmin == vmax:
            vmin, vmax = mncs.min(), mncs.max() + 1e-9
    
        norm = plt.Normalize(vmin=vmin, vmax=vmax)
    
        sc = ax.scatter(
            plot_data['Total_Articles'],
            plot_data['MNCS'],
            s=sizes,
            c=mncs,
            cmap=plt.cm.cividis,
            norm=norm,
            alpha=0.82,
            linewidth=0.6,
            edgecolors='white'
        )
    
        # Axes & reference lines
        ax.set_xscale('log')
        ax.axhline(1.0, color='black', linestyle='--', linewidth=1.2, alpha=0.85)
        med_x = plot_data['Total_Articles'].median()
        ax.axvline(med_x, color='black', linestyle=':', linewidth=1.0, alpha=0.6)
    
        ax.set_xlabel('Total Publications (log scale)')
        ax.set_ylabel('Mean Normalized Citation Score (MNCS)')
        ax.set_title(
            f'Scientometric Performance Matrix: Productivity vs. Impact\n'
            f'(Subset: $N={filtered_n}$ Countries with $\\geq$ {min_pubs} Articles)',
            fontweight='bold',
            pad=14
        )
    
        # Subtle grid + clean spines
        ax.grid(True, which='both', linestyle=':', alpha=0.25)
        ax.spines[['top', 'right']].set_visible(False)
    
        # Quadrant labels (axes coords -> stable placement)
        ax.text(0.02, 0.99, 'Low Productivity / High Impact (Niche)',
                transform=ax.transAxes, ha='left', va='top', fontsize=8, fontweight='bold')
        ax.text(0.98, 0.99, 'High Productivity / High Impact',
                transform=ax.transAxes, ha='right', va='top', fontsize=8, fontweight='bold')
        ax.text(0.02, 0.01, 'Low Productivity / Low Impact',
                transform=ax.transAxes, ha='left', va='bottom', fontsize=8, fontweight='bold', alpha=0.85)
        ax.text(0.98, 0.01, 'High Productivity / Low Impact',
                transform=ax.transAxes, ha='right', va='bottom', fontsize=8, fontweight='bold', alpha=0.85)
    
        # Colorbar (MNCS)
        cbar = plt.colorbar(sc, ax=ax, pad=0.02)
        cbar.set_label('MNCS (Cividis)', rotation=90)
    
    
        # --- Smart labeling (limited, readable) ---
        labels_to_plot = pd.concat([
            plot_data.nlargest(15, 'Total_Articles'),
            plot_data.nlargest(10, 'MNCS'),
            plot_data.nlargest(10, 'Total_Citations')
        ]).drop_duplicates(subset=['Country'])
    
        texts = []
        for _, row in labels_to_plot.iterrows():
            label = textwrap.fill(row['Country'], 18)
            t = ax.text(
                row['Total_Articles'],
                row['MNCS'],
                label,
                fontsize=9,
                ha='center',
                va='center',
                bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, pad=1.5)
            )
            texts.append(t)
    
        if ADJUST_TEXT_AVAILABLE and texts:
            try:
                adjust_text(
                    texts,
                    ax=ax,
                    arrowprops=dict(arrowstyle='-', color='gray', lw=0.6, alpha=0.7),
                    expand_points=(1.2, 1.2),
                    expand_text=(1.1, 1.2)
                )
            except Exception:
                pass
    
        plt.tight_layout()
        plt.savefig(os.path.join(plots_dir, 'Scientometric_Performance_Matrix_Productivity_vs_Impact.pdf'),
                    format='pdf', bbox_inches='tight')
        plt.savefig(os.path.join(plots_dir, 'Scientometric_Performance_Matrix_Productivity_vs_Impact.png'),
                    format='png', bbox_inches='tight')
        plt.show()


    # =========================================================
    # PLOT C: COLLABORATION NETWORK (WIDE) - Journal-ready
    #   size = collaboration strength (weighted degree)
    #   color = MNCS (cividis) from country_stats
    #   edges = top-k strongest ties per node (reduces hairball)
    # =========================================================
    print("Generating Collaborative Network Plot...")
    
    # 1) Build Graph (same logic)
    G = nx.Graph()
    if 'Countries_Extracted_List' not in df.columns:
        df['Countries_Extracted_List'] = df['Countries_Extracted'].apply(ensure_list_safe)
    
    for country_list in df['Countries_Extracted_List']:
        clean_list = sorted(list(set(c for c in country_list if c and str(c).strip())))
        if len(clean_list) > 1:
            for u, v in itertools.combinations(clean_list, 2):
                if G.has_edge(u, v):
                    G[u][v]['weight'] += 1
                else:
                    G.add_edge(u, v, weight=1)
    
    if G.number_of_edges() == 0:
        print("Not enough connections to plot network.")
    else:
        # 2) Subgraph: top 30 by weighted degree (collaboration strength)
        degrees_w = dict(G.degree(weight='weight'))
        top_nodes = sorted(degrees_w, key=degrees_w.get, reverse=True)[:30]
        G_sub = G.subgraph(top_nodes).copy()
    
        if G_sub.number_of_nodes() == 0:
            print("Filtered subgraph is empty.")
        else:
            # 3) Edge thinning: keep top-k edges per node (strongest ties)
            TOP_K_EDGES_PER_NODE = 5
            keep = set()
            for n in G_sub.nodes():
                nbr_edges = sorted(G_sub.edges(n, data=True), key=lambda x: x[2].get('weight', 1), reverse=True)[:TOP_K_EDGES_PER_NODE]
                for u, v, _ in nbr_edges:
                    keep.add(tuple(sorted((u, v))))
    
            edges_keep = [(u, v) for (u, v) in keep if G_sub.has_edge(u, v)]
            weights_keep = np.array([G_sub[u][v]['weight'] for u, v in edges_keep], dtype=float)
    
            # 4) Node aesthetics
            wdeg_sub = dict(G_sub.degree(weight='weight'))
            wvals = np.array([wdeg_sub[n] for n in G_sub.nodes()], dtype=float)
    
            # Size scaling (sqrt keeps spread, avoids monster nodes)
            nmin, nmax = 300, 2600
            w_sqrt = np.sqrt(wvals)
            node_sizes = nmin + (w_sqrt / w_sqrt.max()) * (nmax - nmin)
    
            # Color = MNCS (meaningful variation) using cividis
            mncs_map = {}
            if 'country_stats' in locals() and not country_stats.empty and 'Country' in country_stats.columns:
                mncs_map = country_stats.set_index('Country')['MNCS'].to_dict()
    
            mncs_vals = np.array([mncs_map.get(n, np.nan) for n in G_sub.nodes()], dtype=float)
            if np.all(np.isnan(mncs_vals)):
                mncs_vals = np.zeros(len(G_sub.nodes()))
    
            vmin = np.nanpercentile(mncs_vals, 2)
            vmax = np.nanpercentile(mncs_vals, 98)
            if vmin == vmax:
                vmin, vmax = np.nanmin(mncs_vals), np.nanmax(mncs_vals) + 1e-9
            norm = plt.Normalize(vmin=vmin, vmax=vmax)
            node_colors = plt.cm.cividis(norm(np.nan_to_num(mncs_vals, nan=vmin)))
    
            # 5) Layout (weight-aware)
            pos = nx.spring_layout(G_sub, weight='weight', k=1.2, iterations=250, seed=42)
    
            fig, ax = plt.subplots(figsize=(18, 11))
            ax.set_axis_off()
    
            # Edge widths (log-scaled) + subtle alpha
            if len(weights_keep) > 0:
                wmax = weights_keep.max()
                widths = 0.4 + 3.0 * (np.log1p(weights_keep) / np.log1p(wmax))
            else:
                widths = []
    
            nx.draw_networkx_edges(
                G_sub, pos,
                edgelist=edges_keep,
                width=widths,
                edge_color='black',
                alpha=0.12,
                ax=ax
            )
    
            nx.draw_networkx_nodes(
                G_sub, pos,
                node_size=node_sizes,
                node_color=node_colors,
                edgecolors='black',
                linewidths=0.8,
                alpha=0.95,
                ax=ax
            )
    
            # Labels: only top nodes by weighted degree (keeps it readable)
            label_top = sorted(wdeg_sub, key=wdeg_sub.get, reverse=True)[:40]
            texts = []
            for n in label_top:
                x, y = pos[n]
                t = ax.text(
                    x, y, n,
                    fontsize=11,
                    fontweight='bold',
                    ha='center', va='center',
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=1.2)
                )
                texts.append(t)
    
            if ADJUST_TEXT_AVAILABLE and texts:
                try:
                    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', lw=0.6, alpha=0.6))
                except Exception:
                    pass
    
            # Colorbar (MNCS)
            sm = plt.cm.ScalarMappable(cmap=plt.cm.cividis, norm=norm)
            sm.set_array([])
            cbar = plt.colorbar(sm, ax=ax, shrink=0.75, pad=0.02)
            cbar.set_label('MNCS (Cividis)')
    
            # Size legend (collaboration strength)
            ref_w = np.unique(np.round(np.nanpercentile(wvals, [50, 75, 95])).astype(int))
            handles = []
            for rv in ref_w:
                rv_s = np.sqrt(rv)
                rv_size = nmin + (rv_s / np.sqrt(wvals.max())) * (nmax - nmin)
                handles.append(ax.scatter([], [], s=rv_size, facecolors='none', edgecolors='black', linewidth=0.8))
            labels = [f'Weighted degree ≈ {v:,}' for v in ref_w]
            ax.legend(
                handles, labels,
                title='Node size',
                loc='lower left',
                frameon=True,
                fontsize=8,          # label text size
                title_fontsize=9,    # title text size
                markerscale=0.55,    # scales the marker circles in the legend
                labelspacing=0.25,   # vertical space between entries
                handletextpad=0.5,   # space between marker and text
                borderpad=0.35,      # padding inside legend frame
                handlelength=1.0     # length of handle (mostly affects line handles)
            )
    
            ax.set_title(
            f'International Collaboration Network (Co-authorship)\n'
            f'Top 30 Nodes by Collaboration Strength | Full Network: $N={len(G.nodes())}$ Countries, $E={len(G.edges())}$ Links\n'
            f'Edges shown: strongest ties (top-{TOP_K_EDGES_PER_NODE} per node)',
            fontsize=15, fontweight='bold', pad=16
        )
    
            plt.tight_layout()
            plt.savefig(os.path.join(plots_dir, 'International_Collaboration_Network_Topology.pdf'),
                        format='pdf', bbox_inches='tight')
            plt.savefig(os.path.join(plots_dir, 'International_Collaboration_Network_Topology.png'),
                        format='png', bbox_inches='tight')
            plt.show()


## 19. Longitudinal Productivity Analysis by Cohort

Comparative analysis of country-level production often suffers from **Scale Bias**:  
the growth trajectories of emerging scientific nations are visually overshadowed when plotted together with global hegemons (e.g., USA, China).

---

### Methodological Approach

To enable equitable comparison, countries are stratified into **four Productivity Cohorts** based on publication-volume quartiles (Q1, Q2, Q3).

**Tier 4 (Emerging)**  
- P < Q1  
- Represents niche or early-stage research activity.

**Tier 3 (Developing)**  
- Q1 ≤ P < Q2

**Tier 2 (Established)**  
- Q2 ≤ P < Q3

**Tier 1 (Leaders)**  
- P ≥ Q3  
- Represents global scientific hubs.

---

### Visualization

We generate a **2×2 panel matrix**, plotting longitudinal growth trends separately for each tier.  
This tier-specific view enables the identification of **"Rising Stars"**—countries in lower tiers that display rapid or exponential growth relative to their peers.

---

### What to change (user-facing parameters)

Most users do not need to change anything. If desired, the following are the main “safe knobs”:

```python
# Change tier labels (purely cosmetic)
def categorize_country(count):
    ...

# Increase/Decrease how many countries get legends
if len(countries_to_plot) <= 35:
    ax.legend(...)
else:
    ...

# Switch colormap (line palette)
cmap = plt.cm.viridis  # e.g., plt.cm.cividis
```

---



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from IPython.display import display

# =========================================================
# 1. Publications Per Year by Country (Categorized 2x2 Plot)
# =========================================================
print("\n--- Starting Country Trend Tiers Analysis (2x2 Plot) ---")

# --- 1a. Check for necessary data ---
if 'df_exploded_final' not in locals() or 'country_stats' not in locals():
    print("Error: Missing 'df_exploded_final' or 'country_stats'. Run previous Country analysis cells first. Skipping plot.")
else:
    # --- 1b. Create Pivot Table (Publications per Year per Country) ---
    print("Creating publication pivot table...")
    publications_per_year = (
        df_exploded_final
        .groupby(['Publication Year', 'Country_Exploded'])
        .size()
        .reset_index(name='Publications')
    )
    
    publications_pivot = (
        publications_per_year
        .pivot(index='Publication Year', columns='Country_Exploded', values='Publications')
        .fillna(0)
    )

    # --- 1c. Categorize Countries based on Total Productivity ---
    # Use country_stats 'Articles' column (which is the total count)
    country_totals = country_stats.set_index('Country')['Articles']
    
    # Calculate Total N for Title
    total_n_articles = country_totals.sum()
    total_n_countries = len(country_totals)

    if country_totals.empty:
        print("Warning: 'country_totals' is empty. Skipping plot.")
    else:
        # Compute quartile thresholds
        q1, q2, q3 = country_totals.quantile([0.25, 0.50, 0.75])
        print(f"Productivity Tiers: Q1(25%) < {q1:.0f}, Q2(50%) < {q2:.0f}, Q3(75%) < {q3:.0f}")

        def categorize_country(count):
            if count < q1:
                return 'Category 1: Very Low Activity'
            elif q1 <= count < q2:
                return 'Category 2: Low/Moderate Activity'
            elif q2 <= count < q3:
                return 'Category 3: Moderate/High Activity'
            return 'Category 4: Very High Activity'

         # Assign categories to each country

        country_categories = country_totals.apply(categorize_country)
        cat1_countries = country_totals[country_categories == 'Category 1: Very Low Activity'].index
        cat2_countries = country_totals[country_categories == 'Category 2: Low/Moderate Activity'].index
        cat3_countries = country_totals[country_categories == 'Category 3: Moderate/High Activity'].index
        cat4_countries = country_totals[country_categories == 'Category 4: Very High Activity'].index

        # Create a DataFrame for the Tiers
        df_tiers = country_totals.to_frame(name='Total_Articles').reset_index()
        df_tiers['Category'] = df_tiers['Total_Articles'].apply(categorize_country)
        
        # Sort by Total Articles descending for better readability
        df_tiers = df_tiers.sort_values(by='Total_Articles', ascending=False)

        # --- SAVE TIERS TO CSV ---
        tiers_path = os.path.join(results_dir, 'country_productivity_tiers.csv')
        df_tiers.to_csv(tiers_path, index=False)
        print(f"Country Tier Classification saved to: {tiers_path}")

        # --- PRINT TABLES FOR EACH TIER ---
        print("\n" + "="*40)
        print("COUNTRY TIERS SUMMARY")
        print("="*40)
        
        # Define categories in logical order
        categories_order = [
            'Category 4: Very High Activity',
            'Category 3: Moderate/High Activity',
            'Category 2: Low/Moderate Activity',
            'Category 1: Very Low Activity'
        ]
        
        for cat in categories_order:
            subset = df_tiers[df_tiers['Category'] == cat]
            count = len(subset)
            print(f"\n--- {cat} (N={count}) ---")
            if count > 0:
                # Print top 10 examples + "and X others"
                examples = subset.head(10)
                print(examples[['Country', 'Total_Articles']].to_string(index=False))
                if count > 10:
                    print(f"... and {count - 10} more.")
            else:
                print("(No countries in this tier)")

        # --- 1d. Plotting ---
        print("\nGenerating 2x2 plot...")
        fig, axes = plt.subplots(2, 2, figsize=(20, 16), sharex=False, sharey=False)
        axes = axes.flatten()
        
        # Map categories to list of countries for plotting loop
        cat_map = {
            'Category 1: Very Low Activity': df_tiers[df_tiers['Category'] == 'Category 1: Very Low Activity']['Country'],
            'Category 2: Low/Moderate Activity': df_tiers[df_tiers['Category'] == 'Category 2: Low/Moderate Activity']['Country'],
            'Category 3: Moderate/High Activity': df_tiers[df_tiers['Category'] == 'Category 3: Moderate/High Activity']['Country'],
            'Category 4: Very High Activity': df_tiers[df_tiers['Category'] == 'Category 4: Very High Activity']['Country']
        }
        
        # We need this list to iterate axes in correct order (1 -> 4)
        plot_order = [
            'Category 1: Very Low Activity',
            'Category 2: Low/Moderate Activity',
            'Category 3: Moderate/High Activity',
            'Category 4: Very High Activity'
        ]

        for i, title in enumerate(plot_order):
            ax = axes[i]
            countries_list = cat_map[title]
            
            if len(countries_list) == 0:
                ax.set_title(f"{title} (No Countries)")
                ax.set_xlabel('Year')
                ax.set_ylabel('Number of Publications')
                continue
            
            # Filter the pivot table to only countries in this list
            countries_to_plot = [c for c in countries_list if c in publications_pivot.columns]
            if not countries_to_plot:
                continue
                
            subset = publications_pivot[countries_to_plot]
            
            # --- COLOR STRATEGY: Turbo (High Contrast) ---
            # 'Turbo' spans the full visible spectrum (Blue-Green-Red) without the flaws of 'Jet'.
            # It provides maximum differentiation for lines.
            cmap = plt.cm.viridis
            colors = [cmap(j) for j in np.linspace(0, 0.9, len(subset.columns))]
            
            for k, country in enumerate(subset.columns):
                ax.plot(subset.index, subset[country], marker='o', markersize=4, 
                        label=country, color=colors[k], alpha=0.8, linewidth=1.5)
                
            ax.set_title(title, fontsize=14, fontweight='bold')
            ax.set_xlabel('Year')
            ax.set_ylabel('Number of Publications')
            ax.tick_params(axis='x', rotation=45)
            ax.grid(True, linestyle='--', alpha=0.5)
            
            # Use a log scale for better visibility, especially in high-activity categories
            if title == 'Category 4: Very High Activity':
                 ax.set_yscale('log')
                 ax.set_ylabel('Number of Publications (Log Scale)')

            # Only add a legend if there are 35 or fewer countries
            if len(countries_to_plot) <= 35:
                ax.legend(fontsize=8, loc='upper left', bbox_to_anchor=(1.0, 1.0))
            else:
                ax.text(0.05, 0.95, f"{len(countries_to_plot)} countries (legend omitted)",
                        transform=ax.transAxes, fontsize=10, verticalalignment='top', 
                        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.9))

        # UPDATED: Add Super Title with N Context
        plt.suptitle(f'Global Publication Trends by Productivity Tier\n(Analysis of $N={total_n_articles}$ Articles across $N={total_n_countries}$ Countries)', 
                     fontsize=18, fontweight='bold', y=1.02)

        plt.tight_layout()
        
        # Save Outputs (PNG + PDF)
        out_path_png = os.path.join(plots_dir, 'Country_Publication_Trends_by_Tier.png')
        out_path_pdf = os.path.join(plots_dir, 'Country_Publication_Trends_by_Tier.pdf')
        
        plt.savefig(out_path_png, dpi=600, bbox_inches='tight')
        plt.savefig(out_path_pdf, format='pdf', bbox_inches='tight')
        
        plt.show()
        print(f"Country trend tier plot saved to: {out_path_png}")

## 20. Country Case Study: Bibliometric Profiling Protocol

Aggregate metrics provide a global overview, but *country-specific dynamics* require a deeper, contextual view.  
This section defines a reusable function, `plot_country_deep_dive`, which produces a compact **2×2 bibliometric dashboard** for any selected country.

---

### What the dashboard contains (as implemented)

**1) National trajectory**  
Annual publications for the selected country (with value annotations).

**2) Peer comparison within the same cohort**  
The country is compared against all other countries in its **productivity category** (Category 1–4).

**3) Benchmark vs. global leaders**  
The country is plotted against the **Top 10 most productive countries**, with a **log y-scale** for readability.

**4) Rank evolution**  
The country’s rank by yearly publication volume across time (lower rank is better; the axis is inverted).

---



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
import seaborn as sns
from matplotlib.ticker import MaxNLocator
import re
from datetime import datetime # Required for current_year calculation

# =========================================================
# 2. Country Deep Dive Analysis Function (FIXED)
# =========================================================

def plot_country_deep_dive(
    country_name, 
    df_exploded, 
    publications_pivot, 
    country_totals, 
    country_categories, 
    cat_lists
):
    """
    Generates a 2x2 deep-dive dashboard for a single specified country.
    """
    
    print(f"\n--- Starting Deep Dive Analysis for: {country_name} ---")
    
    # --- 1. Check if Country Exists ---
    if country_name not in country_totals.index:
        print(f"Error: Country '{country_name}' not found in 'country_totals'.")
        # Find potential matches
        matches = [c for c in country_totals.index if country_name.lower() in c.lower()]
        if matches:
            print(f"Did you mean one of these? {matches}")
        return

    # Unpack category lists
    (cat1_countries, cat2_countries, cat3_countries, cat4_countries) = cat_lists
    
    # --- 2. Get Country Data ---
    country_data = df_exploded[df_exploded['Country_Exploded'] == country_name]
    
    if country_data.empty:
        print(f"No publications found for {country_name}.")
        return

    # Create timeline
    country_timeline = (
        country_data
        .groupby('Publication Year')
        .size()
        .reset_index(name='Publications')
    )
    
    country_total_pubs = country_totals.get(country_name, 0)
    country_category = country_categories.get(country_name, 'Not found')
    
    print(f"{country_name}'s total publications: {country_total_pubs}")
    print(f"{country_name}'s category: {country_category}")

    # --- 3. Create 2x2 Dashboard ---
    fig, axes = plt.subplots(2, 2, figsize=(20, 16))
    
    # --- Plot 1: Country's timeline (with values) ---
    ax1 = axes[0, 0]
    ax1.plot(country_timeline['Publication Year'], country_timeline['Publications'], 
             marker='o', linewidth=3, markersize=8, color='red', label=country_name)
    
    # Add value annotations
    for i, row in country_timeline.iterrows():
        ax1.annotate(f'{int(row["Publications"])}', 
                     (row['Publication Year'], row['Publications']),
                     textcoords="offset points", 
                     xytext=(0,10), 
                     ha='center', 
                     fontsize=10, 
                     fontweight='bold',
                     color='darkred')
    
    ax1.set_title(f'{country_name} Publications Over Time', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Year')
    ax1.set_ylabel('Number of Publications')
    ax1.grid(True, linestyle='--', alpha=0.5)
    ax1.legend()
    ax1.set_ylim(bottom=0) # Start y-axis at 0

    # --- Plot 2: Country vs. Peers in the same category ---
    ax2 = axes[0, 1]
    if country_category != 'Not found':
        if country_category == 'Category 1: Very Low Activity':
            comparison_countries = cat1_countries
        elif country_category == 'Category 2: Low/Moderate Activity':
            comparison_countries = cat2_countries
        elif country_category == 'Category 3: Moderate/High Activity':
            comparison_countries = cat3_countries
        else:
            comparison_countries = cat4_countries
        
        countries_to_plot = [c for c in comparison_countries if c in publications_pivot.columns]
        if countries_to_plot:
            subset = publications_pivot[countries_to_plot]
            # Initialize sns (or just use matplotlib colors if sns is not available)
            try:
                colors = sns.color_palette('husl', n_colors=len(subset.columns))
            except NameError:
                colors = plt.cm.get_cmap('cividis', len(subset.columns))
            
            for i, country in enumerate(subset.columns):
                if country == country_name:
                    ax2.plot(subset.index, subset[country], marker='o', linewidth=3, 
                             markersize=8, label=country, color='red', zorder=10)
                else:
                    ax2.plot(subset.index, subset[country], marker='o', linewidth=1, 
                             markersize=4, label=country, color=colors[i], alpha=0.7)
            
            ax2.set_title(f'{country_name} vs Peers ({country_category})', fontsize=14, fontweight='bold')
            ax2.set_xlabel('Year')
            ax2.set_ylabel('Number of Publications')
            ax2.grid(True, linestyle='--', alpha=0.5)
            if len(countries_to_plot) <= 20:
                ax2.legend(fontsize=8, bbox_to_anchor=(1.05, 1), loc='upper left')

    # --- Plot 3: Country vs Top 10 countries (FIXED) ---
    ax3 = axes[1, 0]
    # top_10_countries is already an Index of country names
    top_10_countries = country_totals.nlargest(10).index 
    countries_to_plot = [c for c in top_10_countries if c in publications_pivot.columns]
    
    if countries_to_plot:
        top_10_subset = publications_pivot[countries_to_plot]
        try:
            colors = sns.color_palette('husl', n_colors=len(top_10_subset.columns))
        except NameError:
            colors = plt.cm.get_cmap('cividis', len(top_10_subset.columns))

        for i, country in enumerate(top_10_subset.columns):
            ax3.plot(top_10_subset.index, top_10_subset[country], marker='o', 
                     linewidth=2, markersize=5, label=country, color=colors[i], alpha=0.8)
        
        # Plot target country in red, if it's not in the top 10
        # FIX: The redundant '.index' is removed.
        if country_name not in top_10_countries and country_name in publications_pivot.columns:
             ax3.plot(publications_pivot.index, publications_pivot[country_name], marker='o', 
                      linewidth=3, markersize=8, label=f"{country_name} (Not Top 10)", color='red', zorder=10)
        
        ax3.set_title(f'{country_name} vs Top 10 Most Active Countries', fontsize=14, fontweight='bold')
        ax3.set_xlabel('Year')
        ax3.set_ylabel('Number of Publications (Log Scale)')
        ax3.grid(True, linestyle='--', alpha=0.5)
        ax3.set_yscale('log') # Log scale is necessary here
        ax3.legend(fontsize=8, bbox_to_anchor=(1.05, 1), loc='upper left')

    # --- Plot 4: Country's ranking over time (with values) ---
    ax4 = axes[1, 1]
    yearly_rankings = []
    for year in publications_pivot.index:
        year_data = publications_pivot.loc[year].sort_values(ascending=False)
        try:
            # Note: We use list(year_data.index).index(country_name)
            country_rank = list(year_data.index).index(country_name) + 1
            yearly_rankings.append({'Year': year, 'Rank': country_rank})
        except ValueError:
            yearly_rankings.append({'Year': year, 'Rank': np.nan}) # Country had 0 pubs this year
    
    ranking_df = pd.DataFrame(yearly_rankings)
    ranking_df = ranking_df.dropna() 
    
    if not ranking_df.empty:
        ax4.plot(ranking_df['Year'], ranking_df['Rank'], marker='o', 
                 linewidth=3, markersize=8, color='red')
        
        for i, row in ranking_df.iterrows():
            ax4.annotate(f'#{int(row["Rank"])}', 
                         (row['Year'], row['Rank']),
                         textcoords="offset points", 
                         xytext=(0,10), 
                         ha='center', 
                         fontsize=10, 
                         fontweight='bold',
                         color='darkred')
        
        ax4.set_title(f'{country_name}\'s Global Rank Over Time', fontsize=14, fontweight='bold')
        ax4.set_xlabel('Year')
        ax4.set_ylabel('Ranking (1 = Most Publications)')
        ax4.grid(True, linestyle='--', alpha=0.5)
        ax4.invert_yaxis() # Lower rank numbers at top
    
    # --- Final Touches ---
    main_title = f"{country_name}'s Contribution and Collaboration Analysis: A Temporal Deep Dive"
    fig.suptitle(main_title, fontsize=18, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    # Save the plot (PNG + PDF)
    # Ensure 'plots_dir' is defined in your environment
    if 'plots_dir' in globals():
        safe_country_name = re.sub(r'[^\w\-]+', '_', country_name) # Make filename safe
        
        plot_path_png = os.path.join(plots_dir, f'{safe_country_name}_Deep_Dive_Analysis.png')
        plot_path_pdf = os.path.join(plots_dir, f'{safe_country_name}_Deep_Dive_Analysis.pdf')
        
        plt.savefig(plot_path_png, dpi=300, bbox_inches='tight')
        plt.savefig(plot_path_pdf, format='pdf', bbox_inches='tight')
        
        print(f"{country_name}-focused analysis saved to: {plot_path_png}")
    else:
        print("Warning: 'plots_dir' variable not found, skipping plot save.")
    
    plt.show()
    
    # --- 4. Print Summary Stats ---
    print("\n" + "="*50)
    print(f"{country_name.upper()} STATISTICS SUMMARY")
    print("="*50)
    print(f"Total publications (appearances): {country_total_pubs}")
    print(f"Productivity Category: {country_category}")
    if not country_timeline.empty:
        print(f"Years with publications: {country_timeline['Publication Year'].min()} - {country_timeline['Publication Year'].max()}")
        print(f"Peak year: {country_timeline.loc[country_timeline['Publications'].idxmax(), 'Publication Year']} ({country_timeline['Publications'].max()} publications)")
        print(f"Average publications per year (in active years): {country_timeline['Publications'].mean():.2f}")
    
    country_global_rank = list(country_totals.sort_values(ascending=False).index).index(country_name) + 1
    print(f"Overall ranking: #{country_global_rank} out of {len(country_totals)} countries")

## 21. Statistical Country Report Generation

To complement the visual dashboard, we generate a detailed tabular report.  
The function `generate_country_deep_dive_report` aggregates all relevant metadata for a given country, creating a comprehensive **"Field Guide"** to that nation's research output.

---
### Data inputs (important)

This function intentionally uses **two** dataframes:

- `df_main` → the *paper-level* master dataframe (`df`)  
  Used for counts, journals, citations, keywords, funding, and timelines.

- `df_affiliations` → the *affiliation-exploded* dataframe (`df_affiliations_final`)  
  Used for institution ranking within the country.

This avoids a common error in country analytics: counting *affiliations* as if they were *papers*.

---

### Report Dimensions

**Provenance**  
- Distribution of database sources represented in the country's output.

**Venues**  
- Top journals and conferences where the country most frequently publishes.

**Morphology**  
- Breakdown of document types (articles, reviews, conference papers, etc.).

**Institutional Landscape**  
- Leading domestic institutions contributing to research output.

**Collaboration**  
- Major international partners collaborating with the country.

**Cognitive Structure**  
- Most frequent research keywords, reflecting thematic focus areas.

**Funding**  
- Key funding bodies and grant agencies supporting the country's research.

---

### Requirements and dependencies

This report assumes earlier steps have been executed:

- `df_affiliations_final` exists (Institution & Country extraction completed)
- `Countries_Extracted_List` exists in `df` (country list column created in country analysis)
- `OriginalTitle` exists in `df` (created in the journal/source normalization step)
- `normalize_doc_type` exists in scope (from Document Type standardization)

If any are missing, the report will either fail fast or skip certain sections.

---

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

# =========================================================
# 2. Country Deep Dive Analysis Function (Definition)
# =========================================================

def generate_country_deep_dive_report(
    country_name, 
    df_main, 
    df_affiliations
):
    """
    Generates a detailed statistical report for a single country, using the
    correct un-exploded dataframe for paper-level counts and the
    affiliation-exploded dataframe for institution-level counts.
    """
    
    print("\n" + "="*80)
    print(f"COMMENCING DEEP DIVE STATISTICAL REPORT FOR: {country_name.upper()}")
    print("="*80)

    # --- 1. Identify Target Papers ---
    # df_affiliations is 'df_affiliations_final'
    country_affils = df_affiliations[df_affiliations['Country_Standardized'] == country_name]
    if country_affils.empty:
        print(f"No affiliations found for '{country_name}'. Aborting report.")
        return

    unique_paper_indices = country_affils['Article_Index'].unique()
    # df_main is 'df'
    df_country_papers = df_main.loc[unique_paper_indices].copy()
    
    if df_country_papers.empty:
        print(f"Found affiliations, but no matching papers in df_main. Aborting.")
        return
        
    print(f"Found {len(df_country_papers)} unique papers with at least one affiliation from {country_name}.")

    # --- 2. Database Breakdown (ENHANCED) ---
    print("\n" + "="*60)
    print("="*60)
    db_counts = df_country_papers['Database'].value_counts()
    
    
    print(f"Total Papers: {len(df_country_papers)}")
    for db, count in db_counts.items():
        percentage = (count / len(df_country_papers)) * 100
        print(f"  - {db:<23}: {count:4d} papers ({percentage:5.1f}%)")

    # --- 3. Journal Analysis (ENHANCED) ---
    print("\n" + "="*60)
    print("2. JOURNAL & DOCUMENT TYPE ANALYSIS")
    print("="*60)
    
    # --- 3a. Overall Top Journals ---
    # This line requires 'OriginalTitle' to be in df_main ('df')
    journal_counts = df_country_papers['OriginalTitle'].value_counts() 
    print(f"\nTop 10 Journals for {country_name} (Overall):")
    print("-" * 50)
    for i, (journal, count) in enumerate(journal_counts.head(10).items(), 1):
        print(f"{i:2d}. {journal:<45} ({count} papers)")


    # --- 3c. Document Types by Database (FIXED) ---
    print(f"\nDocument Types Distribution (Overall):")
    
    # --- FIX: Re-run standardization logic on the filtered dataframe ---
    # This creates 'Document Type Standardized' locally.
    try:
        df_country_papers_doc = df_country_papers[['Database', 'Document Type']].copy()
        df_country_papers_doc['Document Type'] = df_country_papers_doc['Document Type'].fillna('Missing/Incomplete data')
        df_country_papers_doc['Document Types List'] = df_country_papers_doc['Document Type'].str.split('; ')
        doc_types_exploded = df_country_papers_doc.explode('Document Types List').dropna(subset=['Document Types List'])
        
        # Apply the standardization (assumes 'normalize_doc_type' is in global scope from Cell 13)
        doc_types_exploded['Document Type Standardized'] = doc_types_exploded['Document Types List'].apply(normalize_doc_type)
    except NameError:
         print("FATAL: 'normalize_doc_type' function not found (from Cell 13). Cannot analyze document types.")
         doc_types_exploded = pd.DataFrame(columns=['Database', 'Document Type Standardized']) # empty df
    except Exception as e:
         print(f"Error during doc type standardization: {e}")
         doc_types_exploded = pd.DataFrame(columns=['Database', 'Document Type Standardized']) # empty df
    
    doc_type_counts = doc_types_exploded['Document Type Standardized'].value_counts()
    
    total_papers = len(df_country_papers) 
    if total_papers > 0:
        for doc_type, count in doc_type_counts.items():
            percentage = (count / total_papers) * 100 
            print(f"  - {doc_type:<23}: {count:3d} ({percentage:5.1f}%)")

    

    # --- 4. Conference Analysis ---
    print("\n" + "="*60)
    print("3. CONFERENCE PARTICIPATION ANALYSIS")
    print("="*60)
    conf_papers = df_country_papers[df_country_papers['Conference Title'].notna()]
    print(f"\nTotal conference papers: {len(conf_papers)}")
    if len(conf_papers) > 0:
        conf_counts = conf_papers['Conference Title'].value_counts()
        print(f"\nTop 10 Conferences:")
        print("-" * 50)
        for i, (conf, count) in enumerate(conf_counts.head(10).items(), 1):
            print(f"{i:2d}. {conf:<60} ({count} papers)")

    # --- 5. Institutional Analysis (from Affiliation data) ---
    print("\n" + "="*60)
    print("4. TOP AFFILIATED INSTITUTIONS")
    print("="*60)
    institution_counts = country_affils['Institution_Extracted'].value_counts()
    print(f"\nTop 15 Institutions in {country_name} (by affiliation count):")
    print("-" * 50)
    for i, (inst, count) in enumerate(institution_counts.head(15).items(), 1):
        print(f"{i:2d}. {inst:<50} ({count} affiliations)")

    # --- 6. Collaboration Analysis ---
    print("\n" + "="*60)
    print("5. INTERNATIONAL COLLABORATION ANALYSIS")
    print("="*60)
    # This column 'Countries_Extracted_List' should be present in df_main ('df')
    all_collab_countries = df_country_papers['Countries_Extracted_List'].explode().str.strip()
    collab_counts = all_collab_countries.value_counts()
    
    print(f"\nTop 15 Collaboration Partner Countries (by co-authored papers):")
    print("-" * 40)
    i = 1
    for country, count in collab_counts.items():
        if country == country_name:
            continue
        if i > 15:
            break
        print(f"{i:2d}. {country:<30} ({count} collaborations)")
        i += 1

    # --- 7. Research Areas and Keywords ---
    print("\n" + "="*60)
    print("6. RESEARCH AREAS AND KEYWORDS ANALYSIS")
    print("="*60)
    all_keywords = df_country_papers['Author Keywords'].dropna().str.split(';').explode().str.strip().str.lower()
    keyword_counts = all_keywords.value_counts()
    print(f"\nTop 20 Author Keywords in {country_name} Papers:")
    print("-" * 50)
    for i, (keyword, count) in enumerate(keyword_counts.head(20).items(), 1):
        if keyword and keyword != 'nan':
            print(f"{i:2d}. {keyword:<40} ({count} occurrences)")

    # --- 8. Funding Analysis ---
    print("\n" + "="*60)
    print("7. FUNDING ANALYSIS")
    print("="*60)
    funded_papers = df_country_papers[df_country_papers['Funding Orgs'].notna()]
    print(f"\n{country_name} papers with funding information: {len(funded_papers)} ({len(funded_papers)/len(df_country_papers):.1%})")
    if len(funded_papers) > 0:
        funding_orgs = funded_papers['Funding Orgs'].str.split(';').explode().str.strip()
        funding_counts = funding_orgs.value_counts()
        
        print(f"\nTop 10 Funding Organizations:")
        print("-" * 40)
        for i, (org, count) in enumerate(funding_counts.head(10).items(), 1):
            if org and org != 'nan':
                print(f"{i:2d}. {org:<50} ({count} papers)")

    # --- 9. Publication Timeline Summary (FIXED) ---
    print("\n" + "="*60)
    print("8. PUBLICATION TIMELINE SUMMARY")
    print("="*60)
    
    # --- FIX: Use the correct column names from the .agg() call ---
    yearly_summary_agg = df_country_papers.groupby('Publication Year').agg(
        Papers=('Title', 'count'),
        Citations_Sum=('Cited by', 'sum') # Use a different name
    ).round(2)
    
    # Now create the new column
    yearly_summary_agg['Avg Cites per Paper'] = (yearly_summary_agg['Citations_Sum'] / yearly_summary_agg['Papers']).round(2)
    
    print(f"\nYear-by-Year Summary for {country_name}:")
    print("-" * 60)
    print(f"{'Year':<6} {'Papers':<8} {'Total Cites':<12} {'Avg Cites':<10}")
    print("-" * 60)
    
    # Iterate using the correct column names
    for year, row in yearly_summary_agg.iterrows():
        print(f"{year:<6} {int(row['Papers']):<8} {int(row['Citations_Sum']):<12} {row['Avg Cites per Paper']:<10}")
    # --- END OF FIX ---
    
    print(f"\n{'='*80}")
    print(f"END OF {country_name.upper()} DETAILED ANALYSIS")
    print(f"{'='*80}")


---

## 🚨 REQUIRED USER CHANGE (MOST IMPORTANT):
Users must set the country they want the case study/report for.
### Usage example (the only line users normally change)

```python
USER_COUNTRY_TO_ANALYZE = "China"   # e.g., "Romania", "United States", "Germany"
```

---
### Usage example if not interested in a report (the only line users normally change)
```python
USER_COUNTRY_TO_ANALYZE = "None"   
```


In [ ]:
# =========================================================
# 3. EXECUTE the Deep Dive Analysis
# =========================================================

# --- USER: Set the country you want to analyze here ---
USER_COUNTRY_TO_ANALYZE = "Germany" # Example: "United States" or "China"
# ----------------------------------------------------

if 'plot_country_deep_dive' not in locals():
    print("Error: 'plot_country_deep_dive' function not defined. Please run the cell above.")
elif 'df_exploded_final' not in locals():
    print("Error: 'df_exploded_final' not found. Please run your data loading/cleaning cell.")
elif 'publications_pivot' not in locals():
    print("Error: 'publications_pivot' not found. Please run your country pivot table cell.")
elif 'country_totals' not in locals():
    print("Error: 'country_totals' not found. Please run your country totals cell.")
elif 'country_categories' not in locals():
    print("Error: 'country_categories' not found. Please run your country categories cell.")
elif ('cat1_countries' not in locals() or 'cat2_countries' not in locals() or 
      'cat3_countries' not in locals() or 'cat4_countries' not in locals()):
    print("Error: Country category lists (cat1_countries, etc.) are missing.")
else:
    # Pass all the required dataframes to the function
    plot_country_deep_dive(
        country_name=USER_COUNTRY_TO_ANALYZE,
        df_exploded=df_exploded_final,
        publications_pivot=publications_pivot,
        country_totals=country_totals,
        country_categories=country_categories,
        cat_lists=(cat1_countries, cat2_countries, cat3_countries, cat4_countries)
    )

# Check if all the required dataframes from previous cells exist
if 'generate_country_deep_dive_report' not in locals():
    print("Error: 'generate_country_deep_dive_report' function not defined. Please run the cell above.")
elif 'df_final' not in locals() or 'df_affiliations_final' not in locals():
    print("Error: 'df_final' or 'df_affiliations_final' not found. Please run all of Part 1 (Cells 1-8).")
else:
    # Run the report
    generate_country_deep_dive_report(
        country_name=USER_COUNTRY_TO_ANALYZE,
        df_main=df,
        df_affiliations=df_affiliations_final
    )

## 18. Final Summary: Main Information about the Dataset

This final step creates a **grand summary table** for the cleaned corpus.  
It recomputes core descriptors (timespan, growth, citations, sources) and extends them with authorship, keyword, collaboration, and document-type structure.

This table is designed to be directly reusable in the **Dataset / Descriptive Statistics** section of a scientometrics paper.

---

### Metrics included (as implemented)

**Document statistics**
- Documents (N)
- Timespan (min year, max year)
- Average document age
- Annual growth rate (% CAGR)

**Source & reference structure**
- Unique sources (prefers `OriginalTitle`, falls back to `Source Title`)
- Average citations per document
- Total references (intellectual base size)

**Keywords**
- Unique Keywords Plus
- Unique Author Keywords

**Authors**
- Unique authors
- Mean co-authors per document
- Single-authored documents

**Collaboration**
- Percent of internationally co-authored papers  
  (computed from `Countries_Extracted` list diversity)

**Document morphology**
- Normalized document-type breakdown (Article / Review / Conference / …)

---

### Output

- Saves the main table to: `final_summary_metrics.csv`
- Displays:
  - the summary dataframe (`results_df`)
  - optionally a document type breakdown table

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
from IPython.display import display

print("Generating Final Summary Table (Using Pre-calculated Metrics)...")

# ============================
# etup & Data Gathering
# ============================
summary_data = {}
current_year = datetime.now().year

# --- A. General Dataset Metrics ---
summary_data['Total Documents'] = len(df)

# Years & Growth
if 'Publication Year' in df.columns:
    years = df['Publication Year'].dropna().astype(int)
    if not years.empty:
        min_year, max_year = years.min(), years.max()
        summary_data['Timespan'] = f"{min_year} - {max_year}"
        summary_data['Average Document Age'] = (current_year - years).mean()
        
        # CAGR (Compound Annual Growth Rate)
        # We calculate this directly from the years series for speed
        n_years = max_year - min_year
        count_start = len(years[years == min_year])
        count_end = len(years[years == max_year])
        
        if n_years > 0 and count_start > 0:
            cagr = ((count_end / count_start) ** (1 / n_years) - 1) * 100
            summary_data['Annual Growth Rate %'] = cagr
        else:
             summary_data['Annual Growth Rate %'] = np.nan

# --- B. Content & Impact ---
if 'Source Title' in df.columns:
    # Prefer source_stats if available (more accurate processing), else raw unique
    if 'source_stats' in locals() and not source_stats.empty:
        summary_data['Journals/Sources'] = len(source_stats)
    else:
        summary_data['Journals/Sources'] = df['Source Title'].nunique()

if 'Keywords Plus' in df.columns:
    # Fast approximation of unique keywords without full explode if possible
    summary_data['Keywords Plus (ID)'] = df['Keywords Plus'].str.split(';').explode().str.strip().nunique()
    
if 'Author Keywords' in df.columns:
    summary_data['Author Keywords (DE)'] = df['Author Keywords'].str.split(';').explode().str.strip().nunique()

if 'Cited Reference Count' in df.columns:
    summary_data['Total References'] = df['Cited Reference Count'].sum()

# Impact
if 'Cited by' in df.columns:
    summary_data['Average Citations per Doc'] = df['Cited by'].mean()
    summary_data['Total Citations'] = df['Cited by'].sum()

# --- C. Authors & Collaboration ---
# 1. Author Count: Use 'author_stats' if it exists (most accurate)
if 'author_stats' in locals() and not author_stats.empty:
    summary_data['Authors'] = len(author_stats)
elif 'Author Full Names' in df.columns:
    summary_data['Authors'] = df['Author Full Names'].str.split(';').explode().str.strip().nunique()

# 2. Co-Authorship
if 'Author Full Names' in df.columns:
    # Quick calc on the main dataframe
    n_authors_per_doc = df['Author Full Names'].astype(str).str.split(';').apply(len)
    summary_data['Co-Authors per Doc'] = n_authors_per_doc.mean()
    summary_data['Single-authored Docs'] = (n_authors_per_doc == 1).sum()

# 3. International Collaboration
# We check if the 'Country_Classification' column was created in the Country Analysis cell
if 'Country_Classification' in df.columns:
    total_docs = len(df)
    mcp_count = (df['Country_Classification'] == 'MCP').sum()
    summary_data['International Collab %'] = (mcp_count / total_docs) * 100
elif 'Countries_Extracted' in df.columns:
    # Fallback: Fast check without full logic
    def is_intl(x):
        try:
            return len(set(str(x).replace("[","").replace("]","").replace("'","").split(','))) > 1
        except: return False
    summary_data['International Collab %'] = df['Countries_Extracted'].apply(is_intl).mean() * 100

# ============================
# 2. Formatting & Output
# ============================

# Convert to DataFrame
df_summary = pd.DataFrame(list(summary_data.items()), columns=['Description', 'Value'])

# Formatting Function
def nice_format(x):
    if isinstance(x, (int, np.integer)):
        return f"{x:,}"
    if isinstance(x, float):
        if np.isnan(x): return "N/A"
        return f"{x:,.2f}"
    return str(x)

df_summary['Value'] = df_summary['Value'].apply(nice_format)

print("\n### Final Dataset Metrics ###")
display(df_summary)

# Save
results_path = os.path.join(results_dir, 'final_main_summary.csv')
df_summary.to_csv(results_path, index=False)
print(f"Saved to: {results_path}")

# --- Optional: Fast Document Type Summary (if column exists) ---
if 'Document Type' in df.columns:
    print("\n### Document Types Snapshot ###")
    # Clean just for display
    doc_types = df['Document Type'].astype(str).str.lower().str.split(';').explode().str.strip().value_counts().head(5)
    doc_types_df = doc_types.reset_index()
    doc_types_df.columns = ['Document Type', 'Count']
    display(doc_types_df)

## 19. Core Impact Analyses: Articles, Authors, and Sources

This section moves from broad descriptive statistics to more granular impact analysis. We will identify the most significant articles and authors, and analyze the structural properties of the journal landscape.

### 1. Most Cited Articles
This is the most straightforward impact analysis. The script ranks all articles in the corpus by their total citation count (`Cited by`) and presents the Top 10 in a table. A full, ranked list of all articles is saved to `all_articles_ranked_by_citations.csv`.

### 2. Most Influential Authors
This analysis differs from the previous "Author Productivity" (which ranked by *number of papers*). This script ranks authors by their **scientific impact**, using a combination of:
1.  **h-index**: Calculated for each author based on their publications *within this corpus*.
2.  **Total Citations**: The sum of all citations received by an author's papers in this corpus.
3.  **Total Publications**: Used as a tie-breaker.

The script "explodes" the author list for each paper, aggregates all citations and papers for each unique author, calculates their h-index, and then ranks them. A full, ranked list is saved to `all_authors_ranked_by_impact.csv`.

### 3. Bradford's Law of Scattering
This is a classic bibliometric law that describes how articles on a specific topic are "scattered" across different journals. It predicts that a small "core" of journals will publish a large portion (e.g., 1/3) of all articles, while a much larger number of journals will form a "periphery" publishing only a few articles each.

This script:
1.  Calculates the number of articles per journal.
2.  Calculates the cumulative sum of articles.
3.  Divides the journals into three zones, each containing approximately 1/3 of the total articles.
4.  Generates the classic Bradford plot (log of journal rank vs. cumulative articles), which shows the "S-curve" of scattering.
5.  Saves the plot and the underlying data.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from IPython.display import display

# ============================
# 0. Setup & Helper Checks
# ============================
print("Generating Specialized Bibliometric Tables (Using Pre-computed Metrics)...")

# Ensure we have the output directory
if 'results_dir' not in locals(): results_dir = '.'
if 'plots_dir' not in locals(): plots_dir = '.'

# Helper to format tables nicely (reuse if exists, else define simple version)
if 'print_markdown_table' not in locals():
    def print_markdown_table(title, df, columns):
        print(f"\n### {title}")
        print(df[columns].to_markdown(index=False))

# ============================
# 1. Most Cited Articles
# ============================
print("\n--- 1. Most Cited Articles ---")

if 'Cited by' in df.columns and 'Title' in df.columns:
    # Use the main dataframe 'df' directly
    top_cited = df.nlargest(10, 'Cited by').copy()
    
    # Helper to extract first author safely (without re-importing ast)
    top_cited['First_Author'] = top_cited['Authors'].astype(str).apply(lambda x: x.split(';')[0].strip())
    
    # Shorten title for display
    top_cited['Short_Title'] = top_cited['Title'].apply(lambda x: str(x)[:70] + '...' if len(str(x)) > 70 else str(x))
    top_cited['Rank'] = range(1, 11)
    
    cols_display = ['Rank', 'Short_Title', 'First_Author', 'Publication Year', 'Cited by']
    
    print_markdown_table("Top 10 Most Cited Articles", top_cited, cols_display)

    # Save full list
    results_path = os.path.join(results_dir, 'all_articles_ranked_by_citations.csv')
    df[['Title', 'Authors', 'Publication Year', 'Cited by', 'DOI']].sort_values(by='Cited by', ascending=False).to_csv(results_path, index=False)
    print(f"Full ranked list saved to: {results_path}")
else:
    print("Skipping (Missing 'Cited by' or 'Title' column)")

# ============================
# 2. Most Influential Authors (Impact)
# ============================
print("\n--- 2. Most Influential Authors (Impact) ---")

# Prefer pre-calculated 'author_stats' from the Author Analysis cell
if 'author_stats' in locals() and not author_stats.empty:
    print("Using pre-computed 'author_stats' DataFrame.")
    # Sort by H-index (if available) or Total Citations
    sort_col = 'h_index' if 'h_index' in author_stats.columns else 'Total_Citations'
    
    # We need to ensure h_index exists. If it wasn't calculated in the previous cell, we skip it or calc it here.
    # Assuming the previous Author Analysis cell (Cell 30) ran, 'author_stats' has 'Total_Citations' but maybe not 'h_index'
    # if you didn't add it there.
    
    # Let's double check if we need to calculate H-index on the fly using the helper function
    if 'h_index' not in author_stats.columns:
        # If missing in summary, we calculate it quickly here
        print("Calculating H-index for top authors...")
        # Explode again just for this calculation if needed
        df_auth = df[['Authors', 'Cited by']].dropna()
        df_auth['Auth_List'] = df_auth['Authors'].astype(str).str.split(';')
        df_exp = df_auth.explode('Auth_List')
        df_exp['Auth_List'] = df_exp['Auth_List'].str.strip()
        df_exp = df_exp[df_exp['Auth_List'] != '']
        
        # Calculate H-index
        h_indices = df_exp.groupby('Auth_List')['Cited by'].apply(calculate_h_index).to_dict()
        author_stats['h_index'] = author_stats['Author'].map(h_indices)
        
    top_authors = author_stats.sort_values(by=['h_index', 'Total_Citations'], ascending=False).head(10)
    
    cols = ['Author', 'h_index', 'Total_Citations', 'Total_Papers']
    # Filter only columns that actually exist
    cols = [c for c in cols if c in top_authors.columns]
    
    print_markdown_table("Top 10 Authors by H-Index", top_authors, cols)
    
    # Save
    path = os.path.join(results_dir, 'top_authors_impact.csv')
    top_authors.to_csv(path, index=False)
    print(f"Author impact list saved to: {path}")

else:
    print("Warning: 'author_stats' not found. Please run the Author Analysis cell first.")

# ============================
# 3. Bradford's Law of Scattering
# ============================
print("\n--- 3. Bradford's Law of Scattering ---")

# Prefer 'source_stats' from Source Analysis cell
if 'source_stats' in locals() and not source_stats.empty:
    # Use the pre-cleaned OriginalTitle
    bradford_df = source_stats[['OriginalTitle', 'Number_of_Publications']].copy()
    bradford_df = bradford_df.rename(columns={'Number_of_Publications': 'Count'})
elif 'OriginalTitle' in df.columns:
    bradford_df = df['OriginalTitle'].value_counts().reset_index()
    bradford_df.columns = ['OriginalTitle', 'Count']
else:
    bradford_df = pd.DataFrame()

if not bradford_df.empty:
    # 1. Sort Descending
    bradford_df = bradford_df.sort_values(by='Count', ascending=False).reset_index(drop=True)
    
    # 2. Cumulative Calculations
    bradford_df['Rank'] = bradford_df.index + 1
    bradford_df['Cumulative_Articles'] = bradford_df['Count'].cumsum()
    bradford_df['Log_Rank'] = np.log(bradford_df['Rank'])
    
    total_articles = bradford_df['Cumulative_Articles'].max()
    total_sources = len(bradford_df)
    
    # 3. Define Zones (1/3, 2/3 thresholds)
    t1 = total_articles / 3
    t2 = total_articles * 2 / 3
    
    # 4. Count Zones
    # Zone 1 (Core)
    z1 = bradford_df[bradford_df['Cumulative_Articles'] <= t1]
    n1 = len(z1) # Number of journals
    a1 = z1['Count'].sum() # Number of articles
    
    # Zone 2 (Middle)
    z2 = bradford_df[(bradford_df['Cumulative_Articles'] > t1) & (bradford_df['Cumulative_Articles'] <= t2)]
    n2 = len(z2)
    a2 = z2['Count'].sum()
    
    # Zone 3 (Periphery)
    z3 = bradford_df[bradford_df['Cumulative_Articles'] > t2]
    n3 = len(z3)
    a3 = z3['Count'].sum()
    
    # Stats
    print(f"Total Sources (N): {total_sources} | Total Articles: {total_articles}")
    print(f"Zone 1 (Core): {n1} sources ({a1} articles)")
    print(f"Zone 2: {n2} sources ({a2} articles)")
    print(f"Zone 3: {n3} sources ({a3} articles)")
    if n1 > 0:
        print(f"Bradford Ratio: 1 : {n2/n1:.1f} : {n3/n1:.1f}")

    # --- Plotting (Journal Style) ---
    plt.figure(figsize=(10, 6))
    ax = plt.gca()
    
    # The Curve (Black line)
    ax.plot(bradford_df['Rank'], bradford_df['Cumulative_Articles'], color='#333333', linewidth=2, label='Source Distribution')
    
    # Bradford's requirement is a log X axis
    ax.set_xscale('log')
    
    # Zone markers (Dashed lines)
    # Zone 1 Boundary
    ax.axhline(t1, color='#666666', linestyle='--', linewidth=1, alpha=0.8)
    ax.axvline(n1, color='#666666', linestyle='--', linewidth=1, alpha=0.8)
    
    # Zone 2 Boundary
    ax.axhline(t2, color='#666666', linestyle='--', linewidth=1, alpha=0.8)
    ax.axvline(n1 + n2, color='#666666', linestyle='--', linewidth=1, alpha=0.8)
    
    # Annotations (Zone Text)
    # Core
    ax.text(1.1, t1/2, f'Zone 1 (Core)\n{n1} Sources', color='black', fontsize=10, fontweight='bold', va='center')
    
    # Middle (Positioned dynamically)
    mid_rank_2 = n1 + (n2/2) if n2 > 0 else n1
    ax.text(mid_rank_2, (t1+t2)/2, f'Zone 2\n{n2} Sources', color='black', fontsize=10, fontweight='bold', ha='center', va='center')
    
    # Periphery
    mid_rank_3 = (n1 + n2) * 1.5
    ax.text(mid_rank_3, (t2+total_articles)/2, f'Zone 3 (Periphery)\n{n3} Sources', color='black', fontsize=10, fontweight='bold', va='center')

    # Titles and Labels
    ax.set_title(f"Bradford's Law of Scattering: Source Concentration\n(Total $N={total_sources}$ Sources, $N={total_articles}$ Articles)", fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Source Rank (Log Scale)', fontsize=12)
    ax.set_ylabel('Cumulative Number of Articles', fontsize=12)
    
    # Aesthetics
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(True, linestyle=':', alpha=0.4, which="both") # Grid on both major/minor ticks for log scale

    plt.tight_layout()
    
    # Save
    plot_path_png = os.path.join(plots_dir, 'bradford_law_scattering.png')
    plot_path_pdf = os.path.join(plots_dir, 'bradford_law_scattering.pdf')
    plt.savefig(plot_path_png, dpi=600, bbox_inches='tight')
    plt.savefig(plot_path_pdf, format='pdf', bbox_inches='tight')
    
    print(f"Bradford plot saved to: {plot_path_png}")
    plt.show()
    
    # Save Data
    bradford_df.to_csv(os.path.join(results_dir, 'bradford_law_data.csv'), index=False)
else:
    print("Insufficient data for Bradford's Law analysis.")

## 20. Advanced Visualizations: Trends and Landscapes

This final set of analyses provides three powerful, high-level visualizations that are common in bibliometric reports. They help identify emerging research, the effect of team size, and the strategic position of journals.

### 1. "Hot Paper" Analysis
This analysis identifies emerging, high-impact research. It filters the dataset to include only "recent" articles (defined as those published in the last 2 full years + the current year). It then ranks these recent articles by their citation count, presenting a Top 10 table of "Hot Papers."

### 2. Collaboration Impact ("More is Better?")
This analysis explores the relationship between team size and citation impact. It answers the question: "Do papers with more authors tend to get cited more?"
1.  The script calculates the number of authors for each paper.
2.  It groups articles into team size categories (e.g., "1", "2", "3", "4", "5", "6+").
3.  It then calculates and plots the *average citation count* for each category, allowing for a quick visual comparison.

### 3. The Journal Landscape ("Three Fields" Plot)
This is a classic "strategic diagram" used in bibliometrics to map the journal landscape. It plots all journals on a 2D grid:
* **X-axis**: Productivity (Total Number of Publications) [Log Scale]
* **Y-axis**: Impact (Average Citations per Paper) [Log Scale]
* **Bubble Size**: A third metric, typically `h-index`.

This visualization helps to classify journals into four quadrants:
* **Top-Right**: High productivity, high impact (core, high-prestige journals).
* **Top-Left**: Low productivity, high impact (niche, high-prestige journals).
* **Bottom-Right**: High productivity, low impact (e.g., proceedings, rapid-publication journals).
* **Bottom-Left**: Low productivity, low impact (niche or peripheral journals).



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime
from IPython.display import display
import textwrap

# Ensure helper function exists if not already defined
if 'safe_author_split' not in locals():
    def safe_author_split(x):
        if pd.isna(x): return []
        return [a.strip() for a in str(x).split(';') if a.strip()]

#Helper for markdown tables
if 'print_markdown_table' not in locals():
    def print_markdown_table(title, df, columns):
        print(f"\n### {title}")
        print(df[columns].to_markdown(index=False))
        
# ============================
# 1. "Hot Paper" Analysis (Emerging Fronts)
# ============================
print("\n--- 1. Identification of Emerging Research Fronts (Hot Papers) ---")

required_cols = ['Publication Year', 'Cited by', 'Title', 'Authors', 'OriginalTitle']

if 'df' not in locals() or df.empty:
    print("Error: 'df' is missing. Run previous steps.")
elif not all(col in df.columns for col in required_cols):
    print(f"Warning: Missing columns {required_cols}. Skipping Hot Paper analysis.")
else:
    # Define "Recent" as last 2 full years + current year
    recent_year = datetime.now().year - 3
    
    df_recent = df[df['Publication Year'] >= recent_year].copy()
    
    if df_recent.empty:
        print(f"No articles found published since {recent_year}.")
    else:
        # Sort by citations (Impact)
        df_recent_sorted = df_recent.sort_values(by='Cited by', ascending=False)
        
        # Metadata cleaning
        df_recent_sorted['First Author'] = df_recent_sorted['Authors'].apply(lambda x: safe_author_split(x)[0] if safe_author_split(x) else 'N/A')
        df_recent_sorted['Title (Short)'] = df_recent_sorted['Title'].apply(lambda x: (x[:70] + '...') if len(str(x)) > 70 else str(x))
        df_recent_sorted['Rank'] = np.arange(1, len(df_recent_sorted) + 1)
        
        # Select columns
        table_cols = ['Rank', 'Title (Short)', 'First Author', 'OriginalTitle', 'Publication Year', 'Cited by']
        top_10_hot = df_recent_sorted[table_cols].head(10)

        # Output
        print_markdown_table(
            f"Top 10 Highly Cited Papers (Published {recent_year}–Present)",
            top_10_hot,
            columns=table_cols
        )
        
        # Save
        results_path = os.path.join(results_dir, 'emerging_research_fronts_hot_papers.csv')
        df_recent_sorted.to_csv(results_path, index=False)
        print(f"Full list of recent papers saved to: {results_path}")

# ============================
# 2. Team Science: Collaboration Impact
# ============================
print("\n--- 2. Team Science Analysis: Collaboration Size vs. Impact ---")

if 'Authors' in df.columns and 'Cited by' in df.columns:
    # Calculate Team Size
    df['Author_Count'] = df['Authors'].apply(safe_author_split).apply(len)
    
    # Filter valid
    df_collab_impact = df[df['Author_Count'] > 0].copy()
    
    # Calculate N for Title
    total_n_team = len(df_collab_impact)
    
    # Categorize Team Sizes
    # We group 6+ into one bucket to handle "Big Science" papers cleanly
    max_authors_display = 6
    df_collab_impact['Team_Size_Category'] = df_collab_impact['Author_Count'].apply(
        lambda x: str(x) if x < max_authors_display else f"{max_authors_display}+"
    )
    
    # Enforce order
    category_order = [str(i) for i in range(1, max_authors_display)] + [f"{max_authors_display}+"]
    
    # Aggregate Impact
    impact_by_team_size = df_collab_impact.groupby('Team_Size_Category')['Cited by'].agg(
        Average_Citations='mean',
        Number_of_Articles='count'
    ).reindex(category_order).reset_index()

    # Save
    results_path = os.path.join(results_dir, 'collaboration_impact_team_size.csv')
    impact_by_team_size.to_csv(results_path, index=False)

    # --- Plotting (Cividis Style) ---
    plt.figure(figsize=(10, 6))
    ax = plt.gca()
    
    # Use Cividis Palette (Perceptually Uniform, Academic Standard)
    sns.barplot(
        data=impact_by_team_size,
        x='Team_Size_Category',
        y='Average_Citations',
        palette='cividis',
        edgecolor='black',
        linewidth=0.5
    )
    
    # UPDATED TITLE WITH N
    ax.set_title(f'Impact of Team Size on Citation Performance\n(Analysis of $N={total_n_team}$ Articles)', fontsize=14, fontweight='bold', pad=15)
    
    ax.set_xlabel('Number of Authors per Article', fontsize=11)
    ax.set_ylabel('Average Citations per Article', fontsize=11)
    
    # Annotate
    for p in ax.patches:
        height = p.get_height()
        if pd.notna(height):
            ax.annotate(
                f"{height:.1f}", 
                (p.get_x() + p.get_width() / 2., height), 
                ha='center', va='bottom', fontsize=10, fontweight='bold', color='#333333',
                xytext=(0, 3), textcoords='offset points'
            )
        
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    
    # Save
    plot_path_png = os.path.join(plots_dir, 'collaboration_impact_team_size.png')
    plot_path_pdf = os.path.join(plots_dir, 'collaboration_impact_team_size.pdf')
    plt.savefig(plot_path_png, dpi=600, bbox_inches='tight')
    plt.savefig(plot_path_pdf, format='pdf', bbox_inches='tight')
    
    print(f"Team science plot saved to: {plot_path_png}")
    plt.show()
else:
    print("Skipping collaboration impact (missing data).")

# ============================
# 3. Bibliometric Landscape (Productivity vs Impact)
# ============================
print("\n--- 3. Source Dynamics: The Bibliometric Landscape ---")

if 'source_stats' not in locals() or source_stats.empty:
    print("Warning: 'source_stats' missing. Run Journal Analysis (Cell 38) first.")
else:
    # Prepare Data
    source_stats['Avg_Citations'] = source_stats.apply(
        lambda row: row['Total_Citations'] / row['Number_of_Publications'] if row['Number_of_Publications'] > 0 else 0,
        axis=1
    )
    plot_data = source_stats[source_stats['Number_of_Publications'] > 0].copy()
    
    # Calculate N for Title
    total_n_sources = len(plot_data)

    # --- Plotting ---
    plt.figure(figsize=(12, 8))
    ax = plt.gca()
    
    # Scatter: Volume vs Impact (Log-Log)
    # Size = H-index (Prestige)
    # Color = H-index (using Cividis for consistency)
    scatter = ax.scatter(
        x=plot_data['Number_of_Publications'],
        y=plot_data['Avg_Citations'],
        s=plot_data['h_index'] * 10, # Scale bubble
        c=plot_data['h_index'],
        cmap='cividis',
        alpha=0.7,
        edgecolors='white',
        linewidth=0.5
    )
    
    # Log Scales for Power Law distributions
    ax.set_xscale('log')
    ax.set_yscale('log')
    
    # UPDATED TITLE WITH N
    ax.set_title(f'Bibliometric Landscape: Source Productivity vs. Impact Efficiency\n($N={total_n_sources}$ Sources)', fontsize=14, fontweight='bold', pad=20)
    
    ax.set_xlabel('Productivity (Total Publications) [Log Scale]', fontsize=11)
    ax.set_ylabel('Impact (Avg Citations per Paper) [Log Scale]', fontsize=11)
    
    # Labels for Top Journals (by H-index)
    labels_to_plot = plot_data.nlargest(10, 'h_index')
    texts = []
    for i, row in labels_to_plot.iterrows():
        label = textwrap.fill(row['OriginalTitle'], 20) 
        texts.append(ax.text(
            row['Number_of_Publications'], 
            row['Avg_Citations'], 
            label, 
            fontsize=8, fontweight='medium'
        ))
    
    # Adjust Labels
    if 'adjust_text' in globals():
        try: adjust_text(texts, arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))
        except: pass

    # Colorbar
    cbar = plt.colorbar(scatter)
    cbar.set_label('Local H-Index (Accumulated Prestige)', rotation=270, labelpad=15)

    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(True, linestyle='--', alpha=0.3, which='both')

    plt.tight_layout()
    
    # Save
    plot_path_png = os.path.join(plots_dir, 'journal_landscape_cividis.png')
    plot_path_pdf = os.path.join(plots_dir, 'journal_landscape_cividis.pdf')
    plt.savefig(plot_path_png, dpi=600, bbox_inches='tight')
    plt.savefig(plot_path_pdf, format='pdf', bbox_inches='tight')
    
    print(f"Landscape plot saved to: {plot_path_png}")
    plt.show()
    

# ----------------------------------------
# PART 3: THEMATIC ANALYSIS

## 21. Thematic Analysis: NLTK Setup

This section begins **Part 3: Thematic Analysis**, which aims to extract prominent concepts from the corpus using text fields such as:

- **Title**
- **Abstract**
- **Author Keywords**
- **Keywords Plus**

Before we can preprocess text and compute n-grams / topics, we must install and configure **NLTK (Natural Language Toolkit)**.

---

### What this setup cell does (and why)

1. **Downloads required NLTK resources**  
   Typical components used later include:
   - `wordnet` → lemmatization  
   - `stopwords` → default noise removal  
   - tokenization and POS tagging resources (`punkt*`, taggers)

2. **Kaggle/Cloud workaround: unzip `wordnet` if needed**  
   In some hosted environments, `wordnet` is present as `wordnet.zip`.  
   The code detects that case and unzips it manually so NLTK can load it correctly.

3. **SSL workaround**  
   In some setups, NLTK downloads can fail due to SSL certificate verification errors.  
   This cell includes a standard workaround so downloads can succeed.

---

### ✅ What you should change

This cell is **Kaggle-optimized**. If you are not running on Kaggle, update the paths:

- Kaggle path used in code:  
  `'/kaggle/working/'`

- Recommended alternatives:
  - **Google Colab:** use default NLTK downloader location (often no unzip needed)
  - **Local machine:** you may prefer a persistent folder, e.g. `./nltk_data`

Example edits (conceptual):

```python
# Kaggle (current)
nltk.download('wordnet', download_dir='/kaggle/working/')

# Local (example)
nltk.download('wordnet', download_dir='./nltk_data')
nltk.data.path.append('./nltk_data')
```

---

### Troubleshooting notes (common issues)

- If you see an error like “Resource punkt not found”:  
  Your NLTK version may require `punkt` rather than `punkt_tab`.  
  You can safely add:

```python
nltk.download('punkt')
```

- If `averaged_perceptron_tagger_eng` fails on older NLTK:  
  use the older resource name:

```python
nltk.download('averaged_perceptron_tagger')
```

---

In [ ]:
## 3. Thematic Analysis ##
import nltk
import subprocess

# NLTK data (like 'wordnet') is sometimes zipped in cloud environments.
# This check finds it and manually unzips it if needed.
try:
    nltk.data.find('wordnet.zip')
except:
    print("Downloading 'wordnet'...")
    nltk.download('wordnet', download_dir='/kaggle/working/')
    command = "unzip /kaggle/working/corpora/wordnet.zip -d /kaggle/working/corpora"
    subprocess.run(command.split())
    nltk.data.path.append('/kaggle/working/')
    print("'wordnet' unzipped and path appended.")
    
from nltk.corpus import wordnet

# This block is a common workaround for SSL certificate errors
# that can occur during NLTK downloads (e.g., [SSL: CERTIFICATE_VERIFY_FAILED]).
import ssl
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

# Download other required NLTK components
print("Downloading NLTK components (stopwords, omw-1.4, etc.)...")
nltk.download('stopwords')
nltk.download('omw-1.4')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

print("NLTK setup complete.")

## 22. Text Preprocessing and N-gram Extraction


## 🚨 REQUIRED USER CHANGE (MOST IMPORTANT): `USER_SEARCH_STRING`

**You must replace `USER_SEARCH_STRING` with your exact database query.**  
The script automatically extracts your query terms and removes them from the analysis so your results are not dominated by the words you used to retrieve the corpus.

Why this matters:
- If you searched for `"exoplanet" OR "hot jupiter"`, those terms will appear *everywhere* in the corpus.
- If not removed, your “top terms” will be mostly your own query, not emergent themes.

✅ Minimum requirement:
- Paste the same query string you used in **Scopus/WoS** (or your source), including quoted phrases.

---


This is the core of the thematic analysis. This cell defines all parameters, functions, and processes required to convert the raw article text (titles, abstracts, keywords) into a clean, analyzable set of high-frequency terms and phrases (n-grams).

### 1. Thematic Analysis Parameters
This section is the primary configuration area for the user.
* **`USER_SEARCH_STRING`**: **This is the most important field for you to change.** You must paste your *exact* database search query here. The script uses this to automatically identify your search terms (e.g., "planet formation", "spatial distribution") and remove them from the analysis. This prevents your results from being cluttered with the terms you searched for.
* **`MASTER NOISE WORD LISTS`**: These are custom lists of generic, non-substantive words (e.g., "study", "result", "method", "elsevier", "propose") that are not useful for thematic analysis.

### 2. Noise List Preparation
This step combines all noise lists (the default NLTK stopwords, all custom lists, and the auto-extracted terms from your `USER_SEARCH_STRING`) into a single, master list called `ALL_NOISE_TERMS`. This is done for processing efficiency.

### 3. Text Preprocessing Function
The `custom_preprocess_text` function is the engine of this analysis. For each article, it:
1.  **Tokenizes** the text (splits it into individual words).
2.  **Tags Part-of-Speech (POS)**: Identifies each word as a noun, verb, adjective, etc.
3.  **Lemmatizes**: Reduces words to their root form (e.g., "using", "uses" -> "use"; "studies" -> "study"). It intelligently lemmatizes verbs *as verbs* (using the `'v'` tag) and other words as nouns.
4.  **Filters**: Removes any word that is in the `ALL_NOISE_TERMS` master list.

### 4. Process and Combine Text
This step applies the `custom_preprocess_text` function to a new column that combines the text from each article's 'Title', 'Abstract', 'Author Keywords', and 'Keywords Plus'.

### 5. N-gram Extraction and Word Clouds
This is the final output. TThe script uses `CountVectorizer` with:

- `ngram_range=(1,4)`  
- **scientometric** `min_df`:  
  `max(5, int(N * 0.01))`  
  (appear in ≥ 1% of docs or at least 5 docs)
- `max_df=0.90` to remove near-universal “field stopwords”
- a hyphen-preserving token pattern:  
  `token_pattern=r"(?u)\b\w[\w-]*\w\b"`

Then it applies additional “clean-map” filters:
- duplicate-token filter (`"star star"`)
- acronym redundancy filter (`"artificial intelligence ai"`)
- explicit publisher/artifact phrase blocklist (`BLOCKLIST_PHRASES`)
:
* **Unigrams** (single words, e.g., "attack")
* **Bigrams** (two-word phrases, e.g., "zero-day attack")
* **Trigrams** (three-word phrases, e.g., "denial of service")

It then prints the top n-grams, saves the full lists to the `results_dir`, and generates word clouds for each n-gram type.

### ✅ What you should change (practical checklist)

#### A) **Replace `USER_SEARCH_STRING` (required)**
Paste your query here:

```python
USER_SEARCH_STRING = """
<PASTE YOUR DATABASE QUERY HERE>
"""
```

#### B) Verify column availability (may require edits)
The combine step assumes these columns exist:

- `Title`
- `Abstract`
- `Author Keywords`
- `Keywords Plus`

If your dataset uses different names, edit `combine_text()` accordingly.

Quick check you can run (optional):

```python
required = ['Title', 'Abstract', 'Author Keywords', 'Keywords Plus']
missing = [c for c in required if c not in df.columns]
print("Missing columns:", missing)
```

#### C) Adjust domain-specific noise (recommended)
- Add publisher artifacts or generic research boilerplate terms you see in your top n-grams.
- For heavily technical domains, you may want to remove terms like:
  - “dataset”, “experiment”, “evaluation”, “performance”, etc.

#### D) Adjust frequency thresholds (optional)
If your corpus is small, `min_df` may be too strict.

- Current rule: `min_df = max(5, 1% of N)`
- For small corpora (<300 docs), you might set:
  - `min_df=2` or `min_df=3`

---

### 🔍 Sanity checks (highly recommended)

#### 1) Confirm your query terms are being removed
After `lemmatized_search_terms` is created:

```python
print(sorted(list(lemmatized_search_terms))[:30])
```

If your query contains complex syntax and you see garbage tokens, simplify parsing by relying more on quoted phrases.

#### 2) Inspect processed examples
The notebook prints:

- `Title`
- `Processed_Text`

If `Processed_Text` is empty for many rows, you are likely:
- missing required columns, or
- over-filtering via noise lists / `min_df`.

---

In [ ]:
import nltk
import subprocess
import ssl
import os
import re
import numpy as np  # Added numpy explicitly
from collections import Counter
import pandas as pd
from wordcloud import WordCloud
import seaborn as sns
import matplotlib.pyplot as plt
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer

# ============================
# 1. THEMATIC ANALYSIS PARAMETERS
# ============================
df_final = df.copy()
N = len(df_final)  # Calculate total number of articles
print(f"Processing Corpus Size: N={N} articles")

# --- 1a. Paste your database search string here ---
# These terms will be *removed* from the analysis,
# as they are trivial (they were used to find the corpus).
USER_SEARCH_STRING = """
( "exoplanet*" OR "extrasolar planet*" OR "planetary transit*" OR "radial velocity" OR "direct imaging" OR "microlensing" ) 
    AND 
    ( "machine learning" OR "deep learning" OR "artificial intelligence" OR "neural network*" 
    OR "convolutional neural network*" OR "recurrent neural network*" OR "transformer model" OR "random forest*" OR "support vector machine*" 
    OR "gaussian process*" OR "xgboost" OR "gradient boosting" OR "generative adversarial network*" OR "bayesian inference" )
"""

# --- 1b. MASTER NOISE WORD LISTS (GENERIC-ONLY) ---
# General academic terms to be removed.
GENERIC_NOISE = {
    # General terms
    "article", "study", "paper", "analysis", "method", "result", "conclusion",
    "introduction", "background", "data", "approach", "model", "system", "review",
    "survey", "case", "example", "framework", "research", "issue", "problem", "purpose",
    "abstract", "objective", "methodology", "discussion", "finding", "limitation",
    "implication", "contribution", "scope", "field", "literature", "chapter", "report",
    "document", "project", "activity", "process", "information",
    
    # Common adjectives/adverbs
    "high", "low", "large", "small", "recent", "new", "old", "current", "future", "previous",
    "different", "difference", "significant", "significantly", "significantly_higher",
    "several", "various", "multiple", "certain", "individual", "particular",
    "good", "better", "best", "effective", "efficient", "novel", "general", "specific",
    "main", "major", "important", "possible", "potential"
}

# Bibliometric, Publisher, and Licensing Noise
BIBLIO_NOISE = {
    # Databases/Tools
    "scopus", "elsevier", "web of science", "wos", "clarivate", "pubmed",
    "journal of informetrics", "scientometrics", "bibliometric", "bibliometrix",
    "vosviewer", "citespace", "incites", "researcherid", "orcid", "ieee", 
    "springer", "wiley", "mdpi", "database", "index", "publication", "author",
    "journal", "conference", "proceeding", "university", "institute", "society",
    
    # Publisher/Copyright Artifacts
    "nature", "switzerland", "ag", "license", "exclusive", "taylor", "francis", 
    "group", "oxford", "press", "behalf", "reserved", "right", "publishing", "ltd",
    "copyright", "homepage", "visit", "download", "online", "open", "access",
    "creative", "commons", "attribution", "distribution", "reproduction", "medium",
    "american", "japan", "japanese", "chinese", "european"
}

# LaTeX and Mathematical Artifacts
LATEX_NOISE = {
    "amsbsy", "mathrsfs", "upgreek", "minimal", "amsmath", "wasysym", 
    "amsfonts", "amssymb", "document", "mathrm", "mathbf", "mathcal", 
    "text", "begin", "end", "equation", "array", "comment"
}

SKLEARN_STOPWORDS = {
    "whether", "also", "still", "every", "across", "through", "however", "although",
    "maybe", "could", "would", "may", "without", "among", "amongst", "various",
    "well", "within", "often", "mainly", "rather", "either", "neither", "another",
    "already", "always", "due", "first", "second", "third", "one", "two", "three",
    "fig", "figure", "table", "et", "al", "etal", "inc", "ltd", "corp", "via", "around"
}

# Non-substantive verbs
VERB_NOISE = {
    "use", "using", "used", "show", "provide", "propose", "suggest", "present",
    "train", "training", "trained", "implement", "implementing", "implemented",
    "perform", "performing", "performed", "apply", "applying", "applied",
    "develop", "developing", "developed", "allow", "base", "include", "require",
    "achieve", "identify", "evaluate", "compare", "demonstrate", "find", "found",
    "describe", "consider", "investigate", "explore", "aim", "obtain", "report",
    # Added per request
    "change", "contain", "occur", "observe", "exist", "remain", "seem", "appear",
    "indicate", "associate", "relate", "consist", "compose", "comprise", "increase", "measure",
    "take"
}

# Explicit Blocklist for Licensing Artifacts that survive stopword removal
BLOCKLIST_PHRASES = {
    "nature switzerland ag", "license nature switzerland", "exclusive license nature", 
    "taylor francis", "taylor francis group", "oxford press"
}


In [ ]:
# ============================
# Noise List Preparation
# ============================
print("Preparing master noise removal list...")
lemmatizer = WordNetLemmatizer()
nltk_stopwords = set(stopwords.words('english'))

def parse_search_string(query, lemmatizer):
    """Extracts keywords from the search string to exclude them."""
    query = query.lower()
    quoted_phrases = re.findall(r'"([^"]*)"', query)
    no_quotes = re.sub(r'"[^"]*"', ' ', query)
    operators = {'and', 'or', 'not', 'title-abs-key', 'title', 'abs', 'key', 'all', 'tak', 'near', 'pre', 'w/'}
    potential_words = re.split(r'[\s()\[\]{}]_?"\']+', no_quotes)
    loose_keywords = [k for k in potential_words if k and k.isalpha() and k not in operators and not k.isdigit()]
    
    all_terms = quoted_phrases + loose_keywords
    lemmatized_terms = set()
    for term in all_terms:
        lemmatized_words = [lemmatizer.lemmatize(word) for word in term.split()]
        lemmatized_terms.add(" ".join(lemmatized_words))
        lemmatized_terms.update(lemmatized_words)
    return lemmatized_terms

# --- Lemmatize Lists ---
lemmatized_search_terms = parse_search_string(USER_SEARCH_STRING, lemmatizer)
lemmatized_biblio_noise = {lemmatizer.lemmatize(w) for w in BIBLIO_NOISE}
lemmatized_generic_noise = {lemmatizer.lemmatize(w) for w in GENERIC_NOISE}
lemmatized_latex_noise = {lemmatizer.lemmatize(w) for w in LATEX_NOISE}
lemmatized_sklearn_noise = {lemmatizer.lemmatize(w) for w in SKLEARN_STOPWORDS}
lemmatized_verb_noise = {lemmatizer.lemmatize(w, 'v') for w in VERB_NOISE}

ALL_NOISE_TERMS = nltk_stopwords.union(
    lemmatized_search_terms, 
    lemmatized_biblio_noise, 
    lemmatized_generic_noise,
    lemmatized_latex_noise,
    lemmatized_sklearn_noise,
    lemmatized_verb_noise
)

print(f"Total unique noise terms: {len(ALL_NOISE_TERMS)}")

# ============================
# Text Preprocessing Function
# ============================

def custom_preprocess_text(text):
    if pd.isnull(text): return ''
    text = text.lower()
    
    # Custom replacements
    text = re.sub(r'\bartificial intelligence\b', 'AI', text)
    text = re.sub(r'\bai\b', 'AI', text)
    
    tokens = word_tokenize(text)
    tagged_tokens = pos_tag(tokens)

    final_tokens = []
    for word, tag in tagged_tokens:
        if not word.isalpha() and word != 'AI': continue

        # POS-aware Lemmatization
        pos_char = 'v' if tag.startswith('VB') else 'n'
        lemma = lemmatizer.lemmatize(word, pos_char)
        
        if lemma not in ALL_NOISE_TERMS:
            final_tokens.append(lemma)
    
    return ' '.join(final_tokens)

# ============================
# Process & Combine Text
# ============================
print("Combining and processing text...")

def combine_text(row):
    return ' '.join([
        str(row['Keywords Plus']) if pd.notnull(row['Keywords Plus']) else '',
        str(row['Author Keywords']) if pd.notnull(row['Author Keywords']) else '',
        str(row['Abstract']) if pd.notnull(row['Abstract']) else '',
        str(row['Title']) if pd.notnull(row['Title']) else ''
    ])

df_final['Combined_Raw_Text'] = df_final.apply(combine_text, axis=1)
df_final['Processed_Text'] = df_final['Combined_Raw_Text'].apply(custom_preprocess_text)

print("Text processing complete.")
display(df_final[['Title', 'Processed_Text']].head())


# =========================================================
# N-gram Extraction and Advanced Filtering 
# =========================================================
print("\n--- 5. N-gram Frequency and Word Cloud Analysis ---")

print("\n--- 5. N-gram Frequency and Word Cloud Analysis ---")

# --- Scientometric Parameter Setup ---
# Dynamic Minimum Frequency: Term must appear in at least 1% of docs or 5 docs (whichever is higher)
# This filters out "tail" noise which is critical for clean maps (VOSviewer style logic).
MIN_DF_SCIENTOMETRIC = max(5, int(N * 0.01)) 

print(f"Applying Scientometric filters:")
print(f" - Minimum Document Frequency: {MIN_DF_SCIENTOMETRIC} (approx 1% of corpus)")
print(f" - Maximum Document Frequency: 90% (removes field-specific stopwords)")
print(f" - Token Pattern: Preserving hyphens (e.g., 'co-occurrence', 'h-index')")

ngram_vectorizer = CountVectorizer(
    ngram_range=(1, 4), # Limit to Trigrams. 4-grams are usually noise/boilerplate.
    max_df=0.90,        # Stricter ceiling: Remove terms appearing in >90% of docs
    min_df=MIN_DF_SCIENTOMETRIC, # Robustness threshold
    # CRITICAL UPDATE: Regex to capture hyphenated words (e.g., "co-citation")
    token_pattern=r"(?u)\b\w[\w-]*\w\b" 
)

ngram_dtm = ngram_vectorizer.fit_transform(df_final['Processed_Text'])
ngram_features = ngram_vectorizer.get_feature_names_out()
ngram_counts = np.array(ngram_dtm.sum(axis=0)).flatten()

ngram_freq_df = pd.DataFrame({'ngram': ngram_features, 'count': ngram_counts})
ngram_freq_df = ngram_freq_df.sort_values(by='count', ascending=False)

# --- ADVANCED FILTERS ---

def has_duplicate_tokens(ngram):
    """Removes 'star star' or 'planet planet'."""
    tokens = ngram.split()
    return len(tokens) != len(set(tokens))

def is_acronym_redundancy(ngram):
    """
    General detector for patterns like:
    - "business intelligence bi" (trigram)
    - "service oriented soa" (trigram)
    
    Logic: Checks if the last word is an acronym of the preceding words.
    """
    parts = ngram.split()
    if len(parts) < 2: return False # Only applies to bigrams/trigrams
    
    # Check if last word is the acronym (e.g., "artificial intelligence ai")
    last_word = parts[-1]
    preceding_phrase = parts[:-1]
    
    # Construct potential acronym from preceding words (first letter of each)
    # We strip 'AI' -> 'ai' already, so just check characters
    constructed_acronym = "".join([w[0] for w in preceding_phrase])
    
    if last_word == constructed_acronym:
        return True
        
    return False
    
# Apply the filters defined previously
ngram_freq_df = ngram_freq_df[~ngram_freq_df['ngram'].apply(has_duplicate_tokens)]
ngram_freq_df = ngram_freq_df[~ngram_freq_df['ngram'].apply(is_acronym_redundancy)]
ngram_freq_df = ngram_freq_df[~ngram_freq_df['ngram'].isin(BLOCKLIST_PHRASES)]

# Recalculate n_tokens after vectorization (since hyphens might change token counts)
ngram_freq_df['n_tokens'] = ngram_freq_df['ngram'].apply(lambda x: len(x.split()))

print("Top 30 combined n-grams (Scientometric Standard):")
display(ngram_freq_df.head(30))

# Separate into individual DataFrames
unigrams = ngram_freq_df[ngram_freq_df['n_tokens'] == 1]
bigrams = ngram_freq_df[ngram_freq_df['n_tokens'] == 2]
trigrams = ngram_freq_df[ngram_freq_df['n_tokens'] == 3]
quadgrams = ngram_freq_df[ngram_freq_df['n_tokens'] == 4]

print("\n--- Top 20 Unigrams ---")
display(unigrams.head(20))
print("\n--- Top 20 Bigrams ---")
display(bigrams.head(20))
print("\n--- Top 20 Trigrams ---")
display(trigrams.head(20))
print("\n--- Top 20 Quadgrams ---")
display(quadgrams.head(20))

# Save the frequency lists to the results directory
unigrams.to_csv(os.path.join(results_dir, 'unigram_frequency.csv'), index=False)
bigrams.to_csv(os.path.join(results_dir, 'bigram_frequency.csv'), index=False)
trigrams.to_csv(os.path.join(results_dir, 'trigram_frequency.csv'), index=False)
print("N-gram frequency lists saved to 'results_dir'.")

def create_wordcloud(freq_df, title, filename):
    freq_dict = dict(zip(freq_df['ngram'], freq_df['count']))
    if not freq_dict: return
        
    wc = WordCloud(
        width=1600, height=800, 
        background_color='white', 
        colormap='cividis', # Cividis Palette
        collocations=False
    )
    wc.generate_from_frequencies(freq_dict)
    
    plt.figure(figsize=(12, 6))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    
    # Title with N count
    full_title = f"{title} (N={N} articles)"
    plt.title(full_title, fontsize=20)
    plt.tight_layout()
    
    plot_path_png = os.path.join(plots_dir, f"{filename}.png")
    plot_path_pdf = os.path.join(plots_dir, f"{filename}.pdf")
    plt.savefig(plot_path_png, dpi=300)
    plt.savefig(plot_path_pdf, format='pdf')
    print(f"Word cloud saved: {filename}")
    plt.show()

create_wordcloud(unigrams.head(150), "Top 150 Unigrams", "wordcloud_unigrams")
create_wordcloud(bigrams.head(150), "Top 150 Bigrams", "wordcloud_bigrams")
create_wordcloud(trigrams.head(150), "Top 150 Trigrams", "wordcloud_trigrams")
create_wordcloud(ngram_freq_df.head(150), "Top 150 Combined N-grams", "wordcloud_combined_ngrams")

## 23. Thematic Analysis: Topic Model Evaluation (Finding 'k')

This cell performs one of the most important steps in topic modeling: finding the **optimal number of topics (k)**. Instead of just guessing, this script systematically tests a range of `k` values (e.g., from 3 to 10 topics) for two different algorithms:

1.  **LDA (Latent Dirichlet Allocation)**: A probabilistic model that assumes each document is a mixture of various topics.
2.  **NMF (Non-negative Matrix Factorization)**: An algebraic model that factorizes the document-term matrix.

### Methodology
For each value of `k`, the script trains a temporary model and calculates its quality using two primary metrics:

* **Perplexity (for LDA)**: A measure of how well the model predicts unseen data. **Lower is better**.
* **Coherence Score (c_v)**: A more human-interpretable metric that measures how semantically similar the top words in a given topic are. **Higher is better**.


This script generates plots for both LDA and NMF. We will typically select the `k` value that corresponds to the **highest coherence score**, as this indicates the most human-interpretable and semantically meaningful set of topics. The script automatically identifies this value as `best_k` and `best_nmf_k` for use in the next step.

---

### ✅ What you should change

#### A) Choose your `k_values` range (important)
Current:

```python
k_values = [3, 4, 5, 6, 7, 8, 9, 10]
```

Guidelines:
- **Small corpora (N < 500):** test 3–10 (current default is fine)
- **Medium corpora (500–5000):** test 5–20
- **Large corpora:** test 10–40 (but consider runtime)

#### B) Vectorizer thresholds (optional)
This step uses `CountVectorizer(max_df=0.95, min_df=2)` on **already-clean** `Processed_Text`.

If your corpus is noisy or huge:
- increase `min_df` to reduce dimensionality
- reduce `max_df` to remove near-universal tokens

Example:

```python
vectorizer = CountVectorizer(max_df=0.90, min_df=5)
```

#### C) Runtime controls (optional)
Topic evaluation can be slow. You can reduce cost by:
- testing fewer k values
- limiting coherence top words
- switching LDA to `learning_method='online'` (faster, sometimes noisier)

---

### 🧪 Interpretation: how to pick a final k

- Prefer the `k` with **highest coherence** (most interpretable topics).
- Treat perplexity as supporting evidence, not the final decision.
- If coherence is flat across a range, pick the smaller `k` (simpler model).

A common outcome:
- LDA coherence peaks at a moderate value (e.g., k=6–8)
- NMF may peak at a different value (e.g., k=10)

That’s normal — pick the model + k that produces the most meaningful topics for your research question.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display
import os
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.corpora.dictionary import Dictionary
from gensim.models import CoherenceModel
from sklearn.decomposition import LatentDirichletAllocation, NMF

# ============================
# Setup
# ============================
print("Setting up thematic analysis...")

# --- NEW: Calculate N for Titles ---
N = len(df_final)
print(f"Topic Modeling Corpus Size: N={N} articles")

# Define color palettes for consistent visualization
# Using colors compatible with Cividis for the lines
# Dark Blue/Purple (Start of Cividis) -> Yellow (End of Cividis)
cividis_colors = ["#00204d", "#414d6b", "#7c7b78", "#b9ac70", "#ffe945"]

COLORS = {
    'perplexity': cividis_colors[0],  # Dark Blue
    'coherence': cividis_colors[3],   # Muddy Yellow
    'nmf_line': cividis_colors[1],    # Lighter Blue
    'neutral_dark': '#333333',
    'neutral_mid': '#666666',
    'neutral_light': '#f8f8f8',
}

# Ensure plots_dir exists (defined in previous steps)
if 'plots_dir' not in locals():
    plots_dir = '/kaggle/working/output/Plots/'
    os.makedirs(plots_dir, exist_ok=True)
    print(f"Created plots directory: {plots_dir}")

# ============================
# Prepare Data for Topic Modeling
# ============================
print("Preparing data for topic modeling...")

if 'Processed_Text' not in df_final.columns:
    print("Error: 'Processed_Text' column not found. Run the n-gram analysis step first.")
else:
    # --- This is the key change ---
    # We use the 'Processed_Text' column which is ALREADY fully cleaned.
    # Therefore, we DO NOT pass a stop_words list to the vectorizer.
    
    print("Using pre-processed text from the previous step. No additional stopwords needed.")
    vectorizer = CountVectorizer(
        max_df=0.95, # Ignore terms in > 95% of docs
        min_df=2     # Ignore terms in < 2 docs
        # stop_words=... is correctly REMOVED
    )
    # --- End of change ---

    # Create the Document-Term Matrix (DTM)
    dtm = vectorizer.fit_transform(df_final['Processed_Text'])
    feature_names = vectorizer.get_feature_names_out()

    # Prepare texts for coherence calculation (needs list of lists of tokens)
    split_texts = [text.split() for text in df_final['Processed_Text'] if text.strip()]
    gensim_dict = Dictionary(split_texts)
    gensim_corpus = [gensim_dict.doc2bow(text) for text in split_texts]

    # ============================
    # Helper Function for Coherence
    # ============================
    def compute_coherence_sklearn(model, feature_names, texts, gensim_dict, gensim_corpus, no_top_words=20):
        """Compute coherence score for sklearn topic models (LDA, NMF)"""
        topics = []
        # Get the top N words for each topic from the model's components
        for idx, topic_vals in enumerate(model.components_):
            top_indices = topic_vals.argsort()[:-no_top_words - 1:-1]
            topics.append([feature_names[i] for i in top_indices])
        
        # Calculate coherence using Gensim's CoherenceModel
        cm = CoherenceModel(
            topics=topics,
            texts=texts,
            dictionary=gensim_dict,
            corpus=gensim_corpus,
            coherence='c_v' # 'c_v' is a standard, robust coherence measure
        )
        return cm.get_coherence()

    # ============================
    # Evaluate LDA Model
    # ============================
    k_values = [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20] # Range of topic numbers to test
    lda_perplexities = []
    lda_coherences = []

    print("Testing different numbers of topics for LDA...")
    for k in k_values:
        print(f"  - Evaluating LDA with {k} topics")
        lda_temp = LatentDirichletAllocation(
            n_components=k,
            random_state=42,
            learning_method='batch' # 'batch' is slower but often more accurate
        )
        lda_temp.fit(dtm)
        # Calculate perplexity (lower is better)
        perp = lda_temp.perplexity(dtm)
        lda_perplexities.append(perp)
        # Calculate coherence (higher is better)
        coh = compute_coherence_sklearn(lda_temp, feature_names, split_texts, gensim_dict, gensim_corpus)
        lda_coherences.append(coh)

    # --- Create dual-axis plot ---
    fig, ax1 = plt.subplots(figsize=(10, 6))
    
    # Use Cividis-compatible colors
    perplexity_color = COLORS['perplexity']
    coherence_color = COLORS['coherence']

    ax1.set_xlabel('Number of Topics (k)', fontsize=12, fontweight='medium')
    ax1.set_ylabel('Perplexity (lower is better)', color=perplexity_color, fontsize=12, fontweight='medium')
    ax1.plot(k_values, lda_perplexities, marker='o', color=perplexity_color, linewidth=2)
    ax1.tick_params(axis='y', labelcolor=perplexity_color)

    # Create a second y-axis sharing the same x-axis
    ax2 = ax1.twinx()
    ax2.set_ylabel('Coherence (c_v) (higher is better)', color=coherence_color, fontsize=12, fontweight='medium')
    ax2.plot(k_values, lda_coherences, marker='s', color=coherence_color, linewidth=2)
    ax2.tick_params(axis='y', labelcolor=coherence_color)

    if lda_coherences:
        # Find the 'k' with the highest coherence
        best_k_idx = lda_coherences.index(max(lda_coherences))
        best_k = k_values[best_k_idx]

        # Add a vertical line and annotation for the best 'k'
        ax1.axvline(x=best_k, color=COLORS['neutral_mid'], linestyle='--', alpha=0.7)
        plt.annotate(f'Optimal k = {best_k}',
                       xy=(best_k, lda_coherences[best_k_idx]),
                       xytext=(10, -20),
                       textcoords='offset points',
                       arrowprops=dict(arrowstyle='->', color=COLORS['neutral_mid']),
                       bbox=dict(boxstyle='round,pad=0.3', fc=COLORS['neutral_light'], ec=COLORS['neutral_mid']),
                       fontsize=10)

        print(f"Selected {best_k} as the optimal number of topics based on coherence")
        print("LDA Coherences:", dict(zip(k_values, lda_coherences)))
    else:
        print("Warning: Could not calculate LDA coherences.")
        best_k = k_values[0] # Default to the lowest k

    # Updated Title with N
    plt.title(f'LDA Topic Model Evaluation Metrics (N={N} articles)', fontsize=16, fontweight='bold', pad=20)
    plt.grid(axis='both', alpha=0.3, linestyle='--')
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, 'LDA_Evaluation_Metrics.png'), dpi=600, bbox_inches='tight')
    plt.show()

    # ============================
    # Evaluate NMF Model
    # ============================
    nmf_coherences = []
    print("\nEvaluating NMF models...")
    for k in k_values:
        print(f"  - Testing NMF with {k} topics")
        nmf_temp = NMF(n_components=k, random_state=42, max_iter=500, init='nndsvda')
        nmf_temp.fit(dtm)
        coh = compute_coherence_sklearn(nmf_temp, feature_names, split_texts, gensim_dict, gensim_corpus)
        nmf_coherences.append(coh)

    # Plot NMF Coherence (single y-axis)
    plt.figure(figsize=(10, 6))
    plt.plot(k_values, nmf_coherences, marker='o', color=COLORS['nmf_line'],
             linewidth=2, markersize=8)
    plt.xlabel('Number of Topics (k)', fontsize=12, fontweight='medium')
    plt.ylabel('Coherence (c_v)', fontsize=12, fontweight='medium')
    # Updated Title with N
    plt.title(f'NMF Model Coherence by Number of Topics (N={N} articles)', fontsize=16, fontweight='bold', pad=20)
    plt.grid(True, alpha=0.3, linestyle='--')

    if nmf_coherences:
        # Find the 'k' with the highest coherence
        best_nmf_k_idx = nmf_coherences.index(max(nmf_coherences))
        best_nmf_k = k_values[best_nmf_k_idx]
        plt.axvline(x=best_nmf_k, color=COLORS['neutral_mid'], linestyle='--', alpha=0.7)
        plt.annotate(f'Best NMF k = {best_nmf_k}',
                       xy=(best_nmf_k, nmf_coherences[best_nmf_k_idx]),
                       xytext=(10, -20),
                       textcoords='offset points',
                       arrowprops=dict(arrowstyle='->', color=COLORS['neutral_mid']),
                       bbox=dict(boxstyle='round,pad=0.3', fc=COLORS['neutral_light'], ec=COLORS['neutral_mid']),
                       fontsize=10)
        print("NMF Coherences:", dict(zip(k_values, nmf_coherences)))
    else:
        print("Warning: Could not calculate NMF coherences.")
        best_nmf_k = k_values[0] # Default to the lowest k

    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, 'NMF_Coherence.png'), dpi=600, bbox_inches='tight')
    plt.show()

## 24. Thematic Analysis: Final Model Training and Topic Evolution

After selecting the optimal number of topics (`best_k` for LDA and `best_nmf_k` for NMF), we now **train the final models** and convert the topic model into **interpretable, time-aware results**.

This cell does four things:

1. **Train Final Models (LDA + NMF)**  
   - Fits one LDA model using `best_k` and one NMF model using `best_nmf_k`.  
   - These are trained on the **same Document-Term Matrix** (`dtm`) created in the evaluation step.

2. **Display Topics (Human Interpretation Layer)**  
   - Prints the **Top N words per topic** (default: 15).  
   - This is where you “name” topics (informally) based on their most representative terms.

3. **Assign Topics to Articles (Document-level Classification)**  
   - Adds a `Dominant_Topic` column (topic with highest probability).  
   - Stores the full probability vector in `Topic_Distribution` for downstream analysis (co-occurrence, impact, specialization).

4. **Topic Evolution Over Time (Trend Dynamics)**  
   - Creates **uniform time bins** (default: 5-year bins).  
   - Produces two complementary plots:
     - **Proportional evolution**: average topic proportion per time slice.
     - **Absolute volume evolution**: count of documents per dominant topic per time slice.

---

### Prerequisites

This cell assumes the following already exist (from previous thematic steps):

- `df_final` with `Processed_Text`
- `dtm` + `feature_names` (vectorizer output)
- `best_k` and `best_nmf_k` (selected from evaluation plots)

If any of these are missing, rerun:

- **Cell 62** (Text preprocessing and n-gram processing)
- **Cell 65** (Topic model evaluation)

---

### What users must change

**Mandatory: none** (if `best_k` / `best_nmf_k` were computed and you accept the automatically selected values).

**Strongly recommended adjustments (common):**

1. **Time bin width**  
   If your dataset is short (e.g., < 10 years), 5-year bins may be too coarse.

   ```python
   # Change this:
   BIN_DURATION = 5

   # Examples:
   BIN_DURATION = 1   # yearly bins (fine-grained)
   BIN_DURATION = 10  # decade bins (very stable trends)
   ```

2. **Topic readability (`no_top_words`)**  
   Use 10–20 depending on how dense topics are.

   ```python
   no_top_words = 15  # default
   ```

3. **Manual override of k (optional)**  
   If the coherence curve is flat or ambiguous, you may choose a “round number” for interpretability.

   ```python
   # Optional: override the automatically selected values
   best_k = 8
   best_nmf_k = 8
   ```

---

### Example: Interpreting and naming topics (optional helper)

You can create a manual mapping after inspection:

```python
# Example mapping after inspecting the printed topic words
TOPIC_NAMES = {
    1: "Atmospheric characterization",
    2: "Detection methods and surveys",
    3: "Orbital dynamics and formation",
    # ...
}

df_final["Dominant_Topic_Name"] = df_final["Dominant_Topic"].map(TOPIC_NAMES).fillna("Unlabeled")
```

This is especially useful for paper writing and figures.

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from matplotlib import colormaps
from sklearn.decomposition import LatentDirichletAllocation, NMF
import os

# ============================
# Final LDA Model & Topic Evolution
# ============================

# --- Helper: Dynamic Color Generator ---
def get_cividis_colors(n):
    """Generates n distinct colors from the Cividis colormap."""
    cmap = colormaps['cividis']
    # Sample n colors evenly from 0.0 to 0.9 (avoiding the very brightest yellow at 1.0)
    return [cmap(i) for i in np.linspace(0, 0.9, n)]

def display_topics(model, feature_names, no_top_words):
    """Format topics from a sklearn model for clear presentation"""
    descriptions = []
    for idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[:-no_top_words - 1:-1]
        top_words = [feature_names[i] for i in top_indices]
        descriptions.append(f"Topic {idx+1}: " + " | ".join(top_words))
    return descriptions

# --- Train final LDA model ---
num_topics = best_k # Use best_k found from coherence
print(f"\nTraining final LDA model with {num_topics} topics...")
lda = LatentDirichletAllocation(
    n_components=num_topics,
    random_state=42,
    learning_method='batch',
    max_iter=50
)
lda.fit(dtm)
lda_output = lda.transform(dtm)

# --- Display LDA topics ---
no_top_words = 15
lda_topic_descriptions = display_topics(lda, feature_names, no_top_words)
print("\nLDA Topic Modeling Results:")
for desc in lda_topic_descriptions:
    print(desc)

# --- Assign topics to main DataFrame ---
df['Dominant_Topic'] = np.argmax(lda_output, axis=1) + 1
df['Topic_Distribution'] = list(lda_output)

if 'df_final' in locals() and isinstance(df_final, pd.DataFrame):
    df_final['Dominant_Topic'] = np.argmax(lda_output, axis=1) + 1
    df_final['Topic_Distribution'] = list(lda_output)

# --- Analyze topic trends over time ---
print("\nAnalyzing topic trends over time...")

N_articles = len(df)
topic_colors = get_cividis_colors(num_topics) # Generate palette for LDA

if 'Publication Year' not in df.columns or df['Publication Year'].dropna().empty:
    print("Warning: 'Publication Year' not found. Skipping topic evolution plots.")
else:
    # --- UNIFORM TIME SLICE GENERATION ---
    print("Creating uniform 5-year time slices...")
    BIN_DURATION = 5  
    
    min_year = int(df['Publication Year'].min())
    max_year = int(df['Publication Year'].max())

    def get_uniform_time_slice(year):
        year = int(year)
        start_year = (year // BIN_DURATION) * BIN_DURATION
        end_year = start_year + BIN_DURATION - 1
        return f"{start_year}-{end_year}"

    df['Time_Slice'] = df['Publication Year'].apply(get_uniform_time_slice)
    
    # Create full list of time slices for continuous axis
    start_bin_year = (min_year // BIN_DURATION) * BIN_DURATION
    end_bin_year = (max_year // BIN_DURATION) * BIN_DURATION
    
    time_slices = []
    for year in range(start_bin_year, end_bin_year + BIN_DURATION, BIN_DURATION):
        time_slices.append(f"{year}-{year + BIN_DURATION - 1}")
    
    print(f"  Created {len(time_slices)} uniform 5-year slices from {time_slices[0]} to {time_slices[-1]}")
    
    # --- Data Prep for Plotting ---
    lda_output_df = pd.DataFrame(lda_output, index=df.index, columns=[f"Topic {i+1}" for i in range(num_topics)])
    df_with_props = pd.concat([df[['Time_Slice']], lda_output_df], axis=1)
    
    avg_props_by_slice = df_with_props.groupby('Time_Slice').mean(numeric_only=True)
    avg_props_by_slice = avg_props_by_slice.reindex(time_slices, fill_value=0)

    # --- Plot 1: Topic Proportion Evolution ---
    plt.figure(figsize=(12, 8))
    for i, topic in enumerate([f"Topic {i+1}" for i in range(num_topics)]):
        plt.plot(avg_props_by_slice.index, avg_props_by_slice[topic],
                 marker='o',
                 linewidth=2,
                 color=topic_colors[i], # Cividis color
                 label=topic)
                 
    plt.xlabel("Time Period (5-Year Bins)", fontsize=12, fontweight='medium')
    plt.ylabel("Average Topic Proportion", fontsize=12, fontweight='medium')
    plt.title(f"Evolution of Research Topics Over Time (Proportional)\n(N={num_topics} topics, N={N_articles} articles)", fontsize=16, fontweight='bold', pad=20)
    plt.legend(title="Topics", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=45)
    plt.grid(alpha=0.3, linestyle='--')
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "Topic_Evolution_Proportion.png"), dpi=600, bbox_inches='tight')
    plt.savefig(os.path.join(plots_dir, "Topic_Evolution_Proportion.pdf"), format='pdf', bbox_inches='tight')
    plt.show()

    # --- Plot 2: Topic Publication Counts ---
    topic_trends_count = df.groupby(['Time_Slice', 'Dominant_Topic']).size().reset_index(name='Count')
    topic_trends_pivot = topic_trends_count.pivot(index='Time_Slice', columns='Dominant_Topic', values='Count').fillna(0)
    topic_trends_pivot = topic_trends_pivot.reindex(time_slices, fill_value=0)

    plt.figure(figsize=(14, 8))
    for t in range(1, num_topics + 1):
        if t in topic_trends_pivot.columns:
            plt.plot(
                topic_trends_pivot.index,
                topic_trends_pivot[t],
                marker='o',
                linewidth=2,
                label=f'Topic {t}',
                color=topic_colors[t-1] # Cividis color
            )
    plt.title(f'Publication Counts by Topic Over Time\n(N={num_topics} topics, N={N_articles} articles)', fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Time Period (5-Year Bins)', fontsize=12, fontweight='medium')
    plt.ylabel('Number of Publications', fontsize=12, fontweight='medium')
    plt.legend(title='Topics', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=45)
    plt.grid(alpha=0.3, linestyle='--')
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, 'Topic_Publication_Trends.png'), dpi=600, bbox_inches='tight')
    plt.savefig(os.path.join(plots_dir, 'Topic_Publication_Trends.pdf'), format='pdf', bbox_inches='tight')
    plt.show()

# ============================
# Final NMF Model (for comparison)
# ============================
print("\nTraining NMF model for comparison...")
num_topics_nmf = best_nmf_k
nmf = NMF(n_components=num_topics_nmf, random_state=42, max_iter=500, init='nndsvda')
W = nmf.fit_transform(dtm) 

nmf_topic_descriptions = display_topics(nmf, feature_names, no_top_words)
print("\nNMF Topic Modeling Results:")
for desc in nmf_topic_descriptions:
    print(desc)

# Assign NMF topics
df['NMF_Dominant_Topic'] = np.argmax(W, axis=1) + 1
if 'df_final' in locals() and isinstance(df_final, pd.DataFrame):
    df_final['NMF_Dominant_Topic'] = np.argmax(W, axis=1) + 1

# Generate colors for NMF (might be different N than LDA)
nmf_colors = get_cividis_colors(num_topics_nmf)

# Plot NMF evolution
if 'Publication Year' in df.columns and not df['Publication Year'].dropna().empty:
    nmf_topic_trends = df.groupby(['Time_Slice', 'NMF_Dominant_Topic']).size().reset_index(name='Count')
    nmf_topic_trends_pivot = nmf_topic_trends.pivot(index='Time_Slice', columns='NMF_Dominant_Topic', values='Count').fillna(0)
    nmf_topic_trends_pivot = nmf_topic_trends_pivot.reindex(time_slices, fill_value=0)

    plt.figure(figsize=(14, 8))
    for t in range(1, num_topics_nmf + 1):
        if t in nmf_topic_trends_pivot.columns:
            plt.plot(
                nmf_topic_trends_pivot.index,
                nmf_topic_trends_pivot[t],
                marker='o',
                linewidth=2,
                label=f'Topic {t}',
                color=nmf_colors[t-1] # Cividis color
            )
    plt.title(f'NMF Topic Distribution Over Time\n(N={num_topics_nmf} topics, N={N_articles} articles)', fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Time Period (5-Year Bins)', fontsize=12, fontweight='medium')
    plt.ylabel('Number of Publications', fontsize=12, fontweight='medium')
    plt.legend(title='NMF Topics', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=45)
    plt.grid(alpha=0.3, linestyle='--')
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, 'NMF_Topic_Trends.png'), dpi=600, bbox_inches='tight')
    plt.savefig(os.path.join(plots_dir, 'NMF_Topic_Trends.pdf'), format='pdf', bbox_inches='tight')
    plt.show()

print("\nTopic modeling and trend analysis complete.")

## 25. Thematic Analysis: Topic Impact, Co-occurrence, and Specialization

This final set of thematic analyses uses the LDA topic modeling results to answer more advanced questions about the research field:

1.  **Topic Impact Analysis (MNCS)**:
    * **Question**: Are all topics created equal? Which topics are associated with higher-impact research?
    * **Method**: Calculates the **Mean Normalized Citation Score (MNCS)** for all articles belonging to each topic. An MNCS > 1.0 means that topic is, on average, cited more than the global average.
    * **Output**: A bar chart ranking topics by their relative citation impact.

2.  **Topic Co-occurrence Heatmap**:
    * **Question**: Which topics are discussed together? What are the intellectual intersections of the field?
    * **Method**: Creates a matrix showing how many documents are *simultaneously* about two different topics (e.g., how many papers have a >10% probability of being about both "Topic 1" and "Topic 3").
    * **Output**: A heatmap visualizing the strength of connection between all topics.

3.  **Most Cited Article per Topic**:
    * **Question**: What is the "canonical" or most-cited paper for each identified topic?
    * **Method**: For each topic group, it finds the single article with the highest citation count.
    * **Output**: A summary table listing the most-cited article for each topic.

4.  **Topic-Country Matrix (Activity Index)**:
    * **Question**: Which countries specialize in which research topics?
    * **Method**: Calculates the **Activity Index (AI)**, a measure of relative specialization. An AI > 1.0 means a country publishes *more* on that topic than the global average, indicating a specialization.
    * **Output**: A heatmap showing the specialization of the top 20 countries across all topics.

---

### Prerequisites

This cell assumes:

- `df_final["Dominant_Topic"]` exists (from Cell 67)
- `lda_output` exists (topic distribution matrix)
- `Countries_Extracted` exists in `df_final`  
  - Ideally as a list-like column (or a clean string per row)

---
### What users must change

**Mandatory: none** (it will run with defaults), but the following are the usual tuning points:

1. **Co-occurrence threshold (controls intersection strictness)**

```python
COOCCURRENCE_THRESHOLD = 0.1
# Try: 0.05 (more overlaps) or 0.2 (stricter overlaps)
```

2. **Country filter (reduces noise in the AI heatmap)**

```python
min_docs = 10  # countries with fewer docs are excluded
# Try: 5 for small datasets, 20+ for very large datasets
```

3. **Missing columns (most common fix)**  
If your dataset does not contain `DOI` or `OriginalTitle`, update the required column lists.

```python
# Example: if DOI is missing, remove it from required_cols_arts
required_cols_arts = ['Dominant_Topic', 'Title', 'Authors', 'Publication Year', 'OriginalTitle', 'Cited by']
```

---
---

### Example: “Top 5 papers per topic” export (optional enhancement)

If you want *more than one* canonical paper per topic:

```python
top_k = 5
top_papers_per_topic = (
    df_all_articles_topics
    .sort_values(["Topic", "Citations"], ascending=[True, False])
    .groupby("Topic")
    .head(top_k)
)
top_papers_per_topic.to_csv(os.path.join(results_dir, f"top_{top_k}_papers_per_topic.csv"), index=False)
```

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ============================
# 1. Topic Citation Impact (MNCS)
# ============================
print("\n--- 1. Topic Citation Impact Analysis (Mean Normalized Citation Score) ---")

# --- 1a. Check for necessary data ---
required_impact_cols = ['Dominant_Topic', 'Cited by', 'Publication Year']

if not all(col in df_final.columns for col in required_impact_cols):
    print(f"Warning: Missing required columns {required_impact_cols}. Skipping Impact Analysis.")
else:
    # --- 1b. Calculate Normalized Citations (Field-Weighted Proxy) ---
    # We normalize citations by the average of the publication year to account for time bias.
    print("Calculating Normalized Citation Scores (NCS)...")
    
    df_year_avg = df_final.groupby('Publication Year')['Cited by'].mean().reset_index(name='Global_Avg_Citations')
    df_impact = df_final.merge(df_year_avg, on='Publication Year', how='left')
    
    df_impact['Normalized_Citations'] = df_impact.apply(
        lambda row: row['Cited by'] / row['Global_Avg_Citations'] if row['Global_Avg_Citations'] > 0 else 0,
        axis=1
    )
    df_impact['Normalized_Citations'] = df_impact['Normalized_Citations'].fillna(0)

    # --- 1c. Aggregate by Topic ---
    topic_impact_stats = df_impact.groupby('Dominant_Topic').agg(
        Topic_MNCS=('Normalized_Citations', 'mean'),
        Number_of_Documents=('Dominant_Topic', 'size')
    ).reset_index()

    topic_impact_stats = topic_impact_stats.sort_values(by='Topic_MNCS', ascending=False)
    topic_impact_stats['Topic_Label'] = "Topic " + topic_impact_stats['Dominant_Topic'].astype(str)

    print("\nTopic Impact Statistics (MNCS):")
    display(topic_impact_stats)

    # Save Data
    results_path = os.path.join(results_dir, 'topic_citation_impact_mncs.csv')
    topic_impact_stats.to_csv(results_path, index=False)
    print(f"Topic impact data saved to: {results_path}")

    # --- 1d. Visualization ---
    plt.figure(figsize=(10, 6))
    ax = plt.gca()

    # Sort for horizontal plotting
    plot_data_impact = topic_impact_stats.sort_values(by='Topic_MNCS', ascending=True)
    
    # Use Cividis "Dark Blue" for professional contrast
    cividis_bar_color = "#00204d"

    bars = ax.barh(
        plot_data_impact['Topic_Label'],
        plot_data_impact['Topic_MNCS'],
        color=cividis_bar_color
    )

    # Global Average Reference Line
    ax.axvline(x=1.0, color='#b9ac70', linestyle='--', linewidth=2, label='Global Baseline (1.0)')

    # Counts for Title
    n_topics_viz = len(topic_impact_stats)
    n_docs_viz = len(df_impact)

    ax.set_title(f'Relative Citation Impact of Research Topics (MNCS)\n(N={n_topics_viz} topics, N={n_docs_viz} articles)', 
                 fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Mean Normalized Citation Score (MNCS)', fontsize=12)
    ax.set_ylabel(None)

    # Styling
    ax.spines[['top', 'right', 'left']].set_visible(False)
    ax.grid(axis='x', linestyle='--', alpha=0.6)
    ax.legend(loc='lower right')
    ax.set_xlim(left=0, right=max(1.1, plot_data_impact['Topic_MNCS'].max() * 1.15))

    # Annotations
    for bar in bars:
        width = bar.get_width()
        ax.annotate(
            f' {width:.2f}',
            xy=(width, bar.get_y() + bar.get_height() / 2),
            xytext=(3, 0), 
            textcoords="offset points",
            ha='left', va='center', fontsize=10
        )

    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, 'topic_citation_impact_mncs.png'), dpi=600, bbox_inches='tight')
    plt.savefig(os.path.join(plots_dir, 'topic_citation_impact_mncs.pdf'), format='pdf', bbox_inches='tight')
    plt.show()


# ============================
# 2. Topic Co-occurrence Matrix
# ============================
print("\n--- 2. Topic Co-occurrence Analysis ---")

if 'lda_output' not in locals() or lda_output is None:
    print("Warning: 'lda_output' matrix not found. Skipping.")
elif num_topics <= 1:
    print("Warning: Only one topic found. Skipping Co-occurrence.")
else:
    # --- 2a. Calculate Co-occurrence ---
    # A document "belongs" to a topic if prob > threshold
    COOCCURRENCE_THRESHOLD = 0.1
    binary_topic_matrix = (lda_output > COOCCURRENCE_THRESHOLD).astype(int)
    
    # Matrix Multiplication (Docs x Topics).T @ (Docs x Topics) = (Topics x Topics)
    cooccurrence_matrix = binary_topic_matrix.T @ binary_topic_matrix
    np.fill_diagonal(cooccurrence_matrix, 0) # Zero out self-loops

    topic_labels = [f"T{i+1}" for i in range(num_topics)]
    cooccurrence_df = pd.DataFrame(cooccurrence_matrix, index=topic_labels, columns=topic_labels)

    # Save Matrix
    results_path = os.path.join(results_dir, 'topic_cooccurrence_matrix.csv')
    cooccurrence_df.to_csv(results_path)
    print(f"Co-occurrence matrix saved to: {results_path}")

    # --- 2b. Visualization (Heatmap) ---
    plt.figure(figsize=(max(8, num_topics * 0.8), max(6, num_topics * 0.7)))
    
    sns.heatmap(
        cooccurrence_df,
        annot=(num_topics <= 12),
        fmt=".0f",
        cmap="cividis", # Cividis Palette
        linewidths=.5,
        linecolor='white',
        cbar_kws={'label': 'Co-occurring Documents'}
    )
    
    n_docs_total = len(df_final)
    plt.title(f'Topic Co-occurrence Matrix\n(Threshold > {COOCCURRENCE_THRESHOLD}, N={num_topics} topics, N={n_docs_total} articles)', 
              fontsize=14, fontweight='bold', pad=20)
    plt.xlabel("Topic", fontsize=12)
    plt.ylabel("Topic", fontsize=12)
    plt.xticks(rotation=0)
    plt.yticks(rotation=0)

    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, 'topic_cooccurrence_heatmap.png'), dpi=600, bbox_inches='tight')
    plt.savefig(os.path.join(plots_dir, 'topic_cooccurrence_heatmap.pdf'), format='pdf', bbox_inches='tight')
    plt.show()


# ==================================
# 3. Topic Article Extraction
# ==================================
print("\n--- 3. Topic Article Extraction ---")

required_cols_arts = ['Dominant_Topic', 'Title', 'Authors', 'Publication Year', 'OriginalTitle', 'Cited by', 'DOI']

if not all(col in df.columns for col in required_cols_arts):
    print(f"Warning: Missing columns {required_cols_arts}. Skipping export.")
else:
    # --- 3a. Full List Export ---
    df_all_articles_topics = df[required_cols_arts].copy()
    df_all_articles_topics.sort_values(by='Cited by', ascending=False, inplace=True)
    df_all_articles_topics.rename(columns={
        'Dominant_Topic': 'Topic',
        'OriginalTitle': 'Source',
        'Cited by': 'Citations'
    }, inplace=True)
    
    path_csv = os.path.join(results_dir, 'all_articles_by_topic_and_citation.csv')
    path_xlsx = os.path.join(results_dir, 'all_articles_by_topic_and_citation.xlsx')
    
    df_all_articles_topics.to_csv(path_csv, index=False, encoding='utf-8-sig')
    df_all_articles_topics.to_excel(path_xlsx, index=False, engine='openpyxl')
    print(f"Exported full article list to: {path_csv}")

    # --- 3b. Most Cited Article per Topic ---
    most_cited_per_topic = df_all_articles_topics.drop_duplicates(subset=['Topic'], keep='first').copy()
    
    # Helper to clean authors for display
    def simple_author_clean(x):
        if pd.isna(x): return "N/A"
        return str(x).split(';')[0].strip()

    most_cited_per_topic['First Author'] = most_cited_per_topic['Authors'].apply(simple_author_clean)
    most_cited_per_topic['Title (Short)'] = most_cited_per_topic['Title'].apply(lambda x: (x[:50] + '...') if len(str(x)) > 50 else str(x))
    most_cited_per_topic = most_cited_per_topic.sort_values(by='Topic')

    display_cols_mca = ['Topic', 'Title (Short)', 'First Author', 'Publication Year', 'Citations']
    
    print("\nMost Cited Article per Topic:")
    display(most_cited_per_topic[display_cols_mca])
    
    path_mca = os.path.join(results_dir, 'most_cited_article_per_topic.csv')
    most_cited_per_topic[display_cols_mca].to_csv(path_mca, index=False)


# ==================================
# 4. Topic-Country Matrix (Activity Index)
# ==================================
print("\n--- 4. Topic Specialization by Country (Activity Index) ---")

required_cols_tc = ['Dominant_Topic', 'Countries_Extracted'] 
if not all(col in df_final.columns for col in required_cols_tc):
    print("Warning: Missing Country/Topic data. Skipping.")
else:
    # --- 4a. Data Preparation ---
    df_topic_country = df_final[required_cols_tc].copy()
    
    # Flatten list of countries
    def ensure_flat_list(x):
        if isinstance(x, list): return [c for c in x if c]
        if isinstance(x, str): return [x]
        return []

    df_topic_country['Countries'] = df_topic_country['Countries_Extracted'].apply(ensure_flat_list)
    df_tc_exploded = df_topic_country.explode('Countries').dropna(subset=['Countries'])

    if df_tc_exploded.empty:
        print("No valid country data found.")
    else:
        # --- 4b. Pivot Table (Raw Counts) ---
        topic_country_pivot = pd.pivot_table(
            df_tc_exploded,
            index='Countries', columns='Dominant_Topic',
            aggfunc='size', fill_value=0
        )
        topic_country_pivot.columns = [f"Topic {int(c)}" for c in topic_country_pivot.columns]
        
        # --- 4c. Calculate Activity Index (AI) ---
        # Filter for countries with significant output to reduce noise (e.g., >= 10 papers)
        min_docs = 10
        country_counts = topic_country_pivot.sum(axis=1)
        valid_countries = country_counts[country_counts >= min_docs].index
        
        filtered_pivot = topic_country_pivot.loc[valid_countries]
        
        # Calculation: (Country Topic Share) / (Global Topic Share)
        global_topic_dist = topic_country_pivot.sum(axis=0) / topic_country_pivot.sum().sum()
        country_topic_dist = filtered_pivot.div(filtered_pivot.sum(axis=1), axis=0)
        
        activity_index = country_topic_dist.div(global_topic_dist, axis=1).fillna(0)
        
        # Save AI Matrix
        path_ai = os.path.join(results_dir, 'topic_country_activity_index.csv')
        activity_index.to_csv(path_ai)
        print(f"Activity Index matrix saved to: {path_ai}")

        # --- 4d. Visualization (Cividis Heatmap) ---
        # Select top 20 countries by volume for the plot
        top_countries = country_counts.loc[valid_countries].nlargest(20).index
        plot_data_ai = activity_index.loc[top_countries]

        n_cntry = len(plot_data_ai)
        n_top = len(plot_data_ai.columns)

        plt.figure(figsize=(10, max(6, n_cntry * 0.4)))
        
        sns.heatmap(
            plot_data_ai,
            annot=False,
            cmap="cividis", # Cividis Palette
            linewidths=.5,
            linecolor='white',
            cbar_kws={'label': 'Activity Index (AI)'}
        )
        
        plt.title(f'Topic Specialization by Country (Activity Index)\n(N={n_cntry} countries, N={n_top} topics)', 
                  fontsize=14, fontweight='bold', pad=20)
        plt.xlabel("Topic", fontsize=12)
        plt.ylabel("Country", fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.yticks(rotation=0)
        
        plt.tight_layout()
        plt.savefig(os.path.join(plots_dir, 'topic_country_specialization_heatmap.png'), dpi=600, bbox_inches='tight')
        plt.savefig(os.path.join(plots_dir, 'topic_country_specialization_heatmap.pdf'), format='pdf', bbox_inches='tight')
        plt.show()

# PART 4. INSTITUTIONAL ANALYSIS

This section focuses on analyzing the productivity and collaboration patterns of institutions. This is a complex analysis that involves identifying all unique institutions, ranking them by publication count, and then building and visualizing their collaboration networks.

## 26. Institutional Analysis: Helper Functions

This first cell defines the "toolkit" of helper functions we will use to perform the institutional analysis. No analysis is run here; these functions are just being defined for use in the next step.

The key functions are:

* **`is_exclusively_eu`**: A new helper function that checks an article's list of countries and returns `True` if *all* affiliations are from EU member states. This allows for EU-specific filtering.
* **`process_and_explode_institutions`**: Takes the main dataframe, assigns a unique `Publication_ID`, and "explodes" the institution list to create a one-row-per-institution-pairing dataframe. It also applies the `standardize_institution_name` function (defined in Part 2) to ensure all names are clean.
* **`get_top_institutions`**: Takes the exploded dataframe and counts the total publications for each unique institution, saving a ranked list to an Excel file.
* **`plot_top_institutions`**: Generates a bar chart of the Top 10 most productive institutions.
* **`generate_institution_pairs`**: For a given article, this function looks at its list of unique, standardized institutions and generates all possible collaboration pairs (e.g., [Harvard, MIT], [Harvard, Stanford], [MIT, Stanford]).
* **`build_collaboration_graph`**: Uses `generate_institution_pairs` on every article to count the total collaborations between each pair of institutions. It then builds a `networkx` graph, where nodes are institutions and edge weights represent the strength of collaboration.
* **`export_network_data`**: A new utility function that saves the final `networkx` graph into formats usable by external software, including a **`.graphml` file for Gephi/Cytoscape**, as well as node and edge lists as `.csv` files.
* **`plot_collaboration_network`**: A complex plotting function that takes the full graph, filters it to show only the Top N institutions (e.g., Top 50), and draws a network diagram. In this plot:
    * **Node size** represents the institution's total productivity.
    * **Node color** represents the institution's centrality (importance) in the network.
    * **Edge thickness** represents the strength of collaboration between two institutions.
* **`analyze_network`**: The first **orchestrator function**. It ties all the above functions together into a single pipeline. It is now upgraded to accept an `is_eu` parameter to filter the dataset before analysis.
* **`run_comprehensive_analysis`**: The second, higher-level **batch orchestrator**. This function automatically runs the `analyze_network` pipeline 6 times for every combination of (`Global`, `EU`) and (`All`, `MCP`, `SCP`).
* **`generate_summary_report`**: Creates a summary `.xlsx` file of the batch analysis, showing the nodes, edges, and top institution for each of the 6 runs.
---

### Prerequisites

Before institutional analysis can work, your pipeline must have already created:

- `df["Institutions_Extracted"]` (or equivalent institution-list column)
- `df["Countries_Extracted_List"]` (or a list-like country column) if you want EU filtering
- `results_dir` and `plots_dir`

Also, earlier in the notebook you likely defined:

- `standardize_institution_name()` (used here if available)

If `standardize_institution_name` is not defined, this toolkit will still run (fallback behavior), but results may contain duplicates due to name variants.

---

### What users must change

**Mandatory (most common):**

1. **Institution column name**  
Later, when calling `run_comprehensive_analysis`, you must provide the correct list-column name.

Examples:

- `inst_col_name = "Institutions_Extracted"` (default in this notebook)
- `inst_col_name = "Institutions_Standardized"` (if you used a different pipeline name)

**Strongly recommended:**

2. **Consortium exclusion threshold**  
Large consortium papers can artificially inflate collaboration edges.  
Tune based on your domain:

```python
MAX_INSTITUTIONS_PER_PAPER = 50
# Try 30 for strict filtering or 100 for permissive filtering
```

3. **EU membership list (only if you use EU filtering)**  
The function uses a static EU membership set. If you require a different definition (EU27 / EU28 / EEA), update the set.

---


### Notes on reproducibility

- Network plots are layout-sensitive (spring layout depends on randomness).  
  This notebook fixes a `seed=42` for stable layouts.
- Exported `.graphml` can be opened in Gephi or Cytoscape for advanced styling and metrics.


In [ ]:
## PART 4. Institutional Analysis ##
### 4.1. Institutional Helper Functions ###

import os
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx
from itertools import combinations
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from IPython.display import display
import textwrap

# Try to import adjustText
try:
    from adjustText import adjust_text
    ADJUST_TEXT_AVAILABLE = True
    print("Loaded 'adjustText' library for non-overlapping labels.")
except ImportError:
    ADJUST_TEXT_AVAILABLE = False
    print("Warning: 'adjustText' not found. Labels may overlap.")

MAX_INSTITUTIONS_PER_PAPER = 50

# ===================================================================
# 1) Basic Helpers and Institution-Processing Functions
# ===================================================================

def assign_publication_id(df, df_name=''):
    """
    Assigns a unique Publication_ID (starting from 1) to each row.
    """
    df = df.reset_index(drop=True) 
    if 'Publication_ID' in df.columns:
        df = df.drop(columns=['Publication_ID'])
        print(f"Warning: Dropped existing 'Publication_ID' column in {df_name}.")
    df['Publication_ID'] = df.index + 1
    return df

def generate_institution_pairs(institutions_list):
    """
    Generates all unique pairs from a list of institutions.
    Assumes filtering decisions are handled upstream.
    """
    if not isinstance(institutions_list, (list, tuple, np.ndarray, pd.Series)):
        return []

    try:
        standardized_institutions = {
            standardize_institution_name(str(inst))
            for inst in institutions_list
            if pd.notnull(inst)
            and str(inst).strip() != ''
            and str(inst) != 'Missing/Incomplete Institution Data'
        }
    except NameError:
        standardized_institutions = {
            str(inst).strip()
            for inst in institutions_list
            if pd.notnull(inst)
            and str(inst).strip() != ''
            and str(inst) != 'Missing/Incomplete Institution Data'
        }

    unique_institutions = sorted(i for i in standardized_institutions if i)

    if len(unique_institutions) < 2:
        return []

    return list(combinations(unique_institutions, 2))

# --- NEW FUNCTION (from v1) ---
def is_exclusively_eu(countries_list):
    """
    Checks if all countries in the list are EU member states.
    """
    # Define EU member states (as of 2024)
    eu_countries = {
        'Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czech Republic',
        'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece',
        'Hungary', 'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg',
        'Malta', 'Netherlands', 'Poland', 'Portugal', 'Romania', 'Slovakia',
        'Slovenia', 'Spain', 'Sweden'
    }
    
    if not isinstance(countries_list, list) or not countries_list:
        return False
    
    # Check if all countries are in EU
    for country in countries_list:
        if not isinstance(country, str) or country.strip().title() not in eu_countries:
            return False # Found a non-EU country
            
    return True # All countries were in the EU list

# ===================================================================
# 2) Processing, Counting, & Exporting Functions
# ===================================================================

def process_and_explode_institutions(df_input, network_type='Global', inst_col='Institutions_Extracted'):
    """
    1. Assigns unique Publication_ID.
    2. Explodes the specified list column (inst_col).
    3. Applies a final standardization pass to the exploded names.
    """
    if inst_col not in df_input.columns:
        print(f"Error: Institution list column '{inst_col}' not found in {network_type}. Cannot process.")
        return pd.DataFrame(), pd.DataFrame()

    df_with_id = df_input.copy() 
    df_with_id = assign_publication_id(df_with_id, f'{network_type}_process_institutions')
    df_exploded = df_with_id.explode(inst_col).rename(columns={inst_col: 'Institution'})

    df_exploded = df_exploded.dropna(subset=['Institution'])
    df_exploded = df_exploded[df_exploded['Institution'] != 'Missing/Incomplete Institution Data']
    df_exploded = df_exploded[df_exploded['Institution'].astype(str).str.strip() != '']

    try:
        df_exploded['Institution'] = df_exploded['Institution'].apply(standardize_institution_name)
        df_exploded = df_exploded.dropna(subset=['Institution'])
        df_exploded = df_exploded[df_exploded['Institution'].astype(str).str.strip() != '']
    except NameError:
        print("Warning: 'standardize_institution_name' function not found. Skipping final standardization pass.")
    except Exception as e:
         print(f"Warning: Error during final standardization pass: {e}. Proceeding without it.")

    return df_with_id, df_exploded


def get_top_institutions(df_institutions_exploded, top_n=1000, network_type='Global', total_articles=0):
    """
    Identifies the top N institutions by publication count.
    """
    if df_institutions_exploded.empty or 'Institution' not in df_institutions_exploded.columns:
         print(f"Warning: Exploded institutions DataFrame is empty for {network_type}. Cannot get top institutions.")
         return pd.DataFrame(columns=['Institution', 'Number_of_Publications'])

    institution_counts = df_institutions_exploded['Institution'].value_counts().reset_index()
    institution_counts.columns = ['Institution', 'Number_of_Publications']

    actual_top_n = min(top_n, len(institution_counts))
    top_institutions_topN_df = institution_counts.head(actual_top_n)

    print(f"\nTop {min(15, actual_top_n)} Institutions ({network_type}) by Number of Publications:")
    display(top_institutions_topN_df.head(15))

    output_path = os.path.join(results_dir, f'Top_Institutions_By_Publications_{network_type}.xlsx')
    try: 
        institution_counts.to_excel(output_path, index=False, engine='openpyxl')
        print(f"Full institution ranking saved to '{output_path}'.")
    except Exception as e:
        print(f"Error saving institution ranking to Excel: {e}")

    return top_institutions_topN_df
    
from collections import Counter

def build_collaboration_graph(df_with_id_and_list,
                              inst_list_col='Institutions_Extracted',
                              network_type='Global'):
    """
    Builds collaboration graph with safeguards against mega-consortium papers.
    """
    if inst_list_col not in df_with_id_and_list.columns:
        print(f"Error: Institution list column '{inst_list_col}' not found for {network_type}.")
        return nx.Graph(), pd.DataFrame(columns=['Institution_Pair', 'Collaboration_Count'])

    pair_counter = Counter()
    skipped_large_papers = 0
    total_papers = len(df_with_id_and_list)

    for inst_list in df_with_id_and_list[inst_list_col].values:

        # --- PRISMA FILTER: mega-consortium exclusion ---
        if not isinstance(inst_list, list):
            continue

        if len(inst_list) > MAX_INSTITUTIONS_PER_PAPER:
            skipped_large_papers += 1
            continue
        # ------------------------------------------------

        pairs = generate_institution_pairs(inst_list)
        if pairs:
            pair_counter.update(pairs)

    if not pair_counter:
        print(f"No valid institution pairs found for {network_type}.")
        return nx.Graph(), pd.DataFrame(columns=['Institution_Pair', 'Collaboration_Count'])

    # Build collaboration counts dataframe
    collaboration_counts = (
        pd.DataFrame(pair_counter.items(),
                     columns=['Institution_Pair', 'Collaboration_Count'])
        .sort_values('Collaboration_Count', ascending=False)
        .reset_index(drop=True)
    )

    # Split institution columns (for export)
    collaboration_counts[['Institution1', 'Institution2']] = pd.DataFrame(
        collaboration_counts['Institution_Pair'].tolist(),
        index=collaboration_counts.index
    )

    # Build graph
    G_collab = nx.Graph()
    for (i1, i2), w in pair_counter.items():
        G_collab.add_edge(i1, i2, weight=int(w))

    G_collab.remove_edges_from(nx.selfloop_edges(G_collab))

    # =======================
    # PRISMA-STYLE REPORTING
    # =======================
    print(f"\n--- Institutional Network Construction ({network_type}) ---")
    print(f"Total publications evaluated: {total_papers:,}")
    
    if skipped_large_papers > 0:
        print(
            f"Excluded mega-consortium papers "
            f"(> {MAX_INSTITUTIONS_PER_PAPER} institutions): {skipped_large_papers:,}"
        )
        print(
            "Exclusion rationale: Prevents artificial edge inflation "
            "from large-scale consortium authorship."
        )
    else:
        print(
            f"No publications exceeded the exclusion threshold "
            f"({MAX_INSTITUTIONS_PER_PAPER} institutions)."
        )
    
    print(
        f"Final network: {G_collab.number_of_nodes():,} institutions, "
        f"{G_collab.number_of_edges():,} collaboration links."
    )

    # Save collaboration counts
    output_path = os.path.join(results_dir, f'Institution_Collaboration_Counts_{network_type}.csv')
    collaboration_counts.to_csv(output_path, index=False)
    print(f"Collaboration pair counts saved to '{output_path}'.")

    return G_collab, collaboration_counts


# --- NEW FUNCTION (from v1) ---
def export_network_data(graph, top_inst_df, filename_prefix):
    """
    Export network data for external analysis (e.g., Gephi, Cytoscape).
    """
    if not isinstance(graph, nx.Graph) or graph.number_of_nodes() == 0:
        print(f"Graph for {filename_prefix} is empty. Skipping export.")
        return
        
    try:
        nx.write_graphml(graph, f"{filename_prefix}.graphml")
        
        node_data = []
        degree_cent = nx.degree_centrality(graph)

        # ✅ ADD THIS GUARD
        if graph.number_of_nodes() > 1000:
            print(f"Skipping betweenness centrality (n={graph.number_of_nodes():,} too large).")
            betweenness_cent = {}
        else:
            betweenness_cent = nx.betweenness_centrality(graph, weight='weight')

        prod_map = top_inst_df.set_index('Institution')['Number_of_Publications'].to_dict()
        
        for node in graph.nodes():
            node_data.append({
                'Id': node,
                'Label': node,
                'Degree': graph.degree(node),
                'Weighted_Degree': graph.degree(node, weight='weight'),
                'Degree_Centrality': degree_cent.get(node, 0.0),
                'Betweenness_Centrality': betweenness_cent.get(node, 0.0),  # defaults to 0 if skipped
                'Productivity': prod_map.get(node, 0)
            })
        pd.DataFrame(node_data).to_csv(f"{filename_prefix}_nodes.csv", index=False)
        
        nx.write_edgelist(graph, f"{filename_prefix}_edges.csv", data=['weight'], delimiter=',')
        
        print(f"Network data (GraphML, nodes, edges) exported with prefix: {filename_prefix}")
        
    except Exception as e:
        print(f"Error exporting network data for {filename_prefix}: {e}")

# ===================================================================
# 3) Plotting Functions
# ===================================================================

def plot_top_institutions(top_institutions_df, network_type='Global', total_articles=0):
    """
    Plots the top 10 institutions based on publication count.
    """
    if top_institutions_df.empty:
         print(f"Warning: No top institutions data to plot for {network_type}.")
         return

    plot_data = top_institutions_df.head(10)
    plt.figure(figsize=(10, 6))
    
    # Cividis Palette
    sns.barplot(data=plot_data, x='Number_of_Publications', y='Institution', palette='cividis')
    
    plt.xlabel('Number of Publications')
    plt.ylabel('Institution')
    
    n_inst_total = len(top_institutions_df)
    plt.title(f'Top 10 Institutions by Publication Count\n({network_type} Level, N={n_inst_total} institutions, N={total_articles} articles)', fontsize=14, fontweight='bold', pad=15)
    
    plt.tight_layout()
    plot_path_png = os.path.join(plots_dir, f'Top_10_Institutions_By_Publications_{network_type}.png')
    plot_path_pdf = os.path.join(plots_dir, f'Top_10_Institutions_By_Publications_{network_type}.pdf')
    try: 
        plt.savefig(plot_path_png, dpi=600, bbox_inches='tight')
        plt.savefig(plot_path_pdf, format='pdf', bbox_inches='tight')
        print(f"Top 10 institutions plot saved to (PNG/PDF): {plots_dir}")
    except Exception as e:
        print(f"Error saving top institutions plot: {e}")
    plt.show()

def plot_collaboration_network(G, top_n, top_institutions_df, network_type='Global'):
    """
    Plots a cleaner collaboration network for the top N institutions.
    """
    if not isinstance(G, nx.Graph) or G.number_of_nodes() == 0:
        print(f"Warning: Collaboration graph for {network_type} is empty. Skipping plot.")
        return G

    # --- 1. Subgraph Creation ---
    top_institutions_list = top_institutions_df.head(top_n)['Institution'].tolist()
    nodes_in_G = [node for node in top_institutions_list if node in G]
    G_top = G.subgraph(nodes_in_G).copy()
    G_top.remove_nodes_from(list(nx.isolates(G_top)))
    
    print(f"Filtered {network_type} network for plotting: {G_top.number_of_nodes()} nodes, {G_top.number_of_edges()} edges.")
    
    if G_top.number_of_nodes() == 0:
         print(f"Warning: Subgraph for Top {top_n} {network_type} is empty. Skipping plot.")
         return G_top

    # --- 2. Layout ---
    fig, ax = plt.subplots(figsize=(18, 16))
    pos = nx.spring_layout(G_top, k=3, scale=2, iterations=50, seed=42) 

    # --- 3. Node Attributes ---
    degree_centrality_top = nx.degree_centrality(G_top)
    if not degree_centrality_top:
        print("Warning: Could not calculate degree centrality. Skipping plot.")
        return G_top

    min_centrality = min(degree_centrality_top.values())
    max_centrality = max(degree_centrality_top.values())
    norm = mcolors.Normalize(vmin=min_centrality, vmax=max(min_centrality + 1e-9, max_centrality))
    
    # Use Cividis Colormap
    cmap = plt.cm.cividis
    node_colors = [cmap(norm(degree_centrality_top[node])) for node in G_top.nodes()]

    degrees_for_size = dict(G_top.degree(weight='weight'))
    max_deg_size = max(degrees_for_size.values()) if degrees_for_size else 1
    node_sizes = [50 + np.sqrt(max(0, degrees_for_size.get(node, 0))) * 40 for node in G_top.nodes()]

    # --- 4. Edge Styling ---
    edge_weights = [d['weight'] for _, _, d in G_top.edges(data=True) if 'weight' in d]
    if edge_weights:
        max_weight = max(edge_weights) if edge_weights else 1.0
        edges_strong = [(u, v) for u, v, d in G_top.edges(data=True) if d['weight'] > max_weight / 2]
        edges_weak    = [(u, v) for u, v, d in G_top.edges(data=True) if d['weight'] <= max_weight / 2]
        
        nx.draw_networkx_edges(G_top, pos, edgelist=edges_weak, width=0.5, edge_color='lightgray', alpha=0.6, ax=ax)
        # Use dark Cividis blue for strong edges
        nx.draw_networkx_edges(G_top, pos, edgelist=edges_strong, width=1.5, edge_color='#00204d', alpha=0.7, ax=ax)
    
    # --- 5. Draw Nodes ---
    nx.draw_networkx_nodes(G_top, pos, node_size=node_sizes, node_color=node_colors,
                           alpha=0.9, edgecolors='black', linewidths=0.5, ax=ax)
    
    # --- 6. Labeling ---
    labels_dict = {n: n for n in G_top.nodes()}
    
    if ADJUST_TEXT_AVAILABLE:
        print("Using 'adjustText' for non-overlapping labels...")
        texts = []
        for node, label in labels_dict.items():
            if node in pos:
                texts.append(ax.text(pos[node][0], pos[node][1], label,
                             fontsize=9, fontweight='bold',
                             bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=0.1)))
        if texts:
            adjust_text(texts, ax=ax,
                        arrowprops=dict(arrowstyle='-', color='grey', lw=0.3, alpha=0.5))
    else:
        print("Using default labels (may overlap). Install 'adjustText' for cleaner plots.")
        nx.draw_networkx_labels(G_top, pos, labels=labels_dict, font_size=9,
                                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=0.1), ax=ax)

    # --- 7. Colorbar and Legend ---
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="3%", pad=0.1)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, cax=cax)
    cbar.set_label('Node Centrality (Degree)', fontsize=12)
    
    legend_handles = [
        plt.Line2D([0], [0], marker='o', color='w', label='Node Size = Productivity (Pub. Count)', markersize=12,
                   markerfacecolor='lightgrey', markeredgewidth=0.5, markeredgecolor='black'),
        plt.Line2D([0], [0], marker='o', color='w', label='High Centrality', markersize=10,
                   markerfacecolor=cmap(1.0), markeredgewidth=0.5, markeredgecolor='black'),
        plt.Line2D([0], [0], marker='o', color='w', label='Low Centrality', markersize=10,
                   markerfacecolor=cmap(0.1), markeredgewidth=0.5, markeredgecolor='black'),
        plt.Line2D([0], [0], color='#00204d', lw=2, label='Strong Collaboration', alpha=0.7),
        plt.Line2D([0], [0], color='lightgray', lw=1, label='Weak Collaboration', alpha=0.6)
    ]
    ax.legend(handles=legend_handles, loc='upper left', frameon=True, facecolor='white', framealpha=0.8, fontsize=12, title='Legend', title_fontsize='13')
    
    # --- 8. Final Touches ---
    n_nodes = G_top.number_of_nodes()
    n_edges = G_top.number_of_edges()
    fig.suptitle(f'Institutional Collaboration Network ({network_type})\n(N={n_nodes} institutions, N={n_edges} links)', fontsize=20)
    plt.axis('off')
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    
    plot_path_png = os.path.join(plots_dir, f'Collaboration_Network_Top_{top_n}_{network_type}.png')
    plot_path_pdf = os.path.join(plots_dir, f'Collaboration_Network_Top_{top_n}_{network_type}.pdf')
    try: 
        plt.savefig(plot_path_png, dpi=600, bbox_inches='tight')
        plt.savefig(plot_path_pdf, format='pdf', bbox_inches='tight')
        print(f"Collaboration network plot ({network_type}, Top {top_n}) saved to (PNG/PDF): {plots_dir}")
    except Exception as e:
        print(f"Error saving collaboration network plot: {e}")

    plt.show()
    
    return G_top

# ===================================================================
# 4) Main Network Analysis Orchestrator
# ===================================================================

def analyze_network(df_main,
                    network_type='Global',
                    publication_type='All',
                    inst_col='Institutions_Extracted', 
                    top_n_for_list=1000,
                    plot_n_for_graph=50,
                    is_eu=False): 
    """
    Orchestrator function to run the full analysis pipeline.
    """
    analysis_name = f"{network_type}_{publication_type}"
    print(f"\n===== Starting Network Analysis: {analysis_name} =====")
    
    if publication_type in ['MCP', 'SCP'] and 'Country_Classification' not in df_main.columns:
         print(f"FATAL ERROR: 'Country_Classification' column missing. Cannot filter for {publication_type}.")
         return nx.Graph(), nx.Graph(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    df_filtered = df_main.copy()

    # --- Filter 1: By Publication Type ---
    if publication_type in ['MCP', 'SCP']:
        df_filtered = df_filtered[df_filtered['Country_Classification'] == publication_type].copy()
        print(f"Filtered to {publication_type} publications: {len(df_filtered)} rows.")
    else:
         print(f"Using All Publications: {len(df_filtered)} rows.")

    # --- NEW Filter 2: By EU ---
    if is_eu:
        if 'Countries_Extracted_List' not in df_filtered.columns:
            # Fallback if list column is not explicitly named "Countries_Extracted_List"
            # It might be "Countries_Extracted" from previous steps
            if 'Countries_Extracted' in df_filtered.columns:
                 df_filtered['Countries_Extracted_List'] = df_filtered['Countries_Extracted']
            else:
                 print("Warning: 'Countries_Extracted' column not found. Cannot apply EU filter.")
        
        if 'Countries_Extracted_List' in df_filtered.columns:
            initial_count = len(df_filtered)
            df_filtered = df_filtered[df_filtered['Countries_Extracted_List'].apply(is_exclusively_eu)].copy()
            print(f"Filtered to EU-only publications: {len(df_filtered)} rows (from {initial_count}).")

    if df_filtered.empty:
        print(f"No publications found for '{analysis_name}'. Aborting analysis.")
        return nx.Graph(), nx.Graph(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    # --- 1) Process & Explode Institutions ---
    df_processed_with_id, df_institutions_exploded = process_and_explode_institutions(
        df_filtered, 
        network_type=analysis_name,
        inst_col=inst_col
    )
    if df_processed_with_id.empty or df_institutions_exploded.empty:
         print(f"Institution processing failed for {analysis_name}. Aborting.")
         return nx.Graph(), nx.Graph(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    # --- 2) Identify top institutions ---
    total_articles = len(df_filtered)
    top_institutions_data = get_top_institutions(
        df_institutions_exploded,
        top_n=top_n_for_list,
        network_type=analysis_name,
        total_articles=total_articles # Pass article count for titles
    )

    # --- 3) Plot top 10 institutions bar chart ---
    plot_top_institutions(
        top_institutions_data,
        network_type=analysis_name,
        total_articles=total_articles
    )

    # --- 4) Build the FULL collaboration graph ---
    G_collab_full, collaboration_counts = build_collaboration_graph(
        df_processed_with_id, 
        inst_list_col=inst_col,
        network_type=analysis_name
    )

    # --- 5) Plot the SUBGRAPH of top N institutions ---
    G_top_plotted = plot_collaboration_network(
        G_collab_full,
        top_n=plot_n_for_graph,
        top_institutions_df=top_institutions_data, 
        network_type=analysis_name
    )
    
    # --- 6) Export Network Data ---
    export_prefix = os.path.join(results_dir, f"network_data_{analysis_name}")
    export_network_data(G_collab_full, top_institutions_data, export_prefix)

    print(f"===== Completed Network Analysis: {analysis_name} =====")
    # Return all artifacts needed for validation
    return G_collab_full, G_top_plotted, df_institutions_exploded, collaboration_counts, top_institutions_data



In [ ]:
# ===================================================================
# 5) NEW: Batch Analysis & Reporting Functions
# ===================================================================

def run_comprehensive_analysis(df_main, inst_col_name, top_n_list=1000, top_n_plot=30):
    """
    Runs comprehensive analysis for all 6 network types (Global/EU x All/MCP/SCP).
    """
    
    # Analysis configurations
    analyses = [
        {'network_type': 'Global', 'publication_type': 'All', 'is_eu': False},
        {'network_type': 'Global', 'publication_type': 'MCP', 'is_eu': False},
        {'network_type': 'Global', 'publication_type': 'SCP', 'is_eu': False},
        {'network_type': 'EU', 'publication_type': 'All', 'is_eu': True},
        {'network_type': 'EU', 'publication_type': 'MCP', 'is_eu': True},
        {'network_type': 'EU', 'publication_type': 'SCP', 'is_eu': True},
    ]
    
    results = {}
    
    for config in analyses:
        analysis_name = f"{config['network_type']}_{config['publication_type']}"
        
        # Run analysis
        try:
            # We must capture all 5 return values for the validation step
            (G_full, G_sub, df_inst, collab_counts, top_inst) = analyze_network(
                df_main,
                network_type=config['network_type'],
                publication_type=config['publication_type'],
                is_eu=config['is_eu'],
                inst_col=inst_col_name,
                top_n_for_list=top_n_list,
                plot_n_for_graph=top_n_plot
            )
            
            results[analysis_name] = {
                'full_graph': G_full,
                'subgraph': G_sub,
                'institutions_df': df_inst,
                'collaboration_counts': collab_counts,
                'top_institutions': top_inst, 
                'config': config,
                'status': 'Success'
            }
            
        except Exception as e:
            print(f"!!! ERROR in analysis {analysis_name}: {str(e)} !!!")
            results[analysis_name] = {'error': str(e), 'config': config, 'status': 'Error'}
    
    return results

def generate_summary_report(results, output_path):
    """
    Generates a summary report of all batch analyses.
    """
    summary_data = []
    
    for analysis_name, result in results.items():
        if result['status'] == 'Error':
            summary_data.append({
                'Analysis': analysis_name,
                'Status': 'Error',
                'Error': result['error'],
                'Nodes': 0,
                'Edges': 0,
                'Top Institution': 'N/A',
                'Top Inst. Count': 0
            })
        else:
            top_inst = result['top_institutions']
            top_inst_name = top_inst.iloc[0]['Institution'] if not top_inst.empty else 'N/A'
            top_inst_count = top_inst.iloc[0]['Number_of_Publications'] if not top_inst.empty else 0
            
            summary_data.append({
                'Analysis': analysis_name,
                'Status': 'Success',
                'Error': '',
                'Nodes': result['full_graph'].number_of_nodes(),
                'Edges': result['full_graph'].number_of_edges(),
                'Top Institution': top_inst_name,
                'Top Inst. Count': top_inst_count
            })
    
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_excel(output_path, index=False)
    print(f"\nBatch analysis summary report saved to '{output_path}'.")
    return summary_df

## 27. Institutional Analysis: Main Execution (Batch Analysis)

This cell executes the entire institutional analysis pipeline using the `run_comprehensive_analysis` batch orchestrator defined in the previous cell.

Instead of running each analysis type manually, this single function will automatically generate all **6 combinations** of network analyses, saving all plots, data files, and network export files with unique filenames.

This allows for a powerful and comprehensive comparative analysis of collaboration patterns:

1.  **Global Networks**:
    * `Global_All`: The "big picture" network of all publications.
    * `Global_MCP`: The international collaboration network.
    * `Global_SCP`: The domestic collaboration network.
2.  **EU-Filtered Networks**:
    * `EU_All`: The network of publications *exclusively* from EU institutions.
    * `EU_MCP`: The *intra-EU* international collaboration network.
    * `EU_SCP`: The domestic collaboration network *within* individual EU countries.


For each run, the pipeline will typically save:

- **Top institutions list** (Excel):  
  `Top_Institutions_By_Publications_<analysis_name>.xlsx`
- **Collaboration pair counts** (CSV):  
  `Institution_Collaboration_Counts_<analysis_name>.csv`
- **Network exports for Gephi/Cytoscape**:
  - `<prefix>.graphml`
  - `<prefix>_nodes.csv`
  - `<prefix>_edges.csv`
- **Plots** (PNG + PDF):
  - Top 10 institution bar chart
  - Collaboration network diagram for Top-N institutions

Finally, a summary workbook is generated:

- `Analysis_Summary_Report.xlsx`

---

### ✅ Mandatory User Changes

This section is driven by the **code cell directly below** (Cell 73).  
Users must check/update:

#### 1) `INST_COL_NAME` (**MANDATORY**)
This must match the column in `df` that contains the **list of institutions per paper**.

Typical examples:
```python
INST_COL_NAME = "Institutions_Extracted"  # default in this notebook
# or (depending on your earlier parsing):
INST_COL_NAME = "Institutions_List"
```

**Why this matters:** if the column name is wrong, the institutional analysis cannot run.

#### 2) `top_n_list` and `top_n_plot` (**optional but recommended**)
These control runtime and readability:

- `top_n_list=1000`: how many institutions to rank/export.
- `top_n_plot=30`: how many nodes to show in the plotted network.

If your dataset is large, reduce the plot size:
```python
top_n_plot = 25  # or 20 for very dense networks
```

---

In [ ]:
# ===================================================================
# 5) Main Execution: Run Comprehensive Batch Analysis
# ===================================================================

# Define the column name containing the LIST of institutions
INST_COL_NAME = 'Institutions_Extracted' 

# --- CRITICAL CHECK ---
if INST_COL_NAME not in df.columns:
      print(f"\n\nFATAL ERROR: The required institution column '{INST_COL_NAME}' was not found in df.")
      print("Cannot run institutional analysis. Check the column name created in the affiliation processing step.")
      print("Available columns in df:", df.columns.tolist())
elif 'Country_Classification' not in df.columns:
      print(f"\n\nFATAL ERROR: The required 'Country_Classification' column was not found in df.")
      print("Cannot filter for MCP/SCP analysis. Run the Country Collaboration Analysis step first.")
elif 'Countries_Extracted_List' not in df.columns:
      print(f"\n\nFATAL ERROR: The required 'Countries_Extracted_List' column was not found in df.")
      print("This is needed for the EU filter. Please run the Country Collaboration Analysis step (Cell 17) first.")
else:
      # Check data type of the column to ensure it's a list
      sample_value = df[INST_COL_NAME].iloc[0] if not df.empty and len(df[INST_COL_NAME]) > 0 else None
      sample_type = type(sample_value)
      print(f"Using institution column: '{INST_COL_NAME}' (Sample data type: {sample_type})")
      if sample_type != list and not df.empty:
          print(f"WARNING: Column '{INST_COL_NAME}' does not appear to contain lists. Ensure affiliation parsing created lists.")

      # --- Run the BATCH analysis for all 6 combinations ---
      print("\n>>> Running Comprehensive Batch Analysis for ALL 6 network types... <<<")
      
      # We use 'df' as the main dataframe
      all_network_results = run_comprehensive_analysis(
          df_main=df, 
          inst_col_name=INST_COL_NAME,
          top_n_list=1000,
          top_n_plot=30 # Plot Top 30 for each network
      )
      
      # --- Generate the Summary Report ---
      summary_report_path = os.path.join(results_dir, 'Analysis_Summary_Report.xlsx')
      summary_df = generate_summary_report(all_network_results, summary_report_path)
      
      print("\n--- Batch Analysis Summary Report ---")
      display(summary_df)
      
      print("\nAll institutional analyses complete.")
      print("Data for advanced validation (community detection) is stored in the 'all_network_results' dictionary.")

## 28. Advanced Institutional Analysis: Setup and Libraries

This cell prepares the environment for advanced network validation. This is an optional but highly recommended step in a rigorous network analysis.

It attempts to install and import specialized libraries:
* **`python-louvain`**: The standard library for performing Louvain community detection, a high-performance algorithm to find clusters (communities) in a network.
* **`scipy`** & **`scikit-learn`**: Used for statistical validation, including K-Fold cross-validation, similarity scores (NMI), and hierarchical clustering.
* **`tqdm`**: A utility for displaying progress bars during long-running calculations.

A flag, `advanced_libs_available`, is set. If these libraries fail to install, the advanced analysis in the following cells will be skipped, but the rest of the notebook will still function.

### ✅ Mandatory User Changes

None.

However, **you may need minor edits depending on environment**:

- In Kaggle: `pip.main([...])` usually works.
- In local Jupyter: you may prefer:
  ```python
  !pip install python-louvain tqdm scipy scikit-learn
  ```

---

### Troubleshooting Notes

If you see import errors:

- Confirm you are using the correct package name:
  - pip package: `python-louvain`
  - import path: `community.community_louvain`

If running in a restricted environment (no installs allowed), set:
```python
advanced_libs_available = False
```
and proceed with the rest of the notebook.

---

In [ ]:

try:
    import pip
    print("Installing/Checking required libraries: python-louvain, tqdm, scipy, scikit-learn...")
    # Install the necessary packages quietly
    pip.main(['install', '-q', 'python-louvain', 'tqdm', 'scipy', 'scikit-learn'])
    print("Libraries checked/installed.")

    # Import the key algorithm
    import community.community_louvain as community_louvain
    # Import validation tools
    from scipy.cluster.hierarchy import linkage, fcluster
    from scipy.spatial.distance import squareform
    from sklearn.model_selection import KFold
    from sklearn.metrics import normalized_mutual_info_score
    print("Advanced analysis libraries imported successfully.")
    advanced_libs_available = True

except ImportError as e:
    print(f"ERROR: Failed to import required libraries ({e}). Please install them manually.")
    print("Required: pip install python-louvain tqdm scipy scikit-learn")
    advanced_libs_available = False
except Exception as e:
    print(f"An unexpected error occurred during library setup: {e}")
    advanced_libs_available = False

## OPTIONAL: Thematic-Institutional Analysis: Collaboration Networks by Topic

This advanced analysis combines the results from Part 3 (Topic Modeling) and Part 4 (Institutional Analysis).

Instead of building one giant network for the entire corpus, this script will **loop through each topic** identified by the LDA model and build a **separate institutional collaboration network for each one**.

This allows us to move from the general question "Who are the most central institutions?" to the much more specific and powerful question: "Who are the key institutions leading the research on *Topic 1: [e.g., Planet Formation]*?" or *Topic 2: [e.g., Orbital Dynamics]*?"

### Methodology
1.  **Iterate**: The script gets the list of all unique `Dominant_Topic` numbers (e.g., 1, 2, 3...).
2.  **Filter**: For each topic, it filters the main `df_final` to create a temporary dataframe containing *only* the articles assigned to that topic.
3.  **Analyze**: It passes this filtered dataframe to the same `analyze_network` orchestrator function we used in Part 4.
4.  **Output**: This process generates a distinct set of outputs for each topic, including:
    * A ranked list of the top institutions *for that topic* (e.g., `Top_Institutions_By_Publications_Topic_1.xlsx`).
    * A bar chart of the top 10 institutions *for that topic*.
    * A full collaboration graph *for that topic* (e.g., `Collaboration_Network_Top_25_Topic_1.png`).


### ✅ Mandatory User Changes

This analysis depends on earlier steps and one key configuration:

#### 1) Topic modeling must already be run (**MANDATORY prerequisite**)
You must have:

- `df_final` available
- `df_final["Dominant_Topic"]` computed (from the final LDA step)

If topic modeling hasn’t run, this section cannot execute.

#### 2) `INST_COL_NAME` must exist (**MANDATORY prerequisite**)
This must match your institution list column in `df_final` (often the same as `df`).

---

### Recommended Tuning (Optional)

This section can generate **many networks** (one per topic).  
Control output volume by adjusting:

- `plot_n_for_graph=25` (default)
  - Lower it for dense topics: `15–20`
- Consider skipping topics with very few papers:
  ```python
  if len(df_topic) < 30:
      continue
  ```

You can also limit topics explicitly:
```python
topics_to_run = [1, 2, 3]  # focus subset
```

---

In [ ]:
# ===================================================================
# Main Execution: Analyze Institutional Networks *per Topic*
# ===================================================================

# This dictionary will store all the results
topic_network_results = {}

# --- CRITICAL CHECK ---
# Check if the necessary components from previous steps are available
if 'df_final' not in locals() or df_final.empty:
    print("FATAL ERROR: 'df_final' not found. Cannot run analysis.")
elif 'Dominant_Topic' not in df_final.columns:
    print("FATAL ERROR: 'Dominant_Topic' column not found. Run the Topic Modeling (Part 3) steps first.")
elif 'INST_COL_NAME' not in locals() or INST_COL_NAME not in df_final.columns:
    print(f"FATAL ERROR: 'INST_COL_NAME' ('{INST_COL_NAME}') not found. Run the Institutional Analysis (Part 4) setup first.")
elif 'analyze_network' not in locals():
    print("FATAL ERROR: 'analyze_network' function not found. Run the Institutional Analysis (Part 4) setup first.")
else:
    # Get all unique topics from the dataframe
    unique_topics = sorted(df_final['Dominant_Topic'].unique())
    print(f"Found {len(unique_topics)} topics to analyze: {unique_topics}")

    # --- Loop through each topic ---
    for topic_num in unique_topics:
        print(f"\n{'='*30} Analyzing Topic {topic_num} {'='*30}")
        
        # 1. Filter the main dataframe to get *only* articles for this topic
        df_topic = df_final[df_final['Dominant_Topic'] == topic_num].copy()
        
        if df_topic.empty:
            print(f"Topic {topic_num} has no articles. Skipping.")
            continue
        
        print(f"Topic {topic_num} has {len(df_topic)} articles.")
        
        # 2. Run the full network analysis pipeline on this filtered subset
        # We only run 'All' type, as MCP/SCP filtering is less relevant here.
        # We also plot a smaller Top 25 graph to keep it readable.
        try:
            (
                G_collab, 
                G_top, 
                df_exploded, 
                coll_counts, 
                top_inst 
            ) = analyze_network(
                df_main=df_topic,
                network_type=f"Topic_{topic_num}",  # Creates unique filenames (e.g., "Top_10_Institutions_Topic_1.png")
                publication_type='All',
                inst_col=INST_COL_NAME,
                top_n_for_list=1000,
                plot_n_for_graph=25 # Plot Top 25 for a less crowded graph
            )
            
            # 3. Store the results in our dictionary
            topic_network_results[topic_num] = {
                'full_graph': G_collab,
                'top_graph': G_top,
                'top_institutions': top_inst,
                'collaboration_counts': coll_counts,
                'exploded_data': df_exploded
            }
            
            print(f"--- Finished analysis for Topic {topic_num} ---")
        
        except Exception as e:
            print(f"!!! ERROR analyzing Topic {topic_num}: {e} !!!")
            # Continue to the next topic
            pass

print("\n\nAll topic-level institutional analyses complete.")
print(f"Results for each topic (graphs, top institutions) are stored in the 'topic_network_results' dictionary.")

## 29. Advanced Institutional Analysis: Validation Function Definitions

This cell defines all the complex functions required to validate the institutional collaboration network. This is a methodologically advanced step that assesses the **stability** and **statistical significance** of the network's structure. No analysis is run in this cell; it only defines the functions.

The workflow is as follows:

1.  **Basic Stats Functions**:
    * `evaluate_network_density`: Calculates how "dense" (interconnected) the network is.
    * `analyze_connected_components`: Checks if the network is one single graph or fragmented into separate components.

2.  **Centrality Validation Function**:
    * `cross_validate_centrality`: Performs **K-Fold cross-validation** on centrality metrics. It repeatedly removes a fraction of the network (collaboration links) and recalculates centrality scores (Degree, Betweenness, etc.). It then measures the correlation between the scores from each fold. A high average correlation suggests the centrality scores are **stable** and robust, not just an artifact of a few specific links.

3.  **Community Detection & Consensus Functions**:
    * `enhanced_community_detection`: Runs the **Louvain algorithm** on the graph to identify clusters (communities) of institutions that collaborate more frequently with each other than with outsiders.
    * `consensus_cross_validation`: This is the core stability function. It runs the Louvain algorithm repeatedly on different K-Fold samples of the network.
    * `create_consensus_clusters`: It builds a "co-occurrence" matrix (tracking how many times two institutions were clustered together across all folds) and uses hierarchical clustering to find the final, **stable "consensus" communities**.


4.  **Stability & Significance Functions**:
    * `create_membership_vector` / `partition_similarity`: Helper functions to calculate the **Normalized Mutual Information (NMI)**, a score from 0 to 1 that measures the similarity between two different community partitions.
    * `calculate_stability_metrics`: Uses NMI to calculate the **"Cross-Fold NMI"** (the average similarity *between* all K-Fold partitions). A high value suggests the community structure is stable.
    * `generate_null_distribution`: A crucial step for statistical significance. It creates many **random graphs** (using a **Configuration Model**) that have the *exact same* node degrees as the real network but random connections. It then calculates the average NMI for these random graphs.
    * `print_validation_results`: A reporting function that prints a summary table comparing the **Observed NMI** of the real network to the **Null Model NMI**. A Z-score and p-value are calculated to show if the observed stability is statistically significant (i.e., not random).


5.  **Main Orchestrator**:
    * `evaluate_and_analyze_network_advanced`: This is the main function that calls all the other functions in the correct order to run the complete validation pipeline.

In [ ]:
# --- Additional Imports for Advanced Analysis ---
from sklearn.metrics import normalized_mutual_info_score
import community.community_louvain as community_louvain
from networkx.algorithms import community as nx_community
from collections import defaultdict, Counter
from tqdm import tqdm
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from itertools import combinations
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from sklearn.model_selection import KFold
import os
import re
import ast
import numpy as np
import pandas as pd
import networkx as nx
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display
from scipy import linalg


# ===================================================================
# Function Definitions for Advanced Network Evaluation & Community Detection
# ===================================================================

def evaluate_network_density(G, network_type='AdvancedAnalysis'):
    """Calculates and prints network density."""
    if not isinstance(G, nx.Graph) or G.number_of_nodes() == 0:
        print(f"[{network_type}] Network is empty or invalid, density is 0.")
        return 0.0
    density = nx.density(G)
    print(f"[{network_type}] Network Density: {density:.4f}")
    return density


def analyze_connected_components(G, network_type='AdvancedAnalysis'):
    """Analyzes and prints connected components, returns list of component subgraphs."""
    if not isinstance(G, nx.Graph) or G.number_of_nodes() == 0:
        print(f"[{network_type}] Network is empty, no connected components.")
        return []
    try:
        connected_components_sets = list(nx.connected_components(G))
        connected_components = sorted(connected_components_sets, key=len, reverse=True)
        print(f"[{network_type}] Number of Connected Components: {len(connected_components)}")
        for i, comp_set in enumerate(connected_components[:5], 1):
            print(f"  Component {i}: {len(comp_set)} nodes")
        return [G.subgraph(c_set).copy() for c_set in connected_components]
    except Exception as e:
        print(f"[{network_type}] Error analyzing connected components: {e}")
        return []


def cross_validate_centrality(df_collab_pairs_input, n_splits=5, network_type='AdvancedAnalysis'):
    """
    Performs K-Fold cross-validation to assess centrality measure stability.
    """
    df_collab_pairs = df_collab_pairs_input.copy()
    required_cols = ['Institution1', 'Institution2', 'Collaboration_Count']
    if not all(col in df_collab_pairs.columns for col in required_cols):
        print(f"[{network_type}] Input DataFrame missing required columns: {required_cols}. Skipping centrality validation.")
        return {}, {}, {}

    df_collab_pairs.dropna(subset=required_cols, inplace=True)
    df_collab_pairs['Institution1'] = df_collab_pairs['Institution1'].astype(str)
    df_collab_pairs['Institution2'] = df_collab_pairs['Institution2'].astype(str)
    df_collab_pairs = df_collab_pairs[
        (df_collab_pairs['Institution1'].str.strip() != '') &
        (df_collab_pairs['Institution2'].str.strip() != '') &
        (df_collab_pairs['Institution1'].str.lower() != 'nan') &
        (df_collab_pairs['Institution2'].str.lower() != 'nan')
    ]
    df_collab_pairs['Collaboration_Count'] = pd.to_numeric(df_collab_pairs['Collaboration_Count'], errors='coerce').fillna(0)

    if df_collab_pairs.empty or len(df_collab_pairs) < n_splits:
        print(f"[{network_type}] Not enough valid collaboration pairs ({len(df_collab_pairs)}) for K-Fold (k={n_splits}). Skipping centrality validation.")
        return {}, {}, {}

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    centrality_results = {}
    all_institutions_set = set(df_collab_pairs['Institution1']) | set(df_collab_pairs['Institution2'])
    all_institutions_list = sorted(list(all_institutions_set))

    print(f"[{network_type}] Starting centrality cross-validation (k={n_splits})...")
    for fold_idx, (train_index, _) in enumerate(kf.split(df_collab_pairs), start=1):
        df_train = df_collab_pairs.iloc[train_index]
        if df_train.empty:
            print(f"[{network_type}] Fold-{fold_idx} training set is empty. Skipping.")
            centrality_results[f'Fold-{fold_idx}'] = pd.DataFrame(0.0, index=all_institutions_list, columns=['Degree_Centrality', 'Betweenness_Centrality', 'Closeness_Centrality', 'Eigenvector_Centrality'])
            continue

        G_fold = nx.Graph()
        for _, row in df_train.iterrows():
            inst1, inst2 = str(row['Institution1']), str(row['Institution2'])
            if inst1 and inst2: 
                G_fold.add_edge(inst1, inst2, weight=float(row['Collaboration_Count']))

        fold_nodes = list(G_fold.nodes())
        if not fold_nodes:
            print(f"[{network_type}] Fold-{fold_idx} graph has no nodes. Skipping.")
            centrality_results[f'Fold-{fold_idx}'] = pd.DataFrame(0.0, index=all_institutions_list, columns=['Degree_Centrality', 'Betweenness_Centrality', 'Closeness_Centrality', 'Eigenvector_Centrality'])
            continue
        
        degree, betweenness, closeness = {}, {}, {}
        eigenvector = {n: 0.0 for n in fold_nodes} 

        try:
            degree = nx.degree_centrality(G_fold)
            betweenness = nx.betweenness_centrality(G_fold, normalized=True, endpoints=False, weight=None)
            closeness = nx.closeness_centrality(G_fold)
            
            valid_nodes_for_eig = [n for n in fold_nodes if G_fold.degree(n) > 0]
            if valid_nodes_for_eig:
                G_fold_sub = G_fold.subgraph(valid_nodes_for_eig)
                for component_nodes in nx.connected_components(G_fold_sub):
                    comp_size = len(component_nodes)
                    if comp_size > 1: 
                        G_comp = G_fold_sub.subgraph(component_nodes)
                        try:
                            if comp_size < 100:
                                adj_matrix_dense = nx.to_numpy_array(G_comp, nodelist=sorted(component_nodes), weight='weight')
                                eigenvalues, eigenvectors_matrix = linalg.eig(adj_matrix_dense)
                                max_eigenvalue_idx = np.argmax(eigenvalues.real)
                                dominant_eigenvector = np.abs(eigenvectors_matrix[:, max_eigenvalue_idx].real)
                                norm_factor = np.linalg.norm(dominant_eigenvector)
                                if norm_factor > 1e-9: 
                                    dominant_eigenvector /= norm_factor
                                eigenvector_comp = dict(zip(sorted(component_nodes), dominant_eigenvector))
                            else: 
                                eigenvector_comp = nx.eigenvector_centrality_numpy(G_comp, weight='weight', max_iter=2000, tol=1e-05)
                            eigenvector.update(eigenvector_comp)
                        except (nx.PowerIterationFailedConvergence, nx.NetworkXError, linalg.LinAlgError) as e_eig:
                            print(f"[{network_type}] Eigenvector on comp. in Fold-{fold_idx} failed: {e_eig}. Nodes set to 0.")
                            for n_comp in component_nodes: eigenvector[n_comp] = 0.0
                        except Exception as e_eig_other:
                            print(f"[{network_type}] UNEXPECTED Eigenvector error on comp. in Fold-{fold_idx}: {e_eig_other}. Nodes set to 0.")
                            for n_comp in component_nodes: eigenvector[n_comp] = 0.0

        except Exception as e:
            print(f"[{network_type}] Error calculating centralities for Fold-{fold_idx}: {e}")
            if not isinstance(degree, dict): degree = {}
            if not isinstance(betweenness, dict): betweenness = {}
            if not isinstance(closeness, dict): closeness = {}
            eigenvector = {n: 0.0 for n in fold_nodes}

        fold_df_data = {
            'Degree_Centrality': [degree.get(n, 0.0) for n in all_institutions_list],
            'Betweenness_Centrality': [betweenness.get(n, 0.0) for n in all_institutions_list],
            'Closeness_Centrality': [closeness.get(n, 0.0) for n in all_institutions_list],
            'Eigenvector_Centrality': [eigenvector.get(n, 0.0) for n in all_institutions_list]
        }
        fold_df = pd.DataFrame(fold_df_data, index=all_institutions_list)
        centrality_results[f'Fold-{fold_idx}'] = fold_df

    correlations = {}
    consistency_scores = {}
    metrics = ["Degree_Centrality", "Betweenness_Centrality", "Closeness_Centrality", "Eigenvector_Centrality"]
    print(f"\n[{network_type}] Calculating centrality consistency scores...")
    for metric in metrics:
        metric_dfs = [centrality_results[f][metric] for f in centrality_results if metric in centrality_results[f].columns]
        if not metric_dfs or len(metric_dfs) < 2:
            correlations[metric], consistency_scores[metric] = pd.DataFrame(), np.nan
            print(f"[{network_type}] Insufficient data for '{metric}'.")
            continue
        
        metric_data = pd.concat(metric_dfs, axis=1)
        metric_data.columns = [f'Fold-{i+1}' for i in range(len(metric_dfs))]
        if metric_data.empty or metric_data.shape[1] < 2:
            correlations[metric], consistency_scores[metric] = pd.DataFrame(), np.nan
            print(f"[{network_type}] Invalid data for '{metric}'.")
            continue
            
        corr_mat = metric_data.corr()
        correlations[metric] = corr_mat
        upper_triangle = corr_mat.where(np.triu(np.ones(corr_mat.shape), k=1).astype(bool)).stack().dropna()
        consistency_scores[metric] = upper_triangle.mean() if not upper_triangle.empty else np.nan
        print(f"[{network_type}] Avg Consistency {metric}: {consistency_scores[metric]:.3f}" if not np.isnan(consistency_scores[metric]) else f"[{network_type}] Avg Consistency {metric}: NaN")

    consistency_df = pd.DataFrame.from_dict(consistency_scores, orient='index', columns=['Average_Correlation'])
    csv_path = os.path.join(results_dir, f'centrality_consistency_scores_{network_type}.csv')
    consistency_df.to_csv(csv_path)
    print(f"[{network_type}] Centrality consistency scores saved to '{csv_path}'.")
    return centrality_results, correlations, consistency_scores
    
def enhanced_community_detection(G, algorithms=None, consensus_threshold=1, network_type='AdvancedAnalysis'):
    """Performs community detection using Louvain algorithm."""
    if not isinstance(G, nx.Graph) or G.number_of_nodes() == 0:
        print(f"[{network_type}] Input graph is invalid or empty. Skipping community detection.")
        return [], G

    if algorithms is None:
        algorithms = {'louvain': community_louvain.best_partition}

    all_partitions = {}
    print(f"\n[{network_type}] Running community detection algorithms...")
    for algo_name, algo_func in algorithms.items():
        partition_comms = []
        try:
            if algo_name == 'louvain':
                weight_key = 'weight' if nx.get_edge_attributes(G, 'weight') else None
                partition_map = algo_func(G, weight=weight_key, random_state=42)
                comms_dict = defaultdict(list)
                for node, comm_id in partition_map.items(): comms_dict[comm_id].append(node)
                partition_comms = [sorted(c_list) for c_list in comms_dict.values() if len(c_list) > 1] 
            else:
                print(f"[{network_type}] Algorithm '{algo_name}' not implemented.")
                continue

            all_partitions[algo_name] = partition_comms
            print(f"[{network_type}] {algo_name.capitalize()} found {len(partition_comms)} non-singleton communities.")

        except Exception as e:
            print(f"[{network_type}] ERROR running {algo_name}: {e}")
            all_partitions[algo_name] = []

    final_consensus_comms = []
    G_sub = G 
    if all_partitions:
        first_successful_algo = next((name for name, res in all_partitions.items() if res is not None), None)
        if first_successful_algo:
            final_consensus_comms = all_partitions[first_successful_algo]
            print(f"[{network_type}] Using results from '{first_successful_algo}' as the final partition.")
            consensus_nodes = {node for comm in final_consensus_comms for node in comm}
            if consensus_nodes:
                G_sub = G.subgraph(consensus_nodes).copy()
                G_sub.remove_nodes_from(list(nx.isolates(G_sub)))
                print(f"[{network_type}] Subgraph for consensus communities: {G_sub.number_of_nodes()} nodes, {G_sub.number_of_edges()} edges.")
            else: G_sub = nx.Graph()
        else:
            print(f"[{network_type}] No algorithm produced valid communities. Returning empty consensus.")
            G_sub = nx.Graph()
    else:
        print(f"[{network_type}] No algorithms were run or all failed. Returning empty consensus.")
        G_sub = nx.Graph()

    return final_consensus_comms, G_sub

def create_membership_vector(communities_list_of_lists, all_nodes_in_scope_list):
    """Creates a membership vector (list) for NMI calculation."""
    node_to_universal_idx = {node: i for i, node in enumerate(all_nodes_in_scope_list)}
    membership_array = np.full(len(all_nodes_in_scope_list), -1, dtype=int) 
    for comm_idx, comm_nodes_list in enumerate(communities_list_of_lists):
        for node in comm_nodes_list:
            if node in node_to_universal_idx:
                membership_array[node_to_universal_idx[node]] = comm_idx
    return membership_array.tolist()


def partition_similarity(partition1_comms, partition2_comms, nodes_scope_set):
    """Calculate Normalized Mutual Information (NMI) between two partitions."""
    if not nodes_scope_set: return 0.0
    list_nodes_in_scope = sorted(list(nodes_scope_set))
    
    p1_scoped = [[n for n in c if n in nodes_scope_set] for c in partition1_comms]
    p1_scoped = [c for c in p1_scoped if c] 
    p2_scoped = [[n for n in c if n in nodes_scope_set] for c in partition2_comms]
    p2_scoped = [c for c in p2_scoped if c]
    
    if not p1_scoped and not p2_scoped: return 1.0
    if not p1_scoped or not p2_scoped: return 0.0
    
    labels1 = create_membership_vector(p1_scoped, list_nodes_in_scope)
    labels2 = create_membership_vector(p2_scoped, list_nodes_in_scope)
    
    if all(l == -1 for l in labels1) and all(l == -1 for l in labels2): return 1.0
    if all(l == -1 for l in labels1) or all(l == -1 for l in labels2): return 0.0

    try:
        return normalized_mutual_info_score(labels1, labels2, average_method='arithmetic')
    except ValueError as e:
        print(f"NMI calc error: {e}. Ret 0.")
        return 0.0

def calculate_stability_metrics(fold_partitions_list, consensus_clusters, all_graph_nodes_set, network_type='AdvancedAnalysis'):
    """Calculate NMI-based stability metrics (Cross-Fold and Consensus Similarity)."""
    metrics = {'cross_fold_nmi': [], 'consensus_similarity': []}

    for i in range(len(fold_partitions_list)):
        for j in range(i + 1, len(fold_partitions_list)):
            nodes_in_p_i = set(n for c in fold_partitions_list[i] for n in c)
            nodes_in_p_j = set(n for c in fold_partitions_list[j] for n in c)
            comparison_scope = (nodes_in_p_i | nodes_in_p_j) & all_graph_nodes_set
            if comparison_scope:
                nmi_val = partition_similarity(fold_partitions_list[i], fold_partitions_list[j], comparison_scope)
                if not np.isnan(nmi_val): metrics['cross_fold_nmi'].append(nmi_val)

    if consensus_clusters:
        nodes_in_consensus_c = set(n for c in consensus_clusters for n in c)
        for fold_partition in fold_partitions_list:
            nodes_in_fold_p = set(n for c in fold_partition for n in c)
            comparison_scope = (nodes_in_fold_p & nodes_in_consensus_c) & all_graph_nodes_set
            if comparison_scope:
                nmi_val = partition_similarity(fold_partition, consensus_clusters, comparison_scope)
                if not np.isnan(nmi_val): metrics['consensus_similarity'].append(nmi_val)

    summary = {}
    for k, v_list in metrics.items():
        v_clean = [v for v in v_list if not np.isnan(v)]
        mean_val = np.mean(v_clean) if v_clean else np.nan
        std_val = np.std(v_clean) if v_clean else np.nan
        summary[k] = (mean_val, std_val)

    print(f"\n[{network_type}] Stability Metrics Summary:")
    for k, (mean_val, std_val) in summary.items():
        print(f"  Mean {k.replace('_', ' ').title()}: {mean_val:.4f} ± {std_val:.4f} (from {len(metrics[k])} comparisons)")
    return summary


def create_consensus_clusters(cooc_matrix_df, n_folds, threshold_factor=0.7):
    """Creates consensus clusters using hierarchical clustering on co-occurrence matrix."""
    if cooc_matrix_df.empty or cooc_matrix_df.shape[0] <= 1:
        print(f"Co-occurrence matrix empty or too small. Cannot create consensus clusters.")
        return []

    dist_matrix_df = 1.0 - (cooc_matrix_df / n_folds)
    np.fill_diagonal(dist_matrix_df.values, 0)
    dist_matrix_df.clip(lower=0, upper=1, inplace=True) 

    dist_values = dist_matrix_df.values
    if not np.allclose(dist_values, dist_values.T, atol=1e-8):
        print("Warning: Distance matrix not symmetric. Forcing symmetry.")
        dist_values = (dist_values + dist_values.T) / 2
        np.fill_diagonal(dist_values, 0)

    try:
        condensed_dist_matrix = squareform(dist_values, checks=True)
    except ValueError as e:
        print(f"Error creating condensed distance matrix: {e}. Cannot create consensus clusters.")
        return []

    if len(condensed_dist_matrix) == 0:
        return [list(dist_matrix_df.index)] if dist_matrix_df.shape[0] == 1 else []

    try:
        Z = linkage(condensed_dist_matrix, method='average')
    except Exception as e_linkage:
        print(f"Error during hierarchical clustering linkage: {e_linkage}. Cannot create consensus clusters.")
        return []

    distance_cutoff = 1.0 - threshold_factor
    if not (0 <= distance_cutoff <= 1):
         print(f"Warning: Invalid threshold factor {threshold_factor}. Using default cutoff 0.4.")
         distance_cutoff = 0.4 

    cluster_labels = fcluster(Z, t=distance_cutoff, criterion='distance')

    consensus_comms_dict = defaultdict(list)
    node_labels = cooc_matrix_df.index.tolist()
    for i, node_label in enumerate(node_labels):
        consensus_comms_dict[cluster_labels[i]].append(node_label)

    final_consensus_communities = sorted(
        [sorted(c) for c in consensus_comms_dict.values() if len(c) > 1],
        key=len, reverse=True
    )
    print(f"Hierarchical clustering (cut at dist={distance_cutoff:.2f}, sim={threshold_factor:.2f}) -> {len(final_consensus_communities)} consensus communities.")
    return final_consensus_communities

def generate_null_distribution(df_edges_input, n_splits, iterations, master_node_list_for_scope, network_type='AdvancedAnalysis'):
    """Generates null distribution of NMI scores using configuration model."""
    null_nmi_means_across_iterations = []
    df_edges = df_edges_input.copy()

    if df_edges.empty:
        print(f"[{network_type}] Null Model: Edge list empty.")
        return {'mean': np.nan, 'std': np.nan, 'values': []}
    df_edges['Institution1'] = df_edges['Institution1'].astype(str)
    df_edges['Institution2'] = df_edges['Institution2'].astype(str)
    df_edges = df_edges[
        (df_edges['Institution1'].str.strip() != '') &
        (df_edges['Institution2'].str.strip() != '') &
        (df_edges['Institution1'].str.lower() != 'nan') &
        (df_edges['Institution2'].str.lower() != 'nan')
    ]
    if df_edges.empty:
        print(f"[{network_type}] Null Model: Edge list empty after cleaning.")
        return {'mean': np.nan, 'std': np.nan, 'values': []}

    G_actual_for_degree = nx.from_pandas_edgelist(df_edges, 'Institution1', 'Institution2', create_using=nx.Graph())
    if not G_actual_for_degree.nodes():
        print(f"[{network_type}] Null Model: Actual graph empty.")
        return {'mean': np.nan, 'std': np.nan, 'values': []}

    node_list_orig_labels = sorted(list(G_actual_for_degree.nodes()))
    label_to_int_map = {label: i for i, label in enumerate(node_list_orig_labels)}
    int_to_label_map = {i: label for i, label in enumerate(node_list_orig_labels)}
    degree_sequence_for_config_model = [d for n, d in G_actual_for_degree.degree()]

    if not degree_sequence_for_config_model or sum(degree_sequence_for_config_model) == 0:
        print(f"[{network_type}] Null Model: Invalid degree sequence.")
        return {'mean': np.nan, 'std': np.nan, 'values': []}
    if sum(degree_sequence_for_config_model) % 2 != 0:
        print(f"[{network_type}] Null Model: Sum of degrees is odd ({sum(degree_sequence_for_config_model)}). Aborting null model.")
        return {'mean': np.nan, 'std': np.nan, 'values': []}

    print(f"[{network_type}] Starting null model generation ({iterations} iterations)...")
    for iter_num in tqdm(range(iterations), desc=f"[{network_type}] Null Model Iterations", leave=False):
        G_null_orig_labels = None 
        try:
            G_null_int_nodes = nx.configuration_model(degree_sequence_for_config_model, seed=iter_num + 777, create_using=nx.Graph())
            G_null_orig_labels = nx.Graph()
            G_null_orig_labels.add_nodes_from(node_list_orig_labels)
            for u_int, v_int in G_null_int_nodes.edges():
                if u_int in int_to_label_map and v_int in int_to_label_map:
                    G_null_orig_labels.add_edge(int_to_label_map[u_int], int_to_label_map[v_int])
            G_null_orig_labels.remove_edges_from(nx.selfloop_edges(G_null_orig_labels))

            if G_null_orig_labels.number_of_edges() == 0: continue

            null_edges_list = [{'Institution1': u, 'Institution2': v, 'Collaboration_Count': 1}
                               for u, v in G_null_orig_labels.edges()]
            if not null_edges_list: continue
            df_null_edges = pd.DataFrame(null_edges_list)
            if df_null_edges.empty or len(df_null_edges) < n_splits: continue

            current_iter_fold_nmis = []
            temp_fold_partitions = []
            kf_null = KFold(n_splits=n_splits, shuffle=True, random_state=iter_num + 888)

            for fold_num_null, (train_idx_null, _) in enumerate(kf_null.split(df_null_edges)):
                df_train_null = df_null_edges.iloc[train_idx_null].copy()
                if df_train_null.empty: continue
                
                df_train_null.loc[:, 'Institution1'] = df_train_null['Institution1'].astype(str)
                df_train_null.loc[:, 'Institution2'] = df_train_null['Institution2'].astype(str)
                
                G_train_null = nx.from_pandas_edgelist(df_train_null, 'Institution1', 'Institution2', create_using=nx.Graph())
                if G_train_null.number_of_nodes() == 0: continue

                try:
                    partition_null = community_louvain.best_partition(G_train_null, random_state=iter_num + fold_num_null + 1)
                    comms_dict_null = defaultdict(list)
                    for node, comm_id in partition_null.items(): comms_dict_null[comm_id].append(node)
                    comms_null = [sorted(c) for c in comms_dict_null.values() if len(c) > 1]
                    if comms_null: temp_fold_partitions.append(comms_null)
                except Exception: continue 

            if len(temp_fold_partitions) >= 2:
                fold_nmis_this_iteration = []
                nodes_in_this_null_graph = set(G_null_orig_labels.nodes())
                for i in range(len(temp_fold_partitions)):
                    for j in range(i + 1, len(temp_fold_partitions)):
                        nodes_p1_null = set(n for c in temp_fold_partitions[i] for n in c)
                        nodes_p2_null = set(n for c in temp_fold_partitions[j] for n in c)
                        scope = (nodes_p1_null | nodes_p2_null) & nodes_in_this_null_graph

                        if scope:
                            nmi_val = partition_similarity(temp_fold_partitions[i], temp_fold_partitions[j], scope)
                            if not np.isnan(nmi_val): fold_nmis_this_iteration.append(nmi_val)

                if fold_nmis_this_iteration:
                    null_nmi_means_across_iterations.append(np.mean(fold_nmis_this_iteration))

        except nx.NetworkXError as net_err:
            print(f"[{network_type}] NetworkXError in null iter {iter_num}: {net_err}")
            continue
        except Exception as e:
            print(f"[{network_type}] General error in null iter {iter_num}: {e}")
            continue

    if not null_nmi_means_across_iterations:
        print(f"[{network_type}] Null model: No valid NMI values generated across {iterations} iterations.")
        return {'mean': np.nan, 'std': np.nan, 'values': []}

    mean_nmi = np.mean(null_nmi_means_across_iterations)
    std_nmi = np.std(null_nmi_means_across_iterations)
    print(f"[{network_type}] Null model complete. Mean NMI: {mean_nmi:.4f} ± {std_nmi:.4f} (from {len(null_nmi_means_across_iterations)} valid iterations)")
    return {'mean': mean_nmi, 'std': std_nmi, 'values': null_nmi_means_across_iterations}

def consensus_cross_validation(df_collab_pairs_input, reference_comms_on_full_graph, n_splits=5, null_iter=100, network_type='AdvancedAnalysis'):
    """
    Orchestrator for K-Fold cross-validation to find consensus communities and their stability.
    """
    df_edges = df_collab_pairs_input.copy()
    required_cols = ['Institution1', 'Institution2', 'Collaboration_Count']
    if not all(col in df_edges.columns for col in required_cols):
        print(f"[{network_type}] df_edges for consensus validation missing required columns. Aborting.")
        return {}

    df_edges.dropna(subset=required_cols, inplace=True)
    df_edges['Institution1'] = df_edges['Institution1'].astype(str)
    df_edges['Institution2'] = df_edges['Institution2'].astype(str)
    df_edges = df_edges[
        (df_edges['Institution1'].str.strip() != '') &
        (df_edges['Institution2'].str.strip() != '') &
        (df_edges['Institution1'].str.lower() != 'nan') &
        (df_edges['Institution2'].str.lower() != 'nan')
    ]
    df_edges['Collaboration_Count'] = pd.to_numeric(df_edges['Collaboration_Count'], errors='coerce').fillna(0)

    if df_edges.empty or len(df_edges) < n_splits:
         print(f"[{network_type}] Not enough valid edges ({len(df_edges)}) for K-Fold (k={n_splits}). Aborting validation.")
         return {}

    all_nodes_in_original_edgelist = sorted(list(set(df_edges['Institution1']) | set(df_edges['Institution2'])))
    if not all_nodes_in_original_edgelist:
        print(f"[{network_type}] No nodes found in edge list for consensus validation. Aborting.")
        return {}

    cooc_matrix = pd.DataFrame(0, index=all_nodes_in_original_edgelist, columns=all_nodes_in_original_edgelist, dtype=int)
    fold_partitions_generated_list = [] 

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=123)
    print(f"[{network_type}] Starting consensus cross-validation (k={n_splits})...")
    
    for fold_num, (train_index, _) in enumerate(kf.split(df_edges), start=1):
        df_train_fold = df_edges.iloc[train_index].copy()

        df_train_fold.dropna(subset=['Institution1', 'Institution2', 'Collaboration_Count'], inplace=True)
        df_train_fold.loc[:, 'Institution1'] = df_train_fold['Institution1'].astype(str)
        df_train_fold.loc[:, 'Institution2'] = df_train_fold['Institution2'].astype(str)
        df_train_fold = df_train_fold[
             (df_train_fold['Institution1'].str.strip() != '') &
             (df_train_fold['Institution2'].str.strip() != '') &
             (df_train_fold['Institution1'].str.lower() != 'nan') &
             (df_train_fold['Institution2'].str.lower() != 'nan')
        ]
        df_train_fold.loc[:, 'Collaboration_Count'] = pd.to_numeric(df_train_fold['Collaboration_Count'], errors='coerce').fillna(0)

        if df_train_fold.empty:
            print(f"[{network_type}] Fold-{fold_num} training set became empty after cleaning. Skipping.")
            continue

        G_train_fold = nx.from_pandas_edgelist(df_train_fold, 'Institution1', 'Institution2',
                                               edge_attr='Collaboration_Count', create_using=nx.Graph())

        if G_train_fold.number_of_nodes() == 0:
            print(f"[{network_type}] Fold-{fold_num} graph has no nodes after creation. Skipping.")
            continue

        try:
            weight_key = 'Collaboration_Count' if nx.get_edge_attributes(G_train_fold, 'Collaboration_Count') else None
            partition_fold = community_louvain.best_partition(G_train_fold, weight=weight_key, random_state=fold_num)
            comms_dict_fold = defaultdict(list)
            for node, comm_id in partition_fold.items(): comms_dict_fold[comm_id].append(node)
            train_comms_this_fold = [sorted(c) for c in comms_dict_fold.values() if len(c) > 1] 

            if not train_comms_this_fold:
                print(f"[{network_type}] Fold-{fold_num} Louvain found no non-singleton communities.")
                for node in G_train_fold.nodes():
                    if node in cooc_matrix.index:
                        cooc_matrix.loc[node, node] += 1
                continue
            else:
                print(f"[{network_type}] Fold-{fold_num} Louvain found {len(train_comms_this_fold)} communities.")
                fold_partitions_generated_list.append(train_comms_this_fold)

            for comm_list_nodes in train_comms_this_fold:
                for i in range(len(comm_list_nodes)):
                    u = comm_list_nodes[i]
                    if u not in cooc_matrix.index: continue 
                    cooc_matrix.loc[u, u] += 1 
                    for j in range(i + 1, len(comm_list_nodes)):
                        v = comm_list_nodes[j]
                        if v not in cooc_matrix.index: continue 
                        cooc_matrix.loc[u, v] += 1
                        cooc_matrix.loc[v, u] += 1 

        except Exception as e:
             print(f"[{network_type}] ERROR during community detection or co-occurrence update for Fold-{fold_num}: {e}")
             for node in G_train_fold.nodes(): 
                 if node in cooc_matrix.index:
                     cooc_matrix.loc[node, node] += 1
             continue 

    print(f"\n[{network_type}] Co-occurrence matrix built ({cooc_matrix.shape[0]} nodes). Max value: {cooc_matrix.values.max()}")

    consensus_clusters = create_consensus_clusters(cooc_matrix, n_splits, threshold_factor=0.6) 

    stability_metrics = calculate_stability_metrics(
        fold_partitions_generated_list, 
        consensus_clusters, 
        set(all_nodes_in_original_edgelist),
        network_type=network_type
    )

    null_dist = generate_null_distribution(
        df_edges, 
        n_splits,
        null_iter,
        all_nodes_in_original_edgelist,
        network_type=network_type
    )

    return {
        'consensus_clusters': consensus_clusters,
        'stability_metrics': stability_metrics,
        'null_model': null_dist,
        'cooccurrence_matrix': cooc_matrix
    }


def print_validation_results(results, top_n_communities_to_show=5, network_type='AdvancedAnalysis'):
    """Prints a formatted report of community validation results."""

    print("\n" + "="*80)
    print(f"====== Community Validation Report for {network_type} Network ======")
    print("="*80)

    communities = results.get('consensus_clusters', [])
    print("\n[1] Consensus Community Structure")
    if communities:
        sizes = sorted([len(c) for c in communities], reverse=True)
        print(f"    Discovered: {len(communities)} consensus communities (non-singleton).")
        if sizes:
            print(f"    Size range: {min(sizes)}-{max(sizes)} institutions.")
            print(f"    Median size: {np.median(sizes):.0f} institutions.")
        else: print("    No consensus communities of size > 1 to report sizes for.")

        print(f"\n    Top {min(top_n_communities_to_show, len(communities))} communities by size:")
        for i, comm in enumerate(sorted(communities, key=len, reverse=True)[:min(top_n_communities_to_show, len(communities))], 1):
            sample_institutions = ', '.join(comm[:min(5,len(comm))]) + ('...' if len(comm) > 5 else '') 
            print(f"      Community {i}: {len(comm)} institutions (Sample: {sample_institutions})")
    else:
        print("    No consensus communities found or provided.")

    sm = results.get('stability_metrics', {})
    print("\n[2] Stability Metrics (Mean ± SD)")
    header = "| {:<28} | {:<18} | {:<40} |".format("Metric", "Value (Mean ± SD)", "Interpretation Guideline")
    print(header)
    print("|" + "-"*29 + "|" + "-"*19 + "|" + "-"*41 + "|")
    interpretations = {
        'cross_fold_nmi': ">0.60: Significant structure",
        'consensus_similarity': ">0.80: High consensus agreement"
    }
    # --- FIX: Iterate over specific keys to avoid v1's 'within_fold_nmi' ---
    for metric_key in ['cross_fold_nmi', 'consensus_similarity']: 
        if metric_key in sm:
            mean, std = sm[metric_key]
            val_str = f"{mean:.3f} ± {std:.3f}" if not (np.isnan(mean) or np.isnan(std)) else "N/A"
            print("| {:<28} | {:<18} | {:<40} |".format( metric_key.replace('_', ' ').title(), val_str, interpretations.get(metric_key, "")))
        else:
            print("| {:<28} | {:<18} | {:<40} |".format( metric_key.replace('_', ' ').title(), "N/A", interpretations.get(metric_key, "")))


    null_model_results = results.get('null_model', {})
    print("\n[3] Statistical Significance (vs. Configuration Null Model)")
    if null_model_results and 'mean' in null_model_results and not np.isnan(null_model_results['mean']):
        observed_nmi_tuple = sm.get('cross_fold_nmi', (np.nan, np.nan))
        observed_nmi = observed_nmi_tuple[0] if not np.isnan(observed_nmi_tuple[0]) else np.nan

        null_mean_nmi = null_model_results['mean']
        null_std_nmi = null_model_results['std']
        null_values = null_model_results.get('values', [])

        print(f"    Observed Cross-Fold NMI: {observed_nmi:.3f}" if not np.isnan(observed_nmi) else "    Observed Cross-Fold NMI: N/A")
        print(f"    Null Model Mean NMI (from {len(null_values)} iterations): {null_mean_nmi:.3f} ± {null_std_nmi:.3f}")

        if not np.isnan(observed_nmi) and null_values:
            cleaned_null_values = np.array([nv for nv in null_values if not np.isnan(nv)])
            if len(cleaned_null_values) > 0:
                p_value = np.mean(cleaned_null_values >= observed_nmi)
                print(f"    Empirical p-value: {p_value:.4f}")

                if not np.isnan(null_std_nmi) and null_std_nmi > 1e-9:
                    z_score = (observed_nmi - null_mean_nmi) / null_std_nmi
                    print(f"    Z-score: {z_score:.2f}")
                else: print("    Z-score: N/A (Null stddev is zero or NaN)")
                print(f"    Interpretation: A low p-value (e.g., < 0.05) suggests the observed community stability is significantly greater than random.")

            else:
                print("    p-value/Z-score: N/A (no valid null model values)")
        else:
            print("    p-value/Z-score: N/A (Observed NMI or null values missing/invalid)")
    else:
        print("    Null model results are unavailable or incomplete.")

    print("\n[4] NMI Interpretation Guide")
    print("| {:<18} | {:<25} |".format("NMI Range", "Interpretation"))
    print("|" + "-"*19 + "|" + "-"*26 + "|")
    print("| {:<18} | {:<25} |".format("0.90 - 1.00", "Excellent / Near Identical"))
    print("| {:<18} | {:<25} |".format("0.75 - 0.89", "Strong / Good Agreement"))
    print("| {:<18} | {:<25} |".format("0.60 - 0.74", "Moderate Agreement"))
    print("| {:<18} | {:<25} |".format("0.40 - 0.59", "Fair / Weak Agreement"))
    print("| {:<18} | {:<25} |".format("< 0.40", "Poor / Little Agreement"))
    print("\n" + "="*80)

## 30. Advanced Institutional Analysis: Main Execution and Validation (Auto-Scaling)

This cell **runs** the advanced validation pipeline on the institutional collaboration networks produced in the previous step.

Because some users may run this notebook on **very large corpora** (hundreds of thousands to millions of papers), this step includes an **auto-scaling validation strategy** so the notebook remains usable across hardware budgets while preserving methodological transparency.

---

### 30.1 What gets validated

By default, we validate the most informative networks:

- **Global_All** — the full institutional collaboration network built from *all included publications*
- **Global_MCP** — the institutional collaboration network built from *internationally co-authored publications (MCP)*
- **EU_All** — EU-only publications *(optional branch, if enabled earlier)*
- **EU_MCP** — EU-only MCP *(optional branch)*

We **skip SCP networks** for advanced validation because SCP networks are typically sparse by definition and community stability metrics are less meaningful there.

---

### 30.2 What this validation pipeline does

For each selected network, we run:

1) **Basic network statistics**
   - density  
   - connected components (fragmentation / largest component size)

2) **Centrality robustness (K-Fold stability)**
   - Degree, Betweenness, Closeness, Eigenvector  
   - the network’s edge list is split into folds, centralities are re-computed per fold, and **cross-fold correlations** are reported

3) **Community robustness (K-Fold + consensus clustering)**
   - Louvain partitions are computed repeatedly on K-Fold samples  
   - a co-occurrence matrix is built (how often two institutions appear in the same community across folds)  
   - hierarchical clustering produces **consensus communities**

4) **Statistical significance vs. a null model**
   - configuration-model graphs preserve node degrees but randomize wiring  
   - observed NMI stability is compared to the null distribution (p-value / z-score)

5) **A “Community Validation Report”**
   - consensus community size summaries  
   - stability metrics (NMI)  
   - null model comparison and interpretation guide

---

### 30.3 Auto-Scaling: why validation may run on a subgraph

Some steps here are intrinsically expensive for large graphs—especially the **N×N co-occurrence matrix** used for consensus clustering.

To ensure this notebook runs reliably for everyone, validation is executed under one of four modes:

#### ✅ Validation modes

- **auto (default):** chooses the most rigorous option that is safe for the graph size  
- **full:** validates the full graph (best for small/medium networks; may be infeasible for very large graphs)  
- **fast:** validates a computationally cheaper representation (typically the Largest Connected Component)  
- **off:** skips advanced validation completely  

#### What “auto” does (in plain terms)

1) **Always prefers the Largest Connected Component (LCC)**  
   - the LCC contains the main collaboration structure and avoids tiny isolated components distorting stability measures

2) If the LCC is still too large, it validates a **core subgraph**  
   - usually the **top-degree institutions** (most connected nodes)  
   - optional edge sampling may be used for cross-validation speed

> **Important:** this does **not** change your exported full network.  
> The full network is still saved for external tools (Gephi/Cytoscape) and descriptive statistics.  
> Auto-scaling only affects advanced validation computation.

---

### ✅ What you may want to change

Most users should not change anything.

If you want to override the auto-scaling behavior, edit only:

```python
ADVANCED_VALIDATION_MODE = "auto"  # "auto", "full", "fast", "off"
```

Optional (advanced users):

```python
MAX_NODES_FOR_FULL = 3500
MAX_EDGES_FOR_FULL = 250000
CORE_MAX_NODES = 3500
CORE_PICK_STRATEGY = "top_degree"  # or "random"
```

---

### Expected outputs

The cell prints:

- per-network density + components summary  
- centrality stability correlations  
- consensus community validation report  

Saved outputs (examples):

**Kaggle**
- `/kaggle/working/output/Results/centrality_consistency_scores_Global_All.csv`

**Google Colab**
- `/content/output/Results/centrality_consistency_scores_Global_All.csv`

**Local Jupyter**
- `./output/Results/centrality_consistency_scores_Global_All.csv`


In [ ]:
# ===================================================================
# Main Orchestrator for Advanced Evaluation
# ===================================================================

def evaluate_and_analyze_network_advanced(G_input_collab, collaboration_counts_df, network_type='AdvancedAnalysis'):
    """Performs density, components, centrality stability, and community validation.
       RETURNS the community_validation_results dictionary."""

    print(f"\n\n{'='*20} Starting Advanced Evaluation for {network_type} Network {'='*20}")
    G_collab = G_input_collab.copy()

    # Initialize return dictionary in case of early exit
    community_validation_results = {}

    if not isinstance(G_collab, nx.Graph) or G_collab.number_of_nodes() == 0:
        print(f"[{network_type}] Input graph is empty or invalid. Skipping all advanced analysis.")
        return community_validation_results # Return empty dict

    # --- Basic Network Stats ---
    density = evaluate_network_density(G_collab, network_type=network_type)
    connected_subgraphs = analyze_connected_components(G_collab, network_type=network_type)

    # --- Prepare df_collab_pairs for Validation ---
    print(f"[{network_type}] Preparing collaboration pairs DataFrame for validation...")
    df_collab_pairs_for_validation = pd.DataFrame()
    required_cols_input = ['Institution1', 'Institution2', 'Collaboration_Count']

    # Check if the input df (collaboration_counts_df) already has the 3 required columns
    if all(col in collaboration_counts_df.columns for col in required_cols_input):
        df_collab_pairs_for_validation = collaboration_counts_df[required_cols_input].copy()
    # Else, check if it has the 'Institution_Pair' tuple
    elif 'Institution_Pair' in collaboration_counts_df.columns and 'Collaboration_Count' in collaboration_counts_df.columns:
        temp_df = collaboration_counts_df.copy()
        temp_df['Institution_Pair'] = temp_df['Institution_Pair'].apply(lambda x: x if isinstance(x, (list, tuple)) else None)
        temp_df.dropna(subset=['Institution_Pair'], inplace=True)
        valid_pairs_mask = temp_df['Institution_Pair'].apply(lambda x: len(x) == 2)
        if valid_pairs_mask.any():
            # Reconstruct the three columns from the pair
            pairs_df = pd.DataFrame(temp_df.loc[valid_pairs_mask, 'Institution_Pair'].tolist(),
                                    index=temp_df[valid_pairs_mask].index,
                                    columns=['Institution1', 'Institution2'])
            df_collab_pairs_for_validation = pd.concat([pairs_df, temp_df.loc[valid_pairs_mask, ['Collaboration_Count']]], axis=1)

    # Final cleaning before validation
    if not df_collab_pairs_for_validation.empty:
        df_collab_pairs_for_validation.dropna(subset=required_cols_input, inplace=True)
        df_collab_pairs_for_validation['Institution1'] = df_collab_pairs_for_validation['Institution1'].astype(str)
        df_collab_pairs_for_validation['Institution2'] = df_collab_pairs_for_validation['Institution2'].astype(str)
        df_collab_pairs_for_validation = df_collab_pairs_for_validation[
            (df_collab_pairs_for_validation['Institution1'].str.strip() != '') &
            (df_collab_pairs_for_validation['Institution2'].str.strip() != '') &
            (df_collab_pairs_for_validation['Institution1'].str.lower() != 'nan') &
            (df_collab_pairs_for_validation['Institution2'].str.lower() != 'nan')
        ]
        df_collab_pairs_for_validation['Collaboration_Count'] = pd.to_numeric(
            df_collab_pairs_for_validation['Collaboration_Count'],
            errors='coerce'
        ).fillna(0).astype(int)
    else:
         print(f"[{network_type}] Collaboration pairs DataFrame is empty after preparation.")

    # --- Centrality Cross-Validation ---
    if not df_collab_pairs_for_validation.empty:
        centrality_results, correlations, consistency_scores = cross_validate_centrality(
            df_collab_pairs_for_validation, n_splits=5, network_type=network_type
        )
    else:
        print(f"[{network_type}] Skipping centrality cross-validation: No valid pairs.")
        consistency_scores = {}

    # --- Community Detection and Validation ---
    if G_collab.number_of_nodes() > 0:
        print(f"\n[{network_type}] Starting Community Detection on the full graph...")
        reference_communities_on_full, G_community_subgraph = enhanced_community_detection(
            G_collab,
            network_type=f"{network_type}_FullGraph"
        )

        if not df_collab_pairs_for_validation.empty and reference_communities_on_full:
            print(f"\n[{network_type}] Starting Community Validation (K-Fold + Consensus)...")
            community_validation_results = consensus_cross_validation(
                 df_collab_pairs_for_validation,
                 reference_communities_on_full,
                 n_splits=5,
                 null_iter=100,
                 network_type=network_type
            )
            community_validation_results['reference_communities'] = reference_communities_on_full
            print_validation_results(community_validation_results, top_n_communities_to_show=5, network_type=network_type)

        elif not reference_communities_on_full:
             print(f"[{network_type}] No reference communities found. Skipping community validation.")
        else:
             print(f"[{network_type}] Skipping community validation due to issues with pairs DataFrame.")
    else:
        print(f"[{network_type}] Graph is empty. Skipping community detection and validation.")

    print(f"--- Advanced Evaluation for {network_type} Network Completed ---")
    return community_validation_results

In [ ]:
# ===================================================================
# AUTO-SCALING WRAPPER (new; does NOT modify your advanced functions)
# ===================================================================

ADVANCED_VALIDATION_MODE = "auto"   # "auto", "full", "fast", "off"

# Safety limits (co-occurrence matrix is N^2 -> keep conservative)
MAX_NODES_FOR_FULL = 3500
MAX_EDGES_FOR_FULL = 250000

MAX_NODES_FOR_FAST = 6000
MAX_EDGES_FOR_FAST = 600000

# If too large, validate only a CORE subgraph (bounded N)
CORE_MAX_NODES     = 3500
CORE_PICK_STRATEGY = "top_degree"   # "top_degree" or "random"
EDGE_SAMPLE_CAP    = 400000         # cap edge DF for KFold if needed

In [ ]:
def choose_lcc(G):
    if G.number_of_nodes() == 0:
        return G
    largest_cc = max(nx.connected_components(G), key=len)
    return G.subgraph(largest_cc).copy()

def induce_core_subgraph(G, max_nodes=3500, strategy="top_degree", seed=42):
    if G.number_of_nodes() <= max_nodes:
        return G

    if strategy == "random":
        rng = np.random.default_rng(seed)
        nodes = list(G.nodes())
        chosen = rng.choice(nodes, size=max_nodes, replace=False).tolist()
        return G.subgraph(chosen).copy()

    deg = dict(G.degree(weight='weight'))
    top_nodes = sorted(deg, key=deg.get, reverse=True)[:max_nodes]
    return G.subgraph(top_nodes).copy()

def filter_edge_df_to_nodes(df_edges, nodes_set):
    return df_edges[
        df_edges["Institution1"].isin(nodes_set) &
        df_edges["Institution2"].isin(nodes_set)
    ].copy()

def maybe_sample_edges(df_edges, cap=400000, seed=42):
    if len(df_edges) <= cap:
        return df_edges
    return df_edges.sample(n=cap, random_state=seed).copy()

def pick_validation_inputs(G_full, df_edges_full, network_key):
    n = G_full.number_of_nodes()
    m = G_full.number_of_edges()

    if ADVANCED_VALIDATION_MODE == "off":
        return None, None, f"[{network_key}] Advanced validation OFF (user setting)."

    if ADVANCED_VALIDATION_MODE == "full":
        return G_full, df_edges_full, f"[{network_key}] FULL validation on full graph (n={n:,}, m={m:,})."

    # Always work on LCC (most meaningful + avoids tiny components distorting metrics)
    G_lcc = choose_lcc(G_full)
    n_lcc = G_lcc.number_of_nodes()
    m_lcc = G_lcc.number_of_edges()

    if ADVANCED_VALIDATION_MODE == "fast":
        if n_lcc > MAX_NODES_FOR_FAST or m_lcc > MAX_EDGES_FOR_FAST:
            G_core = induce_core_subgraph(G_lcc, max_nodes=CORE_MAX_NODES, strategy=CORE_PICK_STRATEGY)
            nodes = set(G_core.nodes())
            df_core = filter_edge_df_to_nodes(df_edges_full, nodes)
            df_core = maybe_sample_edges(df_core, cap=EDGE_SAMPLE_CAP)
            return G_core, df_core, f"[{network_key}] FAST validation on LCC-core (n={G_core.number_of_nodes():,}, m={G_core.number_of_edges():,}); edges capped."
        return G_lcc, df_edges_full, f"[{network_key}] FAST validation on LCC (n={n_lcc:,}, m={m_lcc:,})."

    # AUTO mode
    if n_lcc <= MAX_NODES_FOR_FULL and m_lcc <= MAX_EDGES_FOR_FULL:
        return G_lcc, df_edges_full, f"[{network_key}] AUTO: Full-like validation on LCC (n={n_lcc:,}, m={m_lcc:,})."

    if n_lcc <= MAX_NODES_FOR_FAST and m_lcc <= MAX_EDGES_FOR_FAST:
        return G_lcc, df_edges_full, f"[{network_key}] AUTO: Fast validation on LCC (n={n_lcc:,}, m={m_lcc:,})."

    # Too big -> core
    G_core = induce_core_subgraph(G_lcc, max_nodes=CORE_MAX_NODES, strategy=CORE_PICK_STRATEGY)
    nodes = set(G_core.nodes())
    df_core = filter_edge_df_to_nodes(df_edges_full, nodes)
    df_core = maybe_sample_edges(df_core, cap=EDGE_SAMPLE_CAP)
    return G_core, df_core, (
        f"[{network_key}] AUTO: Downscoped validation to LCC-core "
        f"(n={G_core.number_of_nodes():,}, m={G_core.number_of_edges():,}); "
        f"edges capped for CV; full graph still exported."
    )


In [ ]:
# ===================================================================
# Main Execution of Advanced Evaluation (minimal change here)
# ===================================================================

community_validation_results = {}

if advanced_libs_available:
    print("\n" + "="*80)
    print("====== STARTING ADVANCED NETWORK VALIDATION ======")
    print("="*80)

    if 'all_network_results' not in locals():
        print("FATAL ERROR: 'all_network_results' dictionary not found.")
        print("Please re-run the previous cell (Cell 28 - Batch Analysis) to generate the networks.")
    else:

        # --- helper to avoid repeating the same block ---
        def run_validation_for_key(analysis_key):
            if analysis_key not in all_network_results or all_network_results[analysis_key]['status'] != 'Success':
                print(f"\nSkipping advanced evaluation for '{analysis_key}': Batch analysis failed or was not run.")
                return

            print(f"\n>>> Validating '{analysis_key}' Network <<<")

            G_full = all_network_results[analysis_key]['full_graph']
            df_edges = all_network_results[analysis_key]['collaboration_counts']

            if not isinstance(G_full, nx.Graph) or G_full.number_of_nodes() == 0 or df_edges is None or df_edges.empty:
                print(f"Skipping validation for '{analysis_key}': Graph or counts data is empty.")
                return

            G_val, df_val, note = pick_validation_inputs(G_full, df_edges, analysis_key)
            print(note)

            if G_val is None or df_val is None or G_val.number_of_nodes() == 0 or df_val.empty:
                print(f"Skipping validation for '{analysis_key}': No valid scoped inputs.")
                return

            community_validation_results[analysis_key] = evaluate_and_analyze_network_advanced(
                G_input_collab=G_val,
                collaboration_counts_df=df_val,
                network_type=analysis_key
            )

        # --- Run the 4 meaningful ones ---
        run_validation_for_key('Global_All')
        run_validation_for_key('Global_MCP')
        run_validation_for_key('EU_All')
        run_validation_for_key('EU_MCP')

        # --- SCP Networks (Skipping) ---
        print(f"\n\n{'='*20} Skipping Advanced Evaluation for SCP Networks {'='*20}")
        print("Advanced validation is not meaningful for SCP networks (few inter-institutional collaborations by definition).")

else:
    print("\nSkipping advanced network evaluation because required libraries are missing.")

print("\n" + "="*80)
print("====== ADVANCED NETWORK VALIDATION COMPLETE ======")
print("="*80)


## 31. Advanced Community Visualization: Setup and Libraries

This cell prepares the environment for the final, high-quality network visualizations. Its primary purpose is to import `matplotlib` components and, most importantly, to check for the `adjustText` library.

* **`adjustText`**: This is a powerful plotting library that intelligently repositions text labels on a plot to prevent them from overlapping. This is critical for creating a readable network graph where many node labels might otherwise be clustered together.
* **`ADJUST_TEXT_AVAILABLE`**: A boolean flag is set. If `adjustText` is found, the plotting function in the next cell will use it. If not, it will fall back to the default `networkx` labeling, which may result in overlapping text.
* **Fallback Definitions**: The script also ensures that `plots_dir`, `results_dir`, and a basic `COLORS` palette exist, making this section more robust and self-contained.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import colormaps # CORRECTED IMPORT for modern matplotlib
import matplotlib.colors as mcolors
import networkx as nx
import numpy as np
import pandas as pd
import os
import re # Added for safe filenames
import textwrap # For label wrapping
from IPython.display import display

# --- Check for adjustText library ---
try:
    from adjustText import adjust_text
    ADJUST_TEXT_AVAILABLE = True
    print("Loaded 'adjustText' library for non-overlapping labels.")
except ImportError:
    ADJUST_TEXT_AVAILABLE = False
    print("Warning: 'adjustText' not found. Labels may overlap.")
    print("Install with: !pip install adjustText")


# --- Ensure Output Directories Exist (Fallback) ---
if 'plots_dir' not in locals():
    plots_dir = '/kaggle/working/output/Plots/'
    os.makedirs(plots_dir, exist_ok=True)
if 'results_dir' not in locals():
    results_dir = '/kaggle/working/output/Results/'
    os.makedirs(results_dir, exist_ok=True)
    
# --- Ensure Colors Exist (Fallback) ---
if 'COLORS' not in locals(): 
    # --- CORRECTED colormap access ---
    category_colors = colormaps['tab10'].colors
    COLORS = {
        'primary': category_colors[0],
        'neutral_mid': '#666666',
        'neutral_light': '#f8f8f8',
        'neutral_dark': '#333333'
    }

## 32. Advanced Community Visualization: Plotting Function Definition

This cell defines the upgraded, high-quality network plotting function: `plot_communities_nx`. This function **does not run any analysis itself**; it is a complex plotting template that will be called by the next cell.

It is designed to take the **full graph** (`G_full`) and the **list of consensus communities** (from the previous validation step) and generate a clean, filtered visualization.

Key features of this function:
* **Filtering**: It only plots a subgraph of the `top_n_to_plot` most productive institutions (e.g., the Top 50) to keep the visualization readable.
* **Community Coloring**: It maps the consensus community results to node colors, assigning a unique color to each of the largest communities.
* **Node Sizing**: Node size is scaled by the institution's **total productivity** (its weighted degree from the *full* graph), clearly showing the most important actors.
* **Edge Styling**: It intelligently styles two types of edges:
    * **Intra-community edges** (links *within* a cluster) are drawn darker and thicker.
    * **Inter-community edges** (links *between* clusters) are drawn very light and thin.
    This powerfully highlights the community structure.
* **Intelligent Labeling**: If `adjustText` is available, it is used to add non-overlapping labels with arrows for the most important nodes, resulting in a clean and professional-looking chart.
* **Informative Legend**: It automatically generates a legend that identifies the largest communities by their color and most prominent member institution.

In [ ]:
import matplotlib as mpl

# ===================================================================
# 1. DEFINE  PLOTTING FUNCTION
# ===================================================================

def plot_communities_nx(
    G_full,                 # The *full* graph (e.g., G_collab_all)
    communities_list,       # The list of communities from validation (list of lists)
    top_institutions_df,    # The *ranked list* of top institutions (for filtering)
    top_n_to_plot,          # The number of top nodes to *actually draw* (e.g., 50)
    network_type_label,     # A label for filenames and titles (e.g., "Global_All")
    title_suffix="Community Structure",
    label_top_n_nodes_per_comm=3,  # How many nodes to label *per* community
    label_overall_top_n_nodes=0,   # Alternative: label the top N nodes overall
    show_legend_max_comms=8,     # Max communities to show in the legend
    figure_size=(22, 22), 
    layout_k_factor=0.8,
    layout_iterations=75,        # More iterations for a stable layout
    use_weighted_layout=True,
    node_size_base=40,
    node_size_multiplier=4500  # Multiplier to scale node sizes
):
    """
    Visualizes communities by plotting a FILTERED SUBGRAPH of the top N nodes.
    - Scales edge width by collaboration weight.
    - Uses adjustText with arrows for cleaner labels.
    - Creates a smart legend for the largest communities.
    """
    if not isinstance(G_full, nx.Graph) or G_full.number_of_nodes() == 0:
        print(f"[{network_type_label}] Full graph is empty or invalid. Skipping visualization.")
        return
    if top_institutions_df.empty or 'Institution' not in top_institutions_df.columns:
        print(f"[{network_type_label}] top_institutions_df is empty. Cannot filter nodes. Skipping.")
        return

    # --- 1. Create the Subgraph ---
    # Get the list of the Top N institutions by productivity
    top_n_list = top_institutions_df.head(top_n_to_plot)['Institution'].tolist()
    # Filter to nodes that are *also* in the full graph
    nodes_to_plot = [node for node in top_n_list if node in G_full]
    # Create the subgraph containing only these nodes and their connections
    G_to_plot = G_full.subgraph(nodes_to_plot).copy()
    # Remove any nodes that became isolated after filtering
    G_to_plot.remove_nodes_from(list(nx.isolates(G_to_plot)))

    print(f"[{network_type_label}] Filtering plot to Top {G_to_plot.number_of_nodes()} (of {top_n_to_plot}) linked nodes.")
    if G_to_plot.number_of_nodes() == 0:
        print("Subgraph is empty. Skipping plot.")
        return

    # --- 2. Create Node-to-Community Mapping ---
    # This maps each node in our subgraph to a community ID
    communities_list_sorted = sorted(communities_list, key=len, reverse=True)
    filtered_communities_list = [] # Stores the actual node lists for the legend
    node_to_community_id = {}
    community_id_counter = 0
    valid_subgraph_nodes = set(G_to_plot.nodes())

    # Iterate through the consensus communities (largest first)
    for comm_nodes in communities_list_sorted:
        # Find which nodes from this community are also in our subgraph
        valid_nodes_in_comm_and_subgraph = [node for node in comm_nodes if node in valid_subgraph_nodes]
        
        # If this community is present in the subgraph...
        if valid_nodes_in_comm_and_subgraph:
            filtered_communities_list.append(valid_nodes_in_comm_and_subgraph)
            # Assign the same ID to all these nodes
            for node in valid_nodes_in_comm_and_subgraph:
                if node not in node_to_community_id:
                    node_to_community_id[node] = community_id_counter
            community_id_counter += 1

    num_communities_in_plot = len(filtered_communities_list)

    # --- 2b. Generate Colors (CIVIDIS) ---
    if num_communities_in_plot == 0:
        print(f"[{network_type_label}] No valid communities found within the subgraph. Plotting with default colors.")
        node_to_community_id = {node: 0 for node in G_to_plot.nodes()}
        num_communities_in_plot = 1
        community_colors_map = ['#AAAAAA']  # Neutral grey
        filtered_communities_list = [list(G_to_plot.nodes())]
    else:
        print(f"[{network_type_label}] Visualizing {num_communities_in_plot} communities in the Top {G_to_plot.number_of_nodes()}-node subgraph.")
        try:
            # Continuous colormap -> sample evenly to get categorical colors
            cmap = mpl.colormaps.get_cmap('cividis')

            # Avoid the extreme ends (too dark / too bright) for better label contrast
            if num_communities_in_plot == 1:
                sample_points = [0.60]
            else:
                sample_points = np.linspace(0.10, 0.90, num_communities_in_plot)

            community_colors_map = [cmap(x) for x in sample_points]

        except Exception as e:
            print(f"Warning: Cividis colormap generation failed ({e}). Falling back to gray.")
            community_colors_map = ['#777777'] * max(1, num_communities_in_plot)

    # Map each node in the plot to its assigned color
    node_color_list = [community_colors_map[node_to_community_id[node]]
                       if node in node_to_community_id else '#E0E0E0' # Lighter grey for unclassified
                       for node in G_to_plot.nodes()]

    # --- 3. Layout ---
    plt.figure(figsize=figure_size, facecolor='white')
    ax = plt.gca()
    ax.set_facecolor('white')
    print(f"[{network_type_label}] Calculating layout...")
    # Calculate 'k' for spring layout (controls node repulsion)
    k_val = layout_k_factor / np.sqrt(max(1, G_to_plot.number_of_nodes()))
    # Use edge weights in layout calculation if specified
    layout_weights = 'weight' if use_weighted_layout and nx.get_edge_attributes(G_to_plot, 'weight') else None

    pos = nx.spring_layout(G_to_plot, k=k_val, iterations=layout_iterations, seed=123, scale=3, weight=layout_weights)
    print(f"[{network_type_label}] Layout calculation complete.")

    # --- 4. Node Sizes (Based on FULL graph degrees) ---
    # Get degrees from the *full* graph to show true productivity
    degrees_in_full_graph = dict(G_full.degree(weight='weight'))
    max_deg_full = max(degrees_in_full_graph.values()) if degrees_in_full_graph else 1.0
    # Scale node size based on its degree relative to the max degree
    node_sizes = [
        node_size_base + (degrees_in_full_graph.get(node, 0) / max_deg_full) * node_size_multiplier
        for node in G_to_plot.nodes()
    ]

    # --- 5. Drawing Edges (Scaled by weight) ---
    print(f"[{network_type_label}] Drawing {G_to_plot.number_of_edges()} edges...")
    
    edge_weights = [d['weight'] for _, _, d in G_to_plot.edges(data=True) if 'weight' in d]
    max_w = max(edge_weights) if edge_weights else 1.0
    min_w = min(edge_weights) if edge_weights else 1.0

    # Separate edges into intra-community (within) and inter-community (between)
    intra_community_edges = []
    inter_community_edges = []
    for u, v in G_to_plot.edges():
        u_comm = node_to_community_id.get(u)
        v_comm = node_to_community_id.get(v)
        if u_comm is not None and u_comm == v_comm:
            intra_community_edges.append((u, v))
        else:
            inter_community_edges.append((u, v))

    # Calculate scaled widths
    intra_widths = [0.4 + (G_to_plot[u][v].get('weight', min_w) / max_w) * 2.5 for u, v in intra_community_edges]
    inter_widths = [0.1 + (G_to_plot[u][v].get('weight', min_w) / max_w) * 1.0 for u, v in inter_community_edges]

    # Draw inter-community edges first (faint)
    nx.draw_networkx_edges(G_to_plot, pos, edgelist=inter_community_edges,
                           width=inter_widths,
                           alpha=0.08, edge_color='#AAAAAA', ax=ax)
    # Draw intra-community edges second (darker)
    nx.draw_networkx_edges(G_to_plot, pos, edgelist=intra_community_edges,
                           width=intra_widths,
                           alpha=0.4, edge_color='#333333', ax=ax)

    # --- 6. Drawing Nodes ---
    print(f"[{network_type_label}] Drawing {G_to_plot.number_of_nodes()} nodes...")
    nx.draw_networkx_nodes(G_to_plot, pos,
                           node_color=node_color_list,
                           node_size=node_sizes,
                           alpha=0.9,
                           edgecolors=COLORS.get('neutral_dark', 'black'),
                           linewidths=0.5, # Thinner outlines
                           ax=ax)

    # --- 7. Node Labeling (using adjustText) ---
    labels_to_draw = {}
    degrees_for_ranking = degrees_in_full_graph 

    if label_top_n_nodes_per_comm > 0 and num_communities_in_plot > 0:
        # Label the top N nodes in the largest M communities
        num_comms_to_label = min(num_communities_in_plot, 5) 
        print(f"[{network_type_label}] Identifying top {label_top_n_nodes_per_comm} nodes in largest {num_comms_to_label} communities...")
        for i in range(num_comms_to_label):
            comm_nodes = filtered_communities_list[i]
            # Sort nodes in this community by their degree (productivity)
            nodes_and_degrees = sorted(
                [(n, degrees_for_ranking.get(n, 0)) for n in comm_nodes],
                key=lambda x: x[1], reverse=True
            )
            # Add the top N to the label list
            for node, deg in nodes_and_degrees[:label_top_n_nodes_per_comm]:
                labels_to_draw[node] = node

    elif label_overall_top_n_nodes > 0:
        # Alternative: label the top N nodes overall, regardless of community
        print(f"[{network_type_label}] Identifying overall top {label_overall_top_n_nodes} nodes by degree...")
        sorted_nodes_by_degree = sorted(
            G_to_plot.nodes(),
            key=lambda n: degrees_for_ranking.get(n, 0),
            reverse=True
        )
        for node in sorted_nodes_by_degree[:label_overall_top_n_nodes]:
            labels_to_draw[node] = node

    if labels_to_draw:
        print(f"[{network_type_label}] Adding labels for {len(labels_to_draw)} nodes...")
        
        # --- Use adjustText if available ---
        if ADJUST_TEXT_AVAILABLE:
            texts = []
            for node, label in labels_to_draw.items():
                if node in pos: # Ensure node is in layout
                    texts.append(ax.text(pos[node][0], pos[node][1], label,
                                 fontsize=8,
                                 fontfamily='sans-serif',
                                 fontdict={'weight': 'normal', 'color': COLORS.get('neutral_dark', 'black')},
                                 bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=0.1)))
            if texts:
                print(f"[{network_type_label}] Adjusting text labels...")
                # Run the adjustText algorithm
                adjust_text(texts, ax=ax,
                            arrowprops=dict(arrowstyle='-', color='grey', lw=0.3, alpha=0.5))
        else:
            # Fallback (labels will overlap)
            nx.draw_networkx_labels(G_to_plot, pos, labels=labels_to_draw, font_size=8,
                                   fontfamily='sans-serif',
                                   font_weight='normal',
                                   font_color=COLORS.get('neutral_dark', 'black'),
                                   bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=0.1),
                                   ax=ax)
    else:
        print(f"[{network_type_label}] No node labels requested or identified.")

    # --- 8. Title and Axis ---
    title_text = f"{network_type_label}: {title_suffix} (Top {G_to_plot.number_of_nodes()} Institutions)"
    subtitle_text = f"{num_communities_in_plot} communities identified in subgraph"
    plt.title(f"{title_text}\n{subtitle_text}",
              fontsize=16, y=1.01, fontfamily='sans-serif')
    plt.axis('off')

    # --- 9. Legend (Informative) ---
    if num_communities_in_plot > 1 and num_communities_in_plot <= show_legend_max_comms:
        legend_handles = []
        communities_for_legend = filtered_communities_list[:show_legend_max_comms]
        
        for i in range(len(communities_for_legend)):
            comm_nodes = communities_for_legend[i]
            comm_id = node_to_community_id.get(comm_nodes[0]) # Get ID
            
            if comm_id is not None and comm_id < len(community_colors_map):
                # Find top node in this community to use as a label
                top_node_in_comm = sorted(
                    comm_nodes, 
                    key=lambda n: degrees_for_ranking.get(n, 0), 
                    reverse=True
                )[0]
                
                # Wrap long labels
                label_text = f"Comm. {comm_id + 1}: {top_node_in_comm} ({len(comm_nodes)} nodes)"
                wrapped_label = '\n'.join(textwrap.wrap(label_text, width=30))
                
                legend_handles.append(plt.Line2D([0], [0], marker='o', color='w',
                                     label=wrapped_label,
                                     markerfacecolor=community_colors_map[comm_id],
                                     markersize=9))
        if legend_handles:
            plt.legend(
                handles=legend_handles,
                title=f"Top {len(legend_handles)} Communities",
                loc='upper left',
                bbox_to_anchor=(1.01, 1),
                fontsize=9,
                title_fontproperties={'weight': 'bold', 'size': 10},  # include size here
                frameon=True,
                facecolor='white',
                framealpha=0.9,
                labelspacing=1.2
            )
    elif num_communities_in_plot > show_legend_max_comms:
        plt.text(0.01, 0.01, f"{num_communities_in_plot} communities in subgraph (legend omitted)",
                 transform=ax.transAxes, fontsize=9, color='black',
                 bbox=dict(facecolor='white', alpha=0.5, pad=0.2))

    # --- 10. Saving ---
    # Create safe filenames
    safe_suffix = re.sub(r'[^\w\-]+', '_', title_suffix)
    safe_network_label = re.sub(r'[^\w\-]+', '_', network_type_label)
    plot_filename_base = f"community_plot_filtered_{safe_network_label}_{safe_suffix}"
    plot_path_png = os.path.join(plots_dir, f"{plot_filename_base}.png")
    plot_path_pdf = os.path.join(plots_dir, f"{plot_filename_base}.pdf")

    try:
        # Use bbox_inches='tight' to include the external legend
        plt.savefig(plot_path_png, dpi=300, bbox_inches='tight')
        plt.savefig(plot_path_pdf, format='pdf', bbox_inches='tight')
        print(f"Community visualization for '{network_type_label}' saved to (PNG/PDF): {plots_dir}")
    except Exception as e:
        print(f"Error saving community plot: {e}")

    plt.show()

## 32. Advanced Community Visualization: Main Execution

This final analysis cell executes the `plot_communities_nx` function defined in the previous cell. It generates the final, high-quality visualizations of the institutional community structures.

It leverages the results from the **community validation step** (`community_validation_results_all` and `community_validation_results_mcp`) and the **institutional ranking step** (`top_inst_all` and `top_inst_mcp`) to produce two separate plots:

1.  **Global "All" Network**: Visualizes the consensus communities for the network of *all* publications.
2.  **Global "MCP" Network**: Visualizes the consensus communities for the network of *only international* (Multi-Country) publications.

These plots clearly show the main research clusters (communities) in the field, with node colors indicating community membership, node size indicating productivity, and edge thickness representing collaboration strength.

In [ ]:
# ===================================================================
# 2. EXECUTE the Advanced Community Plotting (FIXED)
# ===================================================================

print("\n" + "="*80)
print("====== STARTING FINAL COMMUNITY VISUALIZATION ======")
print("="*80)

# --- 1. Check if all required dictionaries exist ---
if 'all_network_results' not in locals():
    print("FATAL ERROR: 'all_network_results' dictionary not found. Run Batch Analysis (Cell 28).")
elif 'community_validation_results' not in locals():
    print("FATAL ERROR: 'community_validation_results' dictionary not found. Run Validation (Cell 31).")
elif 'plot_communities_nx' not in locals():
    print("FATAL ERROR: 'plot_communities_nx' function not found. Run Plotting Definition (Cell 30).")
else:
    # --- Plot Consensus Communities for 'Global_All' ---
    analysis_key = 'Global_All'
    print(f"\n>>> Plotting Consensus Communities for '{analysis_key}' <<<")
    
    # Check if all 3 required pieces of data exist
    if (analysis_key in all_network_results and 
        all_network_results[analysis_key]['status'] == 'Success' and
        analysis_key in community_validation_results):
        
        # --- FIX: Unpack data from the correct dictionaries ---
        graph_to_plot = all_network_results[analysis_key]['full_graph']
        top_inst_df = all_network_results[analysis_key]['top_institutions']
        communities_list = community_validation_results[analysis_key].get('consensus_clusters', [])
        # --- End of Fix ---

        if not communities_list:
            print(f"No consensus communities found in results for '{analysis_key}' to plot.")
        elif top_inst_df.empty:
            print(f"Top institutions data ('{analysis_key}') not found or is empty. Cannot plot.")
        elif not isinstance(graph_to_plot, nx.Graph) or graph_to_plot.number_of_nodes() == 0:
            print(f"Graph object for '{analysis_key}' not found or invalid.")
        else:
            plot_communities_nx(
                G_full=graph_to_plot,
                communities_list=communities_list,
                top_institutions_df=top_inst_df,
                top_n_to_plot=50, # Plot the top 50 institutions
                network_type_label="Global_All_Consensus",
                title_suffix="Consensus Communities (Top 50 Nodes)",
                label_top_n_nodes_per_comm=0,    # Turn off per-community labeling
                label_overall_top_n_nodes=50,  # Turn on overall top 50 labeling
                show_legend_max_comms=8,
                figure_size=(22, 22),
                layout_k_factor=0.7,
                use_weighted_layout=True
            )
    else:
        print(f"Validation or Batch results for '{analysis_key}' not available or invalid. Skipping plot.")

    # --- Plot Consensus Communities for 'Global_MCP' ---
    analysis_key = 'Global_MCP'
    print(f"\n>>> Plotting Consensus Communities for '{analysis_key}' <<<")
    
    if (analysis_key in all_network_results and 
        all_network_results[analysis_key]['status'] == 'Success' and
        analysis_key in community_validation_results):

        # --- FIX: Unpack data from the correct dictionaries ---
        graph_to_plot = all_network_results[analysis_key]['full_graph']
        top_inst_df = all_network_results[analysis_key]['top_institutions']
        communities_list = community_validation_results[analysis_key].get('consensus_clusters', [])
        # --- End of Fix ---

        if not communities_list:
            print(f"No consensus communities found in results for '{analysis_key}' to plot.")
        elif top_inst_df.empty:
            print(f"Top institutions data ('{analysis_key}') not found or is empty. Cannot plot.")
        elif not isinstance(graph_to_plot, nx.Graph) or graph_to_plot.number_of_nodes() == 0:
            print(f"Graph object for '{analysis_key}' not found or invalid.")
        else:
            plot_communities_nx(
                G_full=graph_to_plot,
                communities_list=communities_list,
                top_institutions_df=top_inst_df, 
                top_n_to_plot=50,
                network_type_label="Global_MCP_Consensus",
                title_suffix="Consensus Communities (Top 50 Nodes)",
                label_top_n_nodes_per_comm=0,
                label_overall_top_n_nodes=50,
                show_legend_max_comms=8,
                figure_size=(20,20)
            )
    else:
        print(f"Validation or Batch results for '{analysis_key}' not available or invalid. Skipping plot.")
        
    # --- Plot Consensus Communities for 'EU_All' ---
    analysis_key = 'EU_All'
    print(f"\n>>> Plotting Consensus Communities for '{analysis_key}' <<<")
    
    if (analysis_key in all_network_results and 
        all_network_results[analysis_key]['status'] == 'Success' and
        analysis_key in community_validation_results):
        
        graph_to_plot = all_network_results[analysis_key]['full_graph']
        top_inst_df = all_network_results[analysis_key]['top_institutions']
        communities_list = community_validation_results[analysis_key].get('consensus_clusters', [])

        if not communities_list:
            print(f"No consensus communities found in results for '{analysis_key}' to plot.")
        elif top_inst_df.empty:
            print(f"Top institutions data ('{analysis_key}') not found or is empty. Cannot plot.")
        elif not isinstance(graph_to_plot, nx.Graph) or graph_to_plot.number_of_nodes() == 0:
            print(f"Graph object for '{analysis_key}' not found or invalid.")
        else:
            plot_communities_nx(
                G_full=graph_to_plot,
                communities_list=communities_list,
                top_institutions_df=top_inst_df, 
                top_n_to_plot=30, # Plot Top 30 for EU
                network_type_label="EU_All_Consensus",
                title_suffix="Consensus Communities (Top 30 Nodes)",
                label_top_n_nodes_per_comm=0,
                label_overall_top_n_nodes=30,
                show_legend_max_comms=8,
                figure_size=(20,20)
            )
    else:
        print(f"Validation or Batch results for '{analysis_key}' not available or invalid. Skipping plot.")

print("\n" + "="*80)
print("====== FINAL COMMUNITY VISUALIZATION COMPLETE ======")
print("="*80)